<a id="notebook-top"></a>
# FastAPI Service Integration and API Contract Development

**Notebook purpose.** This notebook integrates the frozen vision, Grad-CAM, grounded-language, guardrail, prediction-store, and telemetry components behind a versioned FastAPI service. It makes the API contract—not the user interface—the authoritative orchestration boundary.

**What this notebook covers.** OpenAPI and Pydantic contracts; configuration and controlled errors; reusable services; workflow orchestration; twelve API routes; health, lineage, and LLMOps responses; in-process and standalone testing; Pytest evidence; MLflow registration; and readiness.

**Expected outcome.** A deployment-ready FastAPI application with twelve validated paths, controlled schemas and errors, reusable tests, persisted integration evidence, and complete model lineage.

**Safety boundary.** This work is intended for research and educational demonstration. Model outputs are not clinical diagnoses and must not replace qualified professional review.


<a id="notebook-index"></a>
## Notebook Index

Use the links below to move directly to a section. Each major section ends with a **Back to notebook index** link.

- [1. API Service Configuration](#nb07-1-api-service-configuration)
- [2. Versioned API Endpoint and Response Contract](#nb07-2-versioned-api-endpoint-and-response-contract)
- [3. Pydantic Request and Response Schemas](#nb07-3-pydantic-request-and-response-schemas)
  - [3.1 Common Response Metadata and Controlled Error Contract](#nb07-3-1-common-response-metadata-and-controlled-error-contract)
  - [3.2 Grounded Language Request Contracts](#nb07-3-2-grounded-language-request-contracts)
  - [3.3 Image Classification and Finding-Evidence Schemas](#nb07-3-3-image-classification-and-finding-evidence-schemas)
  - [3.4 Visual Explainability Response Schemas](#nb07-3-4-visual-explainability-response-schemas)
  - [3.5 Grounded Language and Guardrail Response Schemas](#nb07-3-5-grounded-language-and-guardrail-response-schemas)
  - [3.6 Health, Model Lineage, and Metrics Schemas](#nb07-3-6-health-model-lineage-and-metrics-schemas)
  - [3.7 Combined Workflow and Stored Prediction Schemas](#nb07-3-7-combined-workflow-and-stored-prediction-schemas)
  - [3.8 Canonical Schema Package Registry](#nb07-3-8-canonical-schema-package-registry)
- [4. API Core Configuration and Runtime Utilities](#nb07-4-api-core-configuration-and-runtime-utilities)
  - [4.1 Environment-Aware Service Settings and Frozen Lineage](#nb07-4-1-environment-aware-service-settings-and-frozen-lineage)
  - [4.2 Controlled Service Exceptions](#nb07-4-2-controlled-service-exceptions)
  - [4.3 Request Context, Response Metadata, and Error Serialization](#nb07-4-3-request-context-response-metadata-and-error-serialization)
- [5. Reusable API Service Modules](#nb07-5-reusable-api-service-modules)
  - [5.1 Secure Image Validation and Decoding Service](#nb07-5-1-secure-image-validation-and-decoding-service)
  - [5.2 Frozen Finding Ontology and Threshold Registry](#nb07-5-2-frozen-finding-ontology-and-threshold-registry)
  - [5.3 Frozen Computer-Vision Inference Service](#nb07-5-3-frozen-computer-vision-inference-service)
  - [5.4 Isolated Inference Parity Validation](#nb07-5-4-isolated-inference-parity-validation)
  - [5.5 Thread-Safe Prediction Store](#nb07-5-5-thread-safe-prediction-store)
  - [5.6 Thread-Safe Grad-CAM Explainability Service](#nb07-5-6-thread-safe-grad-cam-explainability-service)
  - [5.7 Grounded Language Input Serialization Service](#nb07-5-7-grounded-language-input-serialization-service)
  - [5.8 Frozen Grounded Language Model Inference Service](#nb07-5-8-frozen-grounded-language-model-inference-service)
  - [5.9 Deterministic Grounding and Safety Guardrail Service](#nb07-5-9-deterministic-grounding-and-safety-guardrail-service)
  - [5.10 Thread-Safe Operational Metrics Service](#nb07-5-10-thread-safe-operational-metrics-service)
- [6. FastAPI Workflow Integration](#nb07-6-fastapi-workflow-integration)
  - [6.1 Runtime Service and Schema Interface Registry](#nb07-6-1-runtime-service-and-schema-interface-registry)
  - [6.2 Stored-Prediction Language Workflow Orchestration](#nb07-6-2-stored-prediction-language-workflow-orchestration)
  - [6.3 Image Classification and Explainability Workflow Orchestration](#nb07-6-3-image-classification-and-explainability-workflow-orchestration)
  - [6.4 Complete Analysis Workflow Orchestration](#nb07-6-4-complete-analysis-workflow-orchestration)
  - [6.5 Application Service Container and Dependency Boundary](#nb07-6-5-application-service-container-and-dependency-boundary)
  - [6.6 Image and Grounded Language API Routes](#nb07-6-6-image-and-grounded-language-api-routes)
  - [6.7 Complete Analysis and Stored Prediction API Routes](#nb07-6-7-complete-analysis-and-stored-prediction-api-routes)
  - [6.8 Endpoint-Level API Telemetry Adapter](#nb07-6-8-endpoint-level-api-telemetry-adapter)
  - [6.9 Health, Model Lineage, Evaluation, and LLMOps Routes](#nb07-6-9-health-model-lineage-evaluation-and-llmops-routes)
  - [6.10 FastAPI Application Assembly, Error Handling, and Request Telemetry](#nb07-6-10-fastapi-application-assembly-error-handling-and-request-telemetry)
- [7. In-Process API Integration Testing](#nb07-7-in-process-api-integration-testing)
  - [7.1 System Endpoints and Controlled Error Responses](#nb07-7-1-system-endpoints-and-controlled-error-responses)
  - [7.2 Image Ingestion, Classification, and Explainability Endpoint Tests](#nb07-7-2-image-ingestion-classification-and-explainability-endpoint-tests)
  - [7.3 Grounded Language Endpoint Integration Tests](#nb07-7-3-grounded-language-endpoint-integration-tests)
  - [7.4 Complete Analysis Endpoint Integration Tests](#nb07-7-4-complete-analysis-endpoint-integration-tests)
- [8. Deployment-Ready Service Initialization](#nb07-8-deployment-ready-service-initialization)
  - [8.1 Standalone Service Factory and Cold-Start Validation](#nb07-8-1-standalone-service-factory-and-cold-start-validation)
  - [8.2 Independent Uvicorn Process Smoke Test](#nb07-8-2-independent-uvicorn-process-smoke-test)
- [9. Automated API Test Suite](#nb07-9-automated-api-test-suite)
  - [9.1 Reusable Pytest Integration Suite](#nb07-9-1-reusable-pytest-integration-suite)
- [10. API Artifact Registration and Final Readiness](#nb07-10-api-artifact-registration-and-final-readiness)
  - [10.1 Versioned API Artifact Registry](#nb07-10-1-versioned-api-artifact-registry)
  - [10.2 MLflow API Integration Registration](#nb07-10-2-mlflow-api-integration-registration)
  - [10.3 Final FastAPI Integration Readiness Gate](#nb07-10-3-final-fastapi-integration-readiness-gate)
- [11. Conclusion](#nb07-11-conclusion)


<a id="nb07-1-api-service-configuration"></a>
## 1. API Service Configuration

This section restores only the paths and runtime routing required by the API notebook. It establishes the canonical backend module structure and validates the frozen computer-vision model, fine-tuned language model, prompt registry, explainability metadata, and language guardrail artifacts without loading the models into memory.


In [8]:
import os
import sys
from pathlib import Path

import fastapi
import mlflow
import pydantic
import yaml


# -------------------------------------------------------------------------
# Load the registered solution paths
# -------------------------------------------------------------------------
PATH_REGISTRY_PATH = Path(
    "/home/jovyan/chest-xray-ai-assistant/configs/paths.yaml"
)

if not PATH_REGISTRY_PATH.is_file():
    raise FileNotFoundError(
        f"Path registry was not found: {PATH_REGISTRY_PATH}"
    )

with PATH_REGISTRY_PATH.open("r", encoding="utf-8") as file:
    path_registry = yaml.safe_load(file)

if not isinstance(path_registry, dict):
    raise ValueError(
        "The path registry must contain a YAML mapping."
    )


def collect_registered_paths(value):
    """Recursively collect absolute paths from the registry."""
    collected_paths = []

    if isinstance(value, dict):
        for nested_value in value.values():
            collected_paths.extend(
                collect_registered_paths(nested_value)
            )
    elif isinstance(value, list):
        for nested_value in value:
            collected_paths.extend(
                collect_registered_paths(nested_value)
            )
    elif isinstance(value, str) and value.startswith("/"):
        collected_paths.append(Path(value))

    return collected_paths


registered_paths = collect_registered_paths(
    path_registry
)

expected_solution_root = Path(
    "/home/jovyan/chest-xray-ai-assistant"
)
expected_data_root = Path(
    "/home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data"
)

if expected_solution_root not in registered_paths:
    raise KeyError(
        "The solution root does not match the registered path: "
        f"{expected_solution_root}"
    )

if expected_data_root not in registered_paths:
    raise KeyError(
        "The data root does not match the registered path: "
        f"{expected_data_root}"
    )

SOLUTION_ROOT = expected_solution_root
DATA_ROOT = expected_data_root


# -------------------------------------------------------------------------
# Define the frozen service lineage
# -------------------------------------------------------------------------
API_TITLE = (
    "API-Driven Chest X-Ray Analysis and Explanation Assistant"
)
API_VERSION = "v1"
API_PREFIX = f"/api/{API_VERSION}"

COMPUTER_VISION_MODEL_VERSION = (
    "resnet18-chestmnist-v1"
)
LANGUAGE_MODEL_VERSION = (
    "flan-t5-small-chestmnist-v1"
)
PROMPT_REGISTRY_VERSION = (
    "grounded-language-prompts-v1"
)

COMPUTER_VISION_MODEL_DIR = (
    DATA_ROOT
    / "models"
    / COMPUTER_VISION_MODEL_VERSION
)
LANGUAGE_MODEL_DIR = (
    DATA_ROOT
    / "models"
    / LANGUAGE_MODEL_VERSION
)
EXPLAINABILITY_ROOT = (
    DATA_ROOT / "explainability"
)

COMPUTER_VISION_METADATA_PATH = (
    COMPUTER_VISION_MODEL_DIR
    / "model_metadata.yaml"
)
COMPUTER_VISION_WEIGHTS_PATH = (
    COMPUTER_VISION_MODEL_DIR
    / "model_state_dict.pt"
)

LANGUAGE_MODEL_METADATA_PATH = (
    LANGUAGE_MODEL_DIR
    / "model_metadata.yaml"
)
LANGUAGE_MODEL_WEIGHTS_PATH = (
    LANGUAGE_MODEL_DIR
    / "model.safetensors"
)
LANGUAGE_MODEL_CONFIG_PATH = (
    LANGUAGE_MODEL_DIR
    / "config.json"
)

PROMPT_REGISTRY_PATH = (
    SOLUTION_ROOT
    / "configs"
    / "prompt_registry.yaml"
)
EXPLAINABILITY_METADATA_PATH = (
    EXPLAINABILITY_ROOT
    / "explainability_metadata.yaml"
)

LANGUAGE_GUARDRAIL_SUMMARY_PATH = (
    DATA_ROOT
    / "outputs"
    / "language"
    / "language_guardrail_summary.json"
)
LANGUAGE_EVALUATION_METRICS_PATH = (
    DATA_ROOT
    / "outputs"
    / "language"
    / "language_evaluation_metrics.json"
)


# -------------------------------------------------------------------------
# Establish the canonical API module structure
# -------------------------------------------------------------------------
API_ROOT = SOLUTION_ROOT / "api"
API_CORE_DIR = API_ROOT / "core"
API_ROUTES_DIR = API_ROOT / "routes"
API_SCHEMAS_DIR = API_ROOT / "schemas"

SOURCE_ROOT = SOLUTION_ROOT / "src"
SERVICE_ROOT = SOURCE_ROOT / "services"

TEST_ROOT = SOLUTION_ROOT / "tests"
API_TEST_DIR = TEST_ROOT / "api"

API_OUTPUT_DIR = (
    DATA_ROOT / "outputs" / "api"
)

for directory in (
    API_ROOT,
    API_CORE_DIR,
    API_ROUTES_DIR,
    API_SCHEMAS_DIR,
    SOURCE_ROOT,
    SERVICE_ROOT,
    TEST_ROOT,
    API_TEST_DIR,
    API_OUTPUT_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

if str(SOLUTION_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(SOLUTION_ROOT),
    )


# -------------------------------------------------------------------------
# Restore notebook-local runtime artifact routing
# -------------------------------------------------------------------------
HF_CACHE_DIR = DATA_ROOT / "hf-cache"
HF_DATASETS_CACHE_DIR = (
    HF_CACHE_DIR / "datasets"
)
TORCH_CACHE_DIR = (
    DATA_ROOT / "models" / "torch-cache"
)
MLFLOW_DIRECTORY = DATA_ROOT / "mlflow"
MLFLOW_TRACKING_URI = (
    f"file://{MLFLOW_DIRECTORY}"
)

os.environ["HF_HOME"] = str(
    HF_CACHE_DIR
)
os.environ["HF_DATASETS_CACHE"] = str(
    HF_DATASETS_CACHE_DIR
)
os.environ["TORCH_HOME"] = str(
    TORCH_CACHE_DIR
)
os.environ["MLFLOW_TRACKING_URI"] = (
    MLFLOW_TRACKING_URI
)
os.environ["TOKENIZERS_PARALLELISM"] = (
    "false"
)

mlflow.set_tracking_uri(
    MLFLOW_TRACKING_URI
)


# -------------------------------------------------------------------------
# Validate frozen service dependencies without loading model weights
# -------------------------------------------------------------------------
required_service_artifacts = {
    "Computer-vision metadata": (
        COMPUTER_VISION_METADATA_PATH
    ),
    "Computer-vision weights": (
        COMPUTER_VISION_WEIGHTS_PATH
    ),
    "Language-model metadata": (
        LANGUAGE_MODEL_METADATA_PATH
    ),
    "Language-model weights": (
        LANGUAGE_MODEL_WEIGHTS_PATH
    ),
    "Language-model configuration": (
        LANGUAGE_MODEL_CONFIG_PATH
    ),
    "Prompt registry": (
        PROMPT_REGISTRY_PATH
    ),
    "Explainability metadata": (
        EXPLAINABILITY_METADATA_PATH
    ),
    "Language guardrail summary": (
        LANGUAGE_GUARDRAIL_SUMMARY_PATH
    ),
    "Language evaluation metrics": (
        LANGUAGE_EVALUATION_METRICS_PATH
    ),
}

artifact_availability = {
    artifact_name: artifact_path.is_file()
    for artifact_name, artifact_path
    in required_service_artifacts.items()
}

configuration_checks = {
    "Solution root is available": (
        SOLUTION_ROOT.is_dir()
    ),
    "Data root is available": (
        DATA_ROOT.is_dir()
    ),
    "API module directories are available": all(
        directory.is_dir()
        for directory in (
            API_ROOT,
            API_CORE_DIR,
            API_ROUTES_DIR,
            API_SCHEMAS_DIR,
        )
    ),
    "Service module directory is available": (
        SERVICE_ROOT.is_dir()
    ),
    "API test directory is available": (
        API_TEST_DIR.is_dir()
    ),
    "All frozen artifacts are available": all(
        artifact_availability.values()
    ),
    "MLflow tracking URI is restored": (
        mlflow.get_tracking_uri()
        == MLFLOW_TRACKING_URI
    ),
}


# -------------------------------------------------------------------------
# Report the API service configuration
# -------------------------------------------------------------------------
print("FASTAPI SERVICE CONFIGURATION")
print("-" * 100)
print(f"API title                 : {API_TITLE}")
print(f"API version               : {API_VERSION}")
print(f"API prefix                : {API_PREFIX}")
print(f"Solution root             : {SOLUTION_ROOT}")
print(f"Data root                 : {DATA_ROOT}")
print(f"API root                  : {API_ROOT}")
print(f"Service root              : {SERVICE_ROOT}")
print(f"API test root             : {API_TEST_DIR}")
print(f"API output root           : {API_OUTPUT_DIR}")
print(f"Computer-vision model     : {COMPUTER_VISION_MODEL_VERSION}")
print(f"Language model            : {LANGUAGE_MODEL_VERSION}")
print(f"Prompt registry           : {PROMPT_REGISTRY_VERSION}")
print(f"FastAPI version           : {fastapi.__version__}")
print(f"Pydantic version          : {pydantic.__version__}")
print(f"MLflow tracking URI       : {mlflow.get_tracking_uri()}")
print("-" * 100)
print("FROZEN SERVICE ARTIFACTS")

for artifact_name, available in artifact_availability.items():
    print(
        f"{artifact_name:<34}: "
        f"{'PASS' if available else 'FAIL'}"
    )

print("-" * 100)

for check_name, passed in configuration_checks.items():
    print(
        f"{check_name:<48}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(configuration_checks.values()):
    failed_checks = [
        name
        for name, passed
        in configuration_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "FastAPI service configuration failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: READY FOR API CONTRACT DESIGN")

FASTAPI SERVICE CONFIGURATION
----------------------------------------------------------------------------------------------------
API title                 : API-Driven Chest X-Ray Analysis and Explanation Assistant
API version               : v1
API prefix                : /api/v1
Solution root             : /home/jovyan/chest-xray-ai-assistant
Data root                 : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data
API root                  : /home/jovyan/chest-xray-ai-assistant/api
Service root              : /home/jovyan/chest-xray-ai-assistant/src/services
API test root             : /home/jovyan/chest-xray-ai-assistant/tests/api
API output root           : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api
Computer-vision model     : resnet18-chestmnist-v1
Language model            : flan-t5-small-chestmnist-v1
Prompt registry           : grounded-language-prompts-v1
FastAPI version           : 0.115.12
Pydantic version          : 2.10.6
MLf

**[↑ Back to notebook index](#notebook-index)**


<a id="nb07-2-versioned-api-endpoint-and-response-contract"></a>
## 2. Versioned API Endpoint and Response Contract

The backend uses a versioned `/api/v1` namespace while retaining an unversioned health endpoint for platform probes. The endpoint registry separates image classification, visual analysis, language generation, combined workflow execution, model information, stored prediction retrieval, and operational metrics.

Every successful response uses a common envelope containing a request identifier, UTC timestamp, API version, status, model lineage, prompt version where applicable, latency, warnings, and the educational-use boundary. Errors use a controlled machine-readable contract and must not expose internal tracebacks or filesystem paths.


In [9]:
from datetime import datetime, timezone


# -------------------------------------------------------------------------
# Define the common service response and error contracts
# -------------------------------------------------------------------------
API_CONTRACT_VERSION = "api-contract-v1"
MAXIMUM_UPLOAD_SIZE_MIB = 10

SUPPORTED_IMAGE_MEDIA_TYPES = [
    "image/png",
    "image/jpeg",
]

COMMON_RESPONSE_FIELDS = [
    "request_id",
    "timestamp_utc",
    "api_version",
    "status",
    "model_versions",
    "prompt_registry_version",
    "latency_ms",
    "warnings",
    "educational_use_only",
]

COMMON_ERROR_FIELDS = [
    "request_id",
    "timestamp_utc",
    "api_version",
    "status",
    "error_code",
    "message",
    "details",
    "latency_ms",
    "educational_use_only",
]

SERVICE_COMPONENTS = {
    "prediction_store",
    "computer_vision",
    "gradcam",
    "grounded_language",
    "language_guardrail",
    "operational_metrics",
}


# -------------------------------------------------------------------------
# Register the complete endpoint inventory
# -------------------------------------------------------------------------
API_ENDPOINT_REGISTRY = [
    {
        "name": "health",
        "method": "GET",
        "path": "/health",
        "summary": "Report API and loaded-service health.",
        "request_contract": None,
        "response_contract": "HealthResponse",
        "services": [],
    },
    {
        "name": "model_info",
        "method": "GET",
        "path": f"{API_PREFIX}/model/info",
        "summary": "Return model, prompt, and explainability lineage.",
        "request_contract": None,
        "response_contract": "ModelInfoResponse",
        "services": [
            "computer_vision",
            "grounded_language",
        ],
    },
    {
        "name": "model_metrics",
        "method": "GET",
        "path": f"{API_PREFIX}/model/metrics",
        "summary": "Return frozen computer-vision and language metrics.",
        "request_contract": None,
        "response_contract": "ModelMetricsResponse",
        "services": [
            "computer_vision",
            "grounded_language",
        ],
    },
    {
        "name": "classify_image",
        "method": "POST",
        "path": f"{API_PREFIX}/image/classify",
        "summary": "Classify the supported ChestMNIST findings.",
        "request_contract": "multipart/form-data image",
        "response_contract": "ClassificationResponse",
        "services": [
            "computer_vision",
            "prediction_store",
        ],
    },
    {
        "name": "analyze_image",
        "method": "POST",
        "path": f"{API_PREFIX}/image/analyze",
        "summary": (
            "Classify an image and generate finding-specific "
            "Grad-CAM evidence."
        ),
        "request_contract": "multipart/form-data image",
        "response_contract": "ImageAnalysisResponse",
        "services": [
            "computer_vision",
            "gradcam",
            "prediction_store",
        ],
    },
    {
        "name": "generate_report",
        "method": "POST",
        "path": f"{API_PREFIX}/report/generate",
        "summary": "Generate a guarded structured preliminary report.",
        "request_contract": "GroundedGenerationRequest",
        "response_contract": "LanguageGenerationResponse",
        "services": [
            "prediction_store",
            "grounded_language",
            "language_guardrail",
        ],
    },
    {
        "name": "generate_explanation",
        "method": "POST",
        "path": f"{API_PREFIX}/explanation/generate",
        "summary": "Generate a guarded plain-language explanation.",
        "request_contract": "GroundedGenerationRequest",
        "response_contract": "LanguageGenerationResponse",
        "services": [
            "prediction_store",
            "grounded_language",
            "language_guardrail",
        ],
    },
    {
        "name": "answer_question",
        "method": "POST",
        "path": f"{API_PREFIX}/question/answer",
        "summary": (
            "Answer a question using only stored grounded model evidence."
        ),
        "request_contract": "GroundedQuestionRequest",
        "response_contract": "LanguageGenerationResponse",
        "services": [
            "prediction_store",
            "grounded_language",
            "language_guardrail",
        ],
    },
    {
        "name": "recommend_follow_up",
        "method": "POST",
        "path": f"{API_PREFIX}/follow-up/recommend",
        "summary": "Generate controlled educational follow-up guidance.",
        "request_contract": "GroundedGenerationRequest",
        "response_contract": "LanguageGenerationResponse",
        "services": [
            "prediction_store",
            "grounded_language",
            "language_guardrail",
        ],
    },
    {
        "name": "analyze_complete",
        "method": "POST",
        "path": f"{API_PREFIX}/analyze-complete",
        "summary": (
            "Run classification, Grad-CAM, and all grounded "
            "language tasks."
        ),
        "request_contract": (
            "multipart/form-data image with optional question"
        ),
        "response_contract": "CompleteAnalysisResponse",
        "services": [
            "computer_vision",
            "gradcam",
            "grounded_language",
            "language_guardrail",
            "prediction_store",
            "operational_metrics",
        ],
    },
    {
        "name": "get_prediction",
        "method": "GET",
        "path": f"{API_PREFIX}/predictions/{{prediction_id}}",
        "summary": "Retrieve a previously stored prediction result.",
        "request_contract": "prediction_id path parameter",
        "response_contract": "StoredPredictionResponse",
        "services": [
            "prediction_store",
        ],
    },
    {
        "name": "llmops_metrics",
        "method": "GET",
        "path": f"{API_PREFIX}/llmops/metrics",
        "summary": (
            "Return aggregate request, latency, generation, "
            "and guardrail metrics."
        ),
        "request_contract": None,
        "response_contract": "OperationalMetricsResponse",
        "services": [
            "operational_metrics",
        ],
    },
]


# -------------------------------------------------------------------------
# Build the versioned API contract registry
# -------------------------------------------------------------------------
API_CONTRACT_PATH = (
    SOLUTION_ROOT
    / "configs"
    / "api_contract.yaml"
)

api_contract_registry = {
    "contract_version": API_CONTRACT_VERSION,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "api_title": API_TITLE,
    "api_version": API_VERSION,
    "api_prefix": API_PREFIX,
    "service_lineage": {
        "computer_vision_model_version": (
            COMPUTER_VISION_MODEL_VERSION
        ),
        "language_model_version": (
            LANGUAGE_MODEL_VERSION
        ),
        "prompt_registry_version": (
            PROMPT_REGISTRY_VERSION
        ),
    },
    "upload_contract": {
        "maximum_size_mib": (
            MAXIMUM_UPLOAD_SIZE_MIB
        ),
        "supported_media_types": (
            SUPPORTED_IMAGE_MEDIA_TYPES
        ),
        "image_mode_handling": (
            "convert accepted images to RGB"
        ),
    },
    "common_response_fields": (
        COMMON_RESPONSE_FIELDS
    ),
    "common_error_fields": (
        COMMON_ERROR_FIELDS
    ),
    "global_response_rules": [
        "Generate a UUID request identifier for every request.",
        "Return an ISO-8601 UTC timestamp.",
        "Return model and prompt lineage where applicable.",
        "Return total endpoint latency in milliseconds.",
        "Return guardrail action and trigger reasons for language outputs.",
        "Do not expose internal tracebacks or filesystem paths.",
        "Do not claim that an output is a clinical diagnosis.",
        "Do not claim that Grad-CAM confirms or segments a lesion.",
        "Preserve the no-target-finding interpretation boundary.",
    ],
    "endpoints": API_ENDPOINT_REGISTRY,
}


# -------------------------------------------------------------------------
# Validate endpoint uniqueness and contract completeness
# -------------------------------------------------------------------------
endpoint_keys = [
    (
        endpoint["method"],
        endpoint["path"],
    )
    for endpoint in API_ENDPOINT_REGISTRY
]

required_endpoint_names = {
    "health",
    "model_info",
    "model_metrics",
    "classify_image",
    "analyze_image",
    "generate_report",
    "generate_explanation",
    "answer_question",
    "recommend_follow_up",
    "analyze_complete",
    "get_prediction",
    "llmops_metrics",
}

contract_checks = {
    "Twelve endpoints are registered": (
        len(API_ENDPOINT_REGISTRY) == 12
    ),
    "Required endpoint names are preserved": (
        {
            endpoint["name"]
            for endpoint in API_ENDPOINT_REGISTRY
        }
        == required_endpoint_names
    ),
    "Method and path combinations are unique": (
        len(endpoint_keys)
        == len(set(endpoint_keys))
    ),
    "Health endpoint remains unversioned": (
        API_ENDPOINT_REGISTRY[0]["path"]
        == "/health"
    ),
    "All other endpoints use the API prefix": all(
        endpoint["path"].startswith(
            API_PREFIX
        )
        for endpoint in API_ENDPOINT_REGISTRY
        if endpoint["name"] != "health"
    ),
    "All declared services are registered": all(
        service_name in SERVICE_COMPONENTS
        for endpoint in API_ENDPOINT_REGISTRY
        for service_name in endpoint["services"]
    ),
    "Every endpoint has a response contract": all(
        bool(endpoint["response_contract"])
        for endpoint in API_ENDPOINT_REGISTRY
    ),
    "Common response contract is complete": (
        len(COMMON_RESPONSE_FIELDS)
        == len(set(COMMON_RESPONSE_FIELDS))
        and len(COMMON_RESPONSE_FIELDS) == 9
    ),
    "Common error contract is complete": (
        len(COMMON_ERROR_FIELDS)
        == len(set(COMMON_ERROR_FIELDS))
        and len(COMMON_ERROR_FIELDS) == 9
    ),
}


# -------------------------------------------------------------------------
# Persist the canonical API contract
# -------------------------------------------------------------------------
with API_CONTRACT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        api_contract_registry,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    )


# -------------------------------------------------------------------------
# Report the endpoint registry
# -------------------------------------------------------------------------
print("VERSIONED FASTAPI ENDPOINT CONTRACT")
print("-" * 100)
print(f"Contract version         : {API_CONTRACT_VERSION}")
print(f"Contract path            : {API_CONTRACT_PATH}")
print(f"Registered endpoints     : {len(API_ENDPOINT_REGISTRY)}")
print(
    f"Supported image types    : "
    f"{', '.join(SUPPORTED_IMAGE_MEDIA_TYPES)}"
)
print(
    f"Maximum upload size      : "
    f"{MAXIMUM_UPLOAD_SIZE_MIB} MiB"
)
print("-" * 100)

for endpoint in API_ENDPOINT_REGISTRY:
    service_summary = (
        ", ".join(endpoint["services"])
        if endpoint["services"]
        else "service readiness only"
    )

    print(
        f"{endpoint['method']:<6} "
        f"{endpoint['path']:<42} | "
        f"{service_summary}"
    )

print("-" * 100)

for check_name, passed in contract_checks.items():
    print(
        f"{check_name:<52}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(contract_checks.values()):
    failed_checks = [
        name
        for name, passed
        in contract_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "API endpoint contract validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: VERSIONED API ENDPOINT CONTRACT REGISTERED")

VERSIONED FASTAPI ENDPOINT CONTRACT
----------------------------------------------------------------------------------------------------
Contract version         : api-contract-v1
Contract path            : /home/jovyan/chest-xray-ai-assistant/configs/api_contract.yaml
Registered endpoints     : 12
Supported image types    : image/png, image/jpeg
Maximum upload size      : 10 MiB
----------------------------------------------------------------------------------------------------
GET    /health                                    | service readiness only
GET    /api/v1/model/info                         | computer_vision, grounded_language
GET    /api/v1/model/metrics                      | computer_vision, grounded_language
POST   /api/v1/image/classify                     | computer_vision, prediction_store
POST   /api/v1/image/analyze                      | computer_vision, gradcam, prediction_store
POST   /api/v1/report/generate                    | prediction_store, grounded_languag

**[↑ Back to notebook index](#notebook-index)**


<a id="nb07-3-pydantic-request-and-response-schemas"></a>
## 3. Pydantic Request and Response Schemas

<a id="nb07-3-1-common-response-metadata-and-controlled-error-contract"></a>
### 3.1 Common Response Metadata and Controlled Error Contract

All successful endpoint responses inherit a shared top-level metadata contract. It carries the request identifier, UTC timestamp, API and model versions, prompt lineage, endpoint latency, warnings, and the educational-use boundary.

Controlled API errors use a separate schema with a stable error code and client-safe details. Pydantic forbids undeclared fields, negative latency, incorrect status values, and removal of the educational-use restriction.


In [10]:
import hashlib
import importlib
from datetime import datetime, timezone
from uuid import uuid4

from pydantic import ValidationError


# -------------------------------------------------------------------------
# Initialize the canonical Python package boundaries
# -------------------------------------------------------------------------
package_directories = [
    API_ROOT,
    API_CORE_DIR,
    API_ROUTES_DIR,
    API_SCHEMAS_DIR,
    SOURCE_ROOT,
    SERVICE_ROOT,
]

package_init_paths = []

for package_directory in package_directories:
    init_path = package_directory / "__init__.py"

    if not init_path.exists():
        init_path.write_text(
            '"""Chest X-ray assistant service package."""\n',
            encoding="utf-8",
        )

    package_init_paths.append(init_path)


# -------------------------------------------------------------------------
# Define the reusable common schema module
# -------------------------------------------------------------------------
COMMON_SCHEMA_PATH = (
    API_SCHEMAS_DIR / "common.py"
)

COMMON_SCHEMA_SOURCE = '''"""Common Pydantic contracts for versioned API responses."""

from datetime import datetime
from typing import Any, Literal
from uuid import UUID

from pydantic import BaseModel, ConfigDict, Field


class StrictSchema(BaseModel):
    """Forbid undeclared request and response fields."""

    model_config = ConfigDict(extra="forbid")


class ModelVersions(StrictSchema):
    """Version lineage for the models contributing to a response."""

    computer_vision: str
    language: str | None = None
    explainability_method: str | None = None


class SuccessResponseBase(StrictSchema):
    """Shared top-level fields for successful API responses."""

    request_id: UUID
    timestamp_utc: datetime
    api_version: Literal["v1"] = "v1"
    status: Literal["success"] = "success"
    model_versions: ModelVersions
    prompt_registry_version: str | None = None
    latency_ms: float = Field(ge=0.0)
    warnings: list[str] = Field(default_factory=list)
    educational_use_only: Literal[True] = True


class APIErrorResponse(StrictSchema):
    """Controlled client-safe error response."""

    request_id: UUID
    timestamp_utc: datetime
    api_version: Literal["v1"] = "v1"
    status: Literal["error"] = "error"
    error_code: str = Field(min_length=1, max_length=100)
    message: str = Field(min_length=1, max_length=500)
    details: dict[str, Any] = Field(default_factory=dict)
    latency_ms: float = Field(ge=0.0)
    educational_use_only: Literal[True] = True
'''

COMMON_SCHEMA_PATH.write_text(
    COMMON_SCHEMA_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()

for module_name in [
    "api.schemas.common",
]:
    sys.modules.pop(
        module_name,
        None,
    )

common_schema_module = importlib.import_module(
    "api.schemas.common"
)

ModelVersions = (
    common_schema_module.ModelVersions
)
SuccessResponseBase = (
    common_schema_module.SuccessResponseBase
)
APIErrorResponse = (
    common_schema_module.APIErrorResponse
)


# -------------------------------------------------------------------------
# Instantiate valid success and error examples
# -------------------------------------------------------------------------
schema_test_request_id = uuid4()
schema_test_timestamp = datetime.now(
    timezone.utc
)

success_schema_example = SuccessResponseBase(
    request_id=schema_test_request_id,
    timestamp_utc=schema_test_timestamp,
    model_versions=ModelVersions(
        computer_vision=(
            COMPUTER_VISION_MODEL_VERSION
        ),
        language=LANGUAGE_MODEL_VERSION,
        explainability_method=(
            "LayerGradCam"
        ),
    ),
    prompt_registry_version=(
        PROMPT_REGISTRY_VERSION
    ),
    latency_ms=12.5,
    warnings=[],
)

error_schema_example = APIErrorResponse(
    request_id=schema_test_request_id,
    timestamp_utc=schema_test_timestamp,
    error_code="INVALID_IMAGE",
    message=(
        "The uploaded file is not a supported image."
    ),
    details={
        "supported_media_types": (
            SUPPORTED_IMAGE_MEDIA_TYPES
        )
    },
    latency_ms=1.4,
)


# -------------------------------------------------------------------------
# Verify that unsafe or malformed metadata is rejected
# -------------------------------------------------------------------------
negative_latency_rejected = False
educational_boundary_rejected = False
unknown_field_rejected = False

try:
    SuccessResponseBase(
        request_id=uuid4(),
        timestamp_utc=datetime.now(
            timezone.utc
        ),
        model_versions=ModelVersions(
            computer_vision=(
                COMPUTER_VISION_MODEL_VERSION
            )
        ),
        latency_ms=-1.0,
    )
except ValidationError:
    negative_latency_rejected = True

try:
    APIErrorResponse(
        request_id=uuid4(),
        timestamp_utc=datetime.now(
            timezone.utc
        ),
        error_code="TEST_ERROR",
        message="Controlled validation test.",
        latency_ms=0.1,
        educational_use_only=False,
    )
except ValidationError:
    educational_boundary_rejected = True

try:
    SuccessResponseBase(
        request_id=uuid4(),
        timestamp_utc=datetime.now(
            timezone.utc
        ),
        model_versions=ModelVersions(
            computer_vision=(
                COMPUTER_VISION_MODEL_VERSION
            )
        ),
        latency_ms=0.1,
        undeclared_field="not permitted",
    )
except ValidationError:
    unknown_field_rejected = True


# -------------------------------------------------------------------------
# Validate the written module and serialization contract
# -------------------------------------------------------------------------
common_schema_checksum = hashlib.sha256(
    COMMON_SCHEMA_PATH.read_bytes()
).hexdigest()

success_json = (
    success_schema_example.model_dump(
        mode="json"
    )
)
error_json = (
    error_schema_example.model_dump(
        mode="json"
    )
)

common_schema_checks = {
    "All package boundaries are initialized": all(
        path.is_file()
        for path in package_init_paths
    ),
    "Common schema module was written": (
        COMMON_SCHEMA_PATH.is_file()
    ),
    "Success schema preserves request ID": (
        success_json["request_id"]
        == str(schema_test_request_id)
    ),
    "Success status is fixed": (
        success_json["status"] == "success"
    ),
    "Error status is fixed": (
        error_json["status"] == "error"
    ),
    "Model lineage is preserved": (
        success_json["model_versions"][
            "computer_vision"
        ]
        == COMPUTER_VISION_MODEL_VERSION
        and success_json["model_versions"][
            "language"
        ]
        == LANGUAGE_MODEL_VERSION
    ),
    "Negative latency is rejected": (
        negative_latency_rejected
    ),
    "Educational boundary cannot be disabled": (
        educational_boundary_rejected
    ),
    "Undeclared fields are rejected": (
        unknown_field_rejected
    ),
    "Common schema checksum is available": (
        len(common_schema_checksum) == 64
    ),
}


# -------------------------------------------------------------------------
# Report common schema readiness
# -------------------------------------------------------------------------
print("COMMON API RESPONSE AND ERROR SCHEMAS")
print("-" * 100)
print(f"Schema module             : {COMMON_SCHEMA_PATH}")
print(
    f"Schema SHA-256            : "
    f"{common_schema_checksum[:16]}..."
)
print(
    f"Success response fields   : "
    f"{len(success_json)}"
)
print(
    f"Error response fields     : "
    f"{len(error_json)}"
)
print(
    f"Success request ID        : "
    f"{success_json['request_id']}"
)
print(
    f"Success status            : "
    f"{success_json['status']}"
)
print(
    f"Error status              : "
    f"{error_json['status']}"
)
print("-" * 100)

for check_name, passed in common_schema_checks.items():
    print(
        f"{check_name:<54}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(common_schema_checks.values()):
    failed_checks = [
        name
        for name, passed
        in common_schema_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Common API schema validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: COMMON API SCHEMA CONTRACT READY")

COMMON API RESPONSE AND ERROR SCHEMAS
----------------------------------------------------------------------------------------------------
Schema module             : /home/jovyan/chest-xray-ai-assistant/api/schemas/common.py
Schema SHA-256            : ee855fbeda300725...
Success response fields   : 9
Error response fields     : 9
Success request ID        : 721d5299-8c8a-46ae-b382-ad84ee854863
Success status            : success
Error status              : error
----------------------------------------------------------------------------------------------------
All package boundaries are initialized                : PASS
Common schema module was written                      : PASS
Success schema preserves request ID                   : PASS
Success status is fixed                               : PASS
Error status is fixed                                 : PASS
Model lineage is preserved                            : PASS
Negative latency is rejected                          : PASS
Edu

<a id="nb07-3-2-grounded-language-request-contracts"></a>
### 3.2 Grounded Language Request Contracts

Language endpoints operate only on a stored prediction identifier, preventing clients from directly injecting unsupported finding names, probabilities, thresholds, or descriptions into the generation context.

Question-answering requests additionally accept a bounded user question. Whitespace-only questions, malformed identifiers, control characters, excessive length, and undeclared request fields are rejected before reaching the language model.


In [11]:
# -------------------------------------------------------------------------
# Define the grounded language request-schema module
# -------------------------------------------------------------------------
REQUEST_SCHEMA_PATH = (
    API_SCHEMAS_DIR / "requests.py"
)

REQUEST_SCHEMA_SOURCE = '''"""Pydantic request contracts for grounded language endpoints."""

from uuid import UUID

from pydantic import Field, field_validator

from api.schemas.common import StrictSchema


class GroundedGenerationRequest(StrictSchema):
    """Request a language task using an existing prediction context."""

    prediction_id: UUID


class GroundedQuestionRequest(StrictSchema):
    """Request grounded question answering for an existing prediction."""

    prediction_id: UUID
    question: str = Field(min_length=3, max_length=500)

    @field_validator("question")
    @classmethod
    def validate_question(cls, value: str) -> str:
        cleaned_value = value.strip()

        if len(cleaned_value) < 3:
            raise ValueError(
                "The question must contain at least three non-whitespace characters."
            )

        if any(
            ord(character) < 32
            and character not in {"\\t", "\\n", "\\r"}
            for character in cleaned_value
        ):
            raise ValueError(
                "The question contains unsupported control characters."
            )

        return cleaned_value


class CompleteAnalysisOptions(StrictSchema):
    """Validated optional form values for the combined workflow."""

    question: str | None = Field(
        default=None,
        max_length=500,
    )

    @field_validator("question")
    @classmethod
    def validate_optional_question(
        cls,
        value: str | None,
    ) -> str | None:
        if value is None:
            return None

        cleaned_value = value.strip()

        if not cleaned_value:
            return None

        if len(cleaned_value) < 3:
            raise ValueError(
                "The optional question must contain at least three characters."
            )

        if any(
            ord(character) < 32
            and character not in {"\\t", "\\n", "\\r"}
            for character in cleaned_value
        ):
            raise ValueError(
                "The optional question contains unsupported control characters."
            )

        return cleaned_value
'''

REQUEST_SCHEMA_PATH.write_text(
    REQUEST_SCHEMA_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()
sys.modules.pop(
    "api.schemas.requests",
    None,
)

request_schema_module = importlib.import_module(
    "api.schemas.requests"
)

GroundedGenerationRequest = (
    request_schema_module.GroundedGenerationRequest
)
GroundedQuestionRequest = (
    request_schema_module.GroundedQuestionRequest
)
CompleteAnalysisOptions = (
    request_schema_module.CompleteAnalysisOptions
)


# -------------------------------------------------------------------------
# Create valid request examples
# -------------------------------------------------------------------------
request_test_prediction_id = uuid4()

generation_request_example = (
    GroundedGenerationRequest(
        prediction_id=(
            request_test_prediction_id
        )
    )
)

question_request_example = (
    GroundedQuestionRequest(
        prediction_id=(
            request_test_prediction_id
        ),
        question=(
            "  What does the supplied model "
            "information indicate?  "
        ),
    )
)

complete_options_example = (
    CompleteAnalysisOptions(
        question="   "
    )
)


# -------------------------------------------------------------------------
# Verify malformed requests are rejected
# -------------------------------------------------------------------------
invalid_uuid_rejected = False
blank_question_rejected = False
short_question_rejected = False
long_question_rejected = False
control_character_rejected = False
extra_request_field_rejected = False

try:
    GroundedGenerationRequest(
        prediction_id="not-a-valid-uuid"
    )
except ValidationError:
    invalid_uuid_rejected = True

try:
    GroundedQuestionRequest(
        prediction_id=uuid4(),
        question="     ",
    )
except ValidationError:
    blank_question_rejected = True

try:
    GroundedQuestionRequest(
        prediction_id=uuid4(),
        question="x ",
    )
except ValidationError:
    short_question_rejected = True

try:
    GroundedQuestionRequest(
        prediction_id=uuid4(),
        question="Q" * 501,
    )
except ValidationError:
    long_question_rejected = True

try:
    GroundedQuestionRequest(
        prediction_id=uuid4(),
        question="What does this mean?\x00",
    )
except ValidationError:
    control_character_rejected = True

try:
    GroundedGenerationRequest(
        prediction_id=uuid4(),
        unsupported_context={
            "finding": "not permitted"
        },
    )
except ValidationError:
    extra_request_field_rejected = True


# -------------------------------------------------------------------------
# Validate request normalization and module integrity
# -------------------------------------------------------------------------
request_schema_checksum = hashlib.sha256(
    REQUEST_SCHEMA_PATH.read_bytes()
).hexdigest()

request_schema_checks = {
    "Request schema module was written": (
        REQUEST_SCHEMA_PATH.is_file()
    ),
    "Prediction identifier is preserved": (
        generation_request_example.prediction_id
        == request_test_prediction_id
    ),
    "Question whitespace is normalized": (
        question_request_example.question
        == (
            "What does the supplied model "
            "information indicate?"
        )
    ),
    "Blank optional question becomes absent": (
        complete_options_example.question
        is None
    ),
    "Malformed UUID is rejected": (
        invalid_uuid_rejected
    ),
    "Whitespace-only question is rejected": (
        blank_question_rejected
    ),
    "Question below minimum length is rejected": (
        short_question_rejected
    ),
    "Question above maximum length is rejected": (
        long_question_rejected
    ),
    "Unsupported control character is rejected": (
        control_character_rejected
    ),
    "Client-supplied grounding context is rejected": (
        extra_request_field_rejected
    ),
    "Request schema checksum is available": (
        len(request_schema_checksum) == 64
    ),
}


# -------------------------------------------------------------------------
# Report grounded request readiness
# -------------------------------------------------------------------------
print("GROUNDED LANGUAGE REQUEST SCHEMAS")
print("-" * 100)
print(f"Schema module             : {REQUEST_SCHEMA_PATH}")
print(
    f"Schema SHA-256            : "
    f"{request_schema_checksum[:16]}..."
)
print(
    f"Prediction identifier     : "
    f"{generation_request_example.prediction_id}"
)
print(
    f"Normalized question       : "
    f"{question_request_example.question}"
)
print(
    f"Blank optional question   : "
    f"{complete_options_example.question}"
)
print("-" * 100)

for check_name, passed in request_schema_checks.items():
    print(
        f"{check_name:<56}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(request_schema_checks.values()):
    failed_checks = [
        name
        for name, passed
        in request_schema_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Grounded request schema validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: GROUNDED REQUEST CONTRACTS READY")

GROUNDED LANGUAGE REQUEST SCHEMAS
----------------------------------------------------------------------------------------------------
Schema module             : /home/jovyan/chest-xray-ai-assistant/api/schemas/requests.py
Schema SHA-256            : f2b276869a95d489...
Prediction identifier     : a51728c6-b928-4aba-aceb-961ec7116a1f
Normalized question       : What does the supplied model information indicate?
Blank optional question   : None
----------------------------------------------------------------------------------------------------
Request schema module was written                       : PASS
Prediction identifier is preserved                      : PASS
Question whitespace is normalized                       : PASS
Blank optional question becomes absent                  : PASS
Malformed UUID is rejected                              : PASS
Whitespace-only question is rejected                    : PASS
Question below minimum length is rejected               : PASS
Question 

<a id="nb07-3-3-image-classification-and-finding-evidence-schemas"></a>
### 3.3 Image Classification and Finding-Evidence Schemas

The classification response represents all 14 supported ChestMNIST findings rather than returning only positive decisions. Each finding preserves its label identity, model probability, frozen threshold, threshold decision, confidence category, and approved description.

Crossed-finding names and the no-target-finding state are validated against the individual finding decisions. This prevents contradictory responses such as reporting a no-target-finding state while one or more findings have crossed their thresholds.


In [12]:
# -------------------------------------------------------------------------
# Define image and classification response schemas
# -------------------------------------------------------------------------
PREDICTION_SCHEMA_PATH = (
    API_SCHEMAS_DIR / "prediction.py"
)

PREDICTION_SCHEMA_SOURCE = '''"""Classification and finding-evidence response schemas."""

from typing import Literal
from uuid import UUID

from pydantic import Field, model_validator

from api.schemas.common import StrictSchema, SuccessResponseBase


class ImageMetadata(StrictSchema):
    """Validated metadata for an accepted uploaded image."""

    filename: str = Field(min_length=1, max_length=255)
    media_type: Literal["image/png", "image/jpeg"]
    width: int = Field(gt=0)
    height: int = Field(gt=0)
    original_mode: str = Field(min_length=1, max_length=20)
    sha256: str = Field(
        pattern=r"^[a-f0-9]{64}$"
    )


class FindingEvidence(StrictSchema):
    """One model finding with its frozen decision evidence."""

    label_id: int = Field(ge=0, le=13)
    label_name: str = Field(min_length=1, max_length=50)
    display_name: str = Field(min_length=1, max_length=100)
    probability: float = Field(ge=0.0, le=1.0)
    frozen_threshold: float = Field(ge=0.0, le=1.0)
    crossed_threshold: bool
    confidence_category: Literal[
        "below_threshold",
        "borderline",
        "moderate",
        "higher",
    ]
    approved_description: str = Field(
        min_length=1,
        max_length=500,
    )


class ClassificationResponse(SuccessResponseBase):
    """Complete multilabel classification response."""

    prediction_id: UUID
    image: ImageMetadata
    findings: list[FindingEvidence] = Field(
        min_length=14,
        max_length=14,
    )
    crossed_finding_names: list[str]
    no_target_finding: bool
    interpretation: str = Field(
        min_length=1,
        max_length=1000,
    )

    @model_validator(mode="after")
    def validate_finding_contract(
        self,
    ) -> "ClassificationResponse":
        label_ids = [
            finding.label_id
            for finding in self.findings
        ]
        label_names = [
            finding.label_name
            for finding in self.findings
        ]

        if len(set(label_ids)) != 14:
            raise ValueError(
                "Finding label identifiers must be unique."
            )

        if len(set(label_names)) != 14:
            raise ValueError(
                "Finding label names must be unique."
            )

        expected_crossed_names = [
            finding.label_name
            for finding in self.findings
            if finding.crossed_threshold
        ]

        if (
            self.crossed_finding_names
            != expected_crossed_names
        ):
            raise ValueError(
                "Crossed finding names must match the ordered finding decisions."
            )

        expected_no_target_state = (
            len(expected_crossed_names) == 0
        )

        if (
            self.no_target_finding
            != expected_no_target_state
        ):
            raise ValueError(
                "The no-target-finding state conflicts with finding decisions."
            )

        return self
'''

PREDICTION_SCHEMA_PATH.write_text(
    PREDICTION_SCHEMA_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()
sys.modules.pop(
    "api.schemas.prediction",
    None,
)

prediction_schema_module = (
    importlib.import_module(
        "api.schemas.prediction"
    )
)

ImageMetadata = (
    prediction_schema_module.ImageMetadata
)
FindingEvidence = (
    prediction_schema_module.FindingEvidence
)
ClassificationResponse = (
    prediction_schema_module.ClassificationResponse
)


# -------------------------------------------------------------------------
# Create a complete valid 14-label response example
# -------------------------------------------------------------------------
CHESTMNIST_LABEL_NAMES = [
    "atelectasis",
    "cardiomegaly",
    "effusion",
    "infiltration",
    "mass",
    "nodule",
    "pneumonia",
    "pneumothorax",
    "consolidation",
    "edema",
    "emphysema",
    "fibrosis",
    "pleural",
    "hernia",
]

CHESTMNIST_DISPLAY_NAMES = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Pleural Abnormality",
    "Hernia",
]

schema_test_findings = []

for label_id, (
    label_name,
    display_name,
) in enumerate(
    zip(
        CHESTMNIST_LABEL_NAMES,
        CHESTMNIST_DISPLAY_NAMES,
    )
):
    crossed_threshold = label_id == 0

    schema_test_findings.append(
        FindingEvidence(
            label_id=label_id,
            label_name=label_name,
            display_name=display_name,
            probability=(
                0.8000
                if crossed_threshold
                else 0.2000
            ),
            frozen_threshold=0.7000,
            crossed_threshold=(
                crossed_threshold
            ),
            confidence_category=(
                "higher"
                if crossed_threshold
                else "below_threshold"
            ),
            approved_description=(
                f"Controlled description for "
                f"{display_name}."
            ),
        )
    )

classification_response_example = (
    ClassificationResponse(
        request_id=uuid4(),
        timestamp_utc=datetime.now(
            timezone.utc
        ),
        model_versions=ModelVersions(
            computer_vision=(
                COMPUTER_VISION_MODEL_VERSION
            )
        ),
        latency_ms=8.4,
        prediction_id=uuid4(),
        image=ImageMetadata(
            filename="example.png",
            media_type="image/png",
            width=224,
            height=224,
            original_mode="L",
            sha256="a" * 64,
        ),
        findings=schema_test_findings,
        crossed_finding_names=[
            "atelectasis"
        ],
        no_target_finding=False,
        interpretation=(
            "One supplied finding crossed its "
            "frozen threshold. This is not a diagnosis."
        ),
    )
)


# -------------------------------------------------------------------------
# Verify contradictory or malformed evidence is rejected
# -------------------------------------------------------------------------
inconsistent_no_target_rejected = False
duplicate_label_rejected = False
invalid_probability_rejected = False
invalid_hash_rejected = False

valid_response_payload = (
    classification_response_example.model_dump()
)

try:
    contradictory_payload = dict(
        valid_response_payload
    )
    contradictory_payload[
        "no_target_finding"
    ] = True

    ClassificationResponse.model_validate(
        contradictory_payload
    )
except ValidationError:
    inconsistent_no_target_rejected = True

try:
    duplicate_payload = dict(
        valid_response_payload
    )
    duplicate_findings = [
        dict(finding)
        for finding in duplicate_payload[
            "findings"
        ]
    ]
    duplicate_findings[1] = dict(
        duplicate_findings[0]
    )
    duplicate_payload[
        "findings"
    ] = duplicate_findings

    ClassificationResponse.model_validate(
        duplicate_payload
    )
except ValidationError:
    duplicate_label_rejected = True

try:
    FindingEvidence(
        label_id=0,
        label_name="atelectasis",
        display_name="Atelectasis",
        probability=1.5,
        frozen_threshold=0.7,
        crossed_threshold=True,
        confidence_category="higher",
        approved_description=(
            "Controlled description."
        ),
    )
except ValidationError:
    invalid_probability_rejected = True

try:
    ImageMetadata(
        filename="example.png",
        media_type="image/png",
        width=224,
        height=224,
        original_mode="L",
        sha256="invalid-hash",
    )
except ValidationError:
    invalid_hash_rejected = True


# -------------------------------------------------------------------------
# Validate and report the prediction schema module
# -------------------------------------------------------------------------
prediction_schema_checksum = (
    hashlib.sha256(
        PREDICTION_SCHEMA_PATH.read_bytes()
    ).hexdigest()
)

classification_json = (
    classification_response_example.model_dump(
        mode="json"
    )
)

prediction_schema_checks = {
    "Prediction schema module was written": (
        PREDICTION_SCHEMA_PATH.is_file()
    ),
    "Exactly fourteen findings are represented": (
        len(classification_json["findings"])
        == 14
    ),
    "Crossed finding order is preserved": (
        classification_json[
            "crossed_finding_names"
        ]
        == ["atelectasis"]
    ),
    "No-target state matches decisions": (
        classification_json[
            "no_target_finding"
        ]
        is False
    ),
    "Contradictory no-target state is rejected": (
        inconsistent_no_target_rejected
    ),
    "Duplicate finding labels are rejected": (
        duplicate_label_rejected
    ),
    "Out-of-range probability is rejected": (
        invalid_probability_rejected
    ),
    "Malformed image hash is rejected": (
        invalid_hash_rejected
    ),
    "Prediction schema checksum is available": (
        len(prediction_schema_checksum)
        == 64
    ),
}

print("IMAGE CLASSIFICATION RESPONSE SCHEMAS")
print("-" * 100)
print(
    f"Schema module             : "
    f"{PREDICTION_SCHEMA_PATH}"
)
print(
    f"Schema SHA-256            : "
    f"{prediction_schema_checksum[:16]}..."
)
print(
    f"Finding records           : "
    f"{len(classification_json['findings'])}"
)
print(
    f"Crossed findings          : "
    f"{classification_json['crossed_finding_names']}"
)
print(
    f"No-target-finding state   : "
    f"{classification_json['no_target_finding']}"
)
print("-" * 100)

for check_name, passed in prediction_schema_checks.items():
    print(
        f"{check_name:<56}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(prediction_schema_checks.values()):
    failed_checks = [
        name
        for name, passed
        in prediction_schema_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Prediction response schema validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: CLASSIFICATION RESPONSE CONTRACT READY")

IMAGE CLASSIFICATION RESPONSE SCHEMAS
----------------------------------------------------------------------------------------------------
Schema module             : /home/jovyan/chest-xray-ai-assistant/api/schemas/prediction.py
Schema SHA-256            : 7c9c0ff219b46f5c...
Finding records           : 14
Crossed findings          : ['atelectasis']
No-target-finding state   : False
----------------------------------------------------------------------------------------------------
Prediction schema module was written                    : PASS
Exactly fourteen findings are represented               : PASS
Crossed finding order is preserved                      : PASS
No-target state matches decisions                       : PASS
Contradictory no-target state is rejected               : PASS
Duplicate finding labels are rejected                   : PASS
Out-of-range probability is rejected                    : PASS
Malformed image hash is rejected                        : PASS
Predicti

<a id="nb07-3-4-visual-explainability-response-schemas"></a>
### 3.4 Visual Explainability Response Schemas

Visual analysis responses extend the classification contract with finding-specific Grad-CAM evidence. A heatmap and an image overlay are returned as validated base64-encoded PNG content so that the Streamlit client can display them without accessing backend filesystem paths.

Visual evidence is generated only for findings that crossed their frozen thresholds. The schema requires exact agreement between crossed findings and Grad-CAM evidence and carries the fixed boundary that attribution highlights model influence rather than lesion segmentation, anatomical confirmation, or diagnosis.


In [13]:
import base64


# -------------------------------------------------------------------------
# Define Grad-CAM and image-analysis response schemas
# -------------------------------------------------------------------------
EXPLAINABILITY_SCHEMA_PATH = (
    API_SCHEMAS_DIR / "explainability.py"
)

EXPLAINABILITY_SCHEMA_SOURCE = '''"""Visual explainability response schemas."""

import base64
from typing import Literal

from pydantic import Field, field_validator, model_validator

from api.schemas.common import StrictSchema
from api.schemas.prediction import ClassificationResponse


class ExplainabilityContract(StrictSchema):
    """Frozen Grad-CAM method and interpretation boundary."""

    method: Literal["LayerGradCam"] = "LayerGradCam"
    target_layer: str = Field(min_length=1, max_length=200)
    attribution_target: Literal[
        "finding_specific_pre_sigmoid_logit"
    ] = "finding_specific_pre_sigmoid_logit"
    positive_attributions_only: Literal[True] = True
    heatmap_normalization: Literal[
        "independent_zero_to_one"
    ] = "independent_zero_to_one"
    limitation: str = Field(min_length=1, max_length=1000)


class GradCAMEvidence(StrictSchema):
    """Finding-specific visual evidence returned as PNG content."""

    finding_name: str = Field(min_length=1, max_length=50)
    probability: float = Field(ge=0.0, le=1.0)
    frozen_threshold: float = Field(ge=0.0, le=1.0)
    crossed_threshold: Literal[True] = True
    heatmap_png_base64: str = Field(min_length=12)
    overlay_png_base64: str = Field(min_length=12)
    high_attribution_area_percent: float | None = Field(
        default=None,
        ge=0.0,
        le=100.0,
    )

    @field_validator(
        "heatmap_png_base64",
        "overlay_png_base64",
    )
    @classmethod
    def validate_png_base64(cls, value: str) -> str:
        try:
            decoded_value = base64.b64decode(
                value,
                validate=True,
            )
        except Exception as error:
            raise ValueError(
                "Visual evidence must contain valid base64."
            ) from error

        if not decoded_value.startswith(
            b"\\x89PNG\\r\\n\\x1a\\n"
        ):
            raise ValueError(
                "Visual evidence must contain PNG content."
            )

        return value


class ImageAnalysisResponse(ClassificationResponse):
    """Classification response extended with Grad-CAM evidence."""

    explainability: ExplainabilityContract
    visual_evidence: list[GradCAMEvidence]

    @model_validator(mode="after")
    def validate_visual_evidence(
        self,
    ) -> "ImageAnalysisResponse":
        evidence_names = [
            evidence.finding_name
            for evidence in self.visual_evidence
        ]

        if len(evidence_names) != len(
            set(evidence_names)
        ):
            raise ValueError(
                "Visual-evidence finding names must be unique."
            )

        if evidence_names != self.crossed_finding_names:
            raise ValueError(
                "Visual evidence must match the ordered threshold-crossed findings."
            )

        evidence_by_name = {
            evidence.finding_name: evidence
            for evidence in self.visual_evidence
        }

        for finding in self.findings:
            if finding.crossed_threshold:
                evidence = evidence_by_name[
                    finding.label_name
                ]

                if (
                    abs(
                        evidence.probability
                        - finding.probability
                    )
                    > 1e-6
                    or abs(
                        evidence.frozen_threshold
                        - finding.frozen_threshold
                    )
                    > 1e-6
                ):
                    raise ValueError(
                        "Visual evidence must preserve classification values."
                    )

        return self
'''

EXPLAINABILITY_SCHEMA_PATH.write_text(
    EXPLAINABILITY_SCHEMA_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()
sys.modules.pop(
    "api.schemas.explainability",
    None,
)

explainability_schema_module = (
    importlib.import_module(
        "api.schemas.explainability"
    )
)

ExplainabilityContract = (
    explainability_schema_module.ExplainabilityContract
)
GradCAMEvidence = (
    explainability_schema_module.GradCAMEvidence
)
ImageAnalysisResponse = (
    explainability_schema_module.ImageAnalysisResponse
)


# -------------------------------------------------------------------------
# Create valid PNG test content and visual evidence
# -------------------------------------------------------------------------
schema_test_png_bytes = (
    b"\x89PNG\r\n\x1a\n"
    b"controlled-schema-test"
)

schema_test_png_base64 = (
    base64.b64encode(
        schema_test_png_bytes
    ).decode("ascii")
)

explainability_contract_example = (
    ExplainabilityContract(
        target_layer=(
            "layer4[-1].conv2"
        ),
        limitation=(
            "Grad-CAM highlights image regions that "
            "influenced a model output. It does not "
            "confirm a lesion, provide segmentation, "
            "or establish a clinical diagnosis."
        ),
    )
)

gradcam_evidence_example = (
    GradCAMEvidence(
        finding_name="atelectasis",
        probability=0.8000,
        frozen_threshold=0.7000,
        crossed_threshold=True,
        heatmap_png_base64=(
            schema_test_png_base64
        ),
        overlay_png_base64=(
            schema_test_png_base64
        ),
        high_attribution_area_percent=22.5,
    )
)

classification_payload = (
    classification_response_example.model_dump()
)

analysis_response_example = (
    ImageAnalysisResponse(
        **classification_payload,
        explainability=(
            explainability_contract_example
        ),
        visual_evidence=[
            gradcam_evidence_example
        ],
    )
)


# -------------------------------------------------------------------------
# Verify malformed visual evidence is rejected
# -------------------------------------------------------------------------
invalid_base64_rejected = False
non_png_content_rejected = False
missing_crossed_evidence_rejected = False
mismatched_probability_rejected = False

try:
    GradCAMEvidence(
        finding_name="atelectasis",
        probability=0.8,
        frozen_threshold=0.7,
        crossed_threshold=True,
        heatmap_png_base64="not-base64",
        overlay_png_base64=(
            schema_test_png_base64
        ),
    )
except ValidationError:
    invalid_base64_rejected = True

try:
    non_png_base64 = base64.b64encode(
        b"not-png-content"
    ).decode("ascii")

    GradCAMEvidence(
        finding_name="atelectasis",
        probability=0.8,
        frozen_threshold=0.7,
        crossed_threshold=True,
        heatmap_png_base64=(
            non_png_base64
        ),
        overlay_png_base64=(
            schema_test_png_base64
        ),
    )
except ValidationError:
    non_png_content_rejected = True

try:
    invalid_analysis_payload = dict(
        analysis_response_example.model_dump()
    )
    invalid_analysis_payload[
        "visual_evidence"
    ] = []

    ImageAnalysisResponse.model_validate(
        invalid_analysis_payload
    )
except ValidationError:
    missing_crossed_evidence_rejected = True

try:
    invalid_analysis_payload = dict(
        analysis_response_example.model_dump()
    )
    invalid_visual_evidence = [
        dict(evidence)
        for evidence in invalid_analysis_payload[
            "visual_evidence"
        ]
    ]
    invalid_visual_evidence[0][
        "probability"
    ] = 0.9
    invalid_analysis_payload[
        "visual_evidence"
    ] = invalid_visual_evidence

    ImageAnalysisResponse.model_validate(
        invalid_analysis_payload
    )
except ValidationError:
    mismatched_probability_rejected = True


# -------------------------------------------------------------------------
# Validate and report visual schema readiness
# -------------------------------------------------------------------------
explainability_schema_checksum = (
    hashlib.sha256(
        EXPLAINABILITY_SCHEMA_PATH.read_bytes()
    ).hexdigest()
)

analysis_json = (
    analysis_response_example.model_dump(
        mode="json"
    )
)

explainability_schema_checks = {
    "Explainability schema module was written": (
        EXPLAINABILITY_SCHEMA_PATH.is_file()
    ),
    "Method is frozen to LayerGradCam": (
        analysis_json[
            "explainability"
        ]["method"]
        == "LayerGradCam"
    ),
    "Target layer is preserved": (
        analysis_json[
            "explainability"
        ]["target_layer"]
        == "layer4[-1].conv2"
    ),
    "Visual evidence matches crossed findings": (
        [
            evidence["finding_name"]
            for evidence in analysis_json[
                "visual_evidence"
            ]
        ]
        == analysis_json[
            "crossed_finding_names"
        ]
    ),
    "Valid PNG content is accepted": (
        len(
            analysis_json[
                "visual_evidence"
            ]
        )
        == 1
    ),
    "Malformed base64 is rejected": (
        invalid_base64_rejected
    ),
    "Non-PNG content is rejected": (
        non_png_content_rejected
    ),
    "Missing crossed evidence is rejected": (
        missing_crossed_evidence_rejected
    ),
    "Mismatched evidence values are rejected": (
        mismatched_probability_rejected
    ),
    "Explainability checksum is available": (
        len(
            explainability_schema_checksum
        )
        == 64
    ),
}

print("VISUAL EXPLAINABILITY RESPONSE SCHEMAS")
print("-" * 100)
print(
    f"Schema module             : "
    f"{EXPLAINABILITY_SCHEMA_PATH}"
)
print(
    f"Schema SHA-256            : "
    f"{explainability_schema_checksum[:16]}..."
)
print(
    f"Explainability method     : "
    f"{analysis_json['explainability']['method']}"
)
print(
    f"Target layer              : "
    f"{analysis_json['explainability']['target_layer']}"
)
print(
    f"Visual evidence records   : "
    f"{len(analysis_json['visual_evidence'])}"
)
print(
    f"Evidence finding          : "
    f"{analysis_json['visual_evidence'][0]['finding_name']}"
)
print("-" * 100)

for check_name, passed in (
    explainability_schema_checks.items()
):
    print(
        f"{check_name:<56}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(explainability_schema_checks.values()):
    failed_checks = [
        name
        for name, passed
        in explainability_schema_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Visual explainability schema validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: VISUAL EXPLAINABILITY RESPONSE CONTRACT READY")

VISUAL EXPLAINABILITY RESPONSE SCHEMAS
----------------------------------------------------------------------------------------------------
Schema module             : /home/jovyan/chest-xray-ai-assistant/api/schemas/explainability.py
Schema SHA-256            : a3db9676795c5f8b...
Explainability method     : LayerGradCam
Target layer              : layer4[-1].conv2
Visual evidence records   : 1
Evidence finding          : atelectasis
----------------------------------------------------------------------------------------------------
Explainability schema module was written                : PASS
Method is frozen to LayerGradCam                        : PASS
Target layer is preserved                               : PASS
Visual evidence matches crossed findings                : PASS
Valid PNG content is accepted                           : PASS
Malformed base64 is rejected                            : PASS
Non-PNG content is rejected                             : PASS
Missing crossed evi

<a id="nb07-3-5-grounded-language-and-guardrail-response-schemas"></a>
### 3.5 Grounded Language and Guardrail Response Schemas

All four language tasks share one response schema while retaining explicit task identity, grounded finding names, no-target-finding state, generation latency, token count, prompt lineage, and language-model version.

The response exposes whether the model generation was accepted or replaced by the controlled fallback. Accepted generations cannot contain guardrail triggers, while fallback responses must provide one or more registered trigger reasons. Rejected raw model text is retained only in internal audit artifacts and is not returned to API clients.


In [14]:
# -------------------------------------------------------------------------
# Define grounded language and guardrail response schemas
# -------------------------------------------------------------------------
LANGUAGE_SCHEMA_PATH = (
    API_SCHEMAS_DIR / "language.py"
)

LANGUAGE_SCHEMA_SOURCE = '''"""Grounded language generation response schemas."""

from typing import Literal
from uuid import UUID

from pydantic import Field, model_validator

from api.schemas.common import SuccessResponseBase


LanguageTask = Literal[
    "structured_report",
    "plain_language_explanation",
    "grounded_question_answering",
    "educational_follow_up",
]

GuardrailAction = Literal[
    "accepted_model_generation",
    "safe_template_fallback",
]

GuardrailTrigger = Literal[
    "task_routing_issue",
    "section_order_issue",
    "missing_required_finding_issue",
    "unsupported_finding_issue",
    "numeric_grounding_issue",
    "safety_boundary_issue",
    "no_target_boundary_issue",
    "qa_refusal_issue",
    "gradcam_boundary_issue",
    "forbidden_claim_issue",
]


class LanguageGenerationResponse(SuccessResponseBase):
    """Guarded output from one grounded language task."""

    prediction_id: UUID
    task_type: LanguageTask
    question: str | None = Field(
        default=None,
        min_length=3,
        max_length=500,
    )
    grounded_finding_names: list[str]
    no_target_finding: bool
    output_text: str = Field(
        min_length=1,
        max_length=10000,
    )
    guardrail_action: GuardrailAction
    trigger_reasons: list[GuardrailTrigger] = Field(
        default_factory=list
    )
    generated_tokens: int = Field(ge=1)
    generation_latency_ms: float = Field(ge=0.0)

    @model_validator(mode="after")
    def validate_language_contract(
        self,
    ) -> "LanguageGenerationResponse":
        if self.model_versions.language is None:
            raise ValueError(
                "A language-model version is required."
            )

        if (
            self.guardrail_action
            == "accepted_model_generation"
            and self.trigger_reasons
        ):
            raise ValueError(
                "Accepted generations cannot contain guardrail triggers."
            )

        if (
            self.guardrail_action
            == "safe_template_fallback"
            and not self.trigger_reasons
        ):
            raise ValueError(
                "Fallback responses require at least one trigger reason."
            )

        required_sections = {
            "structured_report": [
                "PRELIMINARY MODEL REPORT",
                "MODEL FINDINGS",
                "LIMITATIONS",
            ],
            "plain_language_explanation": [
                "EXPLANATION",
                "LIMITATIONS",
            ],
            "grounded_question_answering": [
                "ANSWER",
                "LIMITATIONS",
            ],
            "educational_follow_up": [
                "EDUCATIONAL FOLLOW-UP",
                "LIMITATIONS",
            ],
        }[self.task_type]

        section_positions = [
            self.output_text.find(section)
            for section in required_sections
        ]

        if (
            any(
                position < 0
                for position in section_positions
            )
            or section_positions
            != sorted(section_positions)
        ):
            raise ValueError(
                "The generated output does not preserve its task sections."
            )

        if (
            self.task_type
            == "grounded_question_answering"
            and self.question is None
        ):
            raise ValueError(
                "Grounded question answering requires a question."
            )

        return self
'''

LANGUAGE_SCHEMA_PATH.write_text(
    LANGUAGE_SCHEMA_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()
sys.modules.pop(
    "api.schemas.language",
    None,
)

language_schema_module = (
    importlib.import_module(
        "api.schemas.language"
    )
)

LanguageGenerationResponse = (
    language_schema_module.LanguageGenerationResponse
)


# -------------------------------------------------------------------------
# Create valid accepted and fallback response examples
# -------------------------------------------------------------------------
language_response_model_versions = (
    ModelVersions(
        computer_vision=(
            COMPUTER_VISION_MODEL_VERSION
        ),
        language=LANGUAGE_MODEL_VERSION,
    )
)

accepted_report_example = (
    LanguageGenerationResponse(
        request_id=uuid4(),
        timestamp_utc=datetime.now(
            timezone.utc
        ),
        model_versions=(
            language_response_model_versions
        ),
        prompt_registry_version=(
            PROMPT_REGISTRY_VERSION
        ),
        latency_ms=81.5,
        prediction_id=uuid4(),
        task_type="structured_report",
        grounded_finding_names=[
            "atelectasis"
        ],
        no_target_finding=False,
        output_text=(
            "PRELIMINARY MODEL REPORT\n"
            "MODEL FINDINGS\n"
            "The supplied model evidence crossed "
            "the frozen threshold for Atelectasis.\n"
            "LIMITATIONS\n"
            "This output is generated by an "
            "educational decision-support prototype. "
            "It is not a diagnosis."
        ),
        guardrail_action=(
            "accepted_model_generation"
        ),
        trigger_reasons=[],
        generated_tokens=58,
        generation_latency_ms=76.4,
    )
)

fallback_qa_example = (
    LanguageGenerationResponse(
        request_id=uuid4(),
        timestamp_utc=datetime.now(
            timezone.utc
        ),
        model_versions=(
            language_response_model_versions
        ),
        prompt_registry_version=(
            PROMPT_REGISTRY_VERSION
        ),
        latency_ms=84.2,
        prediction_id=uuid4(),
        task_type=(
            "grounded_question_answering"
        ),
        question=(
            "Did the finding cross its threshold?"
        ),
        grounded_finding_names=[
            "atelectasis"
        ],
        no_target_finding=False,
        output_text=(
            "ANSWER\n"
            "The supplied probability crossed its "
            "frozen threshold.\n"
            "LIMITATIONS\n"
            "This output is generated by an "
            "educational decision-support prototype. "
            "It is not a diagnosis."
        ),
        guardrail_action=(
            "safe_template_fallback"
        ),
        trigger_reasons=[
            "numeric_grounding_issue"
        ],
        generated_tokens=49,
        generation_latency_ms=78.9,
    )
)


# -------------------------------------------------------------------------
# Verify contradictory language responses are rejected
# -------------------------------------------------------------------------
accepted_with_trigger_rejected = False
fallback_without_trigger_rejected = False
missing_section_rejected = False
missing_question_rejected = False
missing_language_version_rejected = False

try:
    invalid_payload = (
        accepted_report_example.model_dump()
    )
    invalid_payload["trigger_reasons"] = [
        "numeric_grounding_issue"
    ]

    LanguageGenerationResponse.model_validate(
        invalid_payload
    )
except ValidationError:
    accepted_with_trigger_rejected = True

try:
    invalid_payload = (
        fallback_qa_example.model_dump()
    )
    invalid_payload["trigger_reasons"] = []

    LanguageGenerationResponse.model_validate(
        invalid_payload
    )
except ValidationError:
    fallback_without_trigger_rejected = True

try:
    invalid_payload = (
        accepted_report_example.model_dump()
    )
    invalid_payload["output_text"] = (
        "MODEL FINDINGS\n"
        "A result without required sections."
    )

    LanguageGenerationResponse.model_validate(
        invalid_payload
    )
except ValidationError:
    missing_section_rejected = True

try:
    invalid_payload = (
        fallback_qa_example.model_dump()
    )
    invalid_payload["question"] = None

    LanguageGenerationResponse.model_validate(
        invalid_payload
    )
except ValidationError:
    missing_question_rejected = True

try:
    invalid_payload = (
        accepted_report_example.model_dump()
    )
    invalid_payload["model_versions"] = {
        "computer_vision": (
            COMPUTER_VISION_MODEL_VERSION
        ),
        "language": None,
        "explainability_method": None,
    }

    LanguageGenerationResponse.model_validate(
        invalid_payload
    )
except ValidationError:
    missing_language_version_rejected = True


# -------------------------------------------------------------------------
# Validate and report the language response contract
# -------------------------------------------------------------------------
language_schema_checksum = hashlib.sha256(
    LANGUAGE_SCHEMA_PATH.read_bytes()
).hexdigest()

accepted_report_json = (
    accepted_report_example.model_dump(
        mode="json"
    )
)
fallback_qa_json = (
    fallback_qa_example.model_dump(
        mode="json"
    )
)

language_schema_checks = {
    "Language schema module was written": (
        LANGUAGE_SCHEMA_PATH.is_file()
    ),
    "Accepted action is preserved": (
        accepted_report_json[
            "guardrail_action"
        ]
        == "accepted_model_generation"
    ),
    "Fallback action is preserved": (
        fallback_qa_json[
            "guardrail_action"
        ]
        == "safe_template_fallback"
    ),
    "Fallback trigger is exposed": (
        fallback_qa_json[
            "trigger_reasons"
        ]
        == ["numeric_grounding_issue"]
    ),
    "Accepted response with trigger is rejected": (
        accepted_with_trigger_rejected
    ),
    "Fallback without trigger is rejected": (
        fallback_without_trigger_rejected
    ),
    "Missing task section is rejected": (
        missing_section_rejected
    ),
    "Question-answer response requires a question": (
        missing_question_rejected
    ),
    "Language-model version is required": (
        missing_language_version_rejected
    ),
    "Language schema checksum is available": (
        len(language_schema_checksum) == 64
    ),
}

print("GROUNDED LANGUAGE RESPONSE SCHEMAS")
print("-" * 100)
print(
    f"Schema module             : "
    f"{LANGUAGE_SCHEMA_PATH}"
)
print(
    f"Schema SHA-256            : "
    f"{language_schema_checksum[:16]}..."
)
print(
    f"Accepted action           : "
    f"{accepted_report_json['guardrail_action']}"
)
print(
    f"Fallback action           : "
    f"{fallback_qa_json['guardrail_action']}"
)
print(
    f"Fallback trigger          : "
    f"{fallback_qa_json['trigger_reasons'][0]}"
)
print(
    f"Question task             : "
    f"{fallback_qa_json['task_type']}"
)
print("-" * 100)

for check_name, passed in language_schema_checks.items():
    print(
        f"{check_name:<58}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(language_schema_checks.values()):
    failed_checks = [
        name
        for name, passed
        in language_schema_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Language response schema validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: GUARDED LANGUAGE RESPONSE CONTRACT READY")

GROUNDED LANGUAGE RESPONSE SCHEMAS
----------------------------------------------------------------------------------------------------
Schema module             : /home/jovyan/chest-xray-ai-assistant/api/schemas/language.py
Schema SHA-256            : e6aa80b250c42b32...
Accepted action           : accepted_model_generation
Fallback action           : safe_template_fallback
Fallback trigger          : numeric_grounding_issue
Question task             : grounded_question_answering
----------------------------------------------------------------------------------------------------
Language schema module was written                        : PASS
Accepted action is preserved                              : PASS
Fallback action is preserved                              : PASS
Fallback trigger is exposed                               : PASS
Accepted response with trigger is rejected                : PASS
Fallback without trigger is rejected                      : PASS
Missing task section is

<a id="nb07-3-6-health-model-lineage-and-metrics-schemas"></a>
### 3.6 Health, Model Lineage, and Metrics Schemas

System endpoints provide service health, model information, frozen evaluation metrics, and aggregate operational measurements without exposing model filesystem locations or internal exceptions.

Component health distinguishes ready, not-loaded, degraded, and error states. Metric values are constrained to non-negative numbers, while model information remains a sanitized metadata mapping assembled from the frozen registries.


In [15]:
# -------------------------------------------------------------------------
# Define health, model-information, and metrics schemas
# -------------------------------------------------------------------------
SYSTEM_SCHEMA_PATH = (
    API_SCHEMAS_DIR / "system.py"
)

SYSTEM_SCHEMA_SOURCE = '''"""Health, model lineage, and metrics response schemas."""

from datetime import datetime
from typing import Annotated, Any, Literal

from pydantic import Field

from api.schemas.common import StrictSchema, SuccessResponseBase


NonNegativeFloat = Annotated[
    float,
    Field(ge=0.0),
]
NonNegativeInt = Annotated[
    int,
    Field(ge=0),
]


class ComponentHealth(StrictSchema):
    """Runtime readiness of one service component."""

    status: Literal[
        "ready",
        "not_loaded",
        "degraded",
        "error",
    ]
    loaded: bool
    device: str | None = Field(
        default=None,
        max_length=100,
    )
    detail: str = Field(
        min_length=1,
        max_length=500,
    )


class HealthResponse(SuccessResponseBase):
    """API and component health response."""

    service_name: str = Field(
        min_length=1,
        max_length=200,
    )
    uptime_seconds: NonNegativeFloat
    components: dict[str, ComponentHealth]


class ModelInfoResponse(SuccessResponseBase):
    """Sanitized frozen model and prompt lineage."""

    computer_vision: dict[str, Any]
    language: dict[str, Any]
    explainability: dict[str, Any]
    limitations: list[str] = Field(
        min_length=1,
    )


class ModelMetricsResponse(SuccessResponseBase):
    """Frozen development and held-out evaluation metrics."""

    computer_vision_metrics: dict[
        str,
        NonNegativeFloat,
    ]
    language_metrics: dict[
        str,
        NonNegativeFloat,
    ]
    guardrail_metrics: dict[
        str,
        NonNegativeFloat,
    ]


class OperationalMetricsResponse(SuccessResponseBase):
    """Aggregate in-process request and guardrail measurements."""

    service_started_at_utc: datetime
    total_requests: NonNegativeInt
    successful_requests: NonNegativeInt
    failed_requests: NonNegativeInt
    endpoint_request_counts: dict[
        str,
        NonNegativeInt,
    ]
    endpoint_average_latency_ms: dict[
        str,
        NonNegativeFloat,
    ]
    language_generation_requests: NonNegativeInt
    guardrail_action_counts: dict[
        str,
        NonNegativeInt,
    ]
'''

SYSTEM_SCHEMA_PATH.write_text(
    SYSTEM_SCHEMA_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()
sys.modules.pop(
    "api.schemas.system",
    None,
)

system_schema_module = (
    importlib.import_module(
        "api.schemas.system"
    )
)

ComponentHealth = (
    system_schema_module.ComponentHealth
)
HealthResponse = (
    system_schema_module.HealthResponse
)
ModelInfoResponse = (
    system_schema_module.ModelInfoResponse
)
ModelMetricsResponse = (
    system_schema_module.ModelMetricsResponse
)
OperationalMetricsResponse = (
    system_schema_module.OperationalMetricsResponse
)


# -------------------------------------------------------------------------
# Create valid system-response examples
# -------------------------------------------------------------------------
system_model_versions = ModelVersions(
    computer_vision=(
        COMPUTER_VISION_MODEL_VERSION
    ),
    language=LANGUAGE_MODEL_VERSION,
    explainability_method="LayerGradCam",
)

health_response_example = HealthResponse(
    request_id=uuid4(),
    timestamp_utc=datetime.now(
        timezone.utc
    ),
    model_versions=system_model_versions,
    prompt_registry_version=(
        PROMPT_REGISTRY_VERSION
    ),
    latency_ms=0.8,
    service_name=API_TITLE,
    uptime_seconds=15.2,
    components={
        "computer_vision": ComponentHealth(
            status="not_loaded",
            loaded=False,
            device=None,
            detail=(
                "The model artifact is available "
                "but has not been loaded."
            ),
        ),
        "grounded_language": ComponentHealth(
            status="not_loaded",
            loaded=False,
            device=None,
            detail=(
                "The model artifact is available "
                "but has not been loaded."
            ),
        ),
    },
)

model_info_response_example = (
    ModelInfoResponse(
        request_id=uuid4(),
        timestamp_utc=datetime.now(
            timezone.utc
        ),
        model_versions=system_model_versions,
        prompt_registry_version=(
            PROMPT_REGISTRY_VERSION
        ),
        latency_ms=1.1,
        computer_vision={
            "architecture": "ResNet-18",
            "labels": 14,
            "input_resolution": "224x224",
        },
        language={
            "base_model": (
                "google/flan-t5-small"
            ),
            "fine_tuning": (
                "full sequence-to-sequence"
            ),
            "tasks": 4,
        },
        explainability={
            "method": "LayerGradCam",
            "target_layer": (
                "layer4[-1].conv2"
            ),
        },
        limitations=[
            (
                "The service is an educational "
                "decision-support prototype."
            ),
            (
                "Grad-CAM does not confirm or "
                "segment a lesion."
            ),
        ],
    )
)

model_metrics_response_example = (
    ModelMetricsResponse(
        request_id=uuid4(),
        timestamp_utc=datetime.now(
            timezone.utc
        ),
        model_versions=system_model_versions,
        prompt_registry_version=(
            PROMPT_REGISTRY_VERSION
        ),
        latency_ms=1.0,
        computer_vision_metrics={
            "macro_roc_auc": 0.8175,
            "micro_f1": 0.3391,
        },
        language_metrics={
            "rouge_l_f1": 0.8489,
            "bleu_4": 0.7393,
        },
        guardrail_metrics={
            "raw_acceptance_rate": 0.7267,
            "guarded_contract_compliance": 1.0,
        },
    )
)

operational_metrics_example = (
    OperationalMetricsResponse(
        request_id=uuid4(),
        timestamp_utc=datetime.now(
            timezone.utc
        ),
        model_versions=system_model_versions,
        prompt_registry_version=(
            PROMPT_REGISTRY_VERSION
        ),
        latency_ms=0.5,
        service_started_at_utc=datetime.now(
            timezone.utc
        ),
        total_requests=10,
        successful_requests=9,
        failed_requests=1,
        endpoint_request_counts={
            "/health": 4,
            "/api/v1/model/info": 6,
        },
        endpoint_average_latency_ms={
            "/health": 0.7,
            "/api/v1/model/info": 1.2,
        },
        language_generation_requests=0,
        guardrail_action_counts={
            "accepted_model_generation": 0,
            "safe_template_fallback": 0,
        },
    )
)


# -------------------------------------------------------------------------
# Verify malformed system measurements are rejected
# -------------------------------------------------------------------------
negative_uptime_rejected = False
negative_metric_rejected = False
invalid_component_status_rejected = False
negative_request_count_rejected = False

try:
    invalid_payload = (
        health_response_example.model_dump()
    )
    invalid_payload["uptime_seconds"] = -1.0

    HealthResponse.model_validate(
        invalid_payload
    )
except ValidationError:
    negative_uptime_rejected = True

try:
    invalid_payload = (
        model_metrics_response_example.model_dump()
    )
    invalid_payload[
        "language_metrics"
    ]["rouge_l_f1"] = -0.1

    ModelMetricsResponse.model_validate(
        invalid_payload
    )
except ValidationError:
    negative_metric_rejected = True

try:
    ComponentHealth(
        status="unknown",
        loaded=False,
        detail="Invalid status test.",
    )
except ValidationError:
    invalid_component_status_rejected = True

try:
    invalid_payload = (
        operational_metrics_example.model_dump()
    )
    invalid_payload["failed_requests"] = -1

    OperationalMetricsResponse.model_validate(
        invalid_payload
    )
except ValidationError:
    negative_request_count_rejected = True


# -------------------------------------------------------------------------
# Validate and report system schema readiness
# -------------------------------------------------------------------------
system_schema_checksum = hashlib.sha256(
    SYSTEM_SCHEMA_PATH.read_bytes()
).hexdigest()

system_schema_checks = {
    "System schema module was written": (
        SYSTEM_SCHEMA_PATH.is_file()
    ),
    "Health contains component states": (
        len(
            health_response_example.components
        )
        == 2
    ),
    "Model information contains limitations": (
        len(
            model_info_response_example.limitations
        )
        >= 1
    ),
    "Frozen metrics are preserved": (
        model_metrics_response_example
        .language_metrics[
            "rouge_l_f1"
        ]
        == 0.8489
    ),
    "Operational counts are preserved": (
        operational_metrics_example
        .total_requests
        == 10
    ),
    "Negative uptime is rejected": (
        negative_uptime_rejected
    ),
    "Negative model metric is rejected": (
        negative_metric_rejected
    ),
    "Unknown component status is rejected": (
        invalid_component_status_rejected
    ),
    "Negative request count is rejected": (
        negative_request_count_rejected
    ),
    "System schema checksum is available": (
        len(system_schema_checksum) == 64
    ),
}

print("HEALTH, MODEL, AND METRICS RESPONSE SCHEMAS")
print("-" * 100)
print(
    f"Schema module             : "
    f"{SYSTEM_SCHEMA_PATH}"
)
print(
    f"Schema SHA-256            : "
    f"{system_schema_checksum[:16]}..."
)
print(
    f"Health components         : "
    f"{len(health_response_example.components)}"
)
print(
    f"Language ROUGE-L          : "
    f"{model_metrics_response_example.language_metrics['rouge_l_f1']:.4f}"
)
print(
    f"Operational requests      : "
    f"{operational_metrics_example.total_requests}"
)
print("-" * 100)

for check_name, passed in system_schema_checks.items():
    print(
        f"{check_name:<56}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(system_schema_checks.values()):
    failed_checks = [
        name
        for name, passed
        in system_schema_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "System response schema validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: SYSTEM RESPONSE CONTRACTS READY")

HEALTH, MODEL, AND METRICS RESPONSE SCHEMAS
----------------------------------------------------------------------------------------------------
Schema module             : /home/jovyan/chest-xray-ai-assistant/api/schemas/system.py
Schema SHA-256            : 3dd312c4c783f79f...
Health components         : 2
Language ROUGE-L          : 0.8489
Operational requests      : 10
----------------------------------------------------------------------------------------------------
System schema module was written                        : PASS
Health contains component states                        : PASS
Model information contains limitations                  : PASS
Frozen metrics are preserved                            : PASS
Operational counts are preserved                        : PASS
Negative uptime is rejected                             : PASS
Negative model metric is rejected                       : PASS
Unknown component status is rejected                    : PASS
Negative request co

<a id="nb07-3-7-combined-workflow-and-stored-prediction-schemas"></a>
### 3.7 Combined Workflow and Stored Prediction Schemas

The combined workflow returns classification, Grad-CAM evidence, and three mandatory language outputs: structured report, plain-language explanation, and educational follow-up. Grounded question answering is included as a fourth output only when a valid question is supplied.

Stored predictions retain their creation timestamp and may represent classification-only, visual-analysis, or complete-workflow results. Optional visual evidence must remain consistent with the explainability contract and the ordered threshold-crossed findings.


In [16]:
# -------------------------------------------------------------------------
# Define aggregate workflow and stored-prediction schemas
# -------------------------------------------------------------------------
AGGREGATE_SCHEMA_PATH = (
    API_SCHEMAS_DIR / "aggregate.py"
)

AGGREGATE_SCHEMA_SOURCE = '''"""Combined workflow and stored prediction schemas."""

from datetime import datetime
from typing import Literal

from pydantic import Field, model_validator

from api.schemas.common import StrictSchema
from api.schemas.explainability import (
    ExplainabilityContract,
    GradCAMEvidence,
    ImageAnalysisResponse,
)
from api.schemas.language import (
    GuardrailAction,
    GuardrailTrigger,
    LanguageTask,
)
from api.schemas.prediction import ClassificationResponse


class EmbeddedLanguageOutput(StrictSchema):
    """One language result embedded in a larger API response."""

    task_type: LanguageTask
    question: str | None = Field(
        default=None,
        min_length=3,
        max_length=500,
    )
    output_text: str = Field(
        min_length=1,
        max_length=10000,
    )
    guardrail_action: GuardrailAction
    trigger_reasons: list[GuardrailTrigger] = Field(
        default_factory=list
    )
    generated_tokens: int = Field(ge=1)
    generation_latency_ms: float = Field(ge=0.0)

    @model_validator(mode="after")
    def validate_embedded_output(
        self,
    ) -> "EmbeddedLanguageOutput":
        if (
            self.guardrail_action
            == "accepted_model_generation"
            and self.trigger_reasons
        ):
            raise ValueError(
                "Accepted embedded outputs cannot contain guardrail triggers."
            )

        if (
            self.guardrail_action
            == "safe_template_fallback"
            and not self.trigger_reasons
        ):
            raise ValueError(
                "Fallback embedded outputs require trigger reasons."
            )

        required_sections = {
            "structured_report": [
                "PRELIMINARY MODEL REPORT",
                "MODEL FINDINGS",
                "LIMITATIONS",
            ],
            "plain_language_explanation": [
                "EXPLANATION",
                "LIMITATIONS",
            ],
            "grounded_question_answering": [
                "ANSWER",
                "LIMITATIONS",
            ],
            "educational_follow_up": [
                "EDUCATIONAL FOLLOW-UP",
                "LIMITATIONS",
            ],
        }[self.task_type]

        section_positions = [
            self.output_text.find(section)
            for section in required_sections
        ]

        if (
            any(
                position < 0
                for position in section_positions
            )
            or section_positions
            != sorted(section_positions)
        ):
            raise ValueError(
                "Embedded output sections are incomplete or out of order."
            )

        if (
            self.task_type
            == "grounded_question_answering"
            and self.question is None
        ):
            raise ValueError(
                "Embedded question answering requires a question."
            )

        return self


class CompleteAnalysisResponse(ImageAnalysisResponse):
    """Complete image, visual, and guarded language workflow."""

    language_outputs: list[
        EmbeddedLanguageOutput
    ] = Field(
        min_length=3,
        max_length=4,
    )

    @model_validator(mode="after")
    def validate_complete_language_tasks(
        self,
    ) -> "CompleteAnalysisResponse":
        task_names = [
            output.task_type
            for output in self.language_outputs
        ]

        if len(task_names) != len(set(task_names)):
            raise ValueError(
                "Combined language task names must be unique."
            )

        mandatory_tasks = {
            "structured_report",
            "plain_language_explanation",
            "educational_follow_up",
        }

        if not mandatory_tasks.issubset(
            set(task_names)
        ):
            raise ValueError(
                "The combined response is missing a mandatory language task."
            )

        allowed_tasks = mandatory_tasks | {
            "grounded_question_answering"
        }

        if not set(task_names).issubset(
            allowed_tasks
        ):
            raise ValueError(
                "The combined response contains an unsupported language task."
            )

        return self


class StoredPredictionResponse(ClassificationResponse):
    """Retrievable classification with optional later-stage outputs."""

    created_at_utc: datetime
    explainability: ExplainabilityContract | None = None
    visual_evidence: list[GradCAMEvidence] = Field(
        default_factory=list
    )
    language_outputs: list[
        EmbeddedLanguageOutput
    ] = Field(default_factory=list)

    @model_validator(mode="after")
    def validate_stored_extensions(
        self,
    ) -> "StoredPredictionResponse":
        if (
            self.explainability is None
            and self.visual_evidence
        ):
            raise ValueError(
                "Stored visual evidence requires an explainability contract."
            )

        if self.visual_evidence:
            evidence_names = [
                evidence.finding_name
                for evidence in self.visual_evidence
            ]

            if evidence_names != self.crossed_finding_names:
                raise ValueError(
                    "Stored visual evidence must match crossed findings."
                )

        language_task_names = [
            output.task_type
            for output in self.language_outputs
        ]

        if len(language_task_names) != len(
            set(language_task_names)
        ):
            raise ValueError(
                "Stored language task names must be unique."
            )

        return self
'''

AGGREGATE_SCHEMA_PATH.write_text(
    AGGREGATE_SCHEMA_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()
sys.modules.pop(
    "api.schemas.aggregate",
    None,
)

aggregate_schema_module = (
    importlib.import_module(
        "api.schemas.aggregate"
    )
)

EmbeddedLanguageOutput = (
    aggregate_schema_module.EmbeddedLanguageOutput
)
CompleteAnalysisResponse = (
    aggregate_schema_module.CompleteAnalysisResponse
)
StoredPredictionResponse = (
    aggregate_schema_module.StoredPredictionResponse
)


# -------------------------------------------------------------------------
# Build valid embedded language outputs
# -------------------------------------------------------------------------
embedded_report = EmbeddedLanguageOutput(
    task_type="structured_report",
    output_text=(
        "PRELIMINARY MODEL REPORT\n"
        "MODEL FINDINGS\n"
        "A controlled model finding summary.\n"
        "LIMITATIONS\n"
        "This output is not a diagnosis."
    ),
    guardrail_action=(
        "accepted_model_generation"
    ),
    trigger_reasons=[],
    generated_tokens=35,
    generation_latency_ms=72.0,
)

embedded_explanation = (
    EmbeddedLanguageOutput(
        task_type=(
            "plain_language_explanation"
        ),
        output_text=(
            "EXPLANATION\n"
            "A controlled simple explanation.\n"
            "LIMITATIONS\n"
            "This output is not a diagnosis."
        ),
        guardrail_action=(
            "accepted_model_generation"
        ),
        trigger_reasons=[],
        generated_tokens=28,
        generation_latency_ms=70.0,
    )
)

embedded_follow_up = (
    EmbeddedLanguageOutput(
        task_type="educational_follow_up",
        output_text=(
            "EDUCATIONAL FOLLOW-UP\n"
            "A qualified professional can review "
            "the complete clinical context.\n"
            "LIMITATIONS\n"
            "This output is not a diagnosis."
        ),
        guardrail_action=(
            "safe_template_fallback"
        ),
        trigger_reasons=[
            "missing_required_finding_issue"
        ],
        generated_tokens=39,
        generation_latency_ms=74.0,
    )
)

embedded_question_answer = (
    EmbeddedLanguageOutput(
        task_type=(
            "grounded_question_answering"
        ),
        question=(
            "Did the finding cross its threshold?"
        ),
        output_text=(
            "ANSWER\n"
            "The supplied finding crossed its "
            "frozen threshold.\n"
            "LIMITATIONS\n"
            "This output is not a diagnosis."
        ),
        guardrail_action=(
            "accepted_model_generation"
        ),
        trigger_reasons=[],
        generated_tokens=31,
        generation_latency_ms=71.0,
    )
)


# -------------------------------------------------------------------------
# Build complete and stored response examples
# -------------------------------------------------------------------------
analysis_payload = (
    analysis_response_example.model_dump()
)

complete_response_example = (
    CompleteAnalysisResponse(
        **analysis_payload,
        language_outputs=[
            embedded_report,
            embedded_explanation,
            embedded_follow_up,
            embedded_question_answer,
        ],
    )
)

stored_prediction_example = (
    StoredPredictionResponse(
        **classification_payload,
        created_at_utc=datetime.now(
            timezone.utc
        ),
        explainability=None,
        visual_evidence=[],
        language_outputs=[],
    )
)


# -------------------------------------------------------------------------
# Verify malformed aggregate responses are rejected
# -------------------------------------------------------------------------
missing_mandatory_task_rejected = False
duplicate_language_task_rejected = False
orphan_visual_evidence_rejected = False
duplicate_stored_task_rejected = False

try:
    invalid_payload = (
        complete_response_example.model_dump()
    )
    invalid_payload["language_outputs"] = [
        embedded_report.model_dump(),
        embedded_explanation.model_dump(),
        embedded_question_answer.model_dump(),
    ]

    CompleteAnalysisResponse.model_validate(
        invalid_payload
    )
except ValidationError:
    missing_mandatory_task_rejected = True

try:
    invalid_payload = (
        complete_response_example.model_dump()
    )
    invalid_payload["language_outputs"] = [
        embedded_report.model_dump(),
        embedded_report.model_dump(),
        embedded_explanation.model_dump(),
        embedded_follow_up.model_dump(),
    ]

    CompleteAnalysisResponse.model_validate(
        invalid_payload
    )
except ValidationError:
    duplicate_language_task_rejected = True

try:
    invalid_payload = (
        stored_prediction_example.model_dump()
    )
    invalid_payload["visual_evidence"] = [
        gradcam_evidence_example.model_dump()
    ]

    StoredPredictionResponse.model_validate(
        invalid_payload
    )
except ValidationError:
    orphan_visual_evidence_rejected = True

try:
    invalid_payload = (
        stored_prediction_example.model_dump()
    )
    invalid_payload["language_outputs"] = [
        embedded_report.model_dump(),
        embedded_report.model_dump(),
    ]

    StoredPredictionResponse.model_validate(
        invalid_payload
    )
except ValidationError:
    duplicate_stored_task_rejected = True


# -------------------------------------------------------------------------
# Validate and report aggregate schema readiness
# -------------------------------------------------------------------------
aggregate_schema_checksum = hashlib.sha256(
    AGGREGATE_SCHEMA_PATH.read_bytes()
).hexdigest()

complete_response_json = (
    complete_response_example.model_dump(
        mode="json"
    )
)

aggregate_schema_checks = {
    "Aggregate schema module was written": (
        AGGREGATE_SCHEMA_PATH.is_file()
    ),
    "Three mandatory language tasks are present": (
        {
            "structured_report",
            "plain_language_explanation",
            "educational_follow_up",
        }.issubset(
            {
                output["task_type"]
                for output in complete_response_json[
                    "language_outputs"
                ]
            }
        )
    ),
    "Optional question-answer task is present": (
        any(
            output["task_type"]
            == "grounded_question_answering"
            for output in complete_response_json[
                "language_outputs"
            ]
        )
    ),
    "Classification-only stored result is valid": (
        len(
            stored_prediction_example
            .visual_evidence
        )
        == 0
        and len(
            stored_prediction_example
            .language_outputs
        )
        == 0
    ),
    "Missing mandatory task is rejected": (
        missing_mandatory_task_rejected
    ),
    "Duplicate combined task is rejected": (
        duplicate_language_task_rejected
    ),
    "Visual evidence without contract is rejected": (
        orphan_visual_evidence_rejected
    ),
    "Duplicate stored task is rejected": (
        duplicate_stored_task_rejected
    ),
    "Aggregate schema checksum is available": (
        len(aggregate_schema_checksum) == 64
    ),
}

print("COMBINED WORKFLOW AND STORED PREDICTION SCHEMAS")
print("-" * 100)
print(
    f"Schema module             : "
    f"{AGGREGATE_SCHEMA_PATH}"
)
print(
    f"Schema SHA-256            : "
    f"{aggregate_schema_checksum[:16]}..."
)
print(
    f"Combined language outputs : "
    f"{len(complete_response_json['language_outputs'])}"
)
print(
    f"Stored visual evidence    : "
    f"{len(stored_prediction_example.visual_evidence)}"
)
print(
    f"Stored language outputs   : "
    f"{len(stored_prediction_example.language_outputs)}"
)
print("-" * 100)

for check_name, passed in aggregate_schema_checks.items():
    print(
        f"{check_name:<58}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(aggregate_schema_checks.values()):
    failed_checks = [
        name
        for name, passed
        in aggregate_schema_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Aggregate response schema validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: COMPLETE WORKFLOW RESPONSE CONTRACT READY")

COMBINED WORKFLOW AND STORED PREDICTION SCHEMAS
----------------------------------------------------------------------------------------------------
Schema module             : /home/jovyan/chest-xray-ai-assistant/api/schemas/aggregate.py
Schema SHA-256            : 28315f43fb5cf320...
Combined language outputs : 4
Stored visual evidence    : 0
Stored language outputs   : 0
----------------------------------------------------------------------------------------------------
Aggregate schema module was written                       : PASS
Three mandatory language tasks are present                : PASS
Optional question-answer task is present                  : PASS
Classification-only stored result is valid                : PASS
Missing mandatory task is rejected                        : PASS
Duplicate combined task is rejected                       : PASS
Visual evidence without contract is rejected              : PASS
Duplicate stored task is rejected                         : PASS
Ag

<a id="nb07-3-8-canonical-schema-package-registry"></a>
### 3.8 Canonical Schema Package Registry

All request and response models are exported through a single `api.schemas` package interface. The API endpoint registry is then cross-checked against the available Pydantic class names so that every declared request and response contract can be resolved before route development begins.

Machine-readable JSON schemas are also exported for traceability and later comparison with the FastAPI-generated OpenAPI specification.


In [17]:
# -------------------------------------------------------------------------
# Create the canonical schema package interface
# -------------------------------------------------------------------------
SCHEMA_PACKAGE_INIT_PATH = (
    API_SCHEMAS_DIR / "__init__.py"
)

SCHEMA_PACKAGE_INIT_SOURCE = '''"""Canonical Pydantic schema exports for the API."""

from api.schemas.aggregate import (
    CompleteAnalysisResponse,
    EmbeddedLanguageOutput,
    StoredPredictionResponse,
)
from api.schemas.common import (
    APIErrorResponse,
    ModelVersions,
    StrictSchema,
    SuccessResponseBase,
)
from api.schemas.explainability import (
    ExplainabilityContract,
    GradCAMEvidence,
    ImageAnalysisResponse,
)
from api.schemas.language import (
    LanguageGenerationResponse,
)
from api.schemas.prediction import (
    ClassificationResponse,
    FindingEvidence,
    ImageMetadata,
)
from api.schemas.requests import (
    CompleteAnalysisOptions,
    GroundedGenerationRequest,
    GroundedQuestionRequest,
)
from api.schemas.system import (
    ComponentHealth,
    HealthResponse,
    ModelInfoResponse,
    ModelMetricsResponse,
    OperationalMetricsResponse,
)

__all__ = [
    "APIErrorResponse",
    "ClassificationResponse",
    "CompleteAnalysisOptions",
    "CompleteAnalysisResponse",
    "ComponentHealth",
    "EmbeddedLanguageOutput",
    "ExplainabilityContract",
    "FindingEvidence",
    "GradCAMEvidence",
    "GroundedGenerationRequest",
    "GroundedQuestionRequest",
    "HealthResponse",
    "ImageAnalysisResponse",
    "ImageMetadata",
    "LanguageGenerationResponse",
    "ModelInfoResponse",
    "ModelMetricsResponse",
    "ModelVersions",
    "OperationalMetricsResponse",
    "StoredPredictionResponse",
    "StrictSchema",
    "SuccessResponseBase",
]
'''

SCHEMA_PACKAGE_INIT_PATH.write_text(
    SCHEMA_PACKAGE_INIT_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()
sys.modules.pop(
    "api.schemas",
    None,
)

schema_package = importlib.import_module(
    "api.schemas"
)


# -------------------------------------------------------------------------
# Resolve every registered request and response schema
# -------------------------------------------------------------------------
SCHEMA_CLASS_REGISTRY = {
    schema_name: getattr(
        schema_package,
        schema_name,
    )
    for schema_name in schema_package.__all__
}

non_pydantic_request_contracts = {
    "multipart/form-data image",
    (
        "multipart/form-data image "
        "with optional question"
    ),
    "prediction_id path parameter",
}

declared_pydantic_request_names = {
    endpoint["request_contract"]
    for endpoint in API_ENDPOINT_REGISTRY
    if (
        endpoint["request_contract"]
        is not None
        and endpoint["request_contract"]
        not in non_pydantic_request_contracts
    )
}

declared_response_names = {
    endpoint["response_contract"]
    for endpoint in API_ENDPOINT_REGISTRY
}


# -------------------------------------------------------------------------
# Export machine-readable Pydantic JSON schemas
# -------------------------------------------------------------------------
PYDANTIC_SCHEMA_REGISTRY_PATH = (
    API_OUTPUT_DIR
    / "pydantic_schema_registry.json"
)

pydantic_schema_registry = {
    "registry_version": (
        "pydantic-schema-registry-v1"
    ),
    "api_contract_version": (
        API_CONTRACT_VERSION
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "schemas": {
        schema_name: schema_class.model_json_schema()
        for schema_name, schema_class
        in SCHEMA_CLASS_REGISTRY.items()
    },
}

with PYDANTIC_SCHEMA_REGISTRY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    import json

    json.dump(
        pydantic_schema_registry,
        file,
        indent=2,
        ensure_ascii=False,
    )


# -------------------------------------------------------------------------
# Validate endpoint-to-schema resolution
# -------------------------------------------------------------------------
schema_registry_checksum = hashlib.sha256(
    PYDANTIC_SCHEMA_REGISTRY_PATH.read_bytes()
).hexdigest()

schema_registry_checks = {
    "Schema package interface was written": (
        SCHEMA_PACKAGE_INIT_PATH.is_file()
    ),
    "All exported schema names are unique": (
        len(schema_package.__all__)
        == len(set(schema_package.__all__))
    ),
    "All request schemas resolve": (
        declared_pydantic_request_names.issubset(
            set(SCHEMA_CLASS_REGISTRY)
        )
    ),
    "All response schemas resolve": (
        declared_response_names.issubset(
            set(SCHEMA_CLASS_REGISTRY)
        )
    ),
    "Error response schema is registered": (
        "APIErrorResponse"
        in SCHEMA_CLASS_REGISTRY
    ),
    "Classification response is registered": (
        "ClassificationResponse"
        in SCHEMA_CLASS_REGISTRY
    ),
    "Complete workflow response is registered": (
        "CompleteAnalysisResponse"
        in SCHEMA_CLASS_REGISTRY
    ),
    "JSON schema registry was exported": (
        PYDANTIC_SCHEMA_REGISTRY_PATH.is_file()
    ),
    "Every exported model has a JSON schema": (
        len(
            pydantic_schema_registry[
                "schemas"
            ]
        )
        == len(SCHEMA_CLASS_REGISTRY)
    ),
    "Schema registry checksum is available": (
        len(schema_registry_checksum) == 64
    ),
}


# -------------------------------------------------------------------------
# Report canonical schema registry readiness
# -------------------------------------------------------------------------
print("CANONICAL PYDANTIC SCHEMA REGISTRY")
print("-" * 100)
print(
    f"Schema package            : "
    f"{SCHEMA_PACKAGE_INIT_PATH}"
)
print(
    f"Exported schema classes   : "
    f"{len(SCHEMA_CLASS_REGISTRY)}"
)
print(
    f"Endpoint request schemas  : "
    f"{len(declared_pydantic_request_names)}"
)
print(
    f"Endpoint response schemas : "
    f"{len(declared_response_names)}"
)
print(
    f"JSON schema registry      : "
    f"{PYDANTIC_SCHEMA_REGISTRY_PATH}"
)
print(
    f"Registry SHA-256          : "
    f"{schema_registry_checksum[:16]}..."
)
print("-" * 100)
print("ENDPOINT RESPONSE CONTRACTS")

for response_name in sorted(
    declared_response_names
):
    print(f"- {response_name}")

print("-" * 100)

for check_name, passed in schema_registry_checks.items():
    print(
        f"{check_name:<58}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(schema_registry_checks.values()):
    failed_checks = [
        name
        for name, passed
        in schema_registry_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Canonical schema registry validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: API SCHEMA LAYER READY FOR SERVICE DEVELOPMENT")

CANONICAL PYDANTIC SCHEMA REGISTRY
----------------------------------------------------------------------------------------------------
Schema package            : /home/jovyan/chest-xray-ai-assistant/api/schemas/__init__.py
Exported schema classes   : 22
Endpoint request schemas  : 2
Endpoint response schemas : 9
JSON schema registry      : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/pydantic_schema_registry.json
Registry SHA-256          : 86ecef6094e6e770...
----------------------------------------------------------------------------------------------------
ENDPOINT RESPONSE CONTRACTS
- ClassificationResponse
- CompleteAnalysisResponse
- HealthResponse
- ImageAnalysisResponse
- LanguageGenerationResponse
- ModelInfoResponse
- ModelMetricsResponse
- OperationalMetricsResponse
- StoredPredictionResponse
----------------------------------------------------------------------------------------------------
Schema package interface was written                  

**[↑ Back to notebook index](#notebook-index)**


<a id="nb07-4-api-core-configuration-and-runtime-utilities"></a>
## 4. API Core Configuration and Runtime Utilities

<a id="nb07-4-1-environment-aware-service-settings-and-frozen-lineage"></a>
### 4.1 Environment-Aware Service Settings and Frozen Lineage

The API uses one cached, immutable settings object. Deployment paths may later be overridden through environment variables, while the validated Kubeflow locations remain the current defaults.

The settings layer verifies every required frozen artifact during application startup and exposes only sanitized model lineage to API responses. Internal filesystem paths remain available to service modules but are never included in client-facing metadata.


In [18]:
# -------------------------------------------------------------------------
# Define the immutable environment-aware settings module
# -------------------------------------------------------------------------
CORE_CONFIG_PATH = (
    API_CORE_DIR / "config.py"
)

CORE_CONFIG_SOURCE = '''"""Environment-aware immutable API service settings."""

import json
import os
from functools import lru_cache
from pathlib import Path
from typing import Any, Literal

import yaml
from pydantic import BaseModel, ConfigDict, Field, model_validator


DEFAULT_SOLUTION_ROOT = Path(
    "/home/jovyan/chest-xray-ai-assistant"
)
DEFAULT_DATA_ROOT = Path(
    "/home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data"
)


class ServiceSettings(BaseModel):
    """Validated paths, versions, and upload limits for the API."""

    model_config = ConfigDict(
        extra="forbid",
        frozen=True,
        arbitrary_types_allowed=True,
    )

    api_title: str = (
        "API-Driven Chest X-Ray Analysis and Explanation Assistant"
    )
    api_version: Literal["v1"] = "v1"
    api_prefix: Literal["/api/v1"] = "/api/v1"

    computer_vision_model_version: str = (
        "resnet18-chestmnist-v1"
    )
    language_model_version: str = (
        "flan-t5-small-chestmnist-v1"
    )
    prompt_registry_version: str = (
        "grounded-language-prompts-v1"
    )

    solution_root: Path
    data_root: Path
    api_output_dir: Path

    computer_vision_metadata_path: Path
    computer_vision_weights_path: Path
    language_model_dir: Path
    language_model_metadata_path: Path
    prompt_registry_path: Path
    explainability_metadata_path: Path
    language_guardrail_summary_path: Path
    language_evaluation_metrics_path: Path

    mlflow_tracking_uri: str

    maximum_upload_bytes: int = Field(
        gt=0
    )
    supported_image_media_types: tuple[
        Literal["image/png", "image/jpeg"],
        ...,
    ]

    educational_use_only: Literal[True] = True

    @model_validator(mode="after")
    def validate_artifacts(
        self,
    ) -> "ServiceSettings":
        if not self.solution_root.is_dir():
            raise ValueError(
                "The configured solution root is unavailable."
            )

        if not self.data_root.is_dir():
            raise ValueError(
                "The configured data root is unavailable."
            )

        required_files = [
            self.computer_vision_metadata_path,
            self.computer_vision_weights_path,
            self.language_model_metadata_path,
            self.language_model_dir / "config.json",
            self.language_model_dir / "model.safetensors",
            self.prompt_registry_path,
            self.explainability_metadata_path,
            self.language_guardrail_summary_path,
            self.language_evaluation_metrics_path,
        ]

        missing_files = [
            str(file_path)
            for file_path in required_files
            if not file_path.is_file()
        ]

        if missing_files:
            raise ValueError(
                "Required frozen artifacts are missing: "
                + ", ".join(missing_files)
            )

        return self

    def sanitized_lineage(
        self,
    ) -> dict[str, Any]:
        """Return client-safe lineage without internal paths."""

        return {
            "api_version": self.api_version,
            "computer_vision_model_version": (
                self.computer_vision_model_version
            ),
            "language_model_version": (
                self.language_model_version
            ),
            "prompt_registry_version": (
                self.prompt_registry_version
            ),
            "educational_use_only": (
                self.educational_use_only
            ),
        }


def load_yaml_file(
    file_path: Path,
) -> dict[str, Any]:
    """Load one required YAML mapping."""

    with file_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        loaded_value = yaml.safe_load(file)

    if not isinstance(loaded_value, dict):
        raise ValueError(
            f"Expected a YAML mapping: {file_path.name}"
        )

    return loaded_value


def load_json_file(
    file_path: Path,
) -> dict[str, Any]:
    """Load one required JSON object."""

    with file_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        loaded_value = json.load(file)

    if not isinstance(loaded_value, dict):
        raise ValueError(
            f"Expected a JSON object: {file_path.name}"
        )

    return loaded_value


@lru_cache(maxsize=1)
def get_settings() -> ServiceSettings:
    """Create and cache the validated service settings."""

    solution_root = Path(
        os.getenv(
            "CHEST_XRAY_SOLUTION_ROOT",
            str(DEFAULT_SOLUTION_ROOT),
        )
    )
    data_root = Path(
        os.getenv(
            "CHEST_XRAY_DATA_ROOT",
            str(DEFAULT_DATA_ROOT),
        )
    )

    computer_vision_model_version = (
        "resnet18-chestmnist-v1"
    )
    language_model_version = (
        "flan-t5-small-chestmnist-v1"
    )

    return ServiceSettings(
        solution_root=solution_root,
        data_root=data_root,
        api_output_dir=(
            data_root / "outputs" / "api"
        ),
        computer_vision_metadata_path=(
            data_root
            / "models"
            / computer_vision_model_version
            / "model_metadata.yaml"
        ),
        computer_vision_weights_path=(
            data_root
            / "models"
            / computer_vision_model_version
            / "model_state_dict.pt"
        ),
        language_model_dir=(
            data_root
            / "models"
            / language_model_version
        ),
        language_model_metadata_path=(
            data_root
            / "models"
            / language_model_version
            / "model_metadata.yaml"
        ),
        prompt_registry_path=(
            solution_root
            / "configs"
            / "prompt_registry.yaml"
        ),
        explainability_metadata_path=(
            data_root
            / "explainability"
            / "explainability_metadata.yaml"
        ),
        language_guardrail_summary_path=(
            data_root
            / "outputs"
            / "language"
            / "language_guardrail_summary.json"
        ),
        language_evaluation_metrics_path=(
            data_root
            / "outputs"
            / "language"
            / "language_evaluation_metrics.json"
        ),
        mlflow_tracking_uri=(
            "file://"
            + str(data_root / "mlflow")
        ),
        maximum_upload_bytes=(
            10 * 1024 * 1024
        ),
        supported_image_media_types=(
            "image/png",
            "image/jpeg",
        ),
    )
'''

CORE_CONFIG_PATH.write_text(
    CORE_CONFIG_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()
sys.modules.pop(
    "api.core.config",
    None,
)

core_config_module = importlib.import_module(
    "api.core.config"
)

ServiceSettings = (
    core_config_module.ServiceSettings
)
get_settings = (
    core_config_module.get_settings
)
load_yaml_file = (
    core_config_module.load_yaml_file
)
load_json_file = (
    core_config_module.load_json_file
)


# -------------------------------------------------------------------------
# Load and cross-check the frozen registries
# -------------------------------------------------------------------------
service_settings = get_settings()

cv_metadata = load_yaml_file(
    service_settings
    .computer_vision_metadata_path
)
language_metadata = load_yaml_file(
    service_settings
    .language_model_metadata_path
)
prompt_registry = load_yaml_file(
    service_settings
    .prompt_registry_path
)
explainability_metadata = load_yaml_file(
    service_settings
    .explainability_metadata_path
)
guardrail_summary = load_json_file(
    service_settings
    .language_guardrail_summary_path
)
language_evaluation_metrics = (
    load_json_file(
        service_settings
        .language_evaluation_metrics_path
    )
)


def collect_scalar_values(value):
    """Recursively collect scalar registry values."""
    collected_values = []

    if isinstance(value, dict):
        for nested_value in value.values():
            collected_values.extend(
                collect_scalar_values(
                    nested_value
                )
            )
    elif isinstance(value, list):
        for nested_value in value:
            collected_values.extend(
                collect_scalar_values(
                    nested_value
                )
            )
    elif isinstance(
        value,
        (str, int, float, bool),
    ):
        collected_values.append(value)

    return collected_values


cv_metadata_values = collect_scalar_values(
    cv_metadata
)
language_metadata_values = (
    collect_scalar_values(
        language_metadata
    )
)
explainability_metadata_values = (
    collect_scalar_values(
        explainability_metadata
    )
)

sanitized_lineage = (
    service_settings.sanitized_lineage()
)


# -------------------------------------------------------------------------
# Validate immutable settings and version agreement
# -------------------------------------------------------------------------
cached_settings_match = (
    get_settings() is service_settings
)

settings_checksum = hashlib.sha256(
    CORE_CONFIG_PATH.read_bytes()
).hexdigest()

settings_checks = {
    "Core configuration module was written": (
        CORE_CONFIG_PATH.is_file()
    ),
    "Settings object is cached": (
        cached_settings_match
    ),
    "Settings object is immutable": (
        service_settings.model_config[
            "frozen"
        ]
        is True
    ),
    "Computer-vision version matches metadata": (
        COMPUTER_VISION_MODEL_VERSION
        in cv_metadata_values
    ),
    "Language-model version matches metadata": (
        LANGUAGE_MODEL_VERSION
        in language_metadata_values
    ),
    "Prompt registry version matches": (
        prompt_registry.get(
            "registry_version"
        )
        == PROMPT_REGISTRY_VERSION
    ),
    "Explainability method is registered": (
        "LayerGradCam"
        in explainability_metadata_values
    ),
    "Guardrail summary is available": (
        guardrail_summary.get(
            "records_processed"
        )
        == 600
    ),
    "Language evaluation is available": (
        language_evaluation_metrics.get(
            "records_evaluated"
        )
        == 600
    ),
    "Sanitized lineage contains no paths": all(
        "/" not in str(value)
        for key, value
        in sanitized_lineage.items()
        if key != "api_version"
    ),
    "Upload limit equals ten MiB": (
        service_settings
        .maximum_upload_bytes
        == 10 * 1024 * 1024
    ),
    "Configuration checksum is available": (
        len(settings_checksum) == 64
    ),
}


# -------------------------------------------------------------------------
# Report core settings readiness
# -------------------------------------------------------------------------
print("IMMUTABLE API SERVICE SETTINGS")
print("-" * 100)
print(
    f"Configuration module      : "
    f"{CORE_CONFIG_PATH}"
)
print(
    f"Configuration SHA-256     : "
    f"{settings_checksum[:16]}..."
)
print(
    f"API version               : "
    f"{service_settings.api_version}"
)
print(
    f"Computer-vision version   : "
    f"{service_settings.computer_vision_model_version}"
)
print(
    f"Language-model version    : "
    f"{service_settings.language_model_version}"
)
print(
    f"Prompt registry version   : "
    f"{service_settings.prompt_registry_version}"
)
print(
    f"Maximum upload bytes      : "
    f"{service_settings.maximum_upload_bytes:,}"
)
print(
    f"Supported media types     : "
    f"{', '.join(service_settings.supported_image_media_types)}"
)
print("-" * 100)
print("SANITIZED CLIENT LINEAGE")

for key, value in sanitized_lineage.items():
    print(f"{key:<34}: {value}")

print("-" * 100)

for check_name, passed in settings_checks.items():
    print(
        f"{check_name:<58}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(settings_checks.values()):
    failed_checks = [
        name
        for name, passed
        in settings_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "API core settings validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: IMMUTABLE API SETTINGS AND LINEAGE READY")

IMMUTABLE API SERVICE SETTINGS
----------------------------------------------------------------------------------------------------
Configuration module      : /home/jovyan/chest-xray-ai-assistant/api/core/config.py
Configuration SHA-256     : 75537df61959ad4c...
API version               : v1
Computer-vision version   : resnet18-chestmnist-v1
Language-model version    : flan-t5-small-chestmnist-v1
Prompt registry version   : grounded-language-prompts-v1
Maximum upload bytes      : 10,485,760
Supported media types     : image/png, image/jpeg
----------------------------------------------------------------------------------------------------
SANITIZED CLIENT LINEAGE
api_version                       : v1
computer_vision_model_version     : resnet18-chestmnist-v1
language_model_version            : flan-t5-small-chestmnist-v1
prompt_registry_version           : grounded-language-prompts-v1
educational_use_only              : True
----------------------------------------------------------

<a id="nb07-4-2-controlled-service-exceptions"></a>
### 4.2 Controlled Service Exceptions

Expected client and runtime failures are represented through typed service exceptions with stable HTTP status codes and error codes. This keeps validation, model-readiness, prediction lookup, grounding, and execution failures consistent across every route.

Client-safe details are sanitized before serialization. Keys associated with tracebacks, stack traces, filesystem locations, model paths, or internal exception objects are removed from the public error contract.


In [19]:
# -------------------------------------------------------------------------
# Define typed and sanitized service exceptions
# -------------------------------------------------------------------------
CORE_ERRORS_PATH = (
    API_CORE_DIR / "errors.py"
)

CORE_ERRORS_SOURCE = '''"""Typed client-safe exceptions for API services."""

from typing import Any


SENSITIVE_DETAIL_TERMS = {
    "traceback",
    "stack_trace",
    "filesystem_path",
    "file_path",
    "model_path",
    "internal_exception",
}


def sanitize_client_details(
    details: dict[str, Any] | None,
) -> dict[str, Any]:
    """Remove internal paths and exception details from public errors."""

    if not details:
        return {}

    sanitized_details: dict[str, Any] = {}

    for key, value in details.items():
        normalized_key = key.lower()

        if any(
            sensitive_term in normalized_key
            for sensitive_term
            in SENSITIVE_DETAIL_TERMS
        ):
            continue

        if isinstance(value, dict):
            sanitized_details[key] = (
                sanitize_client_details(value)
            )
        elif isinstance(value, (list, tuple)):
            sanitized_details[key] = [
                item
                for item in value
                if not (
                    isinstance(item, str)
                    and item.startswith(
                        ("/home/", "/root/", "/workspace/")
                    )
                )
            ]
        elif (
            isinstance(value, str)
            and value.startswith(
                ("/home/", "/root/", "/workspace/")
            )
        ):
            continue
        elif isinstance(
            value,
            (str, int, float, bool),
        ) or value is None:
            sanitized_details[key] = value
        else:
            sanitized_details[key] = (
                str(value)
            )

    return sanitized_details


class ServiceError(Exception):
    """Base exception with a stable public error contract."""

    status_code = 500
    error_code = "SERVICE_ERROR"
    default_message = (
        "The service could not complete the request."
    )

    def __init__(
        self,
        message: str | None = None,
        details: dict[str, Any] | None = None,
    ) -> None:
        self.message = (
            message or self.default_message
        )
        self.details = sanitize_client_details(
            details
        )
        super().__init__(self.message)


class InvalidImageError(ServiceError):
    status_code = 400
    error_code = "INVALID_IMAGE"
    default_message = (
        "The uploaded content is not a valid image."
    )


class UnsupportedMediaTypeError(ServiceError):
    status_code = 415
    error_code = "UNSUPPORTED_MEDIA_TYPE"
    default_message = (
        "The uploaded image media type is not supported."
    )


class UploadTooLargeError(ServiceError):
    status_code = 413
    error_code = "UPLOAD_TOO_LARGE"
    default_message = (
        "The uploaded image exceeds the permitted size."
    )


class PredictionNotFoundError(ServiceError):
    status_code = 404
    error_code = "PREDICTION_NOT_FOUND"
    default_message = (
        "The requested prediction was not found."
    )


class ModelNotReadyError(ServiceError):
    status_code = 503
    error_code = "MODEL_NOT_READY"
    default_message = (
        "A required model service is not ready."
    )


class GroundingContractError(ServiceError):
    status_code = 500
    error_code = "GROUNDING_CONTRACT_VIOLATION"
    default_message = (
        "The generated output did not satisfy the grounding contract."
    )


class ServiceExecutionError(ServiceError):
    status_code = 500
    error_code = "SERVICE_EXECUTION_ERROR"
    default_message = (
        "The service encountered an internal execution failure."
    )
'''

CORE_ERRORS_PATH.write_text(
    CORE_ERRORS_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()
sys.modules.pop(
    "api.core.errors",
    None,
)

core_errors_module = importlib.import_module(
    "api.core.errors"
)

ServiceError = (
    core_errors_module.ServiceError
)
InvalidImageError = (
    core_errors_module.InvalidImageError
)
UnsupportedMediaTypeError = (
    core_errors_module.UnsupportedMediaTypeError
)
UploadTooLargeError = (
    core_errors_module.UploadTooLargeError
)
PredictionNotFoundError = (
    core_errors_module.PredictionNotFoundError
)
ModelNotReadyError = (
    core_errors_module.ModelNotReadyError
)
GroundingContractError = (
    core_errors_module.GroundingContractError
)
ServiceExecutionError = (
    core_errors_module.ServiceExecutionError
)
sanitize_client_details = (
    core_errors_module.sanitize_client_details
)


# -------------------------------------------------------------------------
# Exercise every typed exception contract
# -------------------------------------------------------------------------
exception_classes = [
    InvalidImageError,
    UnsupportedMediaTypeError,
    UploadTooLargeError,
    PredictionNotFoundError,
    ModelNotReadyError,
    GroundingContractError,
    ServiceExecutionError,
]

exception_contract_rows = []

for exception_class in exception_classes:
    exception_instance = exception_class(
        details={
            "operation": "controlled_test",
            "filesystem_path": (
                "/home/jovyan/internal/file"
            ),
            "traceback": (
                "internal stack trace"
            ),
            "nested": {
                "safe_value": "retained",
                "model_path": (
                    "/home/jovyan/internal/model"
                ),
            },
        }
    )

    exception_contract_rows.append(
        {
            "exception": (
                exception_class.__name__
            ),
            "status_code": (
                exception_instance.status_code
            ),
            "error_code": (
                exception_instance.error_code
            ),
            "message": (
                exception_instance.message
            ),
            "details": (
                exception_instance.details
            ),
        }
    )


# -------------------------------------------------------------------------
# Validate safe detail sanitization
# -------------------------------------------------------------------------
safe_detail_example = (
    sanitize_client_details(
        {
            "supported_types": [
                "image/png",
                "image/jpeg",
            ],
            "maximum_bytes": 10485760,
            "file_path": (
                "/home/jovyan/private/file"
            ),
            "internal_exception": (
                ValueError("internal")
            ),
            "nested": {
                "reason": "invalid content",
                "stack_trace": "private trace",
            },
        }
    )
)

error_module_checksum = hashlib.sha256(
    CORE_ERRORS_PATH.read_bytes()
).hexdigest()

error_contract_checks = {
    "Core error module was written": (
        CORE_ERRORS_PATH.is_file()
    ),
    "Seven typed service errors are registered": (
        len(exception_contract_rows) == 7
    ),
    "HTTP status codes are valid": all(
        400
        <= row["status_code"]
        <= 599
        for row in exception_contract_rows
    ),
    "Error codes are unique": (
        len(
            {
                row["error_code"]
                for row in exception_contract_rows
            }
        )
        == len(exception_contract_rows)
    ),
    "Filesystem path is removed": (
        "file_path"
        not in safe_detail_example
    ),
    "Internal exception is removed": (
        "internal_exception"
        not in safe_detail_example
    ),
    "Nested stack trace is removed": (
        "stack_trace"
        not in safe_detail_example["nested"]
    ),
    "Safe details are preserved": (
        safe_detail_example[
            "maximum_bytes"
        ]
        == 10485760
        and safe_detail_example[
            "nested"
        ]["reason"]
        == "invalid content"
    ),
    "Exception test details contain no internal paths": all(
        "/home/" not in str(row["details"])
        for row in exception_contract_rows
    ),
    "Error module checksum is available": (
        len(error_module_checksum) == 64
    ),
}


# -------------------------------------------------------------------------
# Report controlled error readiness
# -------------------------------------------------------------------------
print("CONTROLLED API SERVICE EXCEPTIONS")
print("-" * 100)
print(
    f"Error module              : "
    f"{CORE_ERRORS_PATH}"
)
print(
    f"Module SHA-256            : "
    f"{error_module_checksum[:16]}..."
)
print(
    f"Typed exceptions          : "
    f"{len(exception_contract_rows)}"
)
print("-" * 100)

for row in exception_contract_rows:
    print(
        f"{row['status_code']} | "
        f"{row['error_code']:<30} | "
        f"{row['exception']}"
    )

print("-" * 100)

for check_name, passed in error_contract_checks.items():
    print(
        f"{check_name:<58}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(error_contract_checks.values()):
    failed_checks = [
        name
        for name, passed
        in error_contract_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Controlled service exception validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: CONTROLLED SERVICE ERROR CONTRACT READY")

CONTROLLED API SERVICE EXCEPTIONS
----------------------------------------------------------------------------------------------------
Error module              : /home/jovyan/chest-xray-ai-assistant/api/core/errors.py
Module SHA-256            : 243403310d177670...
Typed exceptions          : 7
----------------------------------------------------------------------------------------------------
400 | INVALID_IMAGE                  | InvalidImageError
415 | UNSUPPORTED_MEDIA_TYPE         | UnsupportedMediaTypeError
413 | UPLOAD_TOO_LARGE               | UploadTooLargeError
404 | PREDICTION_NOT_FOUND           | PredictionNotFoundError
503 | MODEL_NOT_READY                | ModelNotReadyError
500 | GROUNDING_CONTRACT_VIOLATION   | GroundingContractError
500 | SERVICE_EXECUTION_ERROR        | ServiceExecutionError
----------------------------------------------------------------------------------------------------
Core error module was written                             : PASS
Seven typed

<a id="nb07-4-3-request-context-response-metadata-and-error-serialization"></a>
### 4.3 Request Context, Response Metadata, and Error Serialization

Each incoming request receives an independent UUID, UTC start timestamp, and monotonic performance timer. These values are used consistently for successful responses, controlled errors, logging, prediction storage, and operational metrics.

Metadata builders attach only the model components actually used by an endpoint. Error serialization converts typed service failures into the strict public error schema while preserving the same request identifier and measured latency.


In [20]:
# -------------------------------------------------------------------------
# Define request context and response metadata utilities
# -------------------------------------------------------------------------
CORE_RUNTIME_PATH = (
    API_CORE_DIR / "runtime.py"
)

CORE_RUNTIME_SOURCE = '''"""Request identity, timing, and response metadata utilities."""

import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any
from uuid import UUID, uuid4

from api.core.config import ServiceSettings, get_settings
from api.core.errors import ServiceError
from api.schemas.common import (
    APIErrorResponse,
    ModelVersions,
)


def utc_now() -> datetime:
    """Return a timezone-aware UTC timestamp."""

    return datetime.now(timezone.utc)


@dataclass
class RequestContext:
    """Identity and monotonic timer for one API request."""

    request_id: UUID = field(
        default_factory=uuid4
    )
    timestamp_utc: datetime = field(
        default_factory=utc_now
    )
    _started_at: float = field(
        default_factory=time.perf_counter,
        repr=False,
    )

    def elapsed_ms(self) -> float:
        """Return non-negative elapsed request time."""

        return max(
            0.0,
            (
                time.perf_counter()
                - self._started_at
            )
            * 1000.0,
        )


def build_model_versions(
    settings: ServiceSettings,
    include_language: bool = False,
    include_explainability: bool = False,
) -> ModelVersions:
    """Build endpoint-specific model lineage."""

    return ModelVersions(
        computer_vision=(
            settings
            .computer_vision_model_version
        ),
        language=(
            settings.language_model_version
            if include_language
            else None
        ),
        explainability_method=(
            "LayerGradCam"
            if include_explainability
            else None
        ),
    )


def build_success_metadata(
    context: RequestContext,
    *,
    settings: ServiceSettings | None = None,
    include_language: bool = False,
    include_explainability: bool = False,
    warnings: list[str] | None = None,
) -> dict[str, Any]:
    """Build common successful response fields."""

    resolved_settings = (
        settings or get_settings()
    )

    return {
        "request_id": context.request_id,
        "timestamp_utc": (
            context.timestamp_utc
        ),
        "api_version": (
            resolved_settings.api_version
        ),
        "status": "success",
        "model_versions": (
            build_model_versions(
                resolved_settings,
                include_language=(
                    include_language
                ),
                include_explainability=(
                    include_explainability
                ),
            )
        ),
        "prompt_registry_version": (
            resolved_settings
            .prompt_registry_version
            if include_language
            else None
        ),
        "latency_ms": (
            context.elapsed_ms()
        ),
        "warnings": warnings or [],
        "educational_use_only": True,
    }


def build_error_response(
    context: RequestContext,
    error: ServiceError,
    *,
    settings: ServiceSettings | None = None,
) -> APIErrorResponse:
    """Serialize a typed service error to the public schema."""

    resolved_settings = (
        settings or get_settings()
    )

    return APIErrorResponse(
        request_id=context.request_id,
        timestamp_utc=(
            context.timestamp_utc
        ),
        api_version=(
            resolved_settings.api_version
        ),
        status="error",
        error_code=error.error_code,
        message=error.message,
        details=error.details,
        latency_ms=context.elapsed_ms(),
        educational_use_only=True,
    )
'''

CORE_RUNTIME_PATH.write_text(
    CORE_RUNTIME_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()
sys.modules.pop(
    "api.core.runtime",
    None,
)

core_runtime_module = importlib.import_module(
    "api.core.runtime"
)

RequestContext = (
    core_runtime_module.RequestContext
)
utc_now = core_runtime_module.utc_now
build_model_versions = (
    core_runtime_module.build_model_versions
)
build_success_metadata = (
    core_runtime_module.build_success_metadata
)
build_error_response = (
    core_runtime_module.build_error_response
)


# -------------------------------------------------------------------------
# Exercise successful and error response construction
# -------------------------------------------------------------------------
first_request_context = RequestContext()
second_request_context = RequestContext()

# Perform a small deterministic operation so elapsed time is measurable.
_ = sum(
    value * value
    for value in range(1000)
)

classification_metadata_example = (
    build_success_metadata(
        first_request_context,
        settings=service_settings,
        include_language=False,
        include_explainability=False,
        warnings=[
            (
                "This output is not a "
                "clinical diagnosis."
            )
        ],
    )
)

complete_metadata_example = (
    build_success_metadata(
        second_request_context,
        settings=service_settings,
        include_language=True,
        include_explainability=True,
    )
)

runtime_error_example = InvalidImageError(
    details={
        "reason": "Image decoding failed.",
        "file_path": (
            "/home/jovyan/internal/upload"
        ),
    }
)

serialized_error_example = (
    build_error_response(
        first_request_context,
        runtime_error_example,
        settings=service_settings,
    )
)


# -------------------------------------------------------------------------
# Validate request identity, timing, and lineage
# -------------------------------------------------------------------------
runtime_module_checksum = hashlib.sha256(
    CORE_RUNTIME_PATH.read_bytes()
).hexdigest()

runtime_checks = {
    "Runtime utility module was written": (
        CORE_RUNTIME_PATH.is_file()
    ),
    "Request identifiers are unique": (
        first_request_context.request_id
        != second_request_context.request_id
    ),
    "Request timestamps are timezone-aware": (
        first_request_context
        .timestamp_utc
        .utcoffset()
        is not None
    ),
    "Elapsed latency is non-negative": (
        first_request_context.elapsed_ms()
        >= 0.0
    ),
    "Classification metadata excludes language": (
        classification_metadata_example[
            "model_versions"
        ].language
        is None
        and classification_metadata_example[
            "prompt_registry_version"
        ]
        is None
    ),
    "Complete metadata includes language": (
        complete_metadata_example[
            "model_versions"
        ].language
        == LANGUAGE_MODEL_VERSION
        and complete_metadata_example[
            "prompt_registry_version"
        ]
        == PROMPT_REGISTRY_VERSION
    ),
    "Complete metadata includes explainability": (
        complete_metadata_example[
            "model_versions"
        ].explainability_method
        == "LayerGradCam"
    ),
    "Error preserves the request identifier": (
        serialized_error_example.request_id
        == first_request_context.request_id
    ),
    "Error preserves the stable error code": (
        serialized_error_example.error_code
        == "INVALID_IMAGE"
    ),
    "Error details contain no internal path": (
        "file_path"
        not in serialized_error_example.details
        and "/home/" not in str(
            serialized_error_example.details
        )
    ),
    "Runtime module checksum is available": (
        len(runtime_module_checksum) == 64
    ),
}


# -------------------------------------------------------------------------
# Report runtime utility readiness
# -------------------------------------------------------------------------
print("REQUEST CONTEXT AND RESPONSE RUNTIME UTILITIES")
print("-" * 100)
print(
    f"Runtime module            : "
    f"{CORE_RUNTIME_PATH}"
)
print(
    f"Module SHA-256            : "
    f"{runtime_module_checksum[:16]}..."
)
print(
    f"First request ID          : "
    f"{first_request_context.request_id}"
)
print(
    f"Second request ID         : "
    f"{second_request_context.request_id}"
)
print(
    f"Measured latency          : "
    f"{first_request_context.elapsed_ms():.4f} ms"
)
print(
    f"Classification language   : "
    f"{classification_metadata_example['model_versions'].language}"
)
print(
    f"Complete language version : "
    f"{complete_metadata_example['model_versions'].language}"
)
print(
    f"Serialized error code     : "
    f"{serialized_error_example.error_code}"
)
print("-" * 100)

for check_name, passed in runtime_checks.items():
    print(
        f"{check_name:<58}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(runtime_checks.values()):
    failed_checks = [
        name
        for name, passed
        in runtime_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Request runtime utility validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: REQUEST IDENTITY AND RESPONSE METADATA READY")

REQUEST CONTEXT AND RESPONSE RUNTIME UTILITIES
----------------------------------------------------------------------------------------------------
Runtime module            : /home/jovyan/chest-xray-ai-assistant/api/core/runtime.py
Module SHA-256            : d9dc3cae44ebbcc7...
First request ID          : 845ab98f-72ba-4617-921a-5111f4a8049c
Second request ID         : 610ceb7e-4667-4347-a6ab-4a33c99bd3ba
Measured latency          : 1.5201 ms
Classification language   : None
Complete language version : flan-t5-small-chestmnist-v1
Serialized error code     : INVALID_IMAGE
----------------------------------------------------------------------------------------------------
Runtime utility module was written                        : PASS
Request identifiers are unique                            : PASS
Request timestamps are timezone-aware                     : PASS
Elapsed latency is non-negative                           : PASS
Classification metadata excludes language                 :

**[↑ Back to notebook index](#notebook-index)**


<a id="nb07-5-reusable-api-service-modules"></a>
## 5. Reusable API Service Modules

<a id="nb07-5-1-secure-image-validation-and-decoding-service"></a>
### 5.1 Secure Image Validation and Decoding Service

Uploaded content is validated before it reaches the computer-vision model. The service enforces the permitted media types, 10 MiB size limit, actual PNG or JPEG format, bounded image dimensions, safe filename handling, successful decoding, and SHA-256 integrity tracking.

Accepted images retain their original dimensions and mode for response metadata, while model-facing content is converted to RGB. Invalid, oversized, spoofed, or undecodable uploads raise controlled typed exceptions.


In [21]:
from io import BytesIO

from PIL import Image


# -------------------------------------------------------------------------
# Define the secure image ingestion service
# -------------------------------------------------------------------------
IMAGE_SERVICE_PATH = (
    SERVICE_ROOT / "image_service.py"
)

IMAGE_SERVICE_SOURCE = '''"""Secure validation and decoding for uploaded images."""

import hashlib
from dataclasses import dataclass
from io import BytesIO
from pathlib import Path

from PIL import Image, UnidentifiedImageError

from api.core.config import ServiceSettings, get_settings
from api.core.errors import (
    InvalidImageError,
    UnsupportedMediaTypeError,
    UploadTooLargeError,
)
from api.schemas.prediction import ImageMetadata


MAXIMUM_IMAGE_WIDTH = 4096
MAXIMUM_IMAGE_HEIGHT = 4096
MAXIMUM_IMAGE_PIXELS = (
    MAXIMUM_IMAGE_WIDTH
    * MAXIMUM_IMAGE_HEIGHT
)

MEDIA_TYPE_TO_FORMAT = {
    "image/png": "PNG",
    "image/jpeg": "JPEG",
}


@dataclass
class ValidatedImage:
    """Decoded image and its client-safe metadata."""

    filename: str
    media_type: str
    original_mode: str
    width: int
    height: int
    sha256: str
    rgb_image: Image.Image

    def response_metadata(
        self,
    ) -> ImageMetadata:
        """Build the strict image response metadata."""

        return ImageMetadata(
            filename=self.filename,
            media_type=self.media_type,
            width=self.width,
            height=self.height,
            original_mode=self.original_mode,
            sha256=self.sha256,
        )


class ImageValidationService:
    """Validate upload boundaries and decode safe RGB content."""

    def __init__(
        self,
        settings: ServiceSettings | None = None,
    ) -> None:
        self.settings = (
            settings or get_settings()
        )

    def validate_and_decode(
        self,
        *,
        filename: str,
        media_type: str,
        content: bytes,
    ) -> ValidatedImage:
        """Validate one upload and return decoded RGB image content."""

        if media_type not in (
            self.settings
            .supported_image_media_types
        ):
            raise UnsupportedMediaTypeError(
                details={
                    "received_media_type": media_type,
                    "supported_media_types": list(
                        self.settings
                        .supported_image_media_types
                    ),
                }
            )

        if not content:
            raise InvalidImageError(
                details={
                    "reason": (
                        "The uploaded file is empty."
                    )
                }
            )

        if len(content) > (
            self.settings
            .maximum_upload_bytes
        ):
            raise UploadTooLargeError(
                details={
                    "received_bytes": len(content),
                    "maximum_bytes": (
                        self.settings
                        .maximum_upload_bytes
                    ),
                }
            )

        safe_filename = (
            Path(filename).name.strip()
            if filename
            else "uploaded-image"
        )

        if not safe_filename:
            safe_filename = "uploaded-image"

        try:
            with Image.open(
                BytesIO(content)
            ) as verification_image:
                detected_format = (
                    verification_image.format
                )
                verification_image.verify()

            with Image.open(
                BytesIO(content)
            ) as decoded_image:
                decoded_image.load()

                width, height = (
                    decoded_image.size
                )
                original_mode = (
                    decoded_image.mode
                )
                rgb_image = (
                    decoded_image.convert(
                        "RGB"
                    )
                )

        except (
            UnidentifiedImageError,
            OSError,
            ValueError,
        ) as error:
            raise InvalidImageError(
                details={
                    "reason": (
                        "Image decoding or integrity "
                        "validation failed."
                    )
                }
            ) from error

        expected_format = (
            MEDIA_TYPE_TO_FORMAT[
                media_type
            ]
        )

        if detected_format != expected_format:
            raise UnsupportedMediaTypeError(
                message=(
                    "The declared media type does not "
                    "match the decoded image format."
                ),
                details={
                    "declared_media_type": media_type,
                    "detected_format": (
                        detected_format
                    ),
                },
            )

        if (
            width <= 0
            or height <= 0
            or width > MAXIMUM_IMAGE_WIDTH
            or height > MAXIMUM_IMAGE_HEIGHT
            or width * height
            > MAXIMUM_IMAGE_PIXELS
        ):
            raise InvalidImageError(
                details={
                    "reason": (
                        "Image dimensions exceed "
                        "the supported boundary."
                    ),
                    "width": width,
                    "height": height,
                    "maximum_width": (
                        MAXIMUM_IMAGE_WIDTH
                    ),
                    "maximum_height": (
                        MAXIMUM_IMAGE_HEIGHT
                    ),
                }
            )

        return ValidatedImage(
            filename=safe_filename,
            media_type=media_type,
            original_mode=original_mode,
            width=width,
            height=height,
            sha256=hashlib.sha256(
                content
            ).hexdigest(),
            rgb_image=rgb_image,
        )
'''

IMAGE_SERVICE_PATH.write_text(
    IMAGE_SERVICE_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()
sys.modules.pop(
    "src.services.image_service",
    None,
)

image_service_module = importlib.import_module(
    "src.services.image_service"
)

ImageValidationService = (
    image_service_module.ImageValidationService
)
ValidatedImage = (
    image_service_module.ValidatedImage
)

image_validation_service = (
    ImageValidationService(
        service_settings
    )
)


# -------------------------------------------------------------------------
# Create in-memory valid PNG and JPEG test uploads
# -------------------------------------------------------------------------
synthetic_grayscale_image = Image.new(
    "L",
    (224, 224),
    color=128,
)

png_buffer = BytesIO()
synthetic_grayscale_image.save(
    png_buffer,
    format="PNG",
)
valid_png_bytes = png_buffer.getvalue()

jpeg_buffer = BytesIO()
synthetic_grayscale_image.save(
    jpeg_buffer,
    format="JPEG",
)
valid_jpeg_bytes = jpeg_buffer.getvalue()

validated_png = (
    image_validation_service
    .validate_and_decode(
        filename="../../example.png",
        media_type="image/png",
        content=valid_png_bytes,
    )
)

validated_jpeg = (
    image_validation_service
    .validate_and_decode(
        filename="example.jpg",
        media_type="image/jpeg",
        content=valid_jpeg_bytes,
    )
)


# -------------------------------------------------------------------------
# Verify invalid upload boundaries
# -------------------------------------------------------------------------
empty_upload_rejected = False
unsupported_media_rejected = False
oversized_upload_rejected = False
invalid_content_rejected = False
spoofed_media_rejected = False

try:
    image_validation_service.validate_and_decode(
        filename="empty.png",
        media_type="image/png",
        content=b"",
    )
except InvalidImageError:
    empty_upload_rejected = True

try:
    image_validation_service.validate_and_decode(
        filename="example.gif",
        media_type="image/gif",
        content=valid_png_bytes,
    )
except UnsupportedMediaTypeError:
    unsupported_media_rejected = True

try:
    image_validation_service.validate_and_decode(
        filename="large.png",
        media_type="image/png",
        content=(
            b"x"
            * (
                service_settings
                .maximum_upload_bytes
                + 1
            )
        ),
    )
except UploadTooLargeError:
    oversized_upload_rejected = True

try:
    image_validation_service.validate_and_decode(
        filename="invalid.png",
        media_type="image/png",
        content=b"not-an-image",
    )
except InvalidImageError:
    invalid_content_rejected = True

try:
    image_validation_service.validate_and_decode(
        filename="spoofed.png",
        media_type="image/png",
        content=valid_jpeg_bytes,
    )
except UnsupportedMediaTypeError:
    spoofed_media_rejected = True


# -------------------------------------------------------------------------
# Validate and report image ingestion readiness
# -------------------------------------------------------------------------
image_service_checksum = hashlib.sha256(
    IMAGE_SERVICE_PATH.read_bytes()
).hexdigest()

png_metadata = (
    validated_png.response_metadata()
)
jpeg_metadata = (
    validated_jpeg.response_metadata()
)

image_service_checks = {
    "Image service module was written": (
        IMAGE_SERVICE_PATH.is_file()
    ),
    "PNG image is decoded successfully": (
        png_metadata.media_type
        == "image/png"
    ),
    "JPEG image is decoded successfully": (
        jpeg_metadata.media_type
        == "image/jpeg"
    ),
    "Original grayscale mode is preserved": (
        png_metadata.original_mode == "L"
    ),
    "Model-facing image is converted to RGB": (
        validated_png.rgb_image.mode
        == "RGB"
    ),
    "Unsafe filename components are removed": (
        png_metadata.filename
        == "example.png"
    ),
    "Image SHA-256 is available": (
        len(png_metadata.sha256) == 64
    ),
    "Empty upload is rejected": (
        empty_upload_rejected
    ),
    "Unsupported media type is rejected": (
        unsupported_media_rejected
    ),
    "Oversized upload is rejected": (
        oversized_upload_rejected
    ),
    "Invalid image content is rejected": (
        invalid_content_rejected
    ),
    "Spoofed media type is rejected": (
        spoofed_media_rejected
    ),
    "Image service checksum is available": (
        len(image_service_checksum) == 64
    ),
}

print("SECURE IMAGE VALIDATION AND DECODING SERVICE")
print("-" * 100)
print(
    f"Service module            : "
    f"{IMAGE_SERVICE_PATH}"
)
print(
    f"Module SHA-256            : "
    f"{image_service_checksum[:16]}..."
)
print(
    f"PNG dimensions            : "
    f"{png_metadata.width} x "
    f"{png_metadata.height}"
)
print(
    f"Original image mode       : "
    f"{png_metadata.original_mode}"
)
print(
    f"Model-facing mode         : "
    f"{validated_png.rgb_image.mode}"
)
print(
    f"Sanitized filename        : "
    f"{png_metadata.filename}"
)
print(
    f"PNG SHA-256               : "
    f"{png_metadata.sha256[:16]}..."
)
print("-" * 100)

for check_name, passed in image_service_checks.items():
    print(
        f"{check_name:<58}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(image_service_checks.values()):
    failed_checks = [
        name
        for name, passed
        in image_service_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Image validation service failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: SECURE IMAGE INGESTION SERVICE READY")

SECURE IMAGE VALIDATION AND DECODING SERVICE
----------------------------------------------------------------------------------------------------
Service module            : /home/jovyan/chest-xray-ai-assistant/src/services/image_service.py
Module SHA-256            : 4cda31250b539c9f...
PNG dimensions            : 224 x 224
Original image mode       : L
Model-facing mode         : RGB
Sanitized filename        : example.png
PNG SHA-256               : ef404e596e63d9e9...
----------------------------------------------------------------------------------------------------
Image service module was written                          : PASS
PNG image is decoded successfully                         : PASS
JPEG image is decoded successfully                        : PASS
Original grayscale mode is preserved                      : PASS
Model-facing image is converted to RGB                    : PASS
Unsafe filename components are removed                    : PASS
Image SHA-256 is available      

<a id="nb07-5-2-frozen-finding-ontology-and-threshold-registry"></a>
### 5.2 Frozen Finding Ontology and Threshold Registry

The API must not reconstruct labels, thresholds, or descriptions independently in multiple services. A single versioned finding contract is therefore created from the frozen computer-vision metadata and the approved language ontology.

This registry preserves the exact 14-label order, display names, per-label thresholds, cautious descriptions, no-target-finding interpretation, educational-use boundary, and professional-review wording. Both computer-vision and language services will load this same file.


In [23]:
# -------------------------------------------------------------------------
# Recover the exact frozen threshold vector
# -------------------------------------------------------------------------
FINDING_CONTRACT_VERSION = (
    "chestmnist-finding-contract-v1"
)

FINDING_CONTRACT_PATH = (
    SOLUTION_ROOT
    / "configs"
    / "finding_contract.yaml"
)

try:
    raw_frozen_thresholds = (
        cv_metadata[
            "output_contract"
        ]["thresholds"]
    )
except (
    KeyError,
    TypeError,
) as error:
    raise KeyError(
        "Frozen thresholds were not found at "
        "output_contract.thresholds in the "
        "computer-vision metadata."
    ) from error


# -------------------------------------------------------------------------
# Support both ordered lists and label-to-threshold mappings
# -------------------------------------------------------------------------
if isinstance(
    raw_frozen_thresholds,
    dict,
):
    missing_threshold_labels = [
        label_name
        for label_name
        in CHESTMNIST_LABEL_NAMES
        if label_name
        not in raw_frozen_thresholds
    ]

    if missing_threshold_labels:
        raise ValueError(
            "The threshold mapping is missing labels: "
            + ", ".join(
                missing_threshold_labels
            )
        )

    frozen_threshold_values = [
        float(
            raw_frozen_thresholds[
                label_name
            ]
        )
        for label_name
        in CHESTMNIST_LABEL_NAMES
    ]

elif isinstance(
    raw_frozen_thresholds,
    (list, tuple),
):
    frozen_threshold_values = [
        float(value)
        for value
        in raw_frozen_thresholds
    ]

else:
    raise TypeError(
        "The frozen threshold contract must "
        "be a list, tuple, or label mapping."
    )

if len(frozen_threshold_values) != 14:
    raise ValueError(
        "The frozen threshold contract must "
        "contain exactly fourteen values."
    )


# -------------------------------------------------------------------------
# Define the approved descriptions from the frozen language ontology
# -------------------------------------------------------------------------
APPROVED_FINDING_DESCRIPTIONS = [
    (
        "A pattern associated with reduced expansion "
        "or partial collapse of part of the lung."
    ),
    (
        "A pattern associated with an enlarged "
        "appearance of the heart."
    ),
    (
        "A pattern associated with fluid collecting "
        "in the space around the lungs."
    ),
    (
        "A broad dataset pattern associated with "
        "increased material or opacity within lung tissue."
    ),
    (
        "A dataset pattern associated with a larger "
        "focal opacity or mass-like appearance."
    ),
    (
        "A dataset pattern associated with a small, "
        "rounded focal opacity."
    ),
    (
        "A pattern associated with lung opacity that "
        "may occur with pneumonia, without confirming infection."
    ),
    (
        "A pattern associated with air in the space "
        "between the lung and chest wall."
    ),
    (
        "A pattern associated with an area of lung "
        "airspace becoming filled and appearing denser."
    ),
    (
        "A pattern associated with increased fluid "
        "within the lungs."
    ),
    (
        "A pattern associated with over-expanded lungs "
        "and changes in lung airspaces."
    ),
    (
        "A pattern associated with scarring or "
        "fibrotic change in lung tissue."
    ),
    (
        "A broad dataset label associated with an "
        "abnormal appearance of the lining around the lungs."
    ),
    (
        "A pattern associated with tissue or an organ "
        "projecting through an opening near the diaphragm."
    ),
]

NO_TARGET_FINDING_DESCRIPTION = (
    "None of the 14 supported ChestMNIST findings "
    "crossed its frozen decision threshold. This does "
    "not establish that the chest X-ray is clinically normal."
)

EDUCATIONAL_USE_LIMITATION = (
    "This output is generated by an educational "
    "decision-support prototype. It is not a diagnosis "
    "and should not replace review by a qualified "
    "healthcare professional."
)

GRADCAM_LIMITATION = (
    "Grad-CAM highlights image regions that influenced "
    "a model output. It does not confirm a lesion, "
    "provide segmentation, or establish a clinical diagnosis."
)

PROFESSIONAL_REVIEW_GUIDANCE = (
    "A qualified healthcare professional can interpret "
    "the image together with symptoms, history, "
    "examination findings, and other tests."
)


# -------------------------------------------------------------------------
# Assemble the exact runtime finding contract
# -------------------------------------------------------------------------
finding_records = [
    {
        "label_id": label_id,
        "label_name": label_name,
        "display_name": display_name,
        "frozen_threshold": float(
            frozen_threshold_values[
                label_id
            ]
        ),
        "approved_description": (
            APPROVED_FINDING_DESCRIPTIONS[
                label_id
            ]
        ),
    }
    for label_id, (
        label_name,
        display_name,
    ) in enumerate(
        zip(
            CHESTMNIST_LABEL_NAMES,
            CHESTMNIST_DISPLAY_NAMES,
        )
    )
]

finding_contract = {
    "contract_version": (
        FINDING_CONTRACT_VERSION
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "computer_vision_model_version": (
        COMPUTER_VISION_MODEL_VERSION
    ),
    "language_model_version": (
        LANGUAGE_MODEL_VERSION
    ),
    "label_count": 14,
    "findings": finding_records,
    "controlled_language": {
        "no_target_finding": (
            NO_TARGET_FINDING_DESCRIPTION
        ),
        "educational_use_limitation": (
            EDUCATIONAL_USE_LIMITATION
        ),
        "gradcam_limitation": (
            GRADCAM_LIMITATION
        ),
        "professional_review_guidance": (
            PROFESSIONAL_REVIEW_GUIDANCE
        ),
    },
}


# -------------------------------------------------------------------------
# Validate threshold and ontology agreement
# -------------------------------------------------------------------------
documented_threshold_summary = [
    0.6797,
    0.8438,
    0.6758,
    0.5977,
    0.8672,
    0.8125,
    0.7695,
    0.8281,
    0.7461,
    0.8789,
    0.8750,
    0.8594,
    0.8594,
    0.6992,
]

finding_contract_checks = {
    "Fourteen findings are registered": (
        len(finding_records) == 14
    ),
    "Label identifiers preserve order": (
        [
            finding["label_id"]
            for finding in finding_records
        ]
        == list(range(14))
    ),
    "Label names preserve model order": (
        [
            finding["label_name"]
            for finding in finding_records
        ]
        == CHESTMNIST_LABEL_NAMES
    ),
    "Display names are complete": all(
        bool(
            finding["display_name"].strip()
        )
        for finding in finding_records
    ),
    "Descriptions are complete": all(
        bool(
            finding[
                "approved_description"
            ].strip()
        )
        for finding in finding_records
    ),
    "Thresholds are within zero and one": all(
        0.0
        <= finding["frozen_threshold"]
        <= 1.0
        for finding in finding_records
    ),
    "Thresholds match frozen evaluation": all(
        abs(
            finding["frozen_threshold"]
            - documented_threshold_summary[
                finding["label_id"]
            ]
        )
        <= 0.001
        for finding in finding_records
    ),
    "No-target boundary is defined": bool(
        NO_TARGET_FINDING_DESCRIPTION
    ),
    "Educational limitation is defined": bool(
        EDUCATIONAL_USE_LIMITATION
    ),
    "Grad-CAM limitation is defined": bool(
        GRADCAM_LIMITATION
    ),
}


# -------------------------------------------------------------------------
# Persist and reload the runtime finding contract
# -------------------------------------------------------------------------
with FINDING_CONTRACT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        finding_contract,
        file,
        sort_keys=False,
        allow_unicode=True,
        width=100,
    )

reloaded_finding_contract = (
    load_yaml_file(
        FINDING_CONTRACT_PATH
    )
)

finding_contract_checksum = (
    hashlib.sha256(
        FINDING_CONTRACT_PATH.read_bytes()
    ).hexdigest()
)

finding_contract_checks.update(
    {
        "Finding contract was exported": (
            FINDING_CONTRACT_PATH.is_file()
        ),
        "Reloaded contract preserves version": (
            reloaded_finding_contract[
                "contract_version"
            ]
            == FINDING_CONTRACT_VERSION
        ),
        "Finding contract checksum is available": (
            len(finding_contract_checksum)
            == 64
        ),
    }
)


# -------------------------------------------------------------------------
# Report the frozen runtime ontology
# -------------------------------------------------------------------------
print("FROZEN FINDING ONTOLOGY AND THRESHOLD REGISTRY")
print("-" * 100)
print(
    f"Threshold metadata type   : "
    f"{type(raw_frozen_thresholds).__name__}"
)
print(
    f"Contract version          : "
    f"{FINDING_CONTRACT_VERSION}"
)
print(
    f"Contract path             : "
    f"{FINDING_CONTRACT_PATH}"
)
print(
    f"Contract SHA-256          : "
    f"{finding_contract_checksum[:16]}..."
)
print(
    f"Registered findings       : "
    f"{len(finding_records)}"
)
print("-" * 100)

for finding in finding_records:
    print(
        f"{finding['label_id']:>2} | "
        f"{finding['label_name']:<16} | "
        f"{finding['frozen_threshold']:.4f}"
    )

print("-" * 100)

for check_name, passed in finding_contract_checks.items():
    print(
        f"{check_name:<58}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(finding_contract_checks.values()):
    failed_checks = [
        name
        for name, passed
        in finding_contract_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Frozen finding contract validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: FROZEN FINDING CONTRACT READY FOR SHARED SERVICE USE")

FROZEN FINDING ONTOLOGY AND THRESHOLD REGISTRY
----------------------------------------------------------------------------------------------------
Threshold metadata type   : dict
Contract version          : chestmnist-finding-contract-v1
Contract path             : /home/jovyan/chest-xray-ai-assistant/configs/finding_contract.yaml
Contract SHA-256          : cdda22f2c65e1669...
Registered findings       : 14
----------------------------------------------------------------------------------------------------
 0 | atelectasis      | 0.6797
 1 | cardiomegaly     | 0.8438
 2 | effusion         | 0.6758
 3 | infiltration     | 0.5977
 4 | mass             | 0.8672
 5 | nodule           | 0.8125
 6 | pneumonia        | 0.7695
 7 | pneumothorax     | 0.8281
 8 | consolidation    | 0.7461
 9 | edema            | 0.8789
10 | emphysema        | 0.8750
11 | fibrosis         | 0.8594
12 | pleural          | 0.8594
13 | hernia           | 0.6992
---------------------------------------------------

<a id="nb07-5-3-frozen-computer-vision-inference-service"></a>
### 5.3 Frozen Computer-Vision Inference Service

The computer-vision service reconstructs the exact ResNet-18 architecture, loads the frozen versioned weights, applies grayscale-to-RGB compatible preprocessing, and performs thread-safe bfloat16 GPU inference.

Every prediction returns all 14 finding records with probabilities, frozen thresholds, decisions, confidence categories, approved descriptions, and the controlled no-target-finding interpretation. This block performs a structural smoke test only; isolated inference parity will be verified separately against the frozen Notebook 4 prediction bundle.


In [24]:
# -------------------------------------------------------------------------
# Define the frozen computer-vision inference service
# -------------------------------------------------------------------------
COMPUTER_VISION_SERVICE_PATH = (
    SERVICE_ROOT
    / "computer_vision_service.py"
)

COMPUTER_VISION_SERVICE_SOURCE = '''"""Frozen ResNet-18 ChestMNIST inference service."""

import threading
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

from api.core.config import (
    ServiceSettings,
    get_settings,
    load_yaml_file,
)
from api.core.errors import (
    ModelNotReadyError,
    ServiceExecutionError,
)
from api.schemas.prediction import FindingEvidence
from src.services.image_service import ValidatedImage


IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406,
]
IMAGENET_STANDARD_DEVIATION = [
    0.229,
    0.224,
    0.225,
]


@dataclass
class ClassificationResult:
    """Internal classification result with reusable model tensors."""

    findings: list[FindingEvidence]
    crossed_finding_names: list[str]
    no_target_finding: bool
    interpretation: str
    inference_latency_ms: float
    probabilities: np.ndarray = field(
        repr=False
    )
    logits: torch.Tensor = field(
        repr=False
    )
    input_tensor: torch.Tensor = field(
        repr=False
    )


class ComputerVisionService:
    """Load and execute the frozen multilabel ResNet-18 model."""

    def __init__(
        self,
        settings: ServiceSettings | None = None,
        device: str | None = None,
    ) -> None:
        self.settings = (
            settings or get_settings()
        )
        self.device = torch.device(
            device
            if device is not None
            else (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )
        )
        self._inference_lock = (
            threading.Lock()
        )
        self._ready = False
        self.model_load_latency_ms = 0.0

        finding_contract_path = (
            self.settings.solution_root
            / "configs"
            / "finding_contract.yaml"
        )
        self.finding_contract = (
            load_yaml_file(
                finding_contract_path
            )
        )
        self.findings = (
            self.finding_contract[
                "findings"
            ]
        )
        self.controlled_language = (
            self.finding_contract[
                "controlled_language"
            ]
        )

        self.thresholds = np.asarray(
            [
                finding[
                    "frozen_threshold"
                ]
                for finding in self.findings
            ],
            dtype=np.float32,
        )

        self.preprocess_transform = (
            transforms.Compose(
                [
                    transforms.Resize(
                        (224, 224),
                        antialias=True,
                    ),
                    transforms.ToTensor(),
                    transforms.Normalize(
                        mean=IMAGENET_MEAN,
                        std=(
                            IMAGENET_STANDARD_DEVIATION
                        ),
                    ),
                ]
            )
        )

        self.model = self._load_model()
        self._ready = True

    def _load_model(
        self,
    ) -> nn.Module:
        """Reconstruct and load the frozen ResNet-18 model."""

        load_start_time = (
            time.perf_counter()
        )

        model = models.resnet18(
            weights=None
        )
        model.fc = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(
                model.fc.in_features,
                14,
            ),
        )

        try:
            checkpoint_value = torch.load(
                self.settings
                .computer_vision_weights_path,
                map_location="cpu",
                weights_only=True,
            )

            if (
                isinstance(
                    checkpoint_value,
                    dict,
                )
                and "model_state_dict"
                in checkpoint_value
            ):
                state_dict = checkpoint_value[
                    "model_state_dict"
                ]
            else:
                state_dict = checkpoint_value

            model.load_state_dict(
                state_dict,
                strict=True,
            )

        except Exception as error:
            raise ModelNotReadyError(
                details={
                    "component": (
                        "computer_vision"
                    ),
                    "reason": (
                        "Frozen model loading failed."
                    ),
                }
            ) from error

        model.to(self.device)
        model.eval()

        self.model_load_latency_ms = (
            (
                time.perf_counter()
                - load_start_time
            )
            * 1000.0
        )

        return model

    @property
    def is_ready(self) -> bool:
        """Report whether frozen inference is available."""

        return self._ready

    @staticmethod
    def confidence_category(
        probability: float,
        threshold: float,
    ) -> str:
        """Map probability distance to the frozen language bands."""

        margin = probability - threshold

        if margin < 0.0:
            return "below_threshold"

        if margin < 0.03:
            return "borderline"

        if margin < 0.10:
            return "moderate"

        return "higher"

    def preprocess(
        self,
        image: Image.Image,
    ) -> torch.Tensor:
        """Create one normalized RGB model tensor."""

        rgb_image = image.convert("RGB")
        input_tensor = (
            self.preprocess_transform(
                rgb_image
            )
            .unsqueeze(0)
            .to(self.device)
        )

        return input_tensor

    def predict(
        self,
        validated_image: ValidatedImage,
    ) -> ClassificationResult:
        """Run one frozen multilabel prediction."""

        if not self.is_ready:
            raise ModelNotReadyError(
                details={
                    "component": (
                        "computer_vision"
                    )
                }
            )

        input_tensor = self.preprocess(
            validated_image.rgb_image
        )

        try:
            with self._inference_lock:
                if self.device.type == "cuda":
                    torch.cuda.synchronize()

                inference_start_time = (
                    time.perf_counter()
                )

                with torch.inference_mode():
                    with torch.autocast(
                        device_type=(
                            self.device.type
                        ),
                        dtype=torch.bfloat16,
                        enabled=(
                            self.device.type
                            == "cuda"
                            and torch.cuda
                            .is_bf16_supported()
                        ),
                    ):
                        logits = self.model(
                            input_tensor
                        )
                        probability_tensor = (
                            torch.sigmoid(
                                logits
                            )
                        )

                if self.device.type == "cuda":
                    torch.cuda.synchronize()

                inference_latency_ms = (
                    (
                        time.perf_counter()
                        - inference_start_time
                    )
                    * 1000.0
                )

        except Exception as error:
            raise ServiceExecutionError(
                message=(
                    "Computer-vision inference failed."
                ),
                details={
                    "component": (
                        "computer_vision"
                    )
                },
            ) from error

        probabilities = (
            probability_tensor
            .detach()
            .float()
            .cpu()
            .numpy()[0]
        )

        decisions = (
            probabilities
            >= self.thresholds
        )

        finding_results = [
            FindingEvidence(
                label_id=int(
                    finding["label_id"]
                ),
                label_name=(
                    finding["label_name"]
                ),
                display_name=(
                    finding["display_name"]
                ),
                probability=float(
                    probabilities[
                        finding["label_id"]
                    ]
                ),
                frozen_threshold=float(
                    finding[
                        "frozen_threshold"
                    ]
                ),
                crossed_threshold=bool(
                    decisions[
                        finding["label_id"]
                    ]
                ),
                confidence_category=(
                    self.confidence_category(
                        float(
                            probabilities[
                                finding[
                                    "label_id"
                                ]
                            ]
                        ),
                        float(
                            finding[
                                "frozen_threshold"
                            ]
                        ),
                    )
                ),
                approved_description=(
                    finding[
                        "approved_description"
                    ]
                ),
            )
            for finding in self.findings
        ]

        crossed_finding_names = [
            finding.label_name
            for finding in finding_results
            if finding.crossed_threshold
        ]

        no_target_finding = (
            len(crossed_finding_names)
            == 0
        )

        if no_target_finding:
            interpretation = (
                self.controlled_language[
                    "no_target_finding"
                ]
            )
        else:
            finding_count = len(
                crossed_finding_names
            )
            interpretation = (
                f"{finding_count} of the 14 supported "
                f"findings crossed their frozen "
                f"decision thresholds. "
                f"{self.controlled_language['educational_use_limitation']}"
            )

        return ClassificationResult(
            findings=finding_results,
            crossed_finding_names=(
                crossed_finding_names
            ),
            no_target_finding=(
                no_target_finding
            ),
            interpretation=interpretation,
            inference_latency_ms=(
                inference_latency_ms
            ),
            probabilities=probabilities,
            logits=logits.detach(),
            input_tensor=input_tensor.detach(),
        )
'''

COMPUTER_VISION_SERVICE_PATH.write_text(
    COMPUTER_VISION_SERVICE_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()
sys.modules.pop(
    "src.services.computer_vision_service",
    None,
)

computer_vision_service_module = (
    importlib.import_module(
        "src.services.computer_vision_service"
    )
)

ComputerVisionService = (
    computer_vision_service_module.ComputerVisionService
)
ClassificationResult = (
    computer_vision_service_module.ClassificationResult
)


# -------------------------------------------------------------------------
# Load the frozen model and perform a structural smoke test
# -------------------------------------------------------------------------
computer_vision_service = (
    ComputerVisionService(
        settings=service_settings
    )
)

structural_smoke_result = (
    computer_vision_service.predict(
        validated_png
    )
)


# -------------------------------------------------------------------------
# Validate model structure and response construction
# -------------------------------------------------------------------------
loaded_parameter_count = sum(
    parameter.numel()
    for parameter
    in computer_vision_service
    .model.parameters()
)

cv_service_checksum = hashlib.sha256(
    COMPUTER_VISION_SERVICE_PATH.read_bytes()
).hexdigest()

cv_service_checks = {
    "Computer-vision service module was written": (
        COMPUTER_VISION_SERVICE_PATH.is_file()
    ),
    "Frozen model loaded successfully": (
        computer_vision_service.is_ready
    ),
    "Model is in evaluation mode": (
        not computer_vision_service
        .model.training
    ),
    "Model executes on the GPU": (
        computer_vision_service
        .device.type
        == "cuda"
    ),
    "Model parameter count is preserved": (
        loaded_parameter_count
        == 11_183_694
    ),
    "Fourteen thresholds are loaded": (
        computer_vision_service
        .thresholds.shape
        == (14,)
    ),
    "Fourteen findings are returned": (
        len(
            structural_smoke_result.findings
        )
        == 14
    ),
    "All probabilities are bounded": all(
        0.0
        <= finding.probability
        <= 1.0
        for finding
        in structural_smoke_result.findings
    ),
    "Crossed names match decisions": (
        structural_smoke_result
        .crossed_finding_names
        == [
            finding.label_name
            for finding
            in structural_smoke_result.findings
            if finding.crossed_threshold
        ]
    ),
    "No-target state matches decisions": (
        structural_smoke_result
        .no_target_finding
        == (
            len(
                structural_smoke_result
                .crossed_finding_names
            )
            == 0
        )
    ),
    "Input tensor uses expected shape": (
        tuple(
            structural_smoke_result
            .input_tensor.shape
        )
        == (1, 3, 224, 224)
    ),
    "Inference latency is positive": (
        structural_smoke_result
        .inference_latency_ms
        > 0.0
    ),
    "Service checksum is available": (
        len(cv_service_checksum) == 64
    ),
}


# -------------------------------------------------------------------------
# Report frozen classifier service readiness
# -------------------------------------------------------------------------
print("FROZEN COMPUTER-VISION INFERENCE SERVICE")
print("-" * 100)
print(
    f"Service module            : "
    f"{COMPUTER_VISION_SERVICE_PATH}"
)
print(
    f"Module SHA-256            : "
    f"{cv_service_checksum[:16]}..."
)
print(
    f"Execution device          : "
    f"{computer_vision_service.device}"
)
print(
    f"Model parameters          : "
    f"{loaded_parameter_count:,}"
)
print(
    f"Model load latency        : "
    f"{computer_vision_service.model_load_latency_ms:.2f} ms"
)
print(
    f"Smoke inference latency   : "
    f"{structural_smoke_result.inference_latency_ms:.2f} ms"
)
print(
    f"Crossed findings          : "
    f"{structural_smoke_result.crossed_finding_names}"
)
print(
    f"No-target-finding state   : "
    f"{structural_smoke_result.no_target_finding}"
)
print("-" * 100)

for check_name, passed in cv_service_checks.items():
    print(
        f"{check_name:<60}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(cv_service_checks.values()):
    failed_checks = [
        name
        for name, passed
        in cv_service_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Frozen computer-vision service validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: COMPUTER-VISION SERVICE READY FOR INFERENCE PARITY VALIDATION")

FROZEN COMPUTER-VISION INFERENCE SERVICE
----------------------------------------------------------------------------------------------------
Service module            : /home/jovyan/chest-xray-ai-assistant/src/services/computer_vision_service.py
Module SHA-256            : 8602a7810c5ce9fe...
Execution device          : cuda
Model parameters          : 11,183,694
Model load latency        : 2301.47 ms
Smoke inference latency   : 328.31 ms
Crossed findings          : []
No-target-finding state   : True
----------------------------------------------------------------------------------------------------
Computer-vision service module was written                  : PASS
Frozen model loaded successfully                            : PASS
Model is in evaluation mode                                 : PASS
Model executes on the GPU                                   : PASS
Model parameter count is preserved                          : PASS
Fourteen thresholds are loaded                          

<a id="nb07-5-4-isolated-inference-parity-validation"></a>
### 5.4 Isolated Inference Parity Validation

A single previously evaluated ChestMNIST test image is reconstructed from the memory-mapped test array and passed through the complete API ingestion and computer-vision service path. Its probabilities and threshold decisions are compared with the frozen Notebook 4 prediction bundle.

This is a parity check against an already frozen artifact, not a new model-selection or threshold-calibration step. The expected contract is bfloat16 probability agreement within 0.0025 and complete binary-decision agreement.


In [26]:
import numpy as np
import pandas as pd


# -------------------------------------------------------------------------
# Load the frozen isolated inference artifacts
# -------------------------------------------------------------------------
PARITY_SOURCE_INDEX = 1690

TEST_IMAGE_MEMMAP_PATH = (
    DATA_ROOT
    / "processed"
    / "chestmnist_224_memmap"
    / "test_images.npy"
)

TEST_PREDICTION_BUNDLE_PATH = (
    DATA_ROOT
    / "outputs"
    / "metrics"
    / "test_prediction_bundle.npz"
)

if not TEST_IMAGE_MEMMAP_PATH.is_file():
    raise FileNotFoundError(
        f"Test image memory map was not found: "
        f"{TEST_IMAGE_MEMMAP_PATH}"
    )

if not TEST_PREDICTION_BUNDLE_PATH.is_file():
    raise FileNotFoundError(
        f"Frozen prediction bundle was not found: "
        f"{TEST_PREDICTION_BUNDLE_PATH}"
    )

test_image_memmap = np.load(
    TEST_IMAGE_MEMMAP_PATH,
    mmap_mode="r",
)

with np.load(
    TEST_PREDICTION_BUNDLE_PATH
) as prediction_bundle:
    frozen_source_indices = (
        prediction_bundle[
            "source_indices"
        ].copy()
    )
    frozen_test_probabilities = (
        prediction_bundle[
            "probabilities"
        ].copy()
    )
    frozen_test_predictions = (
        prediction_bundle[
            "predictions"
        ].copy()
    )
    frozen_bundle_thresholds = (
        prediction_bundle[
            "thresholds"
        ].copy()
    )


# -------------------------------------------------------------------------
# Resolve and reconstruct the selected parity image
# -------------------------------------------------------------------------
parity_bundle_positions = np.flatnonzero(
    frozen_source_indices
    == PARITY_SOURCE_INDEX
)

if len(parity_bundle_positions) != 1:
    raise RuntimeError(
        "The parity source index must appear "
        "exactly once in the frozen bundle."
    )

parity_bundle_position = int(
    parity_bundle_positions[0]
)

parity_image_array = np.asarray(
    test_image_memmap[
        PARITY_SOURCE_INDEX
    ]
).squeeze()

if parity_image_array.shape != (
    224,
    224,
):
    raise ValueError(
        "The parity image does not have the "
        "expected 224 x 224 shape."
    )

parity_image_array = (
    parity_image_array.astype(
        np.uint8,
        copy=False,
    )
)

parity_pil_image = Image.fromarray(
    parity_image_array,
    mode="L",
)

parity_png_buffer = BytesIO()
parity_pil_image.save(
    parity_png_buffer,
    format="PNG",
)

parity_validated_image = (
    image_validation_service
    .validate_and_decode(
        filename=(
            f"chestmnist-test-"
            f"{PARITY_SOURCE_INDEX}.png"
        ),
        media_type="image/png",
        content=(
            parity_png_buffer.getvalue()
        ),
    )
)


# -------------------------------------------------------------------------
# Execute the complete API computer-vision path
# -------------------------------------------------------------------------
parity_service_result = (
    computer_vision_service.predict(
        parity_validated_image
    )
)

service_probabilities = np.asarray(
    [
        finding.probability
        for finding
        in parity_service_result.findings
    ],
    dtype=np.float32,
)

service_predictions = np.asarray(
    [
        finding.crossed_threshold
        for finding
        in parity_service_result.findings
    ],
    dtype=bool,
)

reference_probabilities = (
    frozen_test_probabilities[
        parity_bundle_position
    ].astype(
        np.float32,
        copy=False,
    )
)

reference_predictions = (
    frozen_test_predictions[
        parity_bundle_position
    ].astype(
        bool,
        copy=False,
    )
)

probability_absolute_difference = (
    np.abs(
        service_probabilities
        - reference_probabilities
    )
)

maximum_probability_difference = float(
    probability_absolute_difference.max()
)

mean_probability_difference = float(
    probability_absolute_difference.mean()
)

decision_agreement = float(
    np.mean(
        service_predictions
        == reference_predictions
    )
)

threshold_maximum_difference = float(
    np.max(
        np.abs(
            computer_vision_service
            .thresholds
            - frozen_bundle_thresholds
        )
    )
)


# -------------------------------------------------------------------------
# Build the per-finding parity table
# -------------------------------------------------------------------------
parity_rows = []

for label_id, label_name in enumerate(
    CHESTMNIST_LABEL_NAMES
):
    parity_rows.append(
        {
            "label_id": label_id,
            "label_name": label_name,
            "service_probability": float(
                service_probabilities[
                    label_id
                ]
            ),
            "reference_probability": float(
                reference_probabilities[
                    label_id
                ]
            ),
            "absolute_difference": float(
                probability_absolute_difference[
                    label_id
                ]
            ),
            "service_decision": bool(
                service_predictions[
                    label_id
                ]
            ),
            "reference_decision": bool(
                reference_predictions[
                    label_id
                ]
            ),
        }
    )

cv_inference_parity_df = pd.DataFrame(
    parity_rows
)

CV_INFERENCE_PARITY_PATH = (
    API_OUTPUT_DIR
    / "computer_vision_inference_parity.csv"
)

cv_inference_parity_df.to_csv(
    CV_INFERENCE_PARITY_PATH,
    index=False,
)


# -------------------------------------------------------------------------
# Validate the frozen inference contract
# -------------------------------------------------------------------------
parity_checks = {
    "Parity image was read through memory mapping": (
        isinstance(
            test_image_memmap,
            np.memmap,
        )
    ),
    "Source index appears exactly once": (
        len(parity_bundle_positions)
        == 1
    ),
    "API ingestion preserves image dimensions": (
        parity_validated_image.width
        == 224
        and parity_validated_image.height
        == 224
    ),
    "Fourteen service probabilities are available": (
        service_probabilities.shape
        == (14,)
    ),
    "Frozen thresholds match exactly": (
        threshold_maximum_difference
        <= 1e-7
    ),
    "Maximum probability difference is within tolerance": (
        maximum_probability_difference
        <= 0.0025
    ),
    "Binary prediction agreement is complete": (
        decision_agreement == 1.0
    ),
    "Crossed finding names preserve parity": (
        parity_service_result
        .crossed_finding_names
        == [
            CHESTMNIST_LABEL_NAMES[
                label_id
            ]
            for label_id, decision
            in enumerate(
                reference_predictions
            )
            if decision
        ]
    ),
    "Parity artifact was exported": (
        CV_INFERENCE_PARITY_PATH.is_file()
    ),
}


# -------------------------------------------------------------------------
# Report isolated inference parity
# -------------------------------------------------------------------------
print("COMPUTER-VISION SERVICE INFERENCE PARITY")
print("-" * 100)
print(
    f"Source index              : "
    f"{PARITY_SOURCE_INDEX}"
)
print(
    f"Bundle position           : "
    f"{parity_bundle_position}"
)
print(
    f"Execution precision       : "
    f"bfloat16"
)
print(
    f"Maximum probability diff  : "
    f"{maximum_probability_difference:.6f}"
)
print(
    f"Mean probability diff     : "
    f"{mean_probability_difference:.6f}"
)
print(
    f"Threshold maximum diff    : "
    f"{threshold_maximum_difference:.8f}"
)
print(
    f"Binary decision agreement : "
    f"{decision_agreement * 100:.2f}%"
)
print(
    f"Service inference latency : "
    f"{parity_service_result.inference_latency_ms:.2f} ms"
)
print(
    f"Crossed findings          : "
    f"{parity_service_result.crossed_finding_names}"
)
print("-" * 100)
print(
    cv_inference_parity_df[
        [
            "label_name",
            "service_probability",
            "reference_probability",
            "absolute_difference",
            "service_decision",
            "reference_decision",
        ]
    ].to_string(
        index=False,
        formatters={
            "service_probability": (
                "{:.6f}".format
            ),
            "reference_probability": (
                "{:.6f}".format
            ),
            "absolute_difference": (
                "{:.6f}".format
            ),
        },
    )
)
print("-" * 100)

for check_name, passed in parity_checks.items():
    print(
        f"{check_name:<62}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(parity_checks.values()):
    failed_checks = [
        name
        for name, passed
        in parity_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Computer-vision inference parity failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: FROZEN COMPUTER-VISION INFERENCE CONTRACT PRESERVED")

COMPUTER-VISION SERVICE INFERENCE PARITY
----------------------------------------------------------------------------------------------------
Source index              : 1690
Bundle position           : 1690
Execution precision       : bfloat16
Maximum probability diff  : 0.001953
Mean probability diff     : 0.000140
Threshold maximum diff    : 0.00000000
Binary decision agreement : 100.00%
Service inference latency : 4.19 ms
Crossed findings          : ['atelectasis', 'effusion', 'consolidation']
----------------------------------------------------------------------------------------------------
   label_name service_probability reference_probability absolute_difference  service_decision  reference_decision
  atelectasis            0.972656              0.972656            0.000000              True                True
 cardiomegaly            0.503906              0.503906            0.000000             False               False
     effusion            0.968750              0.96875

<a id="nb07-5-5-thread-safe-prediction-store"></a>
### 5.5 Thread-Safe Prediction Store

Language endpoints receive only a prediction identifier, so classification evidence must be retained by the service. A bounded in-memory store maintains validated image metadata, all finding decisions, optional Grad-CAM evidence, and guarded language outputs.

The prototype store uses thread locking, a 24-hour retention window, and a maximum-record limit to prevent unbounded growth. It supports classification creation, visual-evidence attachment, language-output upsert, safe retrieval, stale-record cleanup, and controlled not-found errors. A persistent database can replace this implementation later without changing the API contracts.


In [28]:
from datetime import datetime, timedelta, timezone

# -------------------------------------------------------------------------
# Define the bounded thread-safe prediction store
# -------------------------------------------------------------------------
PREDICTION_STORE_SERVICE_PATH = (
    SERVICE_ROOT
    / "prediction_store_service.py"
)

PREDICTION_STORE_SERVICE_SOURCE = '''"""Bounded thread-safe in-memory prediction storage."""

import threading
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from typing import Any
from uuid import UUID, uuid4

from api.core.errors import PredictionNotFoundError
from api.schemas.aggregate import (
    EmbeddedLanguageOutput,
    StoredPredictionResponse,
)
from api.schemas.explainability import (
    ExplainabilityContract,
    GradCAMEvidence,
)
from api.schemas.prediction import (
    FindingEvidence,
    ImageMetadata,
)


@dataclass
class PredictionRecord:
    """Internal mutable state for one validated prediction."""

    prediction_id: UUID
    created_at_utc: datetime
    image: ImageMetadata
    findings: list[FindingEvidence]
    crossed_finding_names: list[str]
    no_target_finding: bool
    interpretation: str
    explainability: ExplainabilityContract | None = None
    visual_evidence: list[GradCAMEvidence] = field(
        default_factory=list
    )
    language_outputs: list[
        EmbeddedLanguageOutput
    ] = field(default_factory=list)


class PredictionStoreService:
    """Retain a bounded set of prediction results in memory."""

    def __init__(
        self,
        *,
        maximum_records: int = 1000,
        retention_hours: int = 24,
    ) -> None:
        if maximum_records <= 0:
            raise ValueError(
                "Maximum records must be positive."
            )

        if retention_hours <= 0:
            raise ValueError(
                "Retention hours must be positive."
            )

        self.maximum_records = maximum_records
        self.retention = timedelta(
            hours=retention_hours
        )
        self._records: dict[
            UUID,
            PredictionRecord,
        ] = {}
        self._lock = threading.RLock()

    @staticmethod
    def utc_now() -> datetime:
        return datetime.now(timezone.utc)

    def _remove_stale_records(
        self,
        current_time: datetime,
    ) -> int:
        stale_ids = [
            prediction_id
            for prediction_id, record
            in self._records.items()
            if (
                current_time
                - record.created_at_utc
                > self.retention
            )
        ]

        for prediction_id in stale_ids:
            self._records.pop(
                prediction_id,
                None,
            )

        return len(stale_ids)

    def _enforce_capacity(self) -> None:
        while len(self._records) >= (
            self.maximum_records
        ):
            oldest_prediction_id = min(
                self._records,
                key=lambda prediction_id: (
                    self._records[
                        prediction_id
                    ].created_at_utc
                ),
            )
            self._records.pop(
                oldest_prediction_id,
                None,
            )

    def create(
        self,
        *,
        image: ImageMetadata,
        findings: list[FindingEvidence],
        crossed_finding_names: list[str],
        no_target_finding: bool,
        interpretation: str,
        prediction_id: UUID | None = None,
        created_at_utc: datetime | None = None,
    ) -> PredictionRecord:
        """Create and retain one classification record."""

        with self._lock:
            current_time = (
                created_at_utc
                or self.utc_now()
            )

            self._remove_stale_records(
                current_time
            )
            self._enforce_capacity()

            resolved_prediction_id = (
                prediction_id or uuid4()
            )

            record = PredictionRecord(
                prediction_id=(
                    resolved_prediction_id
                ),
                created_at_utc=current_time,
                image=image,
                findings=list(findings),
                crossed_finding_names=list(
                    crossed_finding_names
                ),
                no_target_finding=(
                    no_target_finding
                ),
                interpretation=interpretation,
            )

            self._records[
                resolved_prediction_id
            ] = record

            return record

    def get(
        self,
        prediction_id: UUID,
    ) -> PredictionRecord:
        """Return one retained prediction or a controlled not-found error."""

        with self._lock:
            self._remove_stale_records(
                self.utc_now()
            )

            record = self._records.get(
                prediction_id
            )

            if record is None:
                raise PredictionNotFoundError(
                    details={
                        "prediction_id": str(
                            prediction_id
                        )
                    }
                )

            return record

    def attach_visual_evidence(
        self,
        *,
        prediction_id: UUID,
        explainability: ExplainabilityContract,
        visual_evidence: list[GradCAMEvidence],
    ) -> PredictionRecord:
        """Attach validated Grad-CAM evidence."""

        with self._lock:
            record = self.get(
                prediction_id
            )
            record.explainability = (
                explainability
            )
            record.visual_evidence = list(
                visual_evidence
            )
            return record

    def upsert_language_output(
        self,
        *,
        prediction_id: UUID,
        language_output: EmbeddedLanguageOutput,
    ) -> PredictionRecord:
        """Insert or replace one task-specific language output."""

        with self._lock:
            record = self.get(
                prediction_id
            )

            retained_outputs = [
                output
                for output
                in record.language_outputs
                if output.task_type
                != language_output.task_type
            ]

            retained_outputs.append(
                language_output
            )

            record.language_outputs = (
                retained_outputs
            )

            return record

    def build_response(
        self,
        *,
        prediction_id: UUID,
        response_metadata: dict[str, Any],
    ) -> StoredPredictionResponse:
        """Build the strict retrievable prediction response."""

        record = self.get(
            prediction_id
        )

        return StoredPredictionResponse(
            **response_metadata,
            prediction_id=(
                record.prediction_id
            ),
            created_at_utc=(
                record.created_at_utc
            ),
            image=record.image,
            findings=record.findings,
            crossed_finding_names=(
                record.crossed_finding_names
            ),
            no_target_finding=(
                record.no_target_finding
            ),
            interpretation=(
                record.interpretation
            ),
            explainability=(
                record.explainability
            ),
            visual_evidence=(
                record.visual_evidence
            ),
            language_outputs=(
                record.language_outputs
            ),
        )

    def snapshot_metrics(
        self,
    ) -> dict[str, int]:
        """Return non-sensitive store measurements."""

        with self._lock:
            removed_stale_records = (
                self._remove_stale_records(
                    self.utc_now()
                )
            )

            return {
                "active_records": len(
                    self._records
                ),
                "maximum_records": (
                    self.maximum_records
                ),
                "removed_stale_records": (
                    removed_stale_records
                ),
            }
'''

PREDICTION_STORE_SERVICE_PATH.write_text(
    PREDICTION_STORE_SERVICE_SOURCE,
    encoding="utf-8",
)

importlib.invalidate_caches()
sys.modules.pop(
    "src.services.prediction_store_service",
    None,
)

prediction_store_module = (
    importlib.import_module(
        "src.services.prediction_store_service"
    )
)

PredictionStoreService = (
    prediction_store_module.PredictionStoreService
)
PredictionRecord = (
    prediction_store_module.PredictionRecord
)

prediction_store_service = (
    PredictionStoreService(
        maximum_records=1000,
        retention_hours=24,
    )
)


# -------------------------------------------------------------------------
# Create and enrich one stored prediction
# -------------------------------------------------------------------------
stored_test_record = (
    prediction_store_service.create(
        image=png_metadata,
        findings=schema_test_findings,
        crossed_finding_names=[
            "atelectasis"
        ],
        no_target_finding=False,
        interpretation=(
            "One supported finding crossed "
            "its frozen threshold."
        ),
    )
)

prediction_store_service.attach_visual_evidence(
    prediction_id=(
        stored_test_record.prediction_id
    ),
    explainability=(
        explainability_contract_example
    ),
    visual_evidence=[
        gradcam_evidence_example
    ],
)

prediction_store_service.upsert_language_output(
    prediction_id=(
        stored_test_record.prediction_id
    ),
    language_output=embedded_report,
)

stored_response_metadata = (
    build_success_metadata(
        RequestContext(),
        settings=service_settings,
        include_language=True,
        include_explainability=True,
    )
)

stored_response_example = (
    prediction_store_service.build_response(
        prediction_id=(
            stored_test_record.prediction_id
        ),
        response_metadata=(
            stored_response_metadata
        ),
    )
)


# -------------------------------------------------------------------------
# Verify not-found and bounded-capacity behavior
# -------------------------------------------------------------------------
missing_prediction_rejected = False

try:
    prediction_store_service.get(
        uuid4()
    )
except PredictionNotFoundError:
    missing_prediction_rejected = True

bounded_store = PredictionStoreService(
    maximum_records=2,
    retention_hours=24,
)

bounded_record_ids = []

for record_number in range(3):
    bounded_record = bounded_store.create(
        image=png_metadata,
        findings=schema_test_findings,
        crossed_finding_names=[
            "atelectasis"
        ],
        no_target_finding=False,
        interpretation=(
            f"Capacity test record "
            f"{record_number}."
        ),
        created_at_utc=(
            datetime.now(
                timezone.utc
            )
            + timedelta(
                microseconds=record_number
            )
        ),
    )
    bounded_record_ids.append(
        bounded_record.prediction_id
    )

oldest_record_evicted = False

try:
    bounded_store.get(
        bounded_record_ids[0]
    )
except PredictionNotFoundError:
    oldest_record_evicted = True

bounded_metrics = (
    bounded_store.snapshot_metrics()
)


# -------------------------------------------------------------------------
# Validate and report prediction storage readiness
# -------------------------------------------------------------------------
prediction_store_checksum = hashlib.sha256(
    PREDICTION_STORE_SERVICE_PATH.read_bytes()
).hexdigest()

prediction_store_checks = {
    "Prediction store module was written": (
        PREDICTION_STORE_SERVICE_PATH.is_file()
    ),
    "Stored prediction identifier is preserved": (
        stored_response_example.prediction_id
        == stored_test_record.prediction_id
    ),
    "Stored image metadata is preserved": (
        stored_response_example.image.sha256
        == png_metadata.sha256
    ),
    "Stored finding evidence is complete": (
        len(
            stored_response_example.findings
        )
        == 14
    ),
    "Visual evidence can be attached": (
        len(
            stored_response_example
            .visual_evidence
        )
        == 1
    ),
    "Language output can be upserted": (
        len(
            stored_response_example
            .language_outputs
        )
        == 1
    ),
    "Missing prediction raises controlled error": (
        missing_prediction_rejected
    ),
    "Oldest record is evicted at capacity": (
        oldest_record_evicted
    ),
    "Bounded store retains two records": (
        bounded_metrics[
            "active_records"
        ]
        == 2
    ),
    "Store checksum is available": (
        len(prediction_store_checksum)
        == 64
    ),
}

print("THREAD-SAFE PREDICTION STORE SERVICE")
print("-" * 100)
print(
    f"Service module            : "
    f"{PREDICTION_STORE_SERVICE_PATH}"
)
print(
    f"Module SHA-256            : "
    f"{prediction_store_checksum[:16]}..."
)
print(
    f"Retention period          : "
    f"24 hours"
)
print(
    f"Maximum records           : "
    f"{prediction_store_service.maximum_records}"
)
print(
    f"Stored prediction ID      : "
    f"{stored_test_record.prediction_id}"
)
print(
    f"Stored visual evidence    : "
    f"{len(stored_response_example.visual_evidence)}"
)
print(
    f"Stored language outputs   : "
    f"{len(stored_response_example.language_outputs)}"
)
print("-" * 100)

for check_name, passed in prediction_store_checks.items():
    print(
        f"{check_name:<60}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(prediction_store_checks.values()):
    failed_checks = [
        name
        for name, passed
        in prediction_store_checks.items()
        if not passed
    ]
    raise RuntimeError(
        "Prediction store validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print("STATUS: PREDICTION STORE READY FOR ROUTE INTEGRATION")

THREAD-SAFE PREDICTION STORE SERVICE
----------------------------------------------------------------------------------------------------
Service module            : /home/jovyan/chest-xray-ai-assistant/src/services/prediction_store_service.py
Module SHA-256            : 2c7a1fb92b89e210...
Retention period          : 24 hours
Maximum records           : 1000
Stored prediction ID      : f351e158-3d23-4237-bc6a-8b60224cbd3a
Stored visual evidence    : 1
Stored language outputs   : 1
----------------------------------------------------------------------------------------------------
Prediction store module was written                         : PASS
Stored prediction identifier is preserved                   : PASS
Stored image metadata is preserved                          : PASS
Stored finding evidence is complete                         : PASS
Visual evidence can be attached                             : PASS
Language output can be upserted                             : PASS
Missing pr

<a id="nb07-5-6-thread-safe-grad-cam-explainability-service"></a>
### 5.6 Thread-Safe Grad-CAM Explainability Service

This block converts the frozen computer-vision model’s internal activation gradients into controlled visual evidence. The service uses the approved `LayerGradCam` method and frozen `layer4[-1].conv2` target layer, returns PNG-encoded heatmap and overlay images, and preserves the educational boundary that Grad-CAM indicates model attention rather than confirming anatomical lesions.


In [30]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import base64
import hashlib
import importlib
import io
import sys
import threading
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Sequence

import numpy as np
import torch
import torch.nn.functional as torch_functional
import yaml
from captum.attr import LayerGradCam
from PIL import Image
from torchvision import transforms


# -------------------------------------------------------------------------
# Define the service-module destination
# -------------------------------------------------------------------------
gradcam_service_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "src/services/gradcam_service.py"
)

finding_contract_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "configs/finding_contract.yaml"
)

gradcam_service_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Write the reusable Grad-CAM service
# -------------------------------------------------------------------------
gradcam_service_source = '''
"""Thread-safe visual explainability for the frozen ChestMNIST model."""

from __future__ import annotations

import base64
import io
import threading
import time
from dataclasses import dataclass
from typing import Any, Mapping, Sequence

import numpy as np
import torch
import torch.nn.functional as functional
from captum.attr import LayerGradCam
from PIL import Image
from torchvision import transforms


@dataclass(frozen=True)
class GradCAMResult:
    """Controlled visual evidence for one model-output label."""

    label_id: int
    label_name: str
    probability: float
    frozen_threshold: float
    threshold_decision: bool
    method: str
    target_layer: str
    heatmap_png_base64: str
    overlay_png_base64: str
    generation_latency_ms: float
    limitation: str


class GradCAMService:
    """Generate visual evidence from the frozen classification model."""

    METHOD = "LayerGradCam"
    TARGET_LAYER = "layer4[-1].conv2"

    def __init__(
        self,
        *,
        model: torch.nn.Module,
        device: torch.device | str,
        limitation: str,
        image_size: int = 224,
        overlay_alpha: float = 0.40,
    ) -> None:
        if not limitation.strip():
            raise ValueError(
                "A Grad-CAM limitation statement is required."
            )

        if image_size <= 0:
            raise ValueError(
                "The model-facing image size must be positive."
            )

        if not 0.0 <= overlay_alpha <= 1.0:
            raise ValueError(
                "The overlay alpha must be between zero and one."
            )

        self.model = model
        self.device = torch.device(device)
        self.limitation = limitation.strip()
        self.image_size = int(image_size)
        self.overlay_alpha = float(overlay_alpha)
        self._lock = threading.RLock()

        try:
            self.target_layer = (
                self.model.layer4[-1].conv2
            )
        except (AttributeError, IndexError, TypeError) as error:
            raise ValueError(
                "The frozen Grad-CAM target layer "
                "layer4[-1].conv2 is unavailable."
            ) from error

        self._preprocess = transforms.Compose(
            [
                transforms.Resize(
                    (self.image_size, self.image_size)
                ),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[
                        0.485,
                        0.456,
                        0.406,
                    ],
                    std=[
                        0.229,
                        0.224,
                        0.225,
                    ],
                ),
            ]
        )

        self.model.to(self.device)
        self.model.eval()

        self._layer_gradcam = LayerGradCam(
            self.model,
            self.target_layer,
        )

    @property
    def model_dtype(self) -> torch.dtype:
        """Return the dtype used by the model parameters."""

        return next(
            self.model.parameters()
        ).dtype

    @staticmethod
    def _read_value(
        record: Any,
        *names: str,
    ) -> Any:
        """Read a field from either a mapping or an object."""

        for name in names:
            if isinstance(record, Mapping):
                if name in record:
                    return record[name]
            elif hasattr(record, name):
                return getattr(record, name)

        raise KeyError(
            "Required finding field was not available: "
            + ", ".join(names)
        )

    @staticmethod
    def _encode_png(image: Image.Image) -> str:
        """Encode an image as a base64 PNG string."""

        buffer = io.BytesIO()
        image.save(
            buffer,
            format="PNG",
            optimize=True,
        )

        return base64.b64encode(
            buffer.getvalue()
        ).decode("ascii")

    @staticmethod
    def _normalize_attribution(
        attribution: torch.Tensor,
    ) -> np.ndarray:
        """Normalize attribution values into the zero-to-one range."""

        attribution = attribution.detach().float()

        if attribution.ndim != 4:
            raise RuntimeError(
                "Grad-CAM attribution must contain four dimensions."
            )

        if attribution.shape[1] > 1:
            attribution = attribution.mean(
                dim=1,
                keepdim=True,
            )

        attribution = attribution.squeeze(
            dim=0
        ).squeeze(
            dim=0
        )

        attribution = torch.clamp(
            attribution,
            min=0.0,
        )

        minimum = attribution.min()
        maximum = attribution.max()

        if not torch.isfinite(
            attribution
        ).all():
            raise RuntimeError(
                "Grad-CAM produced non-finite attribution values."
            )

        value_range = maximum - minimum

        if float(value_range.item()) <= 1e-12:
            normalized = torch.zeros_like(
                attribution
            )
        else:
            normalized = (
                attribution - minimum
            ) / value_range

        return normalized.cpu().numpy()

    @staticmethod
    def _build_heatmap(
        normalized_attribution: np.ndarray,
    ) -> Image.Image:
        """Convert normalized attribution into a controlled RGB heatmap."""

        attribution = np.clip(
            normalized_attribution,
            0.0,
            1.0,
        )

        red = attribution
        green = np.sqrt(
            attribution
        )
        blue = np.clip(
            1.0 - (2.0 * attribution),
            0.0,
            1.0,
        )

        heatmap_array = np.stack(
            [
                red,
                green,
                blue,
            ],
            axis=-1,
        )

        heatmap_uint8 = np.round(
            heatmap_array * 255.0
        ).astype(
            np.uint8
        )

        return Image.fromarray(
            heatmap_uint8,
            mode="RGB",
        )

    def _prepare_image(
        self,
        image: Image.Image,
    ) -> tuple[Image.Image, torch.Tensor]:
        """Create the display image and model-facing tensor."""

        if not isinstance(
            image,
            Image.Image,
        ):
            raise TypeError(
                "Grad-CAM input must be a decoded PIL image."
            )

        rgb_image = image.convert(
            "RGB"
        ).resize(
            (
                self.image_size,
                self.image_size,
            ),
            Image.Resampling.BILINEAR,
        )

        input_tensor = self._preprocess(
            rgb_image
        ).unsqueeze(
            dim=0
        )

        input_tensor = input_tensor.to(
            device=self.device,
            dtype=self.model_dtype,
        )

        input_tensor.requires_grad_(
            True
        )

        return rgb_image, input_tensor

    def generate(
        self,
        *,
        image: Image.Image,
        label_id: int,
        label_name: str,
        probability: float,
        frozen_threshold: float,
        threshold_decision: bool,
    ) -> GradCAMResult:
        """Generate visual evidence for one supplied model output."""

        if not 0 <= int(label_id) < 14:
            raise ValueError(
                "Grad-CAM label ID must be between zero and thirteen."
            )

        if not label_name.strip():
            raise ValueError(
                "Grad-CAM requires a finding name."
            )

        if not 0.0 <= float(probability) <= 1.0:
            raise ValueError(
                "Finding probability must be between zero and one."
            )

        if not 0.0 <= float(frozen_threshold) <= 1.0:
            raise ValueError(
                "Frozen threshold must be between zero and one."
            )

        expected_decision = (
            float(probability)
            >= float(frozen_threshold)
        )

        if bool(threshold_decision) != expected_decision:
            raise ValueError(
                "The supplied threshold decision does not match "
                "the probability and frozen threshold."
            )

        started_at = time.perf_counter()

        with self._lock:
            display_image, input_tensor = (
                self._prepare_image(
                    image
                )
            )

            self.model.eval()
            self.model.zero_grad(
                set_to_none=True
            )

            with torch.enable_grad():
                attribution = (
                    self._layer_gradcam.attribute(
                        input_tensor,
                        target=int(label_id),
                        relu_attributions=True,
                    )
                )

                attribution = (
                    functional.interpolate(
                        attribution.float(),
                        size=(
                            self.image_size,
                            self.image_size,
                        ),
                        mode="bilinear",
                        align_corners=False,
                    )
                )

            self.model.zero_grad(
                set_to_none=True
            )

        normalized_attribution = (
            self._normalize_attribution(
                attribution
            )
        )

        heatmap_image = self._build_heatmap(
            normalized_attribution
        )

        overlay_image = Image.blend(
            display_image,
            heatmap_image,
            alpha=self.overlay_alpha,
        )

        latency_ms = (
            time.perf_counter()
            - started_at
        ) * 1000.0

        return GradCAMResult(
            label_id=int(label_id),
            label_name=label_name.strip(),
            probability=float(probability),
            frozen_threshold=float(frozen_threshold),
            threshold_decision=bool(
                threshold_decision
            ),
            method=self.METHOD,
            target_layer=self.TARGET_LAYER,
            heatmap_png_base64=self._encode_png(
                heatmap_image
            ),
            overlay_png_base64=self._encode_png(
                overlay_image
            ),
            generation_latency_ms=float(
                latency_ms
            ),
            limitation=self.limitation,
        )

    def generate_for_crossed_findings(
        self,
        *,
        image: Image.Image,
        findings: Sequence[Any],
    ) -> tuple[GradCAMResult, ...]:
        """Generate evidence only for threshold-crossing findings."""

        results = []

        for finding in findings:
            decision = bool(
                self._read_value(
                    finding,
                    "threshold_decision",
                    "crossed_threshold",
                    "decision",
                )
            )

            if not decision:
                continue

            result = self.generate(
                image=image,
                label_id=int(
                    self._read_value(
                        finding,
                        "label_id",
                        "finding_id",
                    )
                ),
                label_name=str(
                    self._read_value(
                        finding,
                        "label_name",
                        "finding_name",
                    )
                ),
                probability=float(
                    self._read_value(
                        finding,
                        "probability",
                        "score",
                    )
                ),
                frozen_threshold=float(
                    self._read_value(
                        finding,
                        "frozen_threshold",
                        "threshold",
                    )
                ),
                threshold_decision=True,
            )

            results.append(
                result
            )

        return tuple(
            results
        )
'''

gradcam_service_path.write_text(
    gradcam_service_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Import the generated service module
# -------------------------------------------------------------------------
importlib.invalidate_caches()

module_name = (
    "src.services.gradcam_service"
)

if module_name in sys.modules:
    gradcam_service_module = importlib.reload(
        sys.modules[module_name]
    )
else:
    gradcam_service_module = importlib.import_module(
        module_name
    )

GradCAMService = (
    gradcam_service_module.GradCAMService
)

GradCAMResult = (
    gradcam_service_module.GradCAMResult
)


# -------------------------------------------------------------------------
# Load the approved explainability limitation
# -------------------------------------------------------------------------
# -------------------------------------------------------------------------
# Load the approved explainability limitation
# -------------------------------------------------------------------------
with finding_contract_path.open(
    "r",
    encoding="utf-8",
) as contract_file:
    finding_contract = yaml.safe_load(
        contract_file
    )


def find_gradcam_limitation(
    value,
    path=(),
):
    """Locate the approved Grad-CAM limitation across nested YAML fields."""

    if isinstance(value, dict):
        for key, nested_value in value.items():
            normalized_key = "".join(
                character.lower()
                for character in str(key)
                if character.isalnum()
            )

            current_path = (
                *path,
                normalized_key,
            )

            path_text = "".join(
                current_path
            )

            is_gradcam_boundary = (
                "gradcam" in path_text
                and (
                    "limitation" in path_text
                    or "boundary" in path_text
                )
            )

            if (
                is_gradcam_boundary
                and isinstance(
                    nested_value,
                    str,
                )
                and nested_value.strip()
            ):
                return nested_value.strip()

            located_value = find_gradcam_limitation(
                nested_value,
                current_path,
            )

            if located_value is not None:
                return located_value

    elif isinstance(value, list):
        for item in value:
            located_value = find_gradcam_limitation(
                item,
                path,
            )

            if located_value is not None:
                return located_value

    elif (
        isinstance(value, str)
        and value.strip()
        and "gradcam" in "".join(path)
        and (
            "limitation" in "".join(path)
            or "boundary" in "".join(path)
        )
    ):
        return value.strip()

    return None


gradcam_limitation = find_gradcam_limitation(
    finding_contract
)

if gradcam_limitation is None:
    existing_limitation = globals().get(
        "GRADCAM_LIMITATION"
    )

    if (
        isinstance(existing_limitation, str)
        and existing_limitation.strip()
    ):
        gradcam_limitation = (
            existing_limitation.strip()
        )

if not isinstance(
    gradcam_limitation,
    str,
) or not gradcam_limitation.strip():
    raise KeyError(
        "The approved Grad-CAM limitation could not be "
        "resolved from the frozen finding contract."
    )


# -------------------------------------------------------------------------
# Resolve the model-facing smoke-test image
# -------------------------------------------------------------------------
smoke_image = None

for image_attribute in (
    "model_image",
    "rgb_image",
    "image",
    "original_image",
):
    candidate_image = getattr(
        validated_png,
        image_attribute,
        None,
    )

    if isinstance(
        candidate_image,
        Image.Image,
    ):
        smoke_image = candidate_image
        break

if smoke_image is None:
    raise RuntimeError(
        "A decoded PIL image was not available in validated_png."
    )


# -------------------------------------------------------------------------
# Resolve the frozen model and device
# -------------------------------------------------------------------------
frozen_model = getattr(
    computer_vision_service,
    "model",
    None,
)

frozen_device = getattr(
    computer_vision_service,
    "device",
    None,
)

if frozen_model is None:
    raise RuntimeError(
        "The frozen model was not available from "
        "computer_vision_service."
    )

if frozen_device is None:
    frozen_device = next(
        frozen_model.parameters()
    ).device


# -------------------------------------------------------------------------
# Initialize the Grad-CAM service
# -------------------------------------------------------------------------
gradcam_service = GradCAMService(
    model=frozen_model,
    device=frozen_device,
    limitation=gradcam_limitation,
    image_size=224,
    overlay_alpha=0.40,
)


# -------------------------------------------------------------------------
# Select one finding for an algorithm-level smoke test
# -------------------------------------------------------------------------
smoke_findings = getattr(
    structural_smoke_result,
    "findings",
    None,
)

if smoke_findings is None:
    smoke_findings = getattr(
        structural_smoke_result,
        "finding_records",
        None,
    )

if not smoke_findings:
    raise RuntimeError(
        "The structural classification result did not "
        "contain finding records."
    )


def read_finding_value(
    finding,
    *field_names,
):
    for field_name in field_names:
        if isinstance(
            finding,
            dict,
        ) and field_name in finding:
            return finding[field_name]

        if hasattr(
            finding,
            field_name,
        ):
            return getattr(
                finding,
                field_name,
            )

    raise KeyError(
        "Finding field was not available: "
        + ", ".join(field_names)
    )


smoke_finding = max(
    smoke_findings,
    key=lambda finding: float(
        read_finding_value(
            finding,
            "probability",
            "score",
        )
    ),
)

smoke_label_id = int(
    read_finding_value(
        smoke_finding,
        "label_id",
        "finding_id",
    )
)

smoke_label_name = str(
    read_finding_value(
        smoke_finding,
        "label_name",
        "finding_name",
    )
)

smoke_probability = float(
    read_finding_value(
        smoke_finding,
        "probability",
        "score",
    )
)

smoke_threshold = float(
    read_finding_value(
        smoke_finding,
        "frozen_threshold",
        "threshold",
    )
)

smoke_decision = (
    smoke_probability
    >= smoke_threshold
)


# -------------------------------------------------------------------------
# Generate and validate visual evidence
# -------------------------------------------------------------------------
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

gradcam_smoke_result = (
    gradcam_service.generate(
        image=smoke_image,
        label_id=smoke_label_id,
        label_name=smoke_label_name,
        probability=smoke_probability,
        frozen_threshold=smoke_threshold,
        threshold_decision=smoke_decision,
    )
)

peak_gradcam_gpu_gib = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 3)
    if torch.cuda.is_available()
    else 0.0
)


def decode_and_validate_png(
    encoded_value,
):
    try:
        decoded_bytes = base64.b64decode(
            encoded_value,
            validate=True,
        )

        with Image.open(
            io.BytesIO(decoded_bytes)
        ) as decoded_image:
            decoded_image.load()

            return (
                decoded_image.format == "PNG"
                and decoded_image.size
                == (224, 224)
            )
    except Exception:
        return False


heatmap_is_valid_png = (
    decode_and_validate_png(
        gradcam_smoke_result.heatmap_png_base64
    )
)

overlay_is_valid_png = (
    decode_and_validate_png(
        gradcam_smoke_result.overlay_png_base64
    )
)

crossed_smoke_evidence = (
    gradcam_service.generate_for_crossed_findings(
        image=smoke_image,
        findings=smoke_findings,
    )
)

expected_crossed_count = sum(
    bool(
        read_finding_value(
            finding,
            "threshold_decision",
            "crossed_threshold",
            "decision",
        )
    )
    for finding in smoke_findings
)

service_checksum = hashlib.sha256(
    gradcam_service_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Validate the explainability service
# -------------------------------------------------------------------------
gradcam_checks = {
    (
        "Grad-CAM service module was written"
    ): gradcam_service_path.is_file(),
    (
        "Method is frozen to LayerGradCam"
    ): (
        gradcam_smoke_result.method
        == "LayerGradCam"
    ),
    (
        "Target layer is preserved"
    ): (
        gradcam_smoke_result.target_layer
        == "layer4[-1].conv2"
        and gradcam_service.target_layer
        is frozen_model.layer4[-1].conv2
    ),
    (
        "Heatmap is a valid PNG"
    ): heatmap_is_valid_png,
    (
        "Overlay is a valid PNG"
    ): overlay_is_valid_png,
    (
        "Visual output dimensions are preserved"
    ): (
        heatmap_is_valid_png
        and overlay_is_valid_png
    ),
    (
        "Finding lineage is preserved"
    ): (
        gradcam_smoke_result.label_id
        == smoke_label_id
        and gradcam_smoke_result.label_name
        == smoke_label_name
    ),
    (
        "Probability and threshold are preserved"
    ): (
        gradcam_smoke_result.probability
        == smoke_probability
        and gradcam_smoke_result.frozen_threshold
        == smoke_threshold
    ),
    (
        "Threshold decision is preserved"
    ): (
        gradcam_smoke_result.threshold_decision
        == smoke_decision
    ),
    (
        "Only crossed findings receive endpoint evidence"
    ): (
        len(crossed_smoke_evidence)
        == expected_crossed_count
        and all(
            result.threshold_decision
            for result in crossed_smoke_evidence
        )
    ),
    (
        "Educational limitation is preserved"
    ): (
        gradcam_smoke_result.limitation
        == gradcam_limitation.strip()
    ),
    (
        "Generation latency is positive"
    ): (
        gradcam_smoke_result.generation_latency_ms
        > 0.0
    ),
    (
        "Model remains in evaluation mode"
    ): not frozen_model.training,
    (
        "Service checksum is available"
    ): len(service_checksum) == 64,
}


# -------------------------------------------------------------------------
# Present the validation summary
# -------------------------------------------------------------------------
print(
    "THREAD-SAFE GRAD-CAM EXPLAINABILITY SERVICE"
)
print("-" * 100)
print(
    f"{'Service module':27}: "
    f"{gradcam_service_path}"
)
print(
    f"{'Module SHA-256':27}: "
    f"{service_checksum[:16]}..."
)
print(
    f"{'Explainability method':27}: "
    f"{gradcam_smoke_result.method}"
)
print(
    f"{'Target layer':27}: "
    f"{gradcam_smoke_result.target_layer}"
)
print(
    f"{'Smoke-test finding':27}: "
    f"{gradcam_smoke_result.label_name}"
)
print(
    f"{'Threshold decision':27}: "
    f"{gradcam_smoke_result.threshold_decision}"
)
print(
    f"{'Generated evidence records':27}: "
    f"{len(crossed_smoke_evidence)}"
)
print(
    f"{'Generation latency':27}: "
    f"{gradcam_smoke_result.generation_latency_ms:.2f} ms"
)
print(
    f"{'Peak GPU memory':27}: "
    f"{peak_gradcam_gpu_gib:.2f} GiB"
)
print("-" * 100)

for check_name, passed in gradcam_checks.items():
    print(
        f"{check_name:58}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    gradcam_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in gradcam_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Grad-CAM service validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: GRAD-CAM SERVICE READY FOR API WORKFLOW INTEGRATION"
)

THREAD-SAFE GRAD-CAM EXPLAINABILITY SERVICE
----------------------------------------------------------------------------------------------------
Service module             : /home/jovyan/chest-xray-ai-assistant/src/services/gradcam_service.py
Module SHA-256             : 7238446e8d95ff89...
Explainability method      : LayerGradCam
Target layer               : layer4[-1].conv2
Smoke-test finding         : atelectasis
Threshold decision         : False
Generated evidence records : 0
Generation latency         : 1242.21 ms
Peak GPU memory            : 0.08 GiB
----------------------------------------------------------------------------------------------------
Grad-CAM service module was written                       : PASS
Method is frozen to LayerGradCam                          : PASS
Target layer is preserved                                 : PASS
Heatmap is a valid PNG                                    : PASS
Overlay is a valid PNG                                    : PASS
Visual ou

<a id="nb07-5-7-grounded-language-input-serialization-service"></a>
### 5.7 Grounded Language Input Serialization Service

This block creates the shared serialization layer used by every language endpoint. It converts stored computer-vision findings into the same ordered, threshold-grounded input format used during FLAN-T5 fine-tuning, ensuring that inference prompts preserve task routing, model lineage, approved descriptions, confidence categories, and safety limitations.


In [31]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import importlib
import sys
from pathlib import Path

import yaml


# -------------------------------------------------------------------------
# Define service and configuration paths
# -------------------------------------------------------------------------
grounding_service_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "src/services/grounding_service.py"
)

prompt_registry_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "configs/prompt_registry.yaml"
)

finding_contract_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "configs/finding_contract.yaml"
)

grounding_service_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Write the reusable grounding serialization service
# -------------------------------------------------------------------------
grounding_service_source = '''
"""Grounded language input serialization for API inference."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Mapping, Sequence


TASK_PREFIXES = {
    "structured_report": "generate structured report:",
    "plain_language_explanation": "explain in simple language:",
    "grounded_question_answering": "answer grounded question:",
    "educational_follow_up": "generate educational follow-up:",
}

INPUT_FIELD_ORDER = (
    "task_type",
    "finding_names",
    "probabilities",
    "frozen_thresholds",
    "threshold_decisions",
    "no_target_finding",
    "confidence_categories",
    "model_version",
    "approved_descriptions",
    "limitation_boundary",
    "user_question",
)


@dataclass(frozen=True)
class GroundedFinding:
    """One finding supplied to the grounded language model."""

    label_id: int
    label_name: str
    probability: float
    frozen_threshold: float
    threshold_decision: bool
    confidence_category: str
    approved_description: str


@dataclass(frozen=True)
class SerializedGroundingInput:
    """Versioned language-model input and its grounding lineage."""

    task_type: str
    instruction_prefix: str
    serialized_input: str
    finding_names: tuple[str, ...]
    no_target_finding: bool
    model_version: str
    user_question: str | None


class GroundedInputSerializer:
    """Serialize frozen model output using the training-time contract."""

    def __init__(
        self,
        *,
        language_model_version: str,
        limitation_boundary: str,
        optional_value_marker: str,
        task_prefixes: Mapping[str, str] | None = None,
    ) -> None:
        self.language_model_version = (
            language_model_version.strip()
        )
        self.limitation_boundary = (
            limitation_boundary.strip()
        )
        self.optional_value_marker = (
            optional_value_marker.strip()
        )
        self.task_prefixes = dict(
            task_prefixes or TASK_PREFIXES
        )

        if not self.language_model_version:
            raise ValueError(
                "A language-model version is required."
            )

        if not self.limitation_boundary:
            raise ValueError(
                "The educational limitation is required."
            )

        if not self.optional_value_marker:
            raise ValueError(
                "The optional-value marker is required."
            )

        if set(self.task_prefixes) != set(TASK_PREFIXES):
            raise ValueError(
                "Exactly four registered language tasks are required."
            )

        if any(
            not prefix.endswith(":")
            for prefix in self.task_prefixes.values()
        ):
            raise ValueError(
                "Every language-task prefix must end with a colon."
            )

    @staticmethod
    def _read_value(
        record: Any,
        *field_names: str,
        default: Any = None,
    ) -> Any:
        """Read a value from a mapping or object."""

        for field_name in field_names:
            if isinstance(record, Mapping):
                if field_name in record:
                    return record[field_name]
            elif hasattr(record, field_name):
                return getattr(
                    record,
                    field_name,
                )

        return default

    @staticmethod
    def _derive_confidence_category(
        *,
        probability: float,
        frozen_threshold: float,
        threshold_decision: bool,
    ) -> str:
        """Apply the frozen confidence-category relationship."""

        if not threshold_decision:
            return "below_threshold"

        threshold_margin = (
            probability - frozen_threshold
        )

        if threshold_margin <= 0.02:
            return "borderline"

        if threshold_margin <= 0.10:
            return "moderate"

        return "higher"

    def normalize_finding(
        self,
        record: Any,
    ) -> GroundedFinding:
        """Normalize one service or Pydantic finding record."""

        label_id = self._read_value(
            record,
            "label_id",
            "finding_id",
        )

        label_name = self._read_value(
            record,
            "label_name",
            "finding_name",
        )

        probability = self._read_value(
            record,
            "probability",
            "score",
        )

        frozen_threshold = self._read_value(
            record,
            "frozen_threshold",
            "threshold",
        )

        threshold_decision = self._read_value(
            record,
            "threshold_decision",
            "crossed_threshold",
            "decision",
        )

        approved_description = self._read_value(
            record,
            "approved_description",
            "description",
        )

        confidence_category = self._read_value(
            record,
            "confidence_category",
            "confidence",
        )

        required_values = {
            "label_id": label_id,
            "label_name": label_name,
            "probability": probability,
            "frozen_threshold": frozen_threshold,
            "threshold_decision": threshold_decision,
            "approved_description": approved_description,
        }

        missing_fields = [
            field_name
            for field_name, value
            in required_values.items()
            if value is None
        ]

        if missing_fields:
            raise ValueError(
                "Finding record is missing required fields: "
                + ", ".join(missing_fields)
            )

        label_id = int(label_id)
        label_name = str(label_name).strip()
        probability = float(probability)
        frozen_threshold = float(
            frozen_threshold
        )
        threshold_decision = bool(
            threshold_decision
        )
        approved_description = str(
            approved_description
        ).strip()

        if not 0 <= label_id < 14:
            raise ValueError(
                "Finding label ID must be between zero and thirteen."
            )

        if not label_name:
            raise ValueError(
                "Finding label name cannot be blank."
            )

        if not 0.0 <= probability <= 1.0:
            raise ValueError(
                "Finding probability must be between zero and one."
            )

        if not 0.0 <= frozen_threshold <= 1.0:
            raise ValueError(
                "Frozen threshold must be between zero and one."
            )

        expected_decision = (
            probability >= frozen_threshold
        )

        if threshold_decision != expected_decision:
            raise ValueError(
                f"Threshold decision is inconsistent for {label_name}."
            )

        if not approved_description:
            raise ValueError(
                f"Approved description is missing for {label_name}."
            )

        if confidence_category is None:
            confidence_category = (
                self._derive_confidence_category(
                    probability=probability,
                    frozen_threshold=frozen_threshold,
                    threshold_decision=threshold_decision,
                )
            )
        else:
            confidence_category = str(
                confidence_category
            ).strip()

        allowed_categories = {
            "below_threshold",
            "borderline",
            "moderate",
            "higher",
        }

        if confidence_category not in allowed_categories:
            raise ValueError(
                f"Unsupported confidence category for {label_name}: "
                f"{confidence_category}"
            )

        if (
            not threshold_decision
            and confidence_category != "below_threshold"
        ):
            raise ValueError(
                f"Below-threshold finding {label_name} must use "
                "the below_threshold confidence category."
            )

        return GroundedFinding(
            label_id=label_id,
            label_name=label_name,
            probability=probability,
            frozen_threshold=frozen_threshold,
            threshold_decision=threshold_decision,
            confidence_category=confidence_category,
            approved_description=approved_description,
        )

    def serialize(
        self,
        *,
        task_type: str,
        findings: Sequence[Any],
        no_target_finding: bool,
        user_question: str | None = None,
    ) -> SerializedGroundingInput:
        """Build one deterministic training-compatible model input."""

        if task_type not in self.task_prefixes:
            raise ValueError(
                f"Unsupported language task: {task_type}"
            )

        normalized_findings = tuple(
            self.normalize_finding(
                finding
            )
            for finding in findings
        )

        if not normalized_findings:
            raise ValueError(
                "At least one grounded finding record is required."
            )

        label_ids = [
            finding.label_id
            for finding in normalized_findings
        ]

        label_names = [
            finding.label_name
            for finding in normalized_findings
        ]

        if len(label_ids) != len(set(label_ids)):
            raise ValueError(
                "Duplicate finding label IDs are not allowed."
            )

        if len(label_names) != len(set(label_names)):
            raise ValueError(
                "Duplicate finding label names are not allowed."
            )

        crossed_findings = [
            finding
            for finding in normalized_findings
            if finding.threshold_decision
        ]

        if bool(no_target_finding) != (
            len(crossed_findings) == 0
        ):
            raise ValueError(
                "The no-target-finding state contradicts "
                "the supplied threshold decisions."
            )

        if task_type == "grounded_question_answering":
            if user_question is None:
                raise ValueError(
                    "Grounded question answering requires a question."
                )

            normalized_question = " ".join(
                str(user_question).split()
            )

            if not normalized_question:
                raise ValueError(
                    "Grounded question cannot be blank."
                )
        else:
            if (
                user_question is not None
                and str(user_question).strip()
            ):
                raise ValueError(
                    "Only grounded question answering accepts a question."
                )

            normalized_question = (
                self.optional_value_marker
            )

        finding_names_value = " | ".join(
            finding.label_name
            for finding in normalized_findings
        )

        probabilities_value = " | ".join(
            (
                f"{finding.label_name}="
                f"{finding.probability:.4f}"
            )
            for finding in normalized_findings
        )

        thresholds_value = " | ".join(
            (
                f"{finding.label_name}="
                f"{finding.frozen_threshold:.4f}"
            )
            for finding in normalized_findings
        )

        decisions_value = " | ".join(
            (
                f"{finding.label_name}="
                f"{'crossed' if finding.threshold_decision else 'not_crossed'}"
            )
            for finding in normalized_findings
        )

        confidence_value = " | ".join(
            (
                f"{finding.label_name}="
                f"{finding.confidence_category}"
            )
            for finding in normalized_findings
        )

        descriptions_value = " || ".join(
            (
                f"{finding.label_name}: "
                f"{finding.approved_description}"
            )
            for finding in normalized_findings
        )

        field_values = {
            "task_type": task_type,
            "finding_names": finding_names_value,
            "probabilities": probabilities_value,
            "frozen_thresholds": thresholds_value,
            "threshold_decisions": decisions_value,
            "no_target_finding": str(
                bool(no_target_finding)
            ).lower(),
            "confidence_categories": confidence_value,
            "model_version": self.language_model_version,
            "approved_descriptions": descriptions_value,
            "limitation_boundary": self.limitation_boundary,
            "user_question": normalized_question,
        }

        serialized_lines = [
            self.task_prefixes[task_type]
        ]

        serialized_lines.extend(
            f"{field_name}: {field_values[field_name]}"
            for field_name in INPUT_FIELD_ORDER
        )

        serialized_input = "\\n".join(
            serialized_lines
        )

        return SerializedGroundingInput(
            task_type=task_type,
            instruction_prefix=self.task_prefixes[
                task_type
            ],
            serialized_input=serialized_input,
            finding_names=tuple(label_names),
            no_target_finding=bool(
                no_target_finding
            ),
            model_version=self.language_model_version,
            user_question=(
                normalized_question
                if task_type
                == "grounded_question_answering"
                else None
            ),
        )
'''

grounding_service_path.write_text(
    grounding_service_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Import the generated module
# -------------------------------------------------------------------------
importlib.invalidate_caches()

module_name = (
    "src.services.grounding_service"
)

if module_name in sys.modules:
    grounding_service_module = importlib.reload(
        sys.modules[module_name]
    )
else:
    grounding_service_module = importlib.import_module(
        module_name
    )

GroundedInputSerializer = (
    grounding_service_module.GroundedInputSerializer
)

SerializedGroundingInput = (
    grounding_service_module.SerializedGroundingInput
)

INPUT_FIELD_ORDER = (
    grounding_service_module.INPUT_FIELD_ORDER
)

TASK_PREFIXES = (
    grounding_service_module.TASK_PREFIXES
)


# -------------------------------------------------------------------------
# Load the frozen registries
# -------------------------------------------------------------------------
with prompt_registry_path.open(
    "r",
    encoding="utf-8",
) as registry_file:
    current_prompt_registry = yaml.safe_load(
        registry_file
    )

with finding_contract_path.open(
    "r",
    encoding="utf-8",
) as contract_file:
    current_finding_contract = yaml.safe_load(
        contract_file
    )


# -------------------------------------------------------------------------
# Locate a named text value recursively
# -------------------------------------------------------------------------
def locate_named_text(
    value,
    required_terms,
    path=(),
):
    if isinstance(value, dict):
        for key, nested_value in value.items():
            normalized_key = "".join(
                character.lower()
                for character in str(key)
                if character.isalnum()
            )

            current_path = (
                *path,
                normalized_key,
            )

            path_text = "".join(
                current_path
            )

            if (
                all(
                    term in path_text
                    for term in required_terms
                )
                and isinstance(
                    nested_value,
                    str,
                )
                and nested_value.strip()
            ):
                return nested_value.strip()

            located_value = locate_named_text(
                nested_value,
                required_terms,
                current_path,
            )

            if located_value is not None:
                return located_value

    elif isinstance(value, list):
        for item in value:
            located_value = locate_named_text(
                item,
                required_terms,
                path,
            )

            if located_value is not None:
                return located_value

    return None


# -------------------------------------------------------------------------
# Resolve the optional marker and educational limitation
# -------------------------------------------------------------------------
optional_value_marker = locate_named_text(
    current_prompt_registry,
    (
        "optional",
        "marker",
    ),
)

if optional_value_marker is None:
    optional_value_marker = (
        globals().get(
            "OPTIONAL_VALUE_MARKER"
        )
        or "not_applicable"
    )

educational_limitation = locate_named_text(
    current_finding_contract,
    (
        "educational",
        "limitation",
    ),
)

if educational_limitation is None:
    educational_limitation = (
        globals().get(
            "EDUCATIONAL_USE_LIMITATION"
        )
    )

if not isinstance(
    educational_limitation,
    str,
) or not educational_limitation.strip():
    raise KeyError(
        "The educational limitation could not be resolved "
        "from the frozen finding contract."
    )


# -------------------------------------------------------------------------
# Initialize the serializer
# -------------------------------------------------------------------------
grounded_input_serializer = (
    GroundedInputSerializer(
        language_model_version=(
            "flan-t5-small-chestmnist-v1"
        ),
        limitation_boundary=(
            educational_limitation
        ),
        optional_value_marker=(
            optional_value_marker
        ),
        task_prefixes=TASK_PREFIXES,
    )
)


# -------------------------------------------------------------------------
# Resolve the smoke-test findings
# -------------------------------------------------------------------------
serialization_findings = getattr(
    structural_smoke_result,
    "findings",
    None,
)

if serialization_findings is None:
    serialization_findings = getattr(
        structural_smoke_result,
        "finding_records",
        None,
    )

if not serialization_findings:
    raise RuntimeError(
        "The structural classification result does not "
        "contain finding records."
    )


def notebook_read_value(
    record,
    *field_names,
):
    for field_name in field_names:
        if isinstance(
            record,
            dict,
        ) and field_name in record:
            return record[field_name]

        if hasattr(
            record,
            field_name,
        ):
            return getattr(
                record,
                field_name,
            )

    raise KeyError(
        "Finding field was not available: "
        + ", ".join(field_names)
    )


crossed_serialization_findings = [
    finding
    for finding in serialization_findings
    if bool(
        notebook_read_value(
            finding,
            "threshold_decision",
            "crossed_threshold",
            "decision",
        )
    )
]

if crossed_serialization_findings:
    selected_serialization_findings = (
        crossed_serialization_findings
    )

    serialization_no_target = False
else:
    selected_serialization_findings = sorted(
        serialization_findings,
        key=lambda finding: float(
            notebook_read_value(
                finding,
                "probability",
                "score",
            )
        ),
        reverse=True,
    )[:2]

    serialization_no_target = True


# -------------------------------------------------------------------------
# Serialize all four registered task inputs
# -------------------------------------------------------------------------
serialized_task_inputs = {}

for task_type in TASK_PREFIXES:
    task_question = (
        "What does the supplied model information indicate?"
        if task_type
        == "grounded_question_answering"
        else None
    )

    serialized_task_inputs[task_type] = (
        grounded_input_serializer.serialize(
            task_type=task_type,
            findings=selected_serialization_findings,
            no_target_finding=serialization_no_target,
            user_question=task_question,
        )
    )


# -------------------------------------------------------------------------
# Validate the serialized contract
# -------------------------------------------------------------------------
serialization_checks = {
    (
        "Grounding service module was written"
    ): grounding_service_path.is_file(),
    (
        "Four language tasks were serialized"
    ): len(serialized_task_inputs) == 4,
    (
        "Every input starts with its task prefix"
    ): all(
        record.serialized_input.startswith(
            record.instruction_prefix
        )
        for record in serialized_task_inputs.values()
    ),
    (
        "Every input contains twelve lines"
    ): all(
        len(
            record.serialized_input.splitlines()
        ) == 12
        for record in serialized_task_inputs.values()
    ),
    (
        "Every registered field preserves its order"
    ): all(
        [
            line.split(
                ":",
                maxsplit=1,
            )[0]
            for line in (
                record.serialized_input.splitlines()[1:]
            )
        ]
        == list(INPUT_FIELD_ORDER)
        for record in serialized_task_inputs.values()
    ),
    (
        "Every input preserves model lineage"
    ): all(
        (
            "model_version: "
            "flan-t5-small-chestmnist-v1"
        )
        in record.serialized_input
        for record in serialized_task_inputs.values()
    ),
    (
        "Every input preserves the limitation boundary"
    ): all(
        educational_limitation
        in record.serialized_input
        for record in serialized_task_inputs.values()
    ),
    (
        "Question is included only for grounded QA"
    ): (
        serialized_task_inputs[
            "grounded_question_answering"
        ].user_question
        is not None
        and all(
            record.user_question is None
            for task_name, record
            in serialized_task_inputs.items()
            if task_name
            != "grounded_question_answering"
        )
    ),
    (
        "Non-QA tasks use the optional marker"
    ): all(
        (
            f"user_question: "
            f"{optional_value_marker}"
        )
        in record.serialized_input
        for task_name, record
        in serialized_task_inputs.items()
        if task_name
        != "grounded_question_answering"
    ),
    (
        "No-target state matches supplied findings"
    ): all(
        record.no_target_finding
        == serialization_no_target
        for record in serialized_task_inputs.values()
    ),
}

grounding_service_checksum = hashlib.sha256(
    grounding_service_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Present the validation summary
# -------------------------------------------------------------------------
print(
    "GROUNDED LANGUAGE INPUT SERIALIZATION SERVICE"
)
print("-" * 100)
print(
    f"{'Service module':29}: "
    f"{grounding_service_path}"
)
print(
    f"{'Module SHA-256':29}: "
    f"{grounding_service_checksum[:16]}..."
)
print(
    f"{'Registered language tasks':29}: "
    f"{len(serialized_task_inputs)}"
)
print(
    f"{'Input fields per task':29}: "
    f"{len(INPUT_FIELD_ORDER)}"
)
print(
    f"{'Selected findings':29}: "
    f"{len(selected_serialization_findings)}"
)
print(
    f"{'No-target-finding state':29}: "
    f"{serialization_no_target}"
)
print(
    f"{'Optional-value marker':29}: "
    f"{optional_value_marker}"
)
print("-" * 100)

for task_name, record in serialized_task_inputs.items():
    print(
        f"{task_name:29}: "
        f"{len(record.serialized_input.splitlines()):2d} lines | "
        f"{len(record.serialized_input):4d} characters"
    )

print("-" * 100)

for check_name, passed in serialization_checks.items():
    print(
        f"{check_name:61}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    serialization_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in serialization_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Grounded input serialization validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "GROUNDED QA SERIALIZATION EXAMPLE"
)
print("-" * 100)
print(
    serialized_task_inputs[
        "grounded_question_answering"
    ].serialized_input
)
print("-" * 100)
print(
    "STATUS: GROUNDED LANGUAGE INPUT CONTRACT READY FOR INFERENCE"
)

GROUNDED LANGUAGE INPUT SERIALIZATION SERVICE
----------------------------------------------------------------------------------------------------
Service module               : /home/jovyan/chest-xray-ai-assistant/src/services/grounding_service.py
Module SHA-256               : 791d7e8b0c160aca...
Registered language tasks    : 4
Input fields per task        : 11
Selected findings            : 2
No-target-finding state      : True
Optional-value marker        : not_applicable
----------------------------------------------------------------------------------------------------
structured_report            : 12 lines |  833 characters
plain_language_explanation   : 12 lines |  842 characters
grounded_question_answering  : 12 lines |  877 characters
educational_follow_up        : 12 lines |  841 characters
----------------------------------------------------------------------------------------------------
Grounding service module was written                         : PASS
Four language ta

<a id="nb07-5-8-frozen-grounded-language-model-inference-service"></a>
### 5.8 Frozen Grounded Language Model Inference Service

This block loads the versioned fine-tuned FLAN-T5-small model in inference-only mode and exposes thread-safe greedy generation for all four registered language tasks. The service enforces the frozen 416-token input and 288-token generation limits, preserves serialized grounding inputs, and records model-loading and per-request runtime measurements. Generated text remains subject to the deterministic guardrail before it can be returned by an API endpoint.


In [32]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import importlib
import sys
import threading
import time
from pathlib import Path

import torch
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
)


# -------------------------------------------------------------------------
# Define the service and frozen model paths
# -------------------------------------------------------------------------
language_service_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "src/services/language_model_service.py"
)

language_model_directory = Path(
    "/home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data/models/"
    "flan-t5-small-chestmnist-v1"
)

language_service_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Write the reusable language-model inference service
# -------------------------------------------------------------------------
language_service_source = '''
"""Thread-safe inference for the fine-tuned grounded language model."""

from __future__ import annotations

import threading
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Sequence

import torch
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
)


@dataclass(frozen=True)
class LanguageGenerationResult:
    """One raw generation and its inference lineage."""

    task_type: str
    generated_text: str
    serialized_input: str
    input_tokens: int
    generated_tokens: int
    generation_latency_ms: float
    language_model_version: str
    decoding_strategy: str


class GroundedLanguageModelService:
    """Run the frozen fine-tuned FLAN-T5 model safely."""

    MAXIMUM_INPUT_LENGTH = 416
    MAXIMUM_TARGET_LENGTH = 288
    DECODING_STRATEGY = "greedy"

    def __init__(
        self,
        *,
        model_directory: str | Path,
        model_version: str,
        device: str | torch.device | None = None,
        use_bfloat16: bool = True,
    ) -> None:
        self.model_directory = Path(
            model_directory
        )
        self.model_version = (
            model_version.strip()
        )
        self._lock = threading.RLock()

        if not self.model_directory.is_dir():
            raise FileNotFoundError(
                "The versioned language-model directory "
                "is unavailable."
            )

        if not self.model_version:
            raise ValueError(
                "A language-model version is required."
            )

        if device is None:
            selected_device = (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )
        else:
            selected_device = str(device)

        self.device = torch.device(
            selected_device
        )

        self.use_bfloat16 = bool(
            use_bfloat16
            and self.device.type == "cuda"
            and torch.cuda.is_bf16_supported()
        )

        self.inference_dtype = (
            torch.bfloat16
            if self.use_bfloat16
            else torch.float32
        )

        load_started_at = time.perf_counter()

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_directory,
            local_files_only=True,
            use_fast=True,
        )

        self.model = (
            T5ForConditionalGeneration.from_pretrained(
                self.model_directory,
                local_files_only=True,
                torch_dtype=self.inference_dtype,
            )
        )

        self.model.to(
            self.device
        )
        self.model.eval()

        if self.device.type == "cuda":
            torch.cuda.synchronize(
                self.device
            )

        self.model_load_latency_ms = (
            time.perf_counter()
            - load_started_at
        ) * 1000.0

        self._validate_model_contract()

    def _validate_model_contract(
        self,
    ) -> None:
        """Validate the tokenizer and model compatibility."""

        if not self.model.config.is_encoder_decoder:
            raise RuntimeError(
                "The grounded language model must be "
                "an encoder-decoder model."
            )

        if self.tokenizer.pad_token_id is None:
            raise RuntimeError(
                "The tokenizer must define a padding token."
            )

        if self.tokenizer.eos_token_id is None:
            raise RuntimeError(
                "The tokenizer must define an end-of-sequence token."
            )

        embedding_count = int(
            self.model.get_input_embeddings().num_embeddings
        )

        tokenizer_maximum_id = (
            len(self.tokenizer) - 1
        )

        if tokenizer_maximum_id >= embedding_count:
            raise RuntimeError(
                "Tokenizer IDs exceed the model embedding vocabulary."
            )

    @staticmethod
    def _read_serialized_record(
        record: Any,
    ) -> tuple[str, str]:
        """Read task type and input from a mapping or object."""

        if isinstance(record, dict):
            task_type = record.get(
                "task_type"
            )
            serialized_input = record.get(
                "serialized_input"
            )
        else:
            task_type = getattr(
                record,
                "task_type",
                None,
            )
            serialized_input = getattr(
                record,
                "serialized_input",
                None,
            )

        if not isinstance(
            task_type,
            str,
        ) or not task_type.strip():
            raise ValueError(
                "Serialized grounding input requires a task type."
            )

        if not isinstance(
            serialized_input,
            str,
        ) or not serialized_input.strip():
            raise ValueError(
                "Serialized grounding input cannot be blank."
            )

        return (
            task_type.strip(),
            serialized_input.strip(),
        )

    def generate_many(
        self,
        records: Sequence[Any],
    ) -> tuple[LanguageGenerationResult, ...]:
        """Generate one deterministic output for every grounded input."""

        if not records:
            raise ValueError(
                "At least one serialized grounding input is required."
            )

        normalized_records = [
            self._read_serialized_record(
                record
            )
            for record in records
        ]

        task_types = [
            record[0]
            for record in normalized_records
        ]

        serialized_inputs = [
            record[1]
            for record in normalized_records
        ]

        encoded_batch = self.tokenizer(
            serialized_inputs,
            padding=True,
            truncation=False,
            return_tensors="pt",
            add_special_tokens=True,
        )

        attention_mask = encoded_batch[
            "attention_mask"
        ]

        input_lengths = attention_mask.sum(
            dim=1
        ).tolist()

        if max(input_lengths) > self.MAXIMUM_INPUT_LENGTH:
            raise ValueError(
                "A grounded language input exceeds the frozen "
                f"{self.MAXIMUM_INPUT_LENGTH}-token contract."
            )

        encoded_batch = {
            key: value.to(
                self.device
            )
            for key, value
            in encoded_batch.items()
        }

        if self.device.type == "cuda":
            torch.cuda.synchronize(
                self.device
            )

        generation_started_at = (
            time.perf_counter()
        )

        with self._lock:
            self.model.eval()

            with torch.inference_mode():
                generated_ids = self.model.generate(
                    **encoded_batch,
                    do_sample=False,
                    num_beams=1,
                    max_new_tokens=(
                        self.MAXIMUM_TARGET_LENGTH
                    ),
                    use_cache=True,
                )

            if self.device.type == "cuda":
                torch.cuda.synchronize(
                    self.device
                )

        batch_latency_ms = (
            time.perf_counter()
            - generation_started_at
        ) * 1000.0

        generated_texts = (
            self.tokenizer.batch_decode(
                generated_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )
        )

        generated_lengths = (
            generated_ids.ne(
                self.tokenizer.pad_token_id
            ).sum(
                dim=1
            ).tolist()
        )

        latency_per_record = (
            batch_latency_ms
            / len(normalized_records)
        )

        results = []

        for (
            task_type,
            serialized_input,
            generated_text,
            input_length,
            generated_length,
        ) in zip(
            task_types,
            serialized_inputs,
            generated_texts,
            input_lengths,
            generated_lengths,
            strict=True,
        ):
            normalized_text = (
                generated_text.strip()
            )

            if not normalized_text:
                raise RuntimeError(
                    f"The model returned an empty output for {task_type}."
                )

            results.append(
                LanguageGenerationResult(
                    task_type=task_type,
                    generated_text=normalized_text,
                    serialized_input=serialized_input,
                    input_tokens=int(
                        input_length
                    ),
                    generated_tokens=int(
                        generated_length
                    ),
                    generation_latency_ms=float(
                        latency_per_record
                    ),
                    language_model_version=(
                        self.model_version
                    ),
                    decoding_strategy=(
                        self.DECODING_STRATEGY
                    ),
                )
            )

        return tuple(
            results
        )

    def generate(
        self,
        record: Any,
    ) -> LanguageGenerationResult:
        """Generate a single deterministic grounded output."""

        return self.generate_many(
            [record]
        )[0]
'''

language_service_path.write_text(
    language_service_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Import the generated service module
# -------------------------------------------------------------------------
importlib.invalidate_caches()

module_name = (
    "src.services.language_model_service"
)

if module_name in sys.modules:
    language_service_module = importlib.reload(
        sys.modules[module_name]
    )
else:
    language_service_module = importlib.import_module(
        module_name
    )

GroundedLanguageModelService = (
    language_service_module
    .GroundedLanguageModelService
)

LanguageGenerationResult = (
    language_service_module
    .LanguageGenerationResult
)


# -------------------------------------------------------------------------
# Confirm all frozen model artifacts are available
# -------------------------------------------------------------------------
required_language_model_files = [
    "config.json",
    "generation_config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
]

language_model_file_checks = {
    filename: (
        language_model_directory
        / filename
    ).is_file()
    for filename
    in required_language_model_files
}

if not all(
    language_model_file_checks.values()
):
    missing_files = [
        filename
        for filename, available
        in language_model_file_checks.items()
        if not available
    ]

    raise FileNotFoundError(
        "Required language-model artifacts are missing: "
        + ", ".join(missing_files)
    )


# -------------------------------------------------------------------------
# Release temporary Grad-CAM tensors before loading the language model
# -------------------------------------------------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


# -------------------------------------------------------------------------
# Initialize the fine-tuned language-model service
# -------------------------------------------------------------------------
grounded_language_model_service = (
    GroundedLanguageModelService(
        model_directory=(
            language_model_directory
        ),
        model_version=(
            "flan-t5-small-chestmnist-v1"
        ),
        device=(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        ),
        use_bfloat16=True,
    )
)


# -------------------------------------------------------------------------
# Generate smoke-test outputs for all four language tasks
# -------------------------------------------------------------------------
ordered_serialized_inputs = [
    serialized_task_inputs[
        task_type
    ]
    for task_type in (
        "structured_report",
        "plain_language_explanation",
        "grounded_question_answering",
        "educational_follow_up",
    )
]

language_generation_results = (
    grounded_language_model_service.generate_many(
        ordered_serialized_inputs
    )
)

language_generation_by_task = {
    result.task_type: result
    for result in language_generation_results
}

peak_language_gpu_gib = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 3)
    if torch.cuda.is_available()
    else 0.0
)


# -------------------------------------------------------------------------
# Validate required response sections
# -------------------------------------------------------------------------
required_first_sections = {
    "structured_report": (
        "PRELIMINARY MODEL REPORT"
    ),
    "plain_language_explanation": (
        "EXPLANATION"
    ),
    "grounded_question_answering": (
        "ANSWER"
    ),
    "educational_follow_up": (
        "EDUCATIONAL FOLLOW-UP"
    ),
}

required_task_order = list(
    required_first_sections
)

generated_task_order = [
    result.task_type
    for result in language_generation_results
]

model_parameter_count = sum(
    parameter.numel()
    for parameter
    in grounded_language_model_service
    .model.parameters()
)

model_device = next(
    grounded_language_model_service
    .model.parameters()
).device

model_dtype = next(
    grounded_language_model_service
    .model.parameters()
).dtype

tokenizer_vocabulary = len(
    grounded_language_model_service.tokenizer
)

model_embedding_vocabulary = int(
    grounded_language_model_service
    .model.get_input_embeddings()
    .num_embeddings
)

language_service_checksum = hashlib.sha256(
    language_service_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Evaluate the inference-service contract
# -------------------------------------------------------------------------
language_inference_checks = {
    (
        "Language-model service module was written"
    ): language_service_path.is_file(),
    (
        "All required model artifacts are available"
    ): all(
        language_model_file_checks.values()
    ),
    (
        "Fine-tuned model loaded successfully"
    ): (
        grounded_language_model_service
        .model is not None
    ),
    (
        "Model remains in evaluation mode"
    ): not (
        grounded_language_model_service
        .model.training
    ),
    (
        "Model executes on the selected device"
    ): (
        model_device.type
        == grounded_language_model_service.device.type
    ),
    (
        "Bfloat16 inference is enabled on the GPU"
    ): (
        (
            model_device.type == "cuda"
            and model_dtype == torch.bfloat16
        )
        or model_device.type == "cpu"
    ),
    (
        "Model parameter count is preserved"
    ): model_parameter_count == 76_961_152,
    (
        "Tokenizer IDs fit the model vocabulary"
    ): (
        tokenizer_vocabulary
        <= model_embedding_vocabulary
    ),
    (
        "All four language tasks were generated"
    ): len(
        language_generation_results
    ) == 4,
    (
        "Generated task order is preserved"
    ): (
        generated_task_order
        == required_task_order
    ),
    (
        "Every generated output is non-empty"
    ): all(
        bool(
            result.generated_text.strip()
        )
        for result in language_generation_results
    ),
    (
        "Every output starts with its required section"
    ): all(
        result.generated_text.startswith(
            required_first_sections[
                result.task_type
            ]
        )
        for result in language_generation_results
    ),
    (
        "Every output contains the limitations section"
    ): all(
        "LIMITATIONS"
        in result.generated_text
        for result in language_generation_results
    ),
    (
        "Every input respects the frozen token limit"
    ): all(
        result.input_tokens
        <= 416
        for result in language_generation_results
    ),
    (
        "Every generation respects the frozen token limit"
    ): all(
        result.generated_tokens
        <= 288
        for result in language_generation_results
    ),
    (
        "Greedy decoding is preserved"
    ): all(
        result.decoding_strategy
        == "greedy"
        for result in language_generation_results
    ),
    (
        "Language-model lineage is preserved"
    ): all(
        result.language_model_version
        == "flan-t5-small-chestmnist-v1"
        for result in language_generation_results
    ),
    (
        "Generation latency is positive"
    ): all(
        result.generation_latency_ms
        > 0.0
        for result in language_generation_results
    ),
    (
        "Service checksum is available"
    ): len(
        language_service_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the validation summary
# -------------------------------------------------------------------------
print(
    "FROZEN GROUNDED LANGUAGE MODEL INFERENCE SERVICE"
)
print("-" * 100)
print(
    f"{'Service module':29}: "
    f"{language_service_path}"
)
print(
    f"{'Module SHA-256':29}: "
    f"{language_service_checksum[:16]}..."
)
print(
    f"{'Model directory':29}: "
    f"{language_model_directory}"
)
print(
    f"{'Model class':29}: "
    f"{type(grounded_language_model_service.model).__name__}"
)
print(
    f"{'Execution device':29}: "
    f"{model_device}"
)
print(
    f"{'Inference dtype':29}: "
    f"{model_dtype}"
)
print(
    f"{'Model parameters':29}: "
    f"{model_parameter_count:,}"
)
print(
    f"{'Model load latency':29}: "
    f"{grounded_language_model_service.model_load_latency_ms:.2f} ms"
)
print(
    f"{'Generated task outputs':29}: "
    f"{len(language_generation_results)}"
)
print(
    f"{'Peak GPU memory':29}: "
    f"{peak_language_gpu_gib:.2f} GiB"
)
print("-" * 100)

for result in language_generation_results:
    print(
        f"{result.task_type:29}: "
        f"{result.input_tokens:3d} input tokens | "
        f"{result.generated_tokens:3d} generated tokens | "
        f"{result.generation_latency_ms:.2f} ms"
    )

print("-" * 100)

for check_name, passed in language_inference_checks.items():
    print(
        f"{check_name:62}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    language_inference_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in language_inference_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Grounded language inference validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "GROUNDED QUESTION-ANSWERING RAW GENERATION"
)
print("-" * 100)
print(
    language_generation_by_task[
        "grounded_question_answering"
    ].generated_text
)
print("-" * 100)
print(
    "STATUS: FROZEN LANGUAGE MODEL READY FOR GUARDED API GENERATION"
)

FROZEN GROUNDED LANGUAGE MODEL INFERENCE SERVICE
----------------------------------------------------------------------------------------------------
Service module               : /home/jovyan/chest-xray-ai-assistant/src/services/language_model_service.py
Module SHA-256               : 91ba397b0df2ecae...
Model directory              : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/models/flan-t5-small-chestmnist-v1
Model class                  : T5ForConditionalGeneration
Execution device             : cuda:0
Inference dtype              : torch.bfloat16
Model parameters             : 76,961,152
Model load latency           : 12691.69 ms
Generated task outputs       : 4
Peak GPU memory              : 0.23 GiB
----------------------------------------------------------------------------------------------------
structured_report            : 263 input tokens | 134 generated tokens | 532.43 ms
plain_language_explanation   : 267 input tokens | 122 generated tokens | 532.43 m

<a id="nb07-5-9-deterministic-grounding-and-safety-guardrail-service"></a>
### 5.9 Deterministic Grounding and Safety Guardrail Service

This block places the deterministic guardrail between raw FLAN-T5 generation and every API response. It audits task routing, required sections, finding names, numeric values, threshold relationships, no-target wording, controlled refusals, Grad-CAM limitations, and prohibited medical claims. Any failed check replaces the raw generation with a grounded template while retaining the original output for internal audit lineage.


In [33]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import importlib
import re
import sys
import time
from pathlib import Path


# -------------------------------------------------------------------------
# Define the guardrail service path
# -------------------------------------------------------------------------
language_guardrail_service_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "src/services/language_guardrail_service.py"
)

language_guardrail_service_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Write the deterministic language guardrail service
# -------------------------------------------------------------------------
language_guardrail_service_source = '''
"""Deterministic grounding and safety guardrail for language output."""

from __future__ import annotations

import re
import time
from dataclasses import dataclass
from typing import Any, Mapping, Sequence


SUPPORTED_FINDING_NAMES = (
    "atelectasis",
    "cardiomegaly",
    "effusion",
    "infiltration",
    "mass",
    "nodule",
    "pneumonia",
    "pneumothorax",
    "consolidation",
    "edema",
    "emphysema",
    "fibrosis",
    "pleural",
    "hernia",
)

REQUIRED_FIRST_SECTIONS = {
    "structured_report": "PRELIMINARY MODEL REPORT",
    "plain_language_explanation": "EXPLANATION",
    "grounded_question_answering": "ANSWER",
    "educational_follow_up": "EDUCATIONAL FOLLOW-UP",
}

ALLOWED_TRIGGER_REASONS = (
    "task_routing_issue",
    "section_order_issue",
    "missing_required_finding_issue",
    "unsupported_finding_issue",
    "numeric_grounding_issue",
    "safety_boundary_issue",
    "no_target_boundary_issue",
    "qa_refusal_issue",
    "gradcam_boundary_issue",
    "forbidden_claim_issue",
)


@dataclass(frozen=True)
class GuardrailAudit:
    """Boolean contract checks for one generated output."""

    task_routing_compliant: bool
    section_order_compliant: bool
    required_findings_mentioned: bool
    unsupported_finding_free: bool
    numeric_grounding_compliant: bool
    safety_boundary_compliant: bool
    no_target_boundary_compliant: bool
    qa_refusal_compliant: bool
    gradcam_boundary_compliant: bool
    forbidden_claim_free: bool


@dataclass(frozen=True)
class GuardedLanguageResult:
    """Raw and guarded language output with deterministic audit lineage."""

    task_type: str
    raw_generated_text: str
    final_text: str
    guardrail_action: str
    trigger_reasons: tuple[str, ...]
    question_intent: str | None
    audit: GuardrailAudit
    guardrail_latency_ms: float


class DeterministicLanguageGuardrail:
    """Audit raw language and apply a controlled fallback when required."""

    ACCEPTED_ACTION = "accepted_model_generation"
    FALLBACK_ACTION = "safe_template_fallback"

    def __init__(
        self,
        *,
        educational_limitation: str,
        gradcam_limitation: str,
        professional_review_guidance: str,
    ) -> None:
        self.educational_limitation = (
            educational_limitation.strip()
        )
        self.gradcam_limitation = (
            gradcam_limitation.strip()
        )
        self.professional_review_guidance = (
            professional_review_guidance.strip()
        )

        if not self.educational_limitation:
            raise ValueError(
                "The educational limitation is required."
            )

        if not self.gradcam_limitation:
            raise ValueError(
                "The Grad-CAM limitation is required."
            )

        if not self.professional_review_guidance:
            raise ValueError(
                "Professional-review guidance is required."
            )

    @staticmethod
    def _read_value(
        record: Any,
        *field_names: str,
        default: Any = None,
    ) -> Any:
        for field_name in field_names:
            if isinstance(record, Mapping):
                if field_name in record:
                    return record[field_name]
            elif hasattr(record, field_name):
                return getattr(
                    record,
                    field_name,
                )

        return default

    def _normalize_findings(
        self,
        findings: Sequence[Any],
    ) -> tuple[dict[str, Any], ...]:
        normalized = []

        for record in findings:
            label_name = str(
                self._read_value(
                    record,
                    "label_name",
                    "finding_name",
                )
            ).strip().lower()

            probability = float(
                self._read_value(
                    record,
                    "probability",
                    "score",
                )
            )

            threshold = float(
                self._read_value(
                    record,
                    "frozen_threshold",
                    "threshold",
                )
            )

            decision = bool(
                self._read_value(
                    record,
                    "threshold_decision",
                    "crossed_threshold",
                    "decision",
                )
            )

            confidence = str(
                self._read_value(
                    record,
                    "confidence_category",
                    "confidence",
                    default=(
                        "below_threshold"
                        if not decision
                        else "borderline"
                    ),
                )
            ).strip()

            description = str(
                self._read_value(
                    record,
                    "approved_description",
                    "description",
                )
            ).strip()

            normalized.append(
                {
                    "label_name": label_name,
                    "probability": probability,
                    "threshold": threshold,
                    "decision": decision,
                    "confidence": confidence,
                    "description": description,
                }
            )

        if not normalized:
            raise ValueError(
                "The guardrail requires at least one grounded finding."
            )

        return tuple(
            normalized
        )

    @staticmethod
    def _display_name(
        label_name: str,
    ) -> str:
        if label_name == "pleural":
            return "Pleural Abnormality"

        return label_name.replace(
            "_",
            " ",
        ).title()

    @staticmethod
    def _classify_question_intent(
        question: str | None,
    ) -> str | None:
        if question is None:
            return None

        normalized = " ".join(
            question.lower().split()
        )

        if any(
            term in normalized
            for term in (
                "treat",
                "treatment",
                "medicine",
                "medication",
                "tablet",
                "drug",
                "cure",
            )
        ):
            return "treatment_request"

        if any(
            term in normalized
            for term in (
                "diagnosis",
                "diagnose",
                "do i have",
                "confirm disease",
                "what disease",
            )
        ):
            return "diagnosis_request"

        if any(
            term in normalized
            for term in (
                "grad-cam",
                "gradcam",
                "heatmap",
                "highlighted area",
                "visual evidence",
            )
        ):
            return "gradcam_boundary"

        if (
            "no target" in normalized
            or "no finding" in normalized
        ):
            return "no_target_meaning"

        if "threshold" in normalized:
            return "threshold_meaning"

        if any(
            term in normalized
            for term in (
                "confidence",
                "probability",
                "score",
            )
        ):
            return "confidence_meaning"

        return "output_summary"

    @staticmethod
    def _required_findings_for_audit(
        *,
        findings: Sequence[dict[str, Any]],
        task_type: str,
        question_intent: str | None,
    ) -> tuple[str, ...]:
        if (
            task_type
            == "grounded_question_answering"
            and question_intent
            in {
                "diagnosis_request",
                "treatment_request",
                "gradcam_boundary",
                "no_target_meaning",
            }
        ):
            return ()

        return tuple(
            finding["label_name"]
            for finding in findings
        )

    @staticmethod
    def _finding_is_mentioned(
        text_lower: str,
        label_name: str,
    ) -> bool:
        aliases = {
            "pleural": (
                "pleural",
                "pleural abnormality",
            ),
        }.get(
            label_name,
            (label_name,),
        )

        return any(
            re.search(
                rf"\\b{re.escape(alias)}\\b",
                text_lower,
            )
            is not None
            for alias in aliases
        )

    def _unsupported_finding_names(
        self,
        *,
        text_lower: str,
        supplied_names: set[str],
    ) -> tuple[str, ...]:
        unsupported = []

        for label_name in SUPPORTED_FINDING_NAMES:
            if label_name in supplied_names:
                continue

            if self._finding_is_mentioned(
                text_lower,
                label_name,
            ):
                unsupported.append(
                    label_name
                )

        return tuple(
            unsupported
        )

    @staticmethod
    def _numeric_values_are_grounded(
        *,
        text: str,
        findings: Sequence[dict[str, Any]],
    ) -> bool:
        generated_values = [
            float(match)
            for match in re.findall(
                r"(?<![A-Za-z0-9])0\\.\\d+(?![A-Za-z0-9])",
                text,
            )
        ]

        if not generated_values:
            return True

        allowed_values = []

        for finding in findings:
            allowed_values.extend(
                [
                    finding["probability"],
                    finding["threshold"],
                ]
            )

        return all(
            any(
                abs(
                    generated_value
                    - allowed_value
                )
                <= 0.00005
                for allowed_value in allowed_values
            )
            for generated_value in generated_values
        )

    def _no_target_boundary_is_valid(
        self,
        *,
        text_lower: str,
        findings: Sequence[dict[str, Any]],
        no_target_finding: bool,
    ) -> bool:
        crossed_names = {
            finding["label_name"]
            for finding in findings
            if finding["decision"]
        }

        crossed_claim_pattern = re.compile(
            r"(crossed|above|exceeded)"
            r"(?:\\s+its|\\s+the)?"
            r"\\s+(?:frozen\\s+)?threshold"
        )

        if no_target_finding:
            return (
                not crossed_names
                and crossed_claim_pattern.search(
                    text_lower
                )
                is None
            )

        return bool(
            crossed_names
        )

    @staticmethod
    def _qa_refusal_is_valid(
        *,
        text_lower: str,
        task_type: str,
        question_intent: str | None,
    ) -> bool:
        if task_type != "grounded_question_answering":
            return True

        if question_intent == "diagnosis_request":
            return (
                "cannot confirm" in text_lower
                and "diagnos" in text_lower
            )

        if question_intent == "treatment_request":
            has_decline = any(
                phrase in text_lower
                for phrase in (
                    "cannot recommend treatment",
                    "cannot provide treatment",
                    "cannot recommend medication",
                    "cannot provide medication",
                )
            )

            has_professional_direction = any(
                phrase in text_lower
                for phrase in (
                    "qualified healthcare professional",
                    "healthcare professional",
                    "doctor",
                )
            )

            return (
                has_decline
                and has_professional_direction
            )

        return True

    def _gradcam_boundary_is_valid(
        self,
        *,
        text_lower: str,
        task_type: str,
        question_intent: str | None,
    ) -> bool:
        if (
            task_type
            != "grounded_question_answering"
            or question_intent
            != "gradcam_boundary"
        ):
            return True

        required_terms = (
            "grad-cam",
            "model",
        )

        boundary_present = all(
            term in text_lower
            for term in required_terms
        )

        no_confirmation_claim = any(
            phrase in text_lower
            for phrase in (
                "does not confirm",
                "cannot confirm",
                "not confirm",
            )
        )

        return (
            boundary_present
            and no_confirmation_claim
        )

    @staticmethod
    def _forbidden_claim_free(
        text_lower: str,
    ) -> bool:
        forbidden_patterns = (
            r"\\byou have\\b",
            r"\\bthe patient has\\b",
            r"\\bthis confirms a diagnosis\\b",
            r"\\bthe diagnosis is\\b",
            r"\\bdefinitely (?:has|shows|indicates)\\b",
            r"\\btake \\d+\\s*(?:mg|ml)\\b",
            r"\\bprescribe(?:d|s|ing)?\\b",
        )

        return not any(
            re.search(
                pattern,
                text_lower,
            )
            for pattern in forbidden_patterns
        )

    def audit(
        self,
        *,
        task_type: str,
        generated_text: str,
        findings: Sequence[Any],
        no_target_finding: bool,
        user_question: str | None = None,
    ) -> tuple[
        GuardrailAudit,
        tuple[str, ...],
        str | None,
    ]:
        """Audit one raw model generation."""

        if task_type not in REQUIRED_FIRST_SECTIONS:
            raise ValueError(
                f"Unsupported language task: {task_type}"
            )

        normalized_findings = (
            self._normalize_findings(
                findings
            )
        )

        text = generated_text.strip()
        text_lower = text.lower()

        first_section = REQUIRED_FIRST_SECTIONS[
            task_type
        ]

        task_routing_compliant = (
            text.startswith(
                first_section
            )
        )

        first_section_position = text.find(
            first_section
        )
        limitation_position = text.find(
            "LIMITATIONS"
        )

        section_order_compliant = (
            first_section_position == 0
            and limitation_position
            > first_section_position
        )

        question_intent = (
            self._classify_question_intent(
                user_question
            )
            if task_type
            == "grounded_question_answering"
            else None
        )

        required_findings = (
            self._required_findings_for_audit(
                findings=normalized_findings,
                task_type=task_type,
                question_intent=question_intent,
            )
        )

        required_findings_mentioned = all(
            self._finding_is_mentioned(
                text_lower,
                label_name,
            )
            for label_name in required_findings
        )

        supplied_names = {
            finding["label_name"]
            for finding in normalized_findings
        }

        unsupported_finding_free = not (
            self._unsupported_finding_names(
                text_lower=text_lower,
                supplied_names=supplied_names,
            )
        )

        numeric_grounding_compliant = (
            self._numeric_values_are_grounded(
                text=text,
                findings=normalized_findings,
            )
        )

        safety_boundary_compliant = (
            self.educational_limitation.lower()
            in text_lower
        )

        no_target_boundary_compliant = (
            self._no_target_boundary_is_valid(
                text_lower=text_lower,
                findings=normalized_findings,
                no_target_finding=bool(
                    no_target_finding
                ),
            )
        )

        qa_refusal_compliant = (
            self._qa_refusal_is_valid(
                text_lower=text_lower,
                task_type=task_type,
                question_intent=question_intent,
            )
        )

        gradcam_boundary_compliant = (
            self._gradcam_boundary_is_valid(
                text_lower=text_lower,
                task_type=task_type,
                question_intent=question_intent,
            )
        )

        forbidden_claim_free = (
            self._forbidden_claim_free(
                text_lower
            )
        )

        audit = GuardrailAudit(
            task_routing_compliant=(
                task_routing_compliant
            ),
            section_order_compliant=(
                section_order_compliant
            ),
            required_findings_mentioned=(
                required_findings_mentioned
            ),
            unsupported_finding_free=(
                unsupported_finding_free
            ),
            numeric_grounding_compliant=(
                numeric_grounding_compliant
            ),
            safety_boundary_compliant=(
                safety_boundary_compliant
            ),
            no_target_boundary_compliant=(
                no_target_boundary_compliant
            ),
            qa_refusal_compliant=(
                qa_refusal_compliant
            ),
            gradcam_boundary_compliant=(
                gradcam_boundary_compliant
            ),
            forbidden_claim_free=(
                forbidden_claim_free
            ),
        )

        trigger_map = {
            "task_routing_issue": (
                not audit.task_routing_compliant
            ),
            "section_order_issue": (
                not audit.section_order_compliant
            ),
            "missing_required_finding_issue": (
                not audit.required_findings_mentioned
            ),
            "unsupported_finding_issue": (
                not audit.unsupported_finding_free
            ),
            "numeric_grounding_issue": (
                not audit.numeric_grounding_compliant
            ),
            "safety_boundary_issue": (
                not audit.safety_boundary_compliant
            ),
            "no_target_boundary_issue": (
                not audit.no_target_boundary_compliant
            ),
            "qa_refusal_issue": (
                not audit.qa_refusal_compliant
            ),
            "gradcam_boundary_issue": (
                not audit.gradcam_boundary_compliant
            ),
            "forbidden_claim_issue": (
                not audit.forbidden_claim_free
            ),
        }

        trigger_reasons = tuple(
            reason
            for reason in ALLOWED_TRIGGER_REASONS
            if trigger_map[reason]
        )

        return (
            audit,
            trigger_reasons,
            question_intent,
        )

    @staticmethod
    def _finding_sentence(
        finding: Mapping[str, Any],
        *,
        plain_language: bool,
    ) -> str:
        display_name = (
            DeterministicLanguageGuardrail
            ._display_name(
                finding["label_name"]
            )
        )

        if finding["decision"]:
            relationship = (
                "crossed its frozen threshold and was "
                f"categorized as {finding['confidence']}."
            )
        else:
            relationship = (
                "did not cross its frozen threshold."
            )

        if plain_language:
            return (
                f"For {display_name}, the model score was "
                f"{finding['probability']:.4f}, compared with "
                f"a decision threshold of "
                f"{finding['threshold']:.4f}. The score "
                f"{relationship} This label refers to "
                f"{finding['description']}"
            )

        return (
            f"- {display_name}: model probability "
            f"{finding['probability']:.4f}; frozen threshold "
            f"{finding['threshold']:.4f}; the probability "
            f"{relationship} {finding['description']}"
        )

    def _build_threshold_summary(
        self,
        findings: Sequence[Mapping[str, Any]],
    ) -> str:
        statements = []

        for finding in findings:
            display_name = self._display_name(
                finding["label_name"]
            )

            relationship = (
                "crossed"
                if finding["decision"]
                else "did not cross"
            )

            statements.append(
                f"{display_name} had a model probability of "
                f"{finding['probability']:.4f} and {relationship} "
                f"its frozen threshold of "
                f"{finding['threshold']:.4f}."
            )

        return " ".join(
            statements
        )

    def build_fallback(
        self,
        *,
        task_type: str,
        findings: Sequence[Any],
        no_target_finding: bool,
        user_question: str | None = None,
        question_intent: str | None = None,
    ) -> str:
        """Build one controlled response from supplied values only."""

        normalized_findings = (
            self._normalize_findings(
                findings
            )
        )

        if task_type == "structured_report":
            finding_lines = "\\n".join(
                self._finding_sentence(
                    finding,
                    plain_language=False,
                )
                for finding in normalized_findings
            )

            if no_target_finding:
                introduction = (
                    "No supplied finding probability crossed its "
                    "frozen decision threshold. The supplied "
                    "below-threshold relationships are:"
                )
            else:
                introduction = (
                    "The supplied model output reports the "
                    "following threshold relationships:"
                )

            return (
                "PRELIMINARY MODEL REPORT\\n"
                "MODEL FINDINGS\\n"
                f"{introduction}\\n"
                f"{finding_lines}\\n"
                "LIMITATIONS\\n"
                f"{self.educational_limitation} "
                f"{self.professional_review_guidance}"
            )

        if task_type == "plain_language_explanation":
            finding_text = " ".join(
                self._finding_sentence(
                    finding,
                    plain_language=True,
                )
                for finding in normalized_findings
            )

            return (
                "EXPLANATION\\n"
                "Using the provided model information, the result "
                "contains the following information. "
                f"{finding_text}\\n"
                "LIMITATIONS\\n"
                f"{self.educational_limitation} "
                f"{self.professional_review_guidance}"
            )

        if task_type == "educational_follow_up":
            crossed_names = [
                self._display_name(
                    finding["label_name"]
                )
                for finding in normalized_findings
                if finding["decision"]
            ]

            if crossed_names:
                relationship_text = (
                    "Relative to the frozen thresholds, the output "
                    "records threshold crossing for "
                    + ", ".join(crossed_names)
                    + "."
                )
            else:
                supplied_names = [
                    self._display_name(
                        finding["label_name"]
                    )
                    for finding in normalized_findings
                ]

                relationship_text = (
                    "None of the supplied finding probabilities "
                    "crossed their frozen thresholds. The supplied "
                    "below-threshold findings were "
                    + ", ".join(supplied_names)
                    + "."
                )

            return (
                "EDUCATIONAL FOLLOW-UP\\n"
                f"{relationship_text} "
                f"{self.professional_review_guidance} "
                "The supplied model output should be considered "
                "together with the complete clinical context.\\n"
                "LIMITATIONS\\n"
                f"{self.educational_limitation}"
            )

        if task_type != "grounded_question_answering":
            raise ValueError(
                f"Unsupported language task: {task_type}"
            )

        resolved_intent = (
            question_intent
            or self._classify_question_intent(
                user_question
            )
            or "output_summary"
        )

        if resolved_intent == "diagnosis_request":
            answer = (
                "The supplied model information cannot confirm a "
                "diagnosis. It contains threshold-based model "
                "outputs that require professional interpretation "
                "together with relevant clinical information."
            )

        elif resolved_intent == "treatment_request":
            answer = (
                "The supplied model information cannot recommend "
                "treatment or medication. Treatment decisions "
                "require review by a qualified healthcare "
                "professional using the complete clinical context."
            )

        elif resolved_intent == "gradcam_boundary":
            answer = self.gradcam_limitation

        elif resolved_intent == "no_target_meaning":
            if no_target_finding:
                answer = (
                    "The no-target-finding state means that none "
                    "of the supplied finding probabilities crossed "
                    "their frozen decision thresholds. It does not "
                    "rule out disease or replace professional review."
                )
            else:
                answer = (
                    "The no-target-finding state is false because "
                    "one or more supplied probabilities crossed "
                    "their frozen decision thresholds."
                )

        elif resolved_intent == "threshold_meaning":
            answer = (
                "A frozen threshold is the stored decision boundary "
                "used to convert a model probability into a "
                "threshold-crossing or not-crossing result. "
                + self._build_threshold_summary(
                    normalized_findings
                )
            )

        elif resolved_intent == "confidence_meaning":
            answer = (
                "The confidence category describes how each supplied "
                "model probability relates to its frozen threshold. "
                + self._build_threshold_summary(
                    normalized_findings
                )
            )

        else:
            answer = self._build_threshold_summary(
                normalized_findings
            )

        return (
            "ANSWER\\n"
            f"{answer}\\n"
            "LIMITATIONS\\n"
            f"{self.educational_limitation}"
        )

    def apply(
        self,
        *,
        task_type: str,
        raw_generated_text: str,
        findings: Sequence[Any],
        no_target_finding: bool,
        user_question: str | None = None,
    ) -> GuardedLanguageResult:
        """Accept a compliant generation or replace it deterministically."""

        started_at = time.perf_counter()

        (
            raw_audit,
            trigger_reasons,
            question_intent,
        ) = self.audit(
            task_type=task_type,
            generated_text=raw_generated_text,
            findings=findings,
            no_target_finding=no_target_finding,
            user_question=user_question,
        )

        if trigger_reasons:
            final_text = self.build_fallback(
                task_type=task_type,
                findings=findings,
                no_target_finding=no_target_finding,
                user_question=user_question,
                question_intent=question_intent,
            )

            guardrail_action = (
                self.FALLBACK_ACTION
            )
        else:
            final_text = (
                raw_generated_text.strip()
            )

            guardrail_action = (
                self.ACCEPTED_ACTION
            )

        latency_ms = (
            time.perf_counter()
            - started_at
        ) * 1000.0

        return GuardedLanguageResult(
            task_type=task_type,
            raw_generated_text=(
                raw_generated_text.strip()
            ),
            final_text=final_text,
            guardrail_action=guardrail_action,
            trigger_reasons=trigger_reasons,
            question_intent=question_intent,
            audit=raw_audit,
            guardrail_latency_ms=float(
                latency_ms
            ),
        )
'''

language_guardrail_service_path.write_text(
    language_guardrail_service_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Import the generated guardrail module
# -------------------------------------------------------------------------
importlib.invalidate_caches()

module_name = (
    "src.services.language_guardrail_service"
)

if module_name in sys.modules:
    language_guardrail_service_module = (
        importlib.reload(
            sys.modules[module_name]
        )
    )
else:
    language_guardrail_service_module = (
        importlib.import_module(
            module_name
        )
    )

DeterministicLanguageGuardrail = (
    language_guardrail_service_module
    .DeterministicLanguageGuardrail
)

GuardedLanguageResult = (
    language_guardrail_service_module
    .GuardedLanguageResult
)

ALLOWED_TRIGGER_REASONS = (
    language_guardrail_service_module
    .ALLOWED_TRIGGER_REASONS
)


# -------------------------------------------------------------------------
# Resolve professional-review guidance
# -------------------------------------------------------------------------
professional_review_guidance = locate_named_text(
    current_finding_contract,
    (
        "professional",
        "review",
    ),
)

if professional_review_guidance is None:
    professional_review_guidance = (
        globals().get(
            "PROFESSIONAL_REVIEW_GUIDANCE"
        )
    )

if not isinstance(
    professional_review_guidance,
    str,
) or not professional_review_guidance.strip():
    professional_review_guidance = (
        "A qualified healthcare professional can interpret "
        "the image together with symptoms, history, examination "
        "findings, and other tests."
    )


# -------------------------------------------------------------------------
# Initialize the deterministic guardrail
# -------------------------------------------------------------------------
language_guardrail_service = (
    DeterministicLanguageGuardrail(
        educational_limitation=(
            educational_limitation
        ),
        gradcam_limitation=(
            gradcam_limitation
        ),
        professional_review_guidance=(
            professional_review_guidance
        ),
    )
)


# -------------------------------------------------------------------------
# Normalize the findings used by the smoke-test generations
# -------------------------------------------------------------------------
guardrail_findings = tuple(
    grounded_input_serializer.normalize_finding(
        finding
    )
    for finding
    in selected_serialization_findings
)


# -------------------------------------------------------------------------
# Apply the guardrail to all four raw generations
# -------------------------------------------------------------------------
guarded_language_results = {}

for task_type, raw_result in (
    language_generation_by_task.items()
):
    serialized_record = (
        serialized_task_inputs[
            task_type
        ]
    )

    guarded_language_results[task_type] = (
        language_guardrail_service.apply(
            task_type=task_type,
            raw_generated_text=(
                raw_result.generated_text
            ),
            findings=guardrail_findings,
            no_target_finding=(
                serialization_no_target
            ),
            user_question=(
                serialized_record.user_question
            ),
        )
    )


# -------------------------------------------------------------------------
# Re-audit final guarded outputs
# -------------------------------------------------------------------------
post_guardrail_audits = {}

for task_type, guarded_result in (
    guarded_language_results.items()
):
    serialized_record = (
        serialized_task_inputs[
            task_type
        ]
    )

    (
        final_audit,
        final_trigger_reasons,
        _,
    ) = language_guardrail_service.audit(
        task_type=task_type,
        generated_text=(
            guarded_result.final_text
        ),
        findings=guardrail_findings,
        no_target_finding=(
            serialization_no_target
        ),
        user_question=(
            serialized_record.user_question
        ),
    )

    post_guardrail_audits[
        task_type
    ] = (
        final_audit,
        final_trigger_reasons,
    )


# -------------------------------------------------------------------------
# Summarize guardrail actions
# -------------------------------------------------------------------------
accepted_generation_count = sum(
    result.guardrail_action
    == "accepted_model_generation"
    for result
    in guarded_language_results.values()
)

fallback_generation_count = sum(
    result.guardrail_action
    == "safe_template_fallback"
    for result
    in guarded_language_results.values()
)

all_trigger_reasons = [
    trigger_reason
    for result
    in guarded_language_results.values()
    for trigger_reason
    in result.trigger_reasons
]

guardrail_service_checksum = hashlib.sha256(
    language_guardrail_service_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Validate the deterministic output contract
# -------------------------------------------------------------------------
guardrail_checks = {
    (
        "Language guardrail module was written"
    ): language_guardrail_service_path.is_file(),
    (
        "All four raw generations were processed"
    ): len(
        guarded_language_results
    ) == 4,
    (
        "Accepted and fallback actions reconcile"
    ): (
        accepted_generation_count
        + fallback_generation_count
        == 4
    ),
    (
        "Every trigger reason is registered"
    ): all(
        trigger_reason
        in ALLOWED_TRIGGER_REASONS
        for trigger_reason
        in all_trigger_reasons
    ),
    (
        "Fallback action exposes a trigger"
    ): all(
        (
            bool(result.trigger_reasons)
            if result.guardrail_action
            == "safe_template_fallback"
            else not result.trigger_reasons
        )
        for result
        in guarded_language_results.values()
    ),
    (
        "Raw generations remain preserved"
    ): all(
        result.raw_generated_text
        == language_generation_by_task[
            task_type
        ].generated_text
        for task_type, result
        in guarded_language_results.items()
    ),
    (
        "Every final output is non-empty"
    ): all(
        bool(
            result.final_text.strip()
        )
        for result
        in guarded_language_results.values()
    ),
    (
        "Every final output passes all contract checks"
    ): all(
        not final_trigger_reasons
        for (
            final_audit,
            final_trigger_reasons,
        )
        in post_guardrail_audits.values()
    ),
    (
        "Every final output preserves safety boundary"
    ): all(
        final_audit.safety_boundary_compliant
        for (
            final_audit,
            _,
        )
        in post_guardrail_audits.values()
    ),
    (
        "Every final output is numerically grounded"
    ): all(
        final_audit.numeric_grounding_compliant
        for (
            final_audit,
            _,
        )
        in post_guardrail_audits.values()
    ),
    (
        "Every final output is free of unsupported findings"
    ): all(
        final_audit.unsupported_finding_free
        for (
            final_audit,
            _,
        )
        in post_guardrail_audits.values()
    ),
    (
        "Every final output is free of forbidden claims"
    ): all(
        final_audit.forbidden_claim_free
        for (
            final_audit,
            _,
        )
        in post_guardrail_audits.values()
    ),
    (
        "Guardrail latency is positive"
    ): all(
        result.guardrail_latency_ms
        > 0.0
        for result
        in guarded_language_results.values()
    ),
    (
        "Guardrail checksum is available"
    ): len(
        guardrail_service_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the validation summary
# -------------------------------------------------------------------------
print(
    "DETERMINISTIC GROUNDED LANGUAGE OUTPUT GUARDRAIL SERVICE"
)
print("-" * 100)
print(
    f"{'Service module':31}: "
    f"{language_guardrail_service_path}"
)
print(
    f"{'Module SHA-256':31}: "
    f"{guardrail_service_checksum[:16]}..."
)
print(
    f"{'Outputs processed':31}: "
    f"{len(guarded_language_results)}"
)
print(
    f"{'Accepted model generations':31}: "
    f"{accepted_generation_count}"
)
print(
    f"{'Controlled template fallbacks':31}: "
    f"{fallback_generation_count}"
)
print("-" * 100)

for task_type, result in (
    guarded_language_results.items()
):
    trigger_text = (
        ", ".join(
            result.trigger_reasons
        )
        if result.trigger_reasons
        else "none"
    )

    print(
        f"{task_type:29}: "
        f"{result.guardrail_action:25} | "
        f"{trigger_text}"
    )

print("-" * 100)

for check_name, passed in guardrail_checks.items():
    print(
        f"{check_name:64}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    guardrail_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in guardrail_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Language guardrail validation failed: "
        + ", ".join(failed_checks)
    )

qa_guarded_result = guarded_language_results[
    "grounded_question_answering"
]

print("-" * 100)
print(
    "GROUNDED QUESTION-ANSWERING GUARDRAIL RESULT"
)
print("-" * 100)
print(
    f"Action   : "
    f"{qa_guarded_result.guardrail_action}"
)
print(
    "Triggers : "
    + (
        ", ".join(
            qa_guarded_result.trigger_reasons
        )
        if qa_guarded_result.trigger_reasons
        else "none"
    )
)
print("-" * 100)
print(
    qa_guarded_result.final_text
)
print("-" * 100)
print(
    "STATUS: DETERMINISTIC LANGUAGE GUARDRAIL READY "
    "FOR API RESPONSE INTEGRATION"
)

DETERMINISTIC GROUNDED LANGUAGE OUTPUT GUARDRAIL SERVICE
----------------------------------------------------------------------------------------------------
Service module                 : /home/jovyan/chest-xray-ai-assistant/src/services/language_guardrail_service.py
Module SHA-256                 : 8ea7b4bc2597574b...
Outputs processed              : 4
Accepted model generations     : 2
Controlled template fallbacks  : 2
----------------------------------------------------------------------------------------------------
structured_report            : accepted_model_generation | none
plain_language_explanation   : accepted_model_generation | none
grounded_question_answering  : safe_template_fallback    | numeric_grounding_issue
educational_follow_up        : safe_template_fallback    | missing_required_finding_issue
----------------------------------------------------------------------------------------------------
Language guardrail module was written                           : PA

<a id="nb07-5-10-thread-safe-operational-metrics-service"></a>
### 5.10 Thread-Safe Operational Metrics Service

This block creates the in-process metrics layer used by the LLMOps endpoint and API workflow. It records request outcomes, endpoint activity, service invocations, language guardrail actions, error codes, latency statistics, and service uptime without exposing images, prompts, generated text, filesystem paths, or other sensitive request content.


In [34]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import importlib
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path


# -------------------------------------------------------------------------
# Define the operational metrics service path
# -------------------------------------------------------------------------
operational_metrics_service_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "src/services/operational_metrics_service.py"
)

operational_metrics_service_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Write the reusable operational metrics service
# -------------------------------------------------------------------------
operational_metrics_service_source = '''
"""Thread-safe, privacy-preserving operational API metrics."""

from __future__ import annotations

import math
import threading
import time
from collections import Counter, deque
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Iterable


@dataclass(frozen=True)
class OperationalMetricsSnapshot:
    """Immutable snapshot of API operational metrics."""

    service_started_at_utc: datetime
    captured_at_utc: datetime
    uptime_seconds: float
    total_requests: int
    successful_requests: int
    failed_requests: int
    average_latency_ms: float
    p95_latency_ms: float
    maximum_latency_ms: float
    endpoint_request_counts: dict[str, int]
    status_code_counts: dict[str, int]
    error_code_counts: dict[str, int]
    service_invocation_counts: dict[str, int]
    accepted_model_generations: int
    safe_template_fallbacks: int
    raw_generation_acceptance_rate: float
    safe_fallback_rate: float


class OperationalMetricsService:
    """Collect bounded operational measurements without request content."""

    REGISTERED_SERVICES = (
        "computer_vision",
        "gradcam",
        "grounded_language",
        "language_guardrail",
        "prediction_store",
    )

    LANGUAGE_ACTIONS = (
        "accepted_model_generation",
        "safe_template_fallback",
    )

    def __init__(
        self,
        *,
        registered_endpoints: Iterable[str],
        maximum_latency_samples: int = 10_000,
    ) -> None:
        endpoints = tuple(
            dict.fromkeys(
                str(endpoint).strip()
                for endpoint in registered_endpoints
                if str(endpoint).strip()
            )
        )

        if not endpoints:
            raise ValueError(
                "At least one registered endpoint is required."
            )

        if maximum_latency_samples <= 0:
            raise ValueError(
                "Maximum latency samples must be positive."
            )

        self.registered_endpoints = endpoints
        self.maximum_latency_samples = int(
            maximum_latency_samples
        )

        self._lock = threading.RLock()
        self._started_at_utc = datetime.now(
            timezone.utc
        )
        self._started_at_monotonic = (
            time.monotonic()
        )

        self._total_requests = 0
        self._successful_requests = 0
        self._failed_requests = 0

        self._endpoint_counts = Counter(
            {
                endpoint: 0
                for endpoint in endpoints
            }
        )

        self._status_code_counts = Counter()
        self._error_code_counts = Counter()

        self._service_invocation_counts = Counter(
            {
                service_name: 0
                for service_name
                in self.REGISTERED_SERVICES
            }
        )

        self._language_action_counts = Counter(
            {
                action: 0
                for action
                in self.LANGUAGE_ACTIONS
            }
        )

        self._latency_samples = deque(
            maxlen=self.maximum_latency_samples
        )

    def _validate_endpoint(
        self,
        endpoint: str,
    ) -> str:
        normalized_endpoint = str(
            endpoint
        ).strip()

        if normalized_endpoint not in self.registered_endpoints:
            raise ValueError(
                f"Unregistered endpoint metric: "
                f"{normalized_endpoint}"
            )

        return normalized_endpoint

    @staticmethod
    def _validate_latency(
        latency_ms: float,
    ) -> float:
        normalized_latency = float(
            latency_ms
        )

        if (
            not math.isfinite(
                normalized_latency
            )
            or normalized_latency < 0.0
        ):
            raise ValueError(
                "Request latency must be finite and non-negative."
            )

        return normalized_latency

    def record_request(
        self,
        *,
        endpoint: str,
        status_code: int,
        latency_ms: float,
        error_code: str | None = None,
    ) -> None:
        """Record one completed API request."""

        normalized_endpoint = (
            self._validate_endpoint(
                endpoint
            )
        )

        normalized_status_code = int(
            status_code
        )

        if not 100 <= normalized_status_code <= 599:
            raise ValueError(
                "HTTP status code must be between 100 and 599."
            )

        normalized_latency = (
            self._validate_latency(
                latency_ms
            )
        )

        if error_code is not None:
            normalized_error_code = str(
                error_code
            ).strip()

            if not normalized_error_code:
                raise ValueError(
                    "Error code cannot be blank."
                )
        else:
            normalized_error_code = None

        with self._lock:
            self._total_requests += 1
            self._endpoint_counts[
                normalized_endpoint
            ] += 1
            self._status_code_counts[
                str(normalized_status_code)
            ] += 1
            self._latency_samples.append(
                normalized_latency
            )

            if 200 <= normalized_status_code < 400:
                self._successful_requests += 1
            else:
                self._failed_requests += 1

                if normalized_error_code is not None:
                    self._error_code_counts[
                        normalized_error_code
                    ] += 1

    def record_service_invocation(
        self,
        service_name: str,
        count: int = 1,
    ) -> None:
        """Record calls to one controlled internal service."""

        normalized_service_name = str(
            service_name
        ).strip()

        if (
            normalized_service_name
            not in self.REGISTERED_SERVICES
        ):
            raise ValueError(
                f"Unregistered service metric: "
                f"{normalized_service_name}"
            )

        normalized_count = int(
            count
        )

        if normalized_count <= 0:
            raise ValueError(
                "Service invocation count must be positive."
            )

        with self._lock:
            self._service_invocation_counts[
                normalized_service_name
            ] += normalized_count

    def record_language_action(
        self,
        action: str,
        count: int = 1,
    ) -> None:
        """Record accepted generations and safe fallbacks."""

        normalized_action = str(
            action
        ).strip()

        if normalized_action not in self.LANGUAGE_ACTIONS:
            raise ValueError(
                f"Unsupported language guardrail action: "
                f"{normalized_action}"
            )

        normalized_count = int(
            count
        )

        if normalized_count <= 0:
            raise ValueError(
                "Language action count must be positive."
            )

        with self._lock:
            self._language_action_counts[
                normalized_action
            ] += normalized_count

    @staticmethod
    def _percentile_95(
        values: tuple[float, ...],
    ) -> float:
        if not values:
            return 0.0

        ordered_values = sorted(
            values
        )

        rank = max(
            0,
            math.ceil(
                0.95
                * len(ordered_values)
            )
            - 1,
        )

        return float(
            ordered_values[rank]
        )

    def snapshot(
        self,
    ) -> OperationalMetricsSnapshot:
        """Return one internally consistent immutable snapshot."""

        with self._lock:
            total_requests = int(
                self._total_requests
            )
            successful_requests = int(
                self._successful_requests
            )
            failed_requests = int(
                self._failed_requests
            )

            latency_values = tuple(
                self._latency_samples
            )

            endpoint_counts = {
                endpoint: int(
                    self._endpoint_counts[
                        endpoint
                    ]
                )
                for endpoint
                in self.registered_endpoints
            }

            status_code_counts = {
                key: int(value)
                for key, value
                in sorted(
                    self._status_code_counts.items()
                )
            }

            error_code_counts = {
                key: int(value)
                for key, value
                in sorted(
                    self._error_code_counts.items()
                )
            }

            service_invocation_counts = {
                service_name: int(
                    self._service_invocation_counts[
                        service_name
                    ]
                )
                for service_name
                in self.REGISTERED_SERVICES
            }

            accepted_generations = int(
                self._language_action_counts[
                    "accepted_model_generation"
                ]
            )

            fallback_generations = int(
                self._language_action_counts[
                    "safe_template_fallback"
                ]
            )

        if latency_values:
            average_latency = (
                sum(latency_values)
                / len(latency_values)
            )
            maximum_latency = max(
                latency_values
            )
            p95_latency = self._percentile_95(
                latency_values
            )
        else:
            average_latency = 0.0
            maximum_latency = 0.0
            p95_latency = 0.0

        total_language_generations = (
            accepted_generations
            + fallback_generations
        )

        if total_language_generations:
            acceptance_rate = (
                accepted_generations
                / total_language_generations
            )
            fallback_rate = (
                fallback_generations
                / total_language_generations
            )
        else:
            acceptance_rate = 0.0
            fallback_rate = 0.0

        return OperationalMetricsSnapshot(
            service_started_at_utc=(
                self._started_at_utc
            ),
            captured_at_utc=datetime.now(
                timezone.utc
            ),
            uptime_seconds=max(
                0.0,
                time.monotonic()
                - self._started_at_monotonic,
            ),
            total_requests=total_requests,
            successful_requests=successful_requests,
            failed_requests=failed_requests,
            average_latency_ms=float(
                average_latency
            ),
            p95_latency_ms=float(
                p95_latency
            ),
            maximum_latency_ms=float(
                maximum_latency
            ),
            endpoint_request_counts=endpoint_counts,
            status_code_counts=status_code_counts,
            error_code_counts=error_code_counts,
            service_invocation_counts=(
                service_invocation_counts
            ),
            accepted_model_generations=(
                accepted_generations
            ),
            safe_template_fallbacks=(
                fallback_generations
            ),
            raw_generation_acceptance_rate=float(
                acceptance_rate
            ),
            safe_fallback_rate=float(
                fallback_rate
            ),
        )
'''

operational_metrics_service_path.write_text(
    operational_metrics_service_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Import the generated service module
# -------------------------------------------------------------------------
importlib.invalidate_caches()

module_name = (
    "src.services.operational_metrics_service"
)

if module_name in sys.modules:
    operational_metrics_module = importlib.reload(
        sys.modules[module_name]
    )
else:
    operational_metrics_module = importlib.import_module(
        module_name
    )

OperationalMetricsService = (
    operational_metrics_module
    .OperationalMetricsService
)

OperationalMetricsSnapshot = (
    operational_metrics_module
    .OperationalMetricsSnapshot
)


# -------------------------------------------------------------------------
# Freeze the registered endpoint metric names
# -------------------------------------------------------------------------
registered_metric_endpoints = (
    "GET /health",
    "GET /api/v1/model/info",
    "GET /api/v1/model/metrics",
    "POST /api/v1/image/classify",
    "POST /api/v1/image/analyze",
    "POST /api/v1/report/generate",
    "POST /api/v1/explanation/generate",
    "POST /api/v1/question/answer",
    "POST /api/v1/follow-up/recommend",
    "POST /api/v1/analyze-complete",
    "GET /api/v1/predictions/{prediction_id}",
    "GET /api/v1/llmops/metrics",
)


# -------------------------------------------------------------------------
# Exercise an isolated metrics instance concurrently
# -------------------------------------------------------------------------
metrics_test_service = OperationalMetricsService(
    registered_endpoints=(
        registered_metric_endpoints
    ),
    maximum_latency_samples=100,
)


def record_test_metric(
    request_number,
):
    endpoint = registered_metric_endpoints[
        request_number
        % len(registered_metric_endpoints)
    ]

    if request_number % 5 == 0:
        metrics_test_service.record_request(
            endpoint=endpoint,
            status_code=400,
            latency_ms=(
                10.0
                + request_number
            ),
            error_code="INVALID_IMAGE",
        )
    else:
        metrics_test_service.record_request(
            endpoint=endpoint,
            status_code=200,
            latency_ms=(
                10.0
                + request_number
            ),
        )

    metrics_test_service.record_service_invocation(
        "prediction_store"
    )


with ThreadPoolExecutor(
    max_workers=8
) as executor:
    list(
        executor.map(
            record_test_metric,
            range(100),
        )
    )

for _ in range(7):
    metrics_test_service.record_language_action(
        "accepted_model_generation"
    )

for _ in range(3):
    metrics_test_service.record_language_action(
        "safe_template_fallback"
    )

metrics_test_snapshot = (
    metrics_test_service.snapshot()
)


# -------------------------------------------------------------------------
# Confirm controlled validation failures
# -------------------------------------------------------------------------
negative_latency_rejected = False
unknown_endpoint_rejected = False
unknown_service_rejected = False
unknown_action_rejected = False

try:
    metrics_test_service.record_request(
        endpoint="GET /health",
        status_code=200,
        latency_ms=-1.0,
    )
except ValueError:
    negative_latency_rejected = True

try:
    metrics_test_service.record_request(
        endpoint="GET /unregistered",
        status_code=200,
        latency_ms=1.0,
    )
except ValueError:
    unknown_endpoint_rejected = True

try:
    metrics_test_service.record_service_invocation(
        "unregistered_service"
    )
except ValueError:
    unknown_service_rejected = True

try:
    metrics_test_service.record_language_action(
        "unregistered_action"
    )
except ValueError:
    unknown_action_rejected = True


# -------------------------------------------------------------------------
# Create the clean service instance used by the API
# -------------------------------------------------------------------------
operational_metrics_service = (
    OperationalMetricsService(
        registered_endpoints=(
            registered_metric_endpoints
        ),
        maximum_latency_samples=10_000,
    )
)

initial_metrics_snapshot = (
    operational_metrics_service.snapshot()
)

operational_metrics_checksum = hashlib.sha256(
    operational_metrics_service_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Validate the operational metrics contract
# -------------------------------------------------------------------------
operational_metrics_checks = {
    (
        "Operational metrics module was written"
    ): operational_metrics_service_path.is_file(),
    (
        "One hundred concurrent requests were recorded"
    ): (
        metrics_test_snapshot.total_requests
        == 100
    ),
    (
        "Successful request count is preserved"
    ): (
        metrics_test_snapshot.successful_requests
        == 80
    ),
    (
        "Failed request count is preserved"
    ): (
        metrics_test_snapshot.failed_requests
        == 20
    ),
    (
        "Request counts reconcile"
    ): (
        metrics_test_snapshot.successful_requests
        + metrics_test_snapshot.failed_requests
        == metrics_test_snapshot.total_requests
    ),
    (
        "Endpoint counts reconcile"
    ): (
        sum(
            metrics_test_snapshot
            .endpoint_request_counts.values()
        )
        == 100
    ),
    (
        "Status-code counts reconcile"
    ): (
        sum(
            metrics_test_snapshot
            .status_code_counts.values()
        )
        == 100
    ),
    (
        "Error-code count is preserved"
    ): (
        metrics_test_snapshot
        .error_code_counts.get(
            "INVALID_IMAGE"
        )
        == 20
    ),
    (
        "Service invocation count is preserved"
    ): (
        metrics_test_snapshot
        .service_invocation_counts[
            "prediction_store"
        ]
        == 100
    ),
    (
        "Language action counts are preserved"
    ): (
        metrics_test_snapshot
        .accepted_model_generations
        == 7
        and metrics_test_snapshot
        .safe_template_fallbacks
        == 3
    ),
    (
        "Language action rates are preserved"
    ): (
        abs(
            metrics_test_snapshot
            .raw_generation_acceptance_rate
            - 0.7
        )
        < 1e-12
        and abs(
            metrics_test_snapshot
            .safe_fallback_rate
            - 0.3
        )
        < 1e-12
    ),
    (
        "Latency statistics are non-negative"
    ): (
        metrics_test_snapshot
        .average_latency_ms
        >= 0.0
        and metrics_test_snapshot
        .p95_latency_ms
        >= 0.0
        and metrics_test_snapshot
        .maximum_latency_ms
        >= 0.0
    ),
    (
        "Uptime is non-negative"
    ): (
        metrics_test_snapshot
        .uptime_seconds
        >= 0.0
    ),
    (
        "Negative latency is rejected"
    ): negative_latency_rejected,
    (
        "Unknown endpoint is rejected"
    ): unknown_endpoint_rejected,
    (
        "Unknown service is rejected"
    ): unknown_service_rejected,
    (
        "Unknown language action is rejected"
    ): unknown_action_rejected,
    (
        "Production metrics start empty"
    ): (
        initial_metrics_snapshot.total_requests
        == 0
        and initial_metrics_snapshot
        .accepted_model_generations
        == 0
        and initial_metrics_snapshot
        .safe_template_fallbacks
        == 0
    ),
    (
        "Metrics service checksum is available"
    ): len(
        operational_metrics_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the validation summary
# -------------------------------------------------------------------------
print(
    "THREAD-SAFE OPERATIONAL METRICS SERVICE"
)
print("-" * 100)
print(
    f"{'Service module':32}: "
    f"{operational_metrics_service_path}"
)
print(
    f"{'Module SHA-256':32}: "
    f"{operational_metrics_checksum[:16]}..."
)
print(
    f"{'Registered endpoints':32}: "
    f"{len(registered_metric_endpoints)}"
)
print(
    f"{'Concurrent test requests':32}: "
    f"{metrics_test_snapshot.total_requests}"
)
print(
    f"{'Successful test requests':32}: "
    f"{metrics_test_snapshot.successful_requests}"
)
print(
    f"{'Failed test requests':32}: "
    f"{metrics_test_snapshot.failed_requests}"
)
print(
    f"{'Average test latency':32}: "
    f"{metrics_test_snapshot.average_latency_ms:.2f} ms"
)
print(
    f"{'P95 test latency':32}: "
    f"{metrics_test_snapshot.p95_latency_ms:.2f} ms"
)
print(
    f"{'Raw generation acceptance':32}: "
    f"{metrics_test_snapshot.raw_generation_acceptance_rate:.4f}"
)
print(
    f"{'Safe fallback rate':32}: "
    f"{metrics_test_snapshot.safe_fallback_rate:.4f}"
)
print("-" * 100)

for check_name, passed in (
    operational_metrics_checks.items()
):
    print(
        f"{check_name:61}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    operational_metrics_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in operational_metrics_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Operational metrics service validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: OPERATIONAL METRICS SERVICE READY "
    "FOR FASTAPI INTEGRATION"
)

THREAD-SAFE OPERATIONAL METRICS SERVICE
----------------------------------------------------------------------------------------------------
Service module                  : /home/jovyan/chest-xray-ai-assistant/src/services/operational_metrics_service.py
Module SHA-256                  : 2a4a02e4a8843db3...
Registered endpoints            : 12
Concurrent test requests        : 100
Successful test requests        : 80
Failed test requests            : 20
Average test latency            : 59.50 ms
P95 test latency                : 104.00 ms
Raw generation acceptance       : 0.7000
Safe fallback rate              : 0.3000
----------------------------------------------------------------------------------------------------
Operational metrics module was written                       : PASS
One hundred concurrent requests were recorded                : PASS
Successful request count is preserved                        : PASS
Failed request count is preserved                            : PASS

**[↑ Back to notebook index](#notebook-index)**


<a id="nb07-6-fastapi-workflow-integration"></a>
## 6. FastAPI Workflow Integration

<a id="nb07-6-1-runtime-service-and-schema-interface-registry"></a>
### 6.1 Runtime Service and Schema Interface Registry

Before composing the route workflow, this block freezes the actual constructors, public methods, result fields, and Pydantic endpoint fields exposed by the completed service and schema layers. The exported interface registry provides a machine-readable integration contract and prevents route implementation from relying on assumed method or field names.


In [36]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import inspect
import json
from dataclasses import fields, is_dataclass
from pathlib import Path

import api.schemas as api_schemas


# -------------------------------------------------------------------------
# Define the interface-registry artifact
# -------------------------------------------------------------------------
service_interface_registry_path = Path(
    "/home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data/outputs/api/"
    "service_interface_registry.json"
)

service_interface_registry_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Register the implemented service classes
# -------------------------------------------------------------------------
service_classes = {
    "ImageValidationService": type(
        image_validation_service
    ),
    "ComputerVisionService": type(
        computer_vision_service
    ),
    "PredictionStoreService": type(
        prediction_store_service
    ),
    "GradCAMService": type(
        gradcam_service
    ),
    "GroundedInputSerializer": type(
        grounded_input_serializer
    ),
    "GroundedLanguageModelService": type(
        grounded_language_model_service
    ),
    "DeterministicLanguageGuardrail": type(
        language_guardrail_service
    ),
    "OperationalMetricsService": type(
        operational_metrics_service
    ),
}


# -------------------------------------------------------------------------
# Resolve the stored-prediction representative created earlier
# -------------------------------------------------------------------------
stored_prediction_interface_example = None

for candidate_name in (
    "stored_prediction_response",
    "stored_response",
    "stored_prediction_example",
):
    candidate_value = globals().get(
        candidate_name
    )

    if (
        candidate_value is not None
        and type(candidate_value).__name__
        == "StoredPredictionResponse"
    ):
        stored_prediction_interface_example = (
            candidate_value
        )
        break

if stored_prediction_interface_example is None:
    for candidate_value in list(
        globals().values()
    ):
        if (
            candidate_value is not None
            and type(candidate_value).__name__
            == "StoredPredictionResponse"
        ):
            stored_prediction_interface_example = (
                candidate_value
            )
            break

if stored_prediction_interface_example is None:
    raise RuntimeError(
        "A representative StoredPredictionResponse "
        "could not be resolved from the completed schema "
        "or prediction-store cells."
    )


# -------------------------------------------------------------------------
# Register representative service-result objects
# -------------------------------------------------------------------------
representative_results = {
    "validated_image": validated_png,
    "classification_result": (
        structural_smoke_result
    ),
    "gradcam_result": (
        gradcam_smoke_result
    ),
    "serialized_grounding_input": (
        serialized_task_inputs[
            "grounded_question_answering"
        ]
    ),
    "language_generation_result": (
        language_generation_by_task[
            "grounded_question_answering"
        ]
    ),
    "guarded_language_result": (
        guarded_language_results[
            "grounded_question_answering"
        ]
    ),
    "operational_metrics_snapshot": (
        initial_metrics_snapshot
    ),
    "stored_prediction_response": (
        stored_prediction_interface_example
    ),
}


# -------------------------------------------------------------------------
# Register endpoint-facing Pydantic models
# -------------------------------------------------------------------------
request_schema_names = (
    "GroundedGenerationRequest",
    "GroundedQuestionRequest",
    "CompleteAnalysisOptions",
)

response_schema_names = (
    "APIErrorResponse",
    "ClassificationResponse",
    "CompleteAnalysisResponse",
    "HealthResponse",
    "ImageAnalysisResponse",
    "LanguageGenerationResponse",
    "ModelInfoResponse",
    "ModelMetricsResponse",
    "OperationalMetricsResponse",
    "StoredPredictionResponse",
)

registered_schema_names = (
    *request_schema_names,
    *response_schema_names,
)


# -------------------------------------------------------------------------
# Build reusable inspection helpers
# -------------------------------------------------------------------------
def safe_signature(
    callable_object,
):
    try:
        return str(
            inspect.signature(
                callable_object
            )
        )
    except (
        TypeError,
        ValueError,
    ):
        return "signature_unavailable"


def public_method_registry(
    service_class,
):
    public_methods = {}

    for method_name, method in inspect.getmembers(
        service_class,
        predicate=inspect.isfunction,
    ):
        if method_name.startswith("_"):
            continue

        public_methods[method_name] = safe_signature(
            method
        )

    return public_methods


def object_field_registry(
    result_object,
):
    if is_dataclass(
        result_object
    ):
        return {
            field.name: str(
                field.type
            )
            for field in fields(
                result_object
            )
        }

    model_fields = getattr(
        type(result_object),
        "model_fields",
        None,
    )

    if model_fields is not None:
        return {
            field_name: str(
                field_info.annotation
            )
            for field_name, field_info
            in model_fields.items()
        }

    public_attributes = {}

    for attribute_name in dir(
        result_object
    ):
        if attribute_name.startswith("_"):
            continue

        try:
            attribute_value = getattr(
                result_object,
                attribute_name,
            )
        except Exception:
            continue

        if callable(
            attribute_value
        ):
            continue

        public_attributes[
            attribute_name
        ] = type(
            attribute_value
        ).__name__

    return public_attributes


# -------------------------------------------------------------------------
# Build the service interface section
# -------------------------------------------------------------------------
service_registry = {}

for service_name, service_class in (
    service_classes.items()
):
    service_registry[
        service_name
    ] = {
        "module": service_class.__module__,
        "constructor": safe_signature(
            service_class
        ),
        "public_methods": (
            public_method_registry(
                service_class
            )
        ),
    }


# -------------------------------------------------------------------------
# Build the result-object interface section
# -------------------------------------------------------------------------
result_registry = {
    result_name: {
        "class_name": (
            type(result_object).__name__
        ),
        "module": (
            type(result_object).__module__
        ),
        "fields": object_field_registry(
            result_object
        ),
    }
    for result_name, result_object
    in representative_results.items()
}


# -------------------------------------------------------------------------
# Build the Pydantic schema interface section
# -------------------------------------------------------------------------
schema_registry = {}
missing_schema_names = []

for schema_name in registered_schema_names:
    schema_class = getattr(
        api_schemas,
        schema_name,
        None,
    )

    if schema_class is None:
        missing_schema_names.append(
            schema_name
        )
        continue

    schema_registry[
        schema_name
    ] = {
        "module": schema_class.__module__,
        "fields": {
            field_name: {
                "annotation": str(
                    field_info.annotation
                ),
                "required": (
                    field_info.is_required()
                ),
                "default": (
                    None
                    if field_info.is_required()
                    else repr(
                        field_info.default
                    )
                ),
            }
            for field_name, field_info
            in schema_class.model_fields.items()
        },
    }


# -------------------------------------------------------------------------
# Export the complete integration registry
# -------------------------------------------------------------------------
service_interface_registry = {
    "registry_version": (
        "api-service-interface-registry-v1"
    ),
    "services": service_registry,
    "representative_results": (
        result_registry
    ),
    "pydantic_schemas": (
        schema_registry
    ),
}

service_interface_registry_path.write_text(
    json.dumps(
        service_interface_registry,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

service_interface_registry_checksum = (
    hashlib.sha256(
        service_interface_registry_path.read_bytes()
    ).hexdigest()
)


# -------------------------------------------------------------------------
# Validate the interface registry
# -------------------------------------------------------------------------
interface_registry_checks = {
    (
        "Eight reusable services are registered"
    ): len(
        service_registry
    ) == 8,
    (
        "Every service exposes public methods"
    ): all(
        bool(
            service_details[
                "public_methods"
            ]
        )
        for service_details
        in service_registry.values()
    ),
    (
        "Eight representative results are registered"
    ): len(
        result_registry
    ) == 8,
    (
        "Every representative result exposes fields"
    ): all(
        bool(
            result_details[
                "fields"
            ]
        )
        for result_details
        in result_registry.values()
    ),
    (
        "All request schemas are available"
    ): all(
        schema_name
        in schema_registry
        for schema_name
        in request_schema_names
    ),
    (
        "All response schemas are available"
    ): all(
        schema_name
        in schema_registry
        for schema_name
        in response_schema_names
    ),
    (
        "No registered schema is missing"
    ): not missing_schema_names,
    (
        "Interface registry was exported"
    ): (
        service_interface_registry_path.is_file()
    ),
    (
        "Interface registry checksum is available"
    ): len(
        service_interface_registry_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the integration interfaces
# -------------------------------------------------------------------------
print(
    "RUNTIME SERVICE AND SCHEMA INTERFACE REGISTRY"
)
print("-" * 100)
print(
    f"{'Registry version':29}: "
    f"{service_interface_registry['registry_version']}"
)
print(
    f"{'Registry path':29}: "
    f"{service_interface_registry_path}"
)
print(
    f"{'Registry SHA-256':29}: "
    f"{service_interface_registry_checksum[:16]}..."
)
print(
    f"{'Reusable services':29}: "
    f"{len(service_registry)}"
)
print(
    f"{'Representative results':29}: "
    f"{len(result_registry)}"
)
print(
    f"{'Pydantic schemas':29}: "
    f"{len(schema_registry)}"
)
print("-" * 100)
print(
    "SERVICE PUBLIC METHODS"
)
print("-" * 100)

for service_name, service_details in (
    service_registry.items()
):
    method_summary = " | ".join(
        (
            f"{method_name}"
            f"{method_signature}"
        )
        for method_name, method_signature
        in service_details[
            "public_methods"
        ].items()
    )

    print(
        f"{service_name:36}: "
        f"{method_summary}"
    )

print("-" * 100)
print(
    "REPRESENTATIVE RESULT FIELDS"
)
print("-" * 100)

for result_name, result_details in (
    result_registry.items()
):
    print(
        f"{result_name:36}: "
        + ", ".join(
            result_details[
                "fields"
            ].keys()
        )
    )

print("-" * 100)
print(
    "ENDPOINT PYDANTIC FIELDS"
)
print("-" * 100)

for schema_name in registered_schema_names:
    schema_details = schema_registry[
        schema_name
    ]

    print(
        f"{schema_name:36}: "
        + ", ".join(
            schema_details[
                "fields"
            ].keys()
        )
    )

print("-" * 100)

for check_name, passed in (
    interface_registry_checks.items()
):
    print(
        f"{check_name:59}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    interface_registry_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in interface_registry_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Service interface registry validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: SERVICE INTERFACES FROZEN FOR "
    "FASTAPI WORKFLOW COMPOSITION"
)

RUNTIME SERVICE AND SCHEMA INTERFACE REGISTRY
----------------------------------------------------------------------------------------------------
Registry version             : api-service-interface-registry-v1
Registry path                : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/service_interface_registry.json
Registry SHA-256             : 7a45de93c180413b...
Reusable services            : 8
Representative results       : 8
Pydantic schemas             : 13
----------------------------------------------------------------------------------------------------
SERVICE PUBLIC METHODS
----------------------------------------------------------------------------------------------------
ImageValidationService              : validate_and_decode(self, *, filename: str, media_type: str, content: bytes) -> src.services.image_service.ValidatedImage
ComputerVisionService               : confidence_category(probability: float, threshold: float) -> str | predict(sel

<a id="nb07-6-2-stored-prediction-language-workflow-orchestration"></a>
### 6.2 Stored-Prediction Language Workflow Orchestration

This block combines prediction retrieval, finding selection, grounded serialization, fine-tuned model inference, deterministic guardrail processing, language-output storage, and operational metric recording into one reusable workflow. It supports all four language endpoints while ensuring that clients provide only a prediction identifier and an optional validated question; grounding values always come from the server-side prediction store.


In [38]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import importlib
import inspect
import sys
from pathlib import Path

from api.core.runtime import (
    RequestContext,
    build_success_metadata,
)


# -------------------------------------------------------------------------
# Reload the workflow service written by the previous execution
# -------------------------------------------------------------------------
language_workflow_service_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "src/services/language_workflow_service.py"
)

if not language_workflow_service_path.is_file():
    raise FileNotFoundError(
        "The language workflow service module was not written. "
        "Rerun the original workflow cell once to create it."
    )

importlib.invalidate_caches()

module_name = (
    "src.services.language_workflow_service"
)

if module_name in sys.modules:
    language_workflow_module = importlib.reload(
        sys.modules[module_name]
    )
else:
    language_workflow_module = importlib.import_module(
        module_name
    )

StoredPredictionLanguageWorkflow = (
    language_workflow_module
    .StoredPredictionLanguageWorkflow
)

LanguageWorkflowExecution = (
    language_workflow_module
    .LanguageWorkflowExecution
)


# -------------------------------------------------------------------------
# Initialize the workflow using the active service instances
# -------------------------------------------------------------------------
stored_prediction_language_workflow = (
    StoredPredictionLanguageWorkflow(
        prediction_store=(
            prediction_store_service
        ),
        grounding_serializer=(
            grounded_input_serializer
        ),
        language_model_service=(
            grounded_language_model_service
        ),
        language_guardrail=(
            language_guardrail_service
        ),
        operational_metrics=(
            operational_metrics_service
        ),
        metadata_builder=(
            build_success_metadata
        ),
    )
)


# -------------------------------------------------------------------------
# Resolve an existing StoredPredictionResponse example
# -------------------------------------------------------------------------
workflow_schema_example = None

for candidate_name in (
    "stored_prediction_interface_example",
    "stored_prediction_response",
    "stored_response",
    "stored_prediction_example",
):
    candidate_value = globals().get(
        candidate_name
    )

    if (
        candidate_value is not None
        and type(candidate_value).__name__
        == "StoredPredictionResponse"
    ):
        workflow_schema_example = (
            candidate_value
        )
        break

if workflow_schema_example is None:
    for candidate_value in list(
        globals().values()
    ):
        if (
            candidate_value is not None
            and type(candidate_value).__name__
            == "StoredPredictionResponse"
        ):
            workflow_schema_example = (
                candidate_value
            )
            break

if workflow_schema_example is None:
    raise RuntimeError(
        "A representative StoredPredictionResponse "
        "could not be resolved from the completed schema cells."
    )


# -------------------------------------------------------------------------
# Create a fresh prediction in the active prediction store
# -------------------------------------------------------------------------
workflow_seed_record = (
    prediction_store_service.create(
        image=(
            workflow_schema_example.image
        ),
        findings=list(
            workflow_schema_example.findings
        ),
        crossed_finding_names=list(
            workflow_schema_example
            .crossed_finding_names
        ),
        no_target_finding=(
            workflow_schema_example
            .no_target_finding
        ),
        interpretation=(
            workflow_schema_example
            .interpretation
        ),
    )
)

workflow_prediction_id = (
    workflow_seed_record.prediction_id
)

workflow_stored_record = (
    prediction_store_service.get(
        workflow_prediction_id
    )
)


# -------------------------------------------------------------------------
# Execute the grounded question-answering workflow
# -------------------------------------------------------------------------
workflow_request_context = (
    RequestContext()
)

language_workflow_execution = (
    stored_prediction_language_workflow.generate(
        prediction_id=(
            workflow_prediction_id
        ),
        task_type=(
            "grounded_question_answering"
        ),
        request_context=(
            workflow_request_context
        ),
        question=(
            "What does the supplied model information indicate?"
        ),
    )
)

language_workflow_response = (
    language_workflow_execution.response
)

updated_workflow_record = (
    prediction_store_service.get(
        workflow_prediction_id
    )
)

workflow_metrics_snapshot = (
    operational_metrics_service.snapshot()
)

language_workflow_checksum = hashlib.sha256(
    language_workflow_service_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Confirm that client-supplied grounding fields are excluded
# -------------------------------------------------------------------------
workflow_signature = inspect.signature(
    stored_prediction_language_workflow.generate
)

workflow_parameter_names = set(
    workflow_signature.parameters
)

client_grounding_parameters = {
    "findings",
    "probabilities",
    "thresholds",
    "descriptions",
    "no_target_finding",
    "model_version",
}

client_grounding_rejected_by_interface = (
    workflow_parameter_names.isdisjoint(
        client_grounding_parameters
    )
)


# -------------------------------------------------------------------------
# Validate the workflow
# -------------------------------------------------------------------------
stored_task_names = {
    output.task_type
    for output
    in updated_workflow_record.language_outputs
}

language_workflow_checks = {
    (
        "Language workflow module is available"
    ): language_workflow_service_path.is_file(),
    (
        "Fresh prediction exists in the active store"
    ): (
        workflow_stored_record.prediction_id
        == workflow_prediction_id
    ),
    (
        "Stored prediction identifier is preserved"
    ): (
        language_workflow_response.prediction_id
        == workflow_prediction_id
    ),
    (
        "Question-answer task is preserved"
    ): (
        language_workflow_response.task_type
        == "grounded_question_answering"
    ),
    (
        "Normalized question is preserved"
    ): (
        language_workflow_response.question
        == (
            "What does the supplied model "
            "information indicate?"
        )
    ),
    (
        "Grounding comes from stored findings"
    ): set(
        language_workflow_response
        .grounded_finding_names
    ).issubset(
        {
            finding.label_name
            for finding
            in workflow_stored_record.findings
        }
    ),
    (
        "No-target state is preserved"
    ): (
        language_workflow_response
        .no_target_finding
        == workflow_stored_record
        .no_target_finding
    ),
    (
        "Final guarded output is non-empty"
    ): bool(
        language_workflow_response
        .output_text.strip()
    ),
    (
        "Guardrail action is exposed"
    ): (
        language_workflow_response
        .guardrail_action
        in {
            "accepted_model_generation",
            "safe_template_fallback",
        }
    ),
    (
        "Fallback triggers match the action"
    ): (
        bool(
            language_workflow_response
            .trigger_reasons
        )
        if language_workflow_response
        .guardrail_action
        == "safe_template_fallback"
        else not (
            language_workflow_response
            .trigger_reasons
        )
    ),
    (
        "Generated-token count is positive"
    ): (
        language_workflow_response
        .generated_tokens
        > 0
    ),
    (
        "Generation latency is positive"
    ): (
        language_workflow_response
        .generation_latency_ms
        > 0.0
    ),
    (
        "Language output is stored by task"
    ): (
        "grounded_question_answering"
        in stored_task_names
    ),
    (
        "Prediction-store calls are measured"
    ): (
        workflow_metrics_snapshot
        .service_invocation_counts[
            "prediction_store"
        ]
        >= 2
    ),
    (
        "Language-model calls are measured"
    ): (
        workflow_metrics_snapshot
        .service_invocation_counts[
            "grounded_language"
        ]
        >= 1
    ),
    (
        "Guardrail calls are measured"
    ): (
        workflow_metrics_snapshot
        .service_invocation_counts[
            "language_guardrail"
        ]
        >= 1
    ),
    (
        "Guardrail actions are measured"
    ): (
        workflow_metrics_snapshot
        .accepted_model_generations
        + workflow_metrics_snapshot
        .safe_template_fallbacks
        >= 1
    ),
    (
        "Client grounding fields are excluded"
    ): (
        client_grounding_rejected_by_interface
    ),
    (
        "Workflow checksum is available"
    ): len(
        language_workflow_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the workflow summary
# -------------------------------------------------------------------------
print(
    "STORED-PREDICTION GROUNDED LANGUAGE WORKFLOW"
)
print("-" * 100)
print(
    f"{'Workflow module':31}: "
    f"{language_workflow_service_path}"
)
print(
    f"{'Module SHA-256':31}: "
    f"{language_workflow_checksum[:16]}..."
)
print(
    f"{'Prediction ID':31}: "
    f"{workflow_prediction_id}"
)
print(
    f"{'Task type':31}: "
    f"{language_workflow_response.task_type}"
)
print(
    f"{'Grounded findings':31}: "
    f"{list(language_workflow_response.grounded_finding_names)}"
)
print(
    f"{'No-target-finding state':31}: "
    f"{language_workflow_response.no_target_finding}"
)
print(
    f"{'Guardrail action':31}: "
    f"{language_workflow_response.guardrail_action}"
)
print(
    f"{'Guardrail triggers':31}: "
    + (
        ", ".join(
            language_workflow_response
            .trigger_reasons
        )
        if language_workflow_response
        .trigger_reasons
        else "none"
    )
)
print(
    f"{'Generated tokens':31}: "
    f"{language_workflow_response.generated_tokens}"
)
print(
    f"{'Generation latency':31}: "
    f"{language_workflow_response.generation_latency_ms:.2f} ms"
)
print("-" * 100)

for check_name, passed in (
    language_workflow_checks.items()
):
    print(
        f"{check_name:62}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    language_workflow_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in language_workflow_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Stored-prediction language workflow validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "GUARDED LANGUAGE WORKFLOW OUTPUT"
)
print("-" * 100)
print(
    language_workflow_response.output_text
)
print("-" * 100)
print(
    "STATUS: STORED-PREDICTION LANGUAGE WORKFLOW "
    "READY FOR FASTAPI ROUTES"
)

STORED-PREDICTION GROUNDED LANGUAGE WORKFLOW
----------------------------------------------------------------------------------------------------
Workflow module                : /home/jovyan/chest-xray-ai-assistant/src/services/language_workflow_service.py
Module SHA-256                 : 4a099bdeaae89500...
Prediction ID                  : 8839d88e-ed9a-44fe-9437-162d2f01cd45
Task type                      : grounded_question_answering
Grounded findings              : ['atelectasis']
No-target-finding state        : False
Guardrail action               : safe_template_fallback
Guardrail triggers             : numeric_grounding_issue
Generated tokens               : 62
Generation latency             : 666.10 ms
----------------------------------------------------------------------------------------------------
Language workflow module is available                         : PASS
Fresh prediction exists in the active store                   : PASS
Stored prediction identifier is preserv

<a id="nb07-6-3-image-classification-and-explainability-workflow-orchestration"></a>
### 6.3 Image Classification and Explainability Workflow Orchestration

This block composes validated image ingestion, frozen computer-vision inference, prediction persistence, threshold-controlled Grad-CAM generation, explainability schema construction, and operational measurements. It produces the exact classification and image-analysis response contracts required by the corresponding FastAPI endpoints.


In [40]:
# -------------------------------------------------------------------------
# Recompute the workflow checksum
# -------------------------------------------------------------------------
image_workflow_checksum = hashlib.sha256(
    image_workflow_service_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Read compatible fields from Pydantic models or mappings
# -------------------------------------------------------------------------
def read_workflow_field(
    record,
    *field_names,
):
    for field_name in field_names:
        if isinstance(
            record,
            dict,
        ) and field_name in record:
            return record[field_name]

        if hasattr(
            record,
            field_name,
        ):
            return getattr(
                record,
                field_name,
            )

    raise AttributeError(
        f"{type(record).__name__} does not expose any of: "
        + ", ".join(field_names)
    )


# -------------------------------------------------------------------------
# Validate classification and explainability integration
# -------------------------------------------------------------------------
classification_crossed_names = [
    str(
        read_workflow_field(
            finding,
            "label_name",
            "finding_name",
        )
    )
    for finding
    in classification_workflow_response.findings
    if bool(
        read_workflow_field(
            finding,
            "crossed_threshold",
            "threshold_decision",
            "decision",
        )
    )
]

analysis_evidence_names = [
    str(
        read_workflow_field(
            evidence,
            "finding_name",
            "label_name",
        )
    )
    for evidence
    in analysis_workflow_response.visual_evidence
]

expected_analysis_evidence_names = list(
    analysis_workflow_response
    .crossed_finding_names
)

image_workflow_checks = {
    (
        "Image workflow module was written"
    ): image_workflow_service_path.is_file(),
    (
        "Classification response schema is preserved"
    ): (
        type(
            classification_workflow_response
        ).__name__
        == "ClassificationResponse"
    ),
    (
        "Image-analysis response schema is preserved"
    ): (
        type(
            analysis_workflow_response
        ).__name__
        == "ImageAnalysisResponse"
    ),
    (
        "Classification creates a prediction identifier"
    ): (
        classification_workflow_response
        .prediction_id
        == classification_workflow_execution
        .prediction_id
    ),
    (
        "Analysis creates a prediction identifier"
    ): (
        analysis_workflow_response
        .prediction_id
        == analysis_workflow_execution
        .prediction_id
    ),
    (
        "Image metadata is preserved"
    ): (
        analysis_workflow_response
        .image.sha256
        == validated_png.sha256
        and analysis_workflow_response
        .image.width
        == validated_png.width
        and analysis_workflow_response
        .image.height
        == validated_png.height
    ),
    (
        "Exactly fourteen findings are returned"
    ): (
        len(
            classification_workflow_response
            .findings
        )
        == 14
        and len(
            analysis_workflow_response
            .findings
        )
        == 14
    ),
    (
        "Crossed finding names match decisions"
    ): (
        classification_crossed_names
        == list(
            classification_workflow_response
            .crossed_finding_names
        )
    ),
    (
        "No-target state matches decisions"
    ): (
        classification_workflow_response
        .no_target_finding
        == (
            len(
                classification_workflow_response
                .crossed_finding_names
            )
            == 0
        )
    ),
    (
        "Visual evidence matches crossed findings"
    ): (
        analysis_evidence_names
        == expected_analysis_evidence_names
    ),
    (
        "Stored visual evidence count is preserved"
    ): (
        len(
            analysis_stored_record
            .visual_evidence
        )
        == len(
            analysis_workflow_response
            .visual_evidence
        )
    ),
    (
        "Explainability contract is stored"
    ): (
        analysis_stored_record
        .explainability
        is not None
    ),
    (
        "Computer-vision calls are measured"
    ): (
        image_workflow_metrics
        .service_invocation_counts[
            "computer_vision"
        ]
        >= 2
    ),
    (
        "Grad-CAM calls are measured"
    ): (
        image_workflow_metrics
        .service_invocation_counts[
            "gradcam"
        ]
        >= 1
    ),
    (
        "Prediction-store calls are measured"
    ): (
        image_workflow_metrics
        .service_invocation_counts[
            "prediction_store"
        ]
        >= 4
    ),
    (
        "Workflow checksum is available"
    ): len(
        image_workflow_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the validation summary
# -------------------------------------------------------------------------
print(
    "IMAGE CLASSIFICATION AND EXPLAINABILITY WORKFLOW"
)
print("-" * 100)
print(
    f"{'Workflow module':31}: "
    f"{image_workflow_service_path}"
)
print(
    f"{'Module SHA-256':31}: "
    f"{image_workflow_checksum[:16]}..."
)
print(
    f"{'Classification prediction ID':31}: "
    f"{classification_workflow_response.prediction_id}"
)
print(
    f"{'Analysis prediction ID':31}: "
    f"{analysis_workflow_response.prediction_id}"
)
print(
    f"{'Finding records':31}: "
    f"{len(analysis_workflow_response.findings)}"
)
print(
    f"{'Crossed findings':31}: "
    f"{analysis_workflow_response.crossed_finding_names}"
)
print(
    f"{'No-target-finding state':31}: "
    f"{analysis_workflow_response.no_target_finding}"
)
print(
    f"{'Visual evidence records':31}: "
    f"{len(analysis_workflow_response.visual_evidence)}"
)
print("-" * 100)

for check_name, passed in (
    image_workflow_checks.items()
):
    print(
        f"{check_name:62}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    image_workflow_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in image_workflow_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Image workflow validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: IMAGE CLASSIFICATION AND EXPLAINABILITY "
    "WORKFLOWS READY FOR FASTAPI ROUTES"
)

IMAGE CLASSIFICATION AND EXPLAINABILITY WORKFLOW
----------------------------------------------------------------------------------------------------
Workflow module                : /home/jovyan/chest-xray-ai-assistant/src/services/image_workflow_service.py
Module SHA-256                 : 0b891c00c119b2b1...
Classification prediction ID   : 5412309f-b553-4124-940c-2750ae1ba404
Analysis prediction ID         : 6693e416-9756-422e-b758-77c324f9bb58
Finding records                : 14
Crossed findings               : []
No-target-finding state        : True
Visual evidence records        : 0
----------------------------------------------------------------------------------------------------
Image workflow module was written                             : PASS
Classification response schema is preserved                   : PASS
Image-analysis response schema is preserved                   : PASS
Classification creates a prediction identifier                : PASS
Analysis creates a predict

<a id="nb07-6-4-complete-analysis-workflow-orchestration"></a>
### 6.4 Complete Analysis Workflow Orchestration

This block composes the full API workflow behind `/api/v1/analyze-complete`. A single validated image is classified, stored, processed for threshold-controlled visual evidence, and passed through the three mandatory grounded language tasks. Grounded question answering is added only when the client supplies a validated question. The final response is assembled from the same stored prediction so that all outputs share one prediction identifier and consistent model lineage.


In [41]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import importlib
import sys
from pathlib import Path
from typing import Any

from api.core.runtime import (
    RequestContext,
    build_success_metadata,
)
from api.schemas import (
    CompleteAnalysisResponse,
)


# -------------------------------------------------------------------------
# Define the complete-workflow service path
# -------------------------------------------------------------------------
complete_workflow_service_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "src/services/complete_workflow_service.py"
)

complete_workflow_service_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Write the reusable complete-analysis workflow
# -------------------------------------------------------------------------
complete_workflow_service_source = '''
"""Complete image, explainability, and grounded-language workflow."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Callable

from api.schemas import (
    CompleteAnalysisResponse,
)


MANDATORY_LANGUAGE_TASKS = (
    "structured_report",
    "plain_language_explanation",
    "educational_follow_up",
)

OPTIONAL_QUESTION_TASK = (
    "grounded_question_answering"
)


@dataclass(frozen=True)
class CompleteWorkflowExecution:
    """Complete response and its internal execution summary."""

    response: CompleteAnalysisResponse
    prediction_id: Any
    executed_language_tasks: tuple[str, ...]
    guardrail_actions: dict[str, str]
    visual_evidence_count: int


class CompleteAnalysisWorkflow:
    """Compose the complete educational decision-support workflow."""

    def __init__(
        self,
        *,
        image_workflow: Any,
        language_workflow: Any,
        prediction_store: Any,
        operational_metrics: Any,
        metadata_builder: Callable[..., dict[str, Any]],
    ) -> None:
        self.image_workflow = (
            image_workflow
        )
        self.language_workflow = (
            language_workflow
        )
        self.prediction_store = (
            prediction_store
        )
        self.operational_metrics = (
            operational_metrics
        )
        self.metadata_builder = (
            metadata_builder
        )

    def analyze_complete(
        self,
        *,
        validated_image: Any,
        request_context: Any,
        question: str | None = None,
    ) -> CompleteWorkflowExecution:
        """Execute image analysis and all requested language tasks."""

        normalized_question = None

        if question is not None:
            normalized_question = " ".join(
                str(question).split()
            )

            if not normalized_question:
                normalized_question = None

        image_execution = (
            self.image_workflow.analyze(
                validated_image=(
                    validated_image
                ),
                request_context=(
                    request_context
                ),
            )
        )

        prediction_id = (
            image_execution.prediction_id
        )

        executed_tasks = list(
            MANDATORY_LANGUAGE_TASKS
        )

        if normalized_question is not None:
            executed_tasks.append(
                OPTIONAL_QUESTION_TASK
            )

        language_executions = {}

        for task_type in executed_tasks:
            language_executions[
                task_type
            ] = (
                self.language_workflow.generate(
                    prediction_id=(
                        prediction_id
                    ),
                    task_type=task_type,
                    request_context=(
                        request_context
                    ),
                    question=(
                        normalized_question
                        if task_type
                        == OPTIONAL_QUESTION_TASK
                        else None
                    ),
                )
            )

        stored_record = (
            self.prediction_store.get(
                prediction_id
            )
        )

        self.operational_metrics.record_service_invocation(
            "prediction_store"
        )

        response_metadata = (
            self.metadata_builder(
                request_context,
                include_language=True,
                include_explainability=True,
            )
        )

        response = (
            CompleteAnalysisResponse.model_validate(
                {
                    **response_metadata,
                    "prediction_id": (
                        stored_record
                        .prediction_id
                    ),
                    "image": (
                        stored_record.image
                    ),
                    "findings": list(
                        stored_record.findings
                    ),
                    "crossed_finding_names": list(
                        stored_record
                        .crossed_finding_names
                    ),
                    "no_target_finding": (
                        stored_record
                        .no_target_finding
                    ),
                    "interpretation": (
                        stored_record
                        .interpretation
                    ),
                    "explainability": (
                        stored_record
                        .explainability
                    ),
                    "visual_evidence": list(
                        stored_record
                        .visual_evidence
                    ),
                    "language_outputs": list(
                        stored_record
                        .language_outputs
                    ),
                }
            )
        )

        guardrail_actions = {
            task_type: (
                execution
                .guardrail_action
            )
            for task_type, execution
            in language_executions.items()
        }

        return CompleteWorkflowExecution(
            response=response,
            prediction_id=prediction_id,
            executed_language_tasks=tuple(
                executed_tasks
            ),
            guardrail_actions=(
                guardrail_actions
            ),
            visual_evidence_count=len(
                stored_record
                .visual_evidence
            ),
        )
'''

complete_workflow_service_path.write_text(
    complete_workflow_service_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Import the generated workflow module
# -------------------------------------------------------------------------
importlib.invalidate_caches()

module_name = (
    "src.services.complete_workflow_service"
)

if module_name in sys.modules:
    complete_workflow_module = (
        importlib.reload(
            sys.modules[module_name]
        )
    )
else:
    complete_workflow_module = (
        importlib.import_module(
            module_name
        )
    )

CompleteAnalysisWorkflow = (
    complete_workflow_module
    .CompleteAnalysisWorkflow
)

CompleteWorkflowExecution = (
    complete_workflow_module
    .CompleteWorkflowExecution
)

MANDATORY_LANGUAGE_TASKS = (
    complete_workflow_module
    .MANDATORY_LANGUAGE_TASKS
)

OPTIONAL_QUESTION_TASK = (
    complete_workflow_module
    .OPTIONAL_QUESTION_TASK
)


# -------------------------------------------------------------------------
# Initialize the complete workflow
# -------------------------------------------------------------------------
complete_analysis_workflow = (
    CompleteAnalysisWorkflow(
        image_workflow=(
            image_analysis_workflow
        ),
        language_workflow=(
            stored_prediction_language_workflow
        ),
        prediction_store=(
            prediction_store_service
        ),
        operational_metrics=(
            operational_metrics_service
        ),
        metadata_builder=(
            build_success_metadata
        ),
    )
)


# -------------------------------------------------------------------------
# Execute the complete workflow with optional grounded QA
# -------------------------------------------------------------------------
complete_workflow_context = (
    RequestContext()
)

complete_workflow_question = (
    "What does the supplied model information indicate?"
)

complete_workflow_execution = (
    complete_analysis_workflow
    .analyze_complete(
        validated_image=(
            validated_png
        ),
        request_context=(
            complete_workflow_context
        ),
        question=(
            complete_workflow_question
        ),
    )
)

complete_workflow_response = (
    complete_workflow_execution.response
)

complete_workflow_stored_record = (
    prediction_store_service.get(
        complete_workflow_execution
        .prediction_id
    )
)

complete_workflow_metrics = (
    operational_metrics_service.snapshot()
)

complete_workflow_checksum = (
    hashlib.sha256(
        complete_workflow_service_path
        .read_bytes()
    ).hexdigest()
)


# -------------------------------------------------------------------------
# Derive integrated workflow values
# -------------------------------------------------------------------------
complete_language_task_names = [
    language_output.task_type
    for language_output
    in complete_workflow_response
    .language_outputs
]

stored_language_task_names = [
    language_output.task_type
    for language_output
    in complete_workflow_stored_record
    .language_outputs
]

expected_complete_task_order = [
    *MANDATORY_LANGUAGE_TASKS,
    OPTIONAL_QUESTION_TASK,
]

complete_guardrail_actions = [
    language_output.guardrail_action
    for language_output
    in complete_workflow_response
    .language_outputs
]

complete_language_outputs_non_empty = all(
    bool(
        language_output.output_text.strip()
    )
    for language_output
    in complete_workflow_response
    .language_outputs
)

complete_trigger_contract_valid = all(
    (
        bool(
            language_output.trigger_reasons
        )
        if language_output.guardrail_action
        == "safe_template_fallback"
        else not (
            language_output.trigger_reasons
        )
    )
    for language_output
    in complete_workflow_response
    .language_outputs
)


# -------------------------------------------------------------------------
# Validate complete workflow composition
# -------------------------------------------------------------------------
complete_workflow_checks = {
    (
        "Complete workflow module was written"
    ): complete_workflow_service_path.is_file(),
    (
        "Complete response schema is preserved"
    ): (
        type(
            complete_workflow_response
        ).__name__
        == "CompleteAnalysisResponse"
    ),
    (
        "One prediction identifier is preserved"
    ): (
        complete_workflow_response
        .prediction_id
        == complete_workflow_execution
        .prediction_id
        == complete_workflow_stored_record
        .prediction_id
    ),
    (
        "Exactly fourteen findings are preserved"
    ): len(
        complete_workflow_response.findings
    ) == 14,
    (
        "No-target state matches crossed findings"
    ): (
        complete_workflow_response
        .no_target_finding
        == (
            len(
                complete_workflow_response
                .crossed_finding_names
            )
            == 0
        )
    ),
    (
        "Explainability contract is present"
    ): (
        complete_workflow_response
        .explainability
        is not None
    ),
    (
        "Visual evidence matches crossed findings"
    ): (
        len(
            complete_workflow_response
            .visual_evidence
        )
        == len(
            complete_workflow_response
            .crossed_finding_names
        )
    ),
    (
        "Three mandatory language tasks are present"
    ): all(
        task_type
        in complete_language_task_names
        for task_type
        in MANDATORY_LANGUAGE_TASKS
    ),
    (
        "Optional question-answer task is present"
    ): (
        OPTIONAL_QUESTION_TASK
        in complete_language_task_names
    ),
    (
        "Language task order is preserved"
    ): (
        complete_language_task_names
        == expected_complete_task_order
    ),
    (
        "Stored language task order is preserved"
    ): (
        stored_language_task_names
        == expected_complete_task_order
    ),
    (
        "Language task names are unique"
    ): (
        len(
            complete_language_task_names
        )
        == len(
            set(
                complete_language_task_names
            )
        )
    ),
    (
        "Every language output is non-empty"
    ): (
        complete_language_outputs_non_empty
    ),
    (
        "Every guardrail action is controlled"
    ): all(
        action
        in {
            "accepted_model_generation",
            "safe_template_fallback",
        }
        for action
        in complete_guardrail_actions
    ),
    (
        "Fallback triggers follow each action"
    ): (
        complete_trigger_contract_valid
    ),
    (
        "Question is preserved for grounded QA"
    ): (
        next(
            language_output.question
            for language_output
            in complete_workflow_response
            .language_outputs
            if language_output.task_type
            == OPTIONAL_QUESTION_TASK
        )
        == complete_workflow_question
    ),
    (
        "Image metadata is preserved"
    ): (
        complete_workflow_response
        .image.sha256
        == validated_png.sha256
    ),
    (
        "Computer-vision invocation is measured"
    ): (
        complete_workflow_metrics
        .service_invocation_counts[
            "computer_vision"
        ]
        >= 3
    ),
    (
        "Grad-CAM invocation is measured"
    ): (
        complete_workflow_metrics
        .service_invocation_counts[
            "gradcam"
        ]
        >= 2
    ),
    (
        "Language invocations are measured"
    ): (
        complete_workflow_metrics
        .service_invocation_counts[
            "grounded_language"
        ]
        >= 5
    ),
    (
        "Guardrail invocations are measured"
    ): (
        complete_workflow_metrics
        .service_invocation_counts[
            "language_guardrail"
        ]
        >= 5
    ),
    (
        "Workflow checksum is available"
    ): len(
        complete_workflow_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the complete workflow summary
# -------------------------------------------------------------------------
print(
    "COMPLETE CHEST X-RAY ANALYSIS WORKFLOW"
)
print("-" * 100)
print(
    f"{'Workflow module':32}: "
    f"{complete_workflow_service_path}"
)
print(
    f"{'Module SHA-256':32}: "
    f"{complete_workflow_checksum[:16]}..."
)
print(
    f"{'Prediction ID':32}: "
    f"{complete_workflow_response.prediction_id}"
)
print(
    f"{'Finding records':32}: "
    f"{len(complete_workflow_response.findings)}"
)
print(
    f"{'Crossed findings':32}: "
    f"{complete_workflow_response.crossed_finding_names}"
)
print(
    f"{'No-target-finding state':32}: "
    f"{complete_workflow_response.no_target_finding}"
)
print(
    f"{'Visual evidence records':32}: "
    f"{len(complete_workflow_response.visual_evidence)}"
)
print(
    f"{'Language outputs':32}: "
    f"{len(complete_workflow_response.language_outputs)}"
)
print("-" * 100)
print(
    "LANGUAGE WORKFLOW ACTIONS"
)
print("-" * 100)

for language_output in (
    complete_workflow_response
    .language_outputs
):
    trigger_text = (
        ", ".join(
            language_output.trigger_reasons
        )
        if language_output.trigger_reasons
        else "none"
    )

    print(
        f"{language_output.task_type:29}: "
        f"{language_output.guardrail_action:25} | "
        f"{trigger_text}"
    )

print("-" * 100)

for check_name, passed in (
    complete_workflow_checks.items()
):
    print(
        f"{check_name:63}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    complete_workflow_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in complete_workflow_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Complete analysis workflow validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: COMPLETE ANALYSIS WORKFLOW READY "
    "FOR FASTAPI ENDPOINT INTEGRATION"
)

COMPLETE CHEST X-RAY ANALYSIS WORKFLOW
----------------------------------------------------------------------------------------------------
Workflow module                 : /home/jovyan/chest-xray-ai-assistant/src/services/complete_workflow_service.py
Module SHA-256                  : 312b930b56ab147a...
Prediction ID                   : 2e44bbcd-2ca7-442c-9afc-cc8342cbbb3f
Finding records                 : 14
Crossed findings                : []
No-target-finding state         : True
Visual evidence records         : 0
Language outputs                : 4
----------------------------------------------------------------------------------------------------
LANGUAGE WORKFLOW ACTIONS
----------------------------------------------------------------------------------------------------
structured_report            : accepted_model_generation | none
plain_language_explanation   : accepted_model_generation | none
educational_follow_up        : safe_template_fallback    | missing_required_findi

<a id="nb07-6-5-application-service-container-and-dependency-boundary"></a>
### 6.5 Application Service Container and Dependency Boundary

This block creates the central dependency container used by FastAPI routes. It exposes the validated service and workflow instances through one thread-safe application boundary, prevents route modules from loading models independently, and provides controlled readiness information without exposing internal paths or model objects to API clients.


In [42]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import importlib
import sys
from pathlib import Path

from api.core.errors import (
    ModelNotReadyError,
)


# -------------------------------------------------------------------------
# Define the dependency-container module path
# -------------------------------------------------------------------------
dependency_module_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "api/core/dependencies.py"
)

dependency_module_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Write the application dependency container
# -------------------------------------------------------------------------
dependency_module_source = '''
"""Thread-safe application service dependency container."""

from __future__ import annotations

import threading
from dataclasses import dataclass
from typing import Any

from api.core.errors import (
    ModelNotReadyError,
)


@dataclass(frozen=True)
class ServiceContainer:
    """Validated runtime services shared by FastAPI routes."""

    image_validation_service: Any
    computer_vision_service: Any
    prediction_store_service: Any
    gradcam_service: Any
    grounding_serializer: Any
    language_model_service: Any
    language_guardrail_service: Any
    operational_metrics_service: Any
    image_analysis_workflow: Any
    language_workflow: Any
    complete_analysis_workflow: Any

    def readiness(self) -> dict[str, bool]:
        """Return internal component readiness without sensitive values."""

        cv_model = getattr(
            self.computer_vision_service,
            "model",
            None,
        )

        language_model = getattr(
            self.language_model_service,
            "model",
            None,
        )

        return {
            "image_validation": (
                self.image_validation_service
                is not None
            ),
            "computer_vision": (
                cv_model is not None
                and not cv_model.training
            ),
            "prediction_store": (
                self.prediction_store_service
                is not None
            ),
            "gradcam": (
                self.gradcam_service
                is not None
            ),
            "grounding_serializer": (
                self.grounding_serializer
                is not None
            ),
            "grounded_language": (
                language_model is not None
                and not language_model.training
            ),
            "language_guardrail": (
                self.language_guardrail_service
                is not None
            ),
            "operational_metrics": (
                self.operational_metrics_service
                is not None
            ),
            "image_workflow": (
                self.image_analysis_workflow
                is not None
            ),
            "language_workflow": (
                self.language_workflow
                is not None
            ),
            "complete_workflow": (
                self.complete_analysis_workflow
                is not None
            ),
        }

    @property
    def ready(self) -> bool:
        """Return true only when every registered component is ready."""

        return all(
            self.readiness().values()
        )


_container_lock = threading.RLock()
_service_container: ServiceContainer | None = None


def configure_service_container(
    container: ServiceContainer,
    *,
    replace: bool = False,
) -> ServiceContainer:
    """Register one complete service container."""

    if not isinstance(
        container,
        ServiceContainer,
    ):
        raise TypeError(
            "The application dependency must be a ServiceContainer."
        )

    if not container.ready:
        failed_components = [
            component_name
            for component_name, ready
            in container.readiness().items()
            if not ready
        ]

        raise ModelNotReadyError(
            details={
                "components": (
                    failed_components
                )
            }
        )

    global _service_container

    with _container_lock:
        if (
            _service_container is not None
            and not replace
        ):
            raise RuntimeError(
                "The service container is already configured."
            )

        _service_container = container

    return container


def get_service_container() -> ServiceContainer:
    """FastAPI dependency that returns the configured services."""

    with _container_lock:
        container = _service_container

    if container is None:
        raise ModelNotReadyError(
            details={
                "component": (
                    "service_container"
                )
            }
        )

    if not container.ready:
        failed_components = [
            component_name
            for component_name, ready
            in container.readiness().items()
            if not ready
        ]

        raise ModelNotReadyError(
            details={
                "components": (
                    failed_components
                )
            }
        )

    return container


def service_container_is_configured() -> bool:
    """Return the non-sensitive configuration state."""

    with _container_lock:
        return (
            _service_container is not None
            and _service_container.ready
        )


def clear_service_container() -> None:
    """Remove the container for controlled test isolation."""

    global _service_container

    with _container_lock:
        _service_container = None
'''

dependency_module_path.write_text(
    dependency_module_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Import the generated dependency module
# -------------------------------------------------------------------------
importlib.invalidate_caches()

module_name = (
    "api.core.dependencies"
)

if module_name in sys.modules:
    dependency_module = importlib.reload(
        sys.modules[module_name]
    )
else:
    dependency_module = importlib.import_module(
        module_name
    )

ServiceContainer = (
    dependency_module.ServiceContainer
)

configure_service_container = (
    dependency_module
    .configure_service_container
)

get_service_container = (
    dependency_module
    .get_service_container
)

service_container_is_configured = (
    dependency_module
    .service_container_is_configured
)

clear_service_container = (
    dependency_module
    .clear_service_container
)


# -------------------------------------------------------------------------
# Assemble the validated notebook runtime services
# -------------------------------------------------------------------------
api_service_container = ServiceContainer(
    image_validation_service=(
        image_validation_service
    ),
    computer_vision_service=(
        computer_vision_service
    ),
    prediction_store_service=(
        prediction_store_service
    ),
    gradcam_service=(
        gradcam_service
    ),
    grounding_serializer=(
        grounded_input_serializer
    ),
    language_model_service=(
        grounded_language_model_service
    ),
    language_guardrail_service=(
        language_guardrail_service
    ),
    operational_metrics_service=(
        operational_metrics_service
    ),
    image_analysis_workflow=(
        image_analysis_workflow
    ),
    language_workflow=(
        stored_prediction_language_workflow
    ),
    complete_analysis_workflow=(
        complete_analysis_workflow
    ),
)


# -------------------------------------------------------------------------
# Configure the process-wide FastAPI dependency
# -------------------------------------------------------------------------
clear_service_container()

configured_service_container = (
    configure_service_container(
        api_service_container
    )
)

resolved_service_container = (
    get_service_container()
)

container_readiness = (
    resolved_service_container
    .readiness()
)

dependency_module_checksum = hashlib.sha256(
    dependency_module_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Confirm controlled unconfigured behavior using module isolation
# -------------------------------------------------------------------------
clear_service_container()

unconfigured_error_controlled = False

try:
    get_service_container()
except ModelNotReadyError as error:
    unconfigured_error_controlled = (
        error.error_code
        == "MODEL_NOT_READY"
    )

configure_service_container(
    api_service_container
)

final_resolved_container = (
    get_service_container()
)


# -------------------------------------------------------------------------
# Validate the dependency boundary
# -------------------------------------------------------------------------
dependency_checks = {
    (
        "Dependency module was written"
    ): dependency_module_path.is_file(),
    (
        "Service container is immutable"
    ): (
        ServiceContainer.__dataclass_params__
        .frozen
    ),
    (
        "Eleven runtime components are registered"
    ): len(
        container_readiness
    ) == 11,
    (
        "Every runtime component is ready"
    ): all(
        container_readiness.values()
    ),
    (
        "Configured container identity is preserved"
    ): (
        final_resolved_container
        is api_service_container
    ),
    (
        "Computer-vision model remains in evaluation mode"
    ): not (
        final_resolved_container
        .computer_vision_service
        .model.training
    ),
    (
        "Language model remains in evaluation mode"
    ): not (
        final_resolved_container
        .language_model_service
        .model.training
    ),
    (
        "Unconfigured access raises controlled error"
    ): (
        unconfigured_error_controlled
    ),
    (
        "Container is restored after isolation test"
    ): (
        service_container_is_configured()
    ),
    (
        "Dependency checksum is available"
    ): len(
        dependency_module_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the dependency-boundary summary
# -------------------------------------------------------------------------
print(
    "APPLICATION SERVICE CONTAINER AND DEPENDENCY BOUNDARY"
)
print("-" * 100)
print(
    f"{'Dependency module':32}: "
    f"{dependency_module_path}"
)
print(
    f"{'Module SHA-256':32}: "
    f"{dependency_module_checksum[:16]}..."
)
print(
    f"{'Registered components':32}: "
    f"{len(container_readiness)}"
)
print(
    f"{'Container configured':32}: "
    f"{service_container_is_configured()}"
)
print("-" * 100)
print(
    "COMPONENT READINESS"
)
print("-" * 100)

for component_name, ready in (
    container_readiness.items()
):
    print(
        f"{component_name:32}: "
        f"{'READY' if ready else 'NOT READY'}"
    )

print("-" * 100)

for check_name, passed in (
    dependency_checks.items()
):
    print(
        f"{check_name:63}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    dependency_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in dependency_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Service dependency validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: APPLICATION SERVICE DEPENDENCIES READY "
    "FOR ROUTE INJECTION"
)

APPLICATION SERVICE CONTAINER AND DEPENDENCY BOUNDARY
----------------------------------------------------------------------------------------------------
Dependency module               : /home/jovyan/chest-xray-ai-assistant/api/core/dependencies.py
Module SHA-256                  : 38b0735b9715604e...
Registered components           : 11
Container configured            : True
----------------------------------------------------------------------------------------------------
COMPONENT READINESS
----------------------------------------------------------------------------------------------------
image_validation                : READY
computer_vision                 : READY
prediction_store                : READY
gradcam                         : READY
grounding_serializer            : READY
grounded_language               : READY
language_guardrail              : READY
operational_metrics             : READY
image_workflow                  : READY
language_workflow               : REA

<a id="nb07-6-6-image-and-grounded-language-api-routes"></a>
### 6.6 Image and Grounded Language API Routes

This block implements the six primary workflow endpoints for image classification, visual analysis, structured reporting, plain-language explanation, grounded question answering, and educational follow-up. Uploads are size-bounded before decoding, model work is moved outside the asynchronous event loop, and language routes accept only stored prediction identifiers plus the optional validated question.


In [43]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import importlib
import sys
from pathlib import Path

from fastapi.routing import APIRoute


# -------------------------------------------------------------------------
# Define the route-module path
# -------------------------------------------------------------------------
workflow_routes_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "api/routes/workflows.py"
)

workflow_routes_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Write the image and language workflow routes
# -------------------------------------------------------------------------
workflow_routes_source = '''
"""Image and grounded-language FastAPI workflow routes."""

from __future__ import annotations

from fastapi import (
    APIRouter,
    Depends,
    File,
    UploadFile,
)
from starlette.concurrency import (
    run_in_threadpool,
)

from api.core.config import (
    get_settings,
)
from api.core.dependencies import (
    ServiceContainer,
    get_service_container,
)
from api.core.runtime import (
    RequestContext,
)
from api.schemas import (
    ClassificationResponse,
    GroundedGenerationRequest,
    GroundedQuestionRequest,
    ImageAnalysisResponse,
    LanguageGenerationResponse,
)


router = APIRouter(
    prefix="/api/v1",
    tags=["analysis"],
)


async def _read_bounded_upload(
    upload: UploadFile,
) -> bytes:
    """Read at most one byte beyond the configured upload limit."""

    settings = get_settings()

    content = await upload.read(
        settings.maximum_upload_bytes
        + 1
    )

    return content


async def _validate_upload(
    *,
    upload: UploadFile,
    services: ServiceContainer,
):
    """Read, validate, and decode one uploaded image."""

    content = await _read_bounded_upload(
        upload
    )

    return await run_in_threadpool(
        services.image_validation_service
        .validate_and_decode,
        filename=(
            upload.filename
            or "uploaded-image"
        ),
        media_type=(
            upload.content_type
            or ""
        ),
        content=content,
    )


@router.post(
    "/image/classify",
    response_model=ClassificationResponse,
    summary="Classify a chest X-ray image",
)
async def classify_image(
    image: UploadFile = File(...),
    services: ServiceContainer = Depends(
        get_service_container
    ),
) -> ClassificationResponse:
    """Run frozen multilabel classification and store the prediction."""

    request_context = RequestContext()

    validated_image = await _validate_upload(
        upload=image,
        services=services,
    )

    execution = await run_in_threadpool(
        services.image_analysis_workflow
        .classify,
        validated_image=validated_image,
        request_context=request_context,
    )

    return execution.response


@router.post(
    "/image/analyze",
    response_model=ImageAnalysisResponse,
    summary="Classify an image and generate visual evidence",
)
async def analyze_image(
    image: UploadFile = File(...),
    services: ServiceContainer = Depends(
        get_service_container
    ),
) -> ImageAnalysisResponse:
    """Run classification and threshold-controlled Grad-CAM."""

    request_context = RequestContext()

    validated_image = await _validate_upload(
        upload=image,
        services=services,
    )

    execution = await run_in_threadpool(
        services.image_analysis_workflow
        .analyze,
        validated_image=validated_image,
        request_context=request_context,
    )

    return execution.response


@router.post(
    "/report/generate",
    response_model=LanguageGenerationResponse,
    summary="Generate a grounded preliminary model report",
)
async def generate_report(
    request: GroundedGenerationRequest,
    services: ServiceContainer = Depends(
        get_service_container
    ),
) -> LanguageGenerationResponse:
    """Generate a guarded structured report from a stored prediction."""

    request_context = RequestContext()

    execution = await run_in_threadpool(
        services.language_workflow.generate,
        prediction_id=(
            request.prediction_id
        ),
        task_type="structured_report",
        request_context=request_context,
        question=None,
    )

    return execution.response


@router.post(
    "/explanation/generate",
    response_model=LanguageGenerationResponse,
    summary="Generate a grounded plain-language explanation",
)
async def generate_explanation(
    request: GroundedGenerationRequest,
    services: ServiceContainer = Depends(
        get_service_container
    ),
) -> LanguageGenerationResponse:
    """Explain stored model output using guarded simple language."""

    request_context = RequestContext()

    execution = await run_in_threadpool(
        services.language_workflow.generate,
        prediction_id=(
            request.prediction_id
        ),
        task_type=(
            "plain_language_explanation"
        ),
        request_context=request_context,
        question=None,
    )

    return execution.response


@router.post(
    "/question/answer",
    response_model=LanguageGenerationResponse,
    summary="Answer a question using stored model output",
)
async def answer_grounded_question(
    request: GroundedQuestionRequest,
    services: ServiceContainer = Depends(
        get_service_container
    ),
) -> LanguageGenerationResponse:
    """Answer only from server-controlled stored grounding values."""

    request_context = RequestContext()

    execution = await run_in_threadpool(
        services.language_workflow.generate,
        prediction_id=(
            request.prediction_id
        ),
        task_type=(
            "grounded_question_answering"
        ),
        request_context=request_context,
        question=request.question,
    )

    return execution.response


@router.post(
    "/follow-up/recommend",
    response_model=LanguageGenerationResponse,
    summary="Generate controlled educational follow-up",
)
async def recommend_follow_up(
    request: GroundedGenerationRequest,
    services: ServiceContainer = Depends(
        get_service_container
    ),
) -> LanguageGenerationResponse:
    """Generate non-diagnostic educational follow-up guidance."""

    request_context = RequestContext()

    execution = await run_in_threadpool(
        services.language_workflow.generate,
        prediction_id=(
            request.prediction_id
        ),
        task_type="educational_follow_up",
        request_context=request_context,
        question=None,
    )

    return execution.response
'''

workflow_routes_path.write_text(
    workflow_routes_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Import the generated route module
# -------------------------------------------------------------------------
importlib.invalidate_caches()

module_name = (
    "api.routes.workflows"
)

if module_name in sys.modules:
    workflow_routes_module = importlib.reload(
        sys.modules[module_name]
    )
else:
    workflow_routes_module = importlib.import_module(
        module_name
    )

workflow_router = (
    workflow_routes_module.router
)


# -------------------------------------------------------------------------
# Inspect the registered API routes
# -------------------------------------------------------------------------
registered_workflow_routes = [
    route
    for route in workflow_router.routes
    if isinstance(
        route,
        APIRoute,
    )
]

workflow_route_contracts = [
    {
        "path": route.path,
        "methods": tuple(
            sorted(
                route.methods
            )
        ),
        "name": route.name,
        "response_model": (
            route.response_model.__name__
            if route.response_model
            is not None
            else None
        ),
    }
    for route
    in registered_workflow_routes
]

expected_workflow_route_contracts = {
    (
        "POST",
        "/api/v1/image/classify",
    ): "ClassificationResponse",
    (
        "POST",
        "/api/v1/image/analyze",
    ): "ImageAnalysisResponse",
    (
        "POST",
        "/api/v1/report/generate",
    ): "LanguageGenerationResponse",
    (
        "POST",
        "/api/v1/explanation/generate",
    ): "LanguageGenerationResponse",
    (
        "POST",
        "/api/v1/question/answer",
    ): "LanguageGenerationResponse",
    (
        "POST",
        "/api/v1/follow-up/recommend",
    ): "LanguageGenerationResponse",
}

actual_workflow_route_contracts = {}

for route in registered_workflow_routes:
    for method in route.methods:
        actual_workflow_route_contracts[
            (
                method,
                route.path,
            )
        ] = (
            route.response_model.__name__
            if route.response_model
            is not None
            else None
        )

workflow_routes_checksum = hashlib.sha256(
    workflow_routes_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Validate the route contract
# -------------------------------------------------------------------------
workflow_route_checks = {
    (
        "Workflow route module was written"
    ): workflow_routes_path.is_file(),
    (
        "Six workflow endpoints are registered"
    ): len(
        registered_workflow_routes
    ) == 6,
    (
        "Every workflow endpoint uses POST"
    ): all(
        route.methods == {"POST"}
        for route
        in registered_workflow_routes
    ),
    (
        "Every workflow path uses the API prefix"
    ): all(
        route.path.startswith(
            "/api/v1/"
        )
        for route
        in registered_workflow_routes
    ),
    (
        "Image classification response is preserved"
    ): (
        actual_workflow_route_contracts[
            (
                "POST",
                "/api/v1/image/classify",
            )
        ]
        == "ClassificationResponse"
    ),
    (
        "Image analysis response is preserved"
    ): (
        actual_workflow_route_contracts[
            (
                "POST",
                "/api/v1/image/analyze",
            )
        ]
        == "ImageAnalysisResponse"
    ),
    (
        "All four language routes are registered"
    ): all(
        route_contract
        in actual_workflow_route_contracts
        for route_contract
        in (
            (
                "POST",
                "/api/v1/report/generate",
            ),
            (
                "POST",
                "/api/v1/explanation/generate",
            ),
            (
                "POST",
                "/api/v1/question/answer",
            ),
            (
                "POST",
                "/api/v1/follow-up/recommend",
            ),
        )
    ),
    (
        "Every route response matches the frozen contract"
    ): (
        actual_workflow_route_contracts
        == expected_workflow_route_contracts
    ),
    (
        "Route paths and methods are unique"
    ): (
        len(
            actual_workflow_route_contracts
        )
        == len(
            registered_workflow_routes
        )
    ),
    (
        "Workflow route checksum is available"
    ): len(
        workflow_routes_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the route summary
# -------------------------------------------------------------------------
print(
    "IMAGE AND GROUNDED LANGUAGE FASTAPI ROUTES"
)
print("-" * 100)
print(
    f"{'Route module':29}: "
    f"{workflow_routes_path}"
)
print(
    f"{'Module SHA-256':29}: "
    f"{workflow_routes_checksum[:16]}..."
)
print(
    f"{'Registered routes':29}: "
    f"{len(registered_workflow_routes)}"
)
print("-" * 100)

for route_contract in (
    workflow_route_contracts
):
    print(
        f"{','.join(route_contract['methods']):6} "
        f"{route_contract['path']:39} | "
        f"{route_contract['response_model']}"
    )

print("-" * 100)

for check_name, passed in (
    workflow_route_checks.items()
):
    print(
        f"{check_name:62}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    workflow_route_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in workflow_route_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Workflow route validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: IMAGE AND LANGUAGE ROUTES READY "
    "FOR FASTAPI APPLICATION REGISTRATION"
)

IMAGE AND GROUNDED LANGUAGE FASTAPI ROUTES
----------------------------------------------------------------------------------------------------
Route module                 : /home/jovyan/chest-xray-ai-assistant/api/routes/workflows.py
Module SHA-256               : 5985fb08ddc34504...
Registered routes            : 6
----------------------------------------------------------------------------------------------------
POST   /api/v1/image/classify                  | ClassificationResponse
POST   /api/v1/image/analyze                   | ImageAnalysisResponse
POST   /api/v1/report/generate                 | LanguageGenerationResponse
POST   /api/v1/explanation/generate            | LanguageGenerationResponse
POST   /api/v1/question/answer                 | LanguageGenerationResponse
POST   /api/v1/follow-up/recommend             | LanguageGenerationResponse
----------------------------------------------------------------------------------------------------
Workflow route module was writt

<a id="nb07-6-7-complete-analysis-and-stored-prediction-api-routes"></a>
### 6.7 Complete Analysis and Stored Prediction API Routes

This block exposes the complete end-to-end analysis endpoint and stored-prediction retrieval endpoint. The complete route accepts one bounded image upload and an optional validated question, while retrieval reconstructs the response exclusively from the server-side prediction store with lineage determined by the artifacts already attached to that prediction.


In [44]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import importlib
import sys
from pathlib import Path

from fastapi.routing import APIRoute


# -------------------------------------------------------------------------
# Define the aggregate route-module path
# -------------------------------------------------------------------------
aggregate_routes_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "api/routes/aggregate.py"
)

aggregate_routes_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Write complete-analysis and prediction-retrieval routes
# -------------------------------------------------------------------------
aggregate_routes_source = '''
"""Complete-analysis and stored-prediction FastAPI routes."""

from __future__ import annotations

from uuid import UUID

from fastapi import (
    APIRouter,
    Depends,
    File,
    Form,
    UploadFile,
)
from starlette.concurrency import (
    run_in_threadpool,
)

from api.core.dependencies import (
    ServiceContainer,
    get_service_container,
)
from api.core.runtime import (
    RequestContext,
    build_success_metadata,
)
from api.routes.workflows import (
    _validate_upload,
)
from api.schemas import (
    CompleteAnalysisOptions,
    CompleteAnalysisResponse,
    StoredPredictionResponse,
)


router = APIRouter(
    prefix="/api/v1",
    tags=["complete-analysis"],
)


@router.post(
    "/analyze-complete",
    response_model=CompleteAnalysisResponse,
    summary="Run the complete chest X-ray analysis workflow",
)
async def analyze_complete(
    image: UploadFile = File(...),
    question: str | None = Form(
        default=None
    ),
    services: ServiceContainer = Depends(
        get_service_container
    ),
) -> CompleteAnalysisResponse:
    """Run classification, Grad-CAM, and guarded language generation."""

    request_context = RequestContext()

    options = CompleteAnalysisOptions(
        question=question
    )

    validated_image = await _validate_upload(
        upload=image,
        services=services,
    )

    execution = await run_in_threadpool(
        services.complete_analysis_workflow
        .analyze_complete,
        validated_image=validated_image,
        request_context=request_context,
        question=options.question,
    )

    return execution.response


@router.get(
    "/predictions/{prediction_id}",
    response_model=StoredPredictionResponse,
    summary="Retrieve one stored prediction",
)
async def get_prediction(
    prediction_id: UUID,
    services: ServiceContainer = Depends(
        get_service_container
    ),
) -> StoredPredictionResponse:
    """Return one stored prediction and its attached outputs."""

    request_context = RequestContext()

    stored_record = await run_in_threadpool(
        services.prediction_store_service
        .get,
        prediction_id,
    )

    services.operational_metrics_service.record_service_invocation(
        "prediction_store"
    )

    include_language = bool(
        stored_record.language_outputs
    )

    include_explainability = (
        stored_record.explainability
        is not None
    )

    response_metadata = build_success_metadata(
        request_context,
        include_language=include_language,
        include_explainability=(
            include_explainability
        ),
    )

    return await run_in_threadpool(
        services.prediction_store_service
        .build_response,
        prediction_id=prediction_id,
        response_metadata=response_metadata,
    )
'''

aggregate_routes_path.write_text(
    aggregate_routes_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Import the generated route module
# -------------------------------------------------------------------------
importlib.invalidate_caches()

module_name = (
    "api.routes.aggregate"
)

if module_name in sys.modules:
    aggregate_routes_module = importlib.reload(
        sys.modules[module_name]
    )
else:
    aggregate_routes_module = importlib.import_module(
        module_name
    )

aggregate_router = (
    aggregate_routes_module.router
)


# -------------------------------------------------------------------------
# Inspect the registered aggregate routes
# -------------------------------------------------------------------------
registered_aggregate_routes = [
    route
    for route in aggregate_router.routes
    if isinstance(
        route,
        APIRoute,
    )
]

aggregate_route_contracts = [
    {
        "path": route.path,
        "methods": tuple(
            sorted(
                route.methods
            )
        ),
        "name": route.name,
        "response_model": (
            route.response_model.__name__
            if route.response_model
            is not None
            else None
        ),
    }
    for route
    in registered_aggregate_routes
]

expected_aggregate_route_contracts = {
    (
        "POST",
        "/api/v1/analyze-complete",
    ): "CompleteAnalysisResponse",
    (
        "GET",
        "/api/v1/predictions/{prediction_id}",
    ): "StoredPredictionResponse",
}

actual_aggregate_route_contracts = {}

for route in registered_aggregate_routes:
    for method in route.methods:
        actual_aggregate_route_contracts[
            (
                method,
                route.path,
            )
        ] = (
            route.response_model.__name__
            if route.response_model
            is not None
            else None
        )

aggregate_routes_checksum = hashlib.sha256(
    aggregate_routes_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Inspect complete-analysis request parameters
# -------------------------------------------------------------------------
complete_route = next(
    route
    for route
    in registered_aggregate_routes
    if route.path
    == "/api/v1/analyze-complete"
)

complete_parameter_names = {
    parameter.name
    for parameter
    in complete_route.dependant
    .body_params
}


# -------------------------------------------------------------------------
# Validate the aggregate route contract
# -------------------------------------------------------------------------
aggregate_route_checks = {
    (
        "Aggregate route module was written"
    ): aggregate_routes_path.is_file(),
    (
        "Two aggregate endpoints are registered"
    ): len(
        registered_aggregate_routes
    ) == 2,
    (
        "Complete workflow uses POST"
    ): (
        actual_aggregate_route_contracts[
            (
                "POST",
                "/api/v1/analyze-complete",
            )
        ]
        == "CompleteAnalysisResponse"
    ),
    (
        "Stored prediction retrieval uses GET"
    ): (
        actual_aggregate_route_contracts[
            (
                "GET",
                "/api/v1/predictions/{prediction_id}",
            )
        ]
        == "StoredPredictionResponse"
    ),
    (
        "Complete workflow accepts an image"
    ): (
        "image"
        in complete_parameter_names
    ),
    (
        "Complete workflow accepts an optional question"
    ): (
        "question"
        in complete_parameter_names
    ),
    (
        "Aggregate paths use the API prefix"
    ): all(
        route.path.startswith(
            "/api/v1/"
        )
        for route
        in registered_aggregate_routes
    ),
    (
        "Route paths and methods are unique"
    ): (
        len(
            actual_aggregate_route_contracts
        )
        == len(
            registered_aggregate_routes
        )
    ),
    (
        "Responses match the frozen contract"
    ): (
        actual_aggregate_route_contracts
        == expected_aggregate_route_contracts
    ),
    (
        "Aggregate route checksum is available"
    ): len(
        aggregate_routes_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the aggregate-route summary
# -------------------------------------------------------------------------
print(
    "COMPLETE ANALYSIS AND STORED PREDICTION FASTAPI ROUTES"
)
print("-" * 100)
print(
    f"{'Route module':29}: "
    f"{aggregate_routes_path}"
)
print(
    f"{'Module SHA-256':29}: "
    f"{aggregate_routes_checksum[:16]}..."
)
print(
    f"{'Registered routes':29}: "
    f"{len(registered_aggregate_routes)}"
)
print("-" * 100)

for route_contract in (
    aggregate_route_contracts
):
    print(
        f"{','.join(route_contract['methods']):6} "
        f"{route_contract['path']:45} | "
        f"{route_contract['response_model']}"
    )

print("-" * 100)

for check_name, passed in (
    aggregate_route_checks.items()
):
    print(
        f"{check_name:62}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    aggregate_route_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in aggregate_route_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Aggregate route validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: COMPLETE ANALYSIS AND PREDICTION ROUTES "
    "READY FOR APPLICATION REGISTRATION"
)

COMPLETE ANALYSIS AND STORED PREDICTION FASTAPI ROUTES
----------------------------------------------------------------------------------------------------
Route module                 : /home/jovyan/chest-xray-ai-assistant/api/routes/aggregate.py
Module SHA-256               : 9e0f5523046dae85...
Registered routes            : 2
----------------------------------------------------------------------------------------------------
POST   /api/v1/analyze-complete                      | CompleteAnalysisResponse
GET    /api/v1/predictions/{prediction_id}           | StoredPredictionResponse
----------------------------------------------------------------------------------------------------
Aggregate route module was written                            : PASS
Two aggregate endpoints are registered                        : PASS
Complete workflow uses POST                                   : PASS
Stored prediction retrieval uses GET                          : PASS
Complete workflow accepts an i

<a id="nb07-6-8-endpoint-level-api-telemetry-adapter"></a>
### 6.8 Endpoint-Level API Telemetry Adapter

This block extends the operational metrics service with endpoint-specific latency aggregation required by the public LLMOps response schema. The adapter preserves existing model, guardrail, and service counters while adding bounded per-endpoint averages. It is then injected into the existing workflows and application container without reloading either trained model.


In [45]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import importlib
import sys
from pathlib import Path


# -------------------------------------------------------------------------
# Define the telemetry-adapter module path
# -------------------------------------------------------------------------
telemetry_adapter_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "api/core/telemetry.py"
)

telemetry_adapter_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Write the endpoint telemetry adapter
# -------------------------------------------------------------------------
telemetry_adapter_source = '''
"""Endpoint-level telemetry adapter for public API metrics."""

from __future__ import annotations

import math
import threading
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class APIOperationalSnapshot:
    """Public operational values required by the API schema."""

    service_started_at_utc: Any
    total_requests: int
    successful_requests: int
    failed_requests: int
    endpoint_request_counts: dict[str, int]
    endpoint_average_latency_ms: dict[str, float]
    language_generation_requests: int
    guardrail_action_counts: dict[str, int]


class EndpointTelemetryAdapter:
    """Add endpoint-specific latency metrics to the base service."""

    def __init__(
        self,
        base_metrics_service: Any,
    ) -> None:
        self.base_metrics_service = (
            base_metrics_service
        )

        self._lock = threading.RLock()

        registered_endpoints = tuple(
            base_metrics_service
            .registered_endpoints
        )

        self._endpoint_latency_total = {
            endpoint: 0.0
            for endpoint
            in registered_endpoints
        }

        self._endpoint_latency_count = {
            endpoint: 0
            for endpoint
            in registered_endpoints
        }

    @property
    def registered_endpoints(
        self,
    ) -> tuple[str, ...]:
        return tuple(
            self.base_metrics_service
            .registered_endpoints
        )

    def record_request(
        self,
        *,
        endpoint: str,
        status_code: int,
        latency_ms: float,
        error_code: str | None = None,
    ) -> None:
        normalized_latency = float(
            latency_ms
        )

        if (
            not math.isfinite(
                normalized_latency
            )
            or normalized_latency < 0.0
        ):
            raise ValueError(
                "Request latency must be finite and non-negative."
            )

        self.base_metrics_service.record_request(
            endpoint=endpoint,
            status_code=status_code,
            latency_ms=normalized_latency,
            error_code=error_code,
        )

        with self._lock:
            self._endpoint_latency_total[
                endpoint
            ] += normalized_latency

            self._endpoint_latency_count[
                endpoint
            ] += 1

    def record_service_invocation(
        self,
        service_name: str,
        count: int = 1,
    ) -> None:
        self.base_metrics_service.record_service_invocation(
            service_name,
            count,
        )

    def record_language_action(
        self,
        action: str,
        count: int = 1,
    ) -> None:
        self.base_metrics_service.record_language_action(
            action,
            count,
        )

    def snapshot(
        self,
    ):
        return self.base_metrics_service.snapshot()

    def endpoint_average_latency_ms(
        self,
    ) -> dict[str, float]:
        with self._lock:
            averages = {}

            for endpoint in self.registered_endpoints:
                request_count = (
                    self._endpoint_latency_count[
                        endpoint
                    ]
                )

                if request_count:
                    average = (
                        self._endpoint_latency_total[
                            endpoint
                        ]
                        / request_count
                    )
                else:
                    average = 0.0

                averages[endpoint] = float(
                    average
                )

        return averages

    def api_snapshot(
        self,
    ) -> APIOperationalSnapshot:
        base_snapshot = (
            self.base_metrics_service.snapshot()
        )

        accepted_count = (
            base_snapshot
            .accepted_model_generations
        )

        fallback_count = (
            base_snapshot
            .safe_template_fallbacks
        )

        return APIOperationalSnapshot(
            service_started_at_utc=(
                base_snapshot
                .service_started_at_utc
            ),
            total_requests=(
                base_snapshot.total_requests
            ),
            successful_requests=(
                base_snapshot
                .successful_requests
            ),
            failed_requests=(
                base_snapshot.failed_requests
            ),
            endpoint_request_counts=dict(
                base_snapshot
                .endpoint_request_counts
            ),
            endpoint_average_latency_ms=(
                self.endpoint_average_latency_ms()
            ),
            language_generation_requests=(
                accepted_count
                + fallback_count
            ),
            guardrail_action_counts={
                "accepted_model_generation": (
                    accepted_count
                ),
                "safe_template_fallback": (
                    fallback_count
                ),
            },
        )
'''

telemetry_adapter_path.write_text(
    telemetry_adapter_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Import the generated telemetry module
# -------------------------------------------------------------------------
importlib.invalidate_caches()

module_name = (
    "api.core.telemetry"
)

if module_name in sys.modules:
    telemetry_module = importlib.reload(
        sys.modules[module_name]
    )
else:
    telemetry_module = importlib.import_module(
        module_name
    )

EndpointTelemetryAdapter = (
    telemetry_module
    .EndpointTelemetryAdapter
)

APIOperationalSnapshot = (
    telemetry_module
    .APIOperationalSnapshot
)


# -------------------------------------------------------------------------
# Validate the adapter using an isolated metrics service
# -------------------------------------------------------------------------
isolated_base_metrics = (
    OperationalMetricsService(
        registered_endpoints=(
            registered_metric_endpoints
        ),
        maximum_latency_samples=100,
    )
)

isolated_telemetry = (
    EndpointTelemetryAdapter(
        isolated_base_metrics
    )
)

isolated_telemetry.record_request(
    endpoint="GET /health",
    status_code=200,
    latency_ms=10.0,
)

isolated_telemetry.record_request(
    endpoint="GET /health",
    status_code=200,
    latency_ms=30.0,
)

isolated_telemetry.record_request(
    endpoint="GET /api/v1/model/info",
    status_code=500,
    latency_ms=50.0,
    error_code="SERVICE_EXECUTION_ERROR",
)

isolated_telemetry.record_language_action(
    "accepted_model_generation",
    count=2,
)

isolated_telemetry.record_language_action(
    "safe_template_fallback",
    count=1,
)

isolated_api_snapshot = (
    isolated_telemetry.api_snapshot()
)


# -------------------------------------------------------------------------
# Wrap the active operational metrics service
# -------------------------------------------------------------------------
api_operational_metrics_service = (
    EndpointTelemetryAdapter(
        operational_metrics_service
    )
)


# -------------------------------------------------------------------------
# Inject the telemetry adapter into existing workflows
# -------------------------------------------------------------------------
stored_prediction_language_workflow.operational_metrics = (
    api_operational_metrics_service
)

image_analysis_workflow.operational_metrics = (
    api_operational_metrics_service
)

complete_analysis_workflow.operational_metrics = (
    api_operational_metrics_service
)


# -------------------------------------------------------------------------
# Rebuild the application container without reloading models
# -------------------------------------------------------------------------
api_service_container = ServiceContainer(
    image_validation_service=(
        image_validation_service
    ),
    computer_vision_service=(
        computer_vision_service
    ),
    prediction_store_service=(
        prediction_store_service
    ),
    gradcam_service=(
        gradcam_service
    ),
    grounding_serializer=(
        grounded_input_serializer
    ),
    language_model_service=(
        grounded_language_model_service
    ),
    language_guardrail_service=(
        language_guardrail_service
    ),
    operational_metrics_service=(
        api_operational_metrics_service
    ),
    image_analysis_workflow=(
        image_analysis_workflow
    ),
    language_workflow=(
        stored_prediction_language_workflow
    ),
    complete_analysis_workflow=(
        complete_analysis_workflow
    ),
)

configure_service_container(
    api_service_container,
    replace=True,
)

resolved_telemetry_container = (
    get_service_container()
)

initial_api_operational_snapshot = (
    api_operational_metrics_service
    .api_snapshot()
)

telemetry_adapter_checksum = hashlib.sha256(
    telemetry_adapter_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Validate telemetry alignment
# -------------------------------------------------------------------------
telemetry_checks = {
    (
        "Telemetry adapter module was written"
    ): telemetry_adapter_path.is_file(),
    (
        "Three isolated requests were recorded"
    ): (
        isolated_api_snapshot
        .total_requests
        == 3
    ),
    (
        "Successful isolated requests are preserved"
    ): (
        isolated_api_snapshot
        .successful_requests
        == 2
    ),
    (
        "Failed isolated requests are preserved"
    ): (
        isolated_api_snapshot
        .failed_requests
        == 1
    ),
    (
        "Health endpoint average is exact"
    ): (
        isolated_api_snapshot
        .endpoint_average_latency_ms[
            "GET /health"
        ]
        == 20.0
    ),
    (
        "Model-info endpoint average is exact"
    ): (
        isolated_api_snapshot
        .endpoint_average_latency_ms[
            "GET /api/v1/model/info"
        ]
        == 50.0
    ),
    (
        "Unused endpoint averages remain zero"
    ): (
        isolated_api_snapshot
        .endpoint_average_latency_ms[
            "POST /api/v1/image/classify"
        ]
        == 0.0
    ),
    (
        "Language generation count is preserved"
    ): (
        isolated_api_snapshot
        .language_generation_requests
        == 3
    ),
    (
        "Guardrail action counts are preserved"
    ): (
        isolated_api_snapshot
        .guardrail_action_counts[
            "accepted_model_generation"
        ]
        == 2
        and isolated_api_snapshot
        .guardrail_action_counts[
            "safe_template_fallback"
        ]
        == 1
    ),
    (
        "Existing workflows use the telemetry adapter"
    ): (
        stored_prediction_language_workflow
        .operational_metrics
        is api_operational_metrics_service
        and image_analysis_workflow
        .operational_metrics
        is api_operational_metrics_service
        and complete_analysis_workflow
        .operational_metrics
        is api_operational_metrics_service
    ),
    (
        "Application container uses the telemetry adapter"
    ): (
        resolved_telemetry_container
        .operational_metrics_service
        is api_operational_metrics_service
    ),
    (
        "Model instances were not reloaded"
    ): (
        resolved_telemetry_container
        .computer_vision_service
        is computer_vision_service
        and resolved_telemetry_container
        .language_model_service
        is grounded_language_model_service
    ),
    (
        "Telemetry checksum is available"
    ): len(
        telemetry_adapter_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the telemetry summary
# -------------------------------------------------------------------------
print(
    "ENDPOINT-LEVEL API TELEMETRY ADAPTER"
)
print("-" * 100)
print(
    f"{'Telemetry module':32}: "
    f"{telemetry_adapter_path}"
)
print(
    f"{'Module SHA-256':32}: "
    f"{telemetry_adapter_checksum[:16]}..."
)
print(
    f"{'Registered endpoints':32}: "
    f"{len(registered_metric_endpoints)}"
)
print(
    f"{'Isolated requests':32}: "
    f"{isolated_api_snapshot.total_requests}"
)
print(
    f"{'Health average latency':32}: "
    f"{isolated_api_snapshot.endpoint_average_latency_ms['GET /health']:.2f} ms"
)
print(
    f"{'Model-info average latency':32}: "
    f"{isolated_api_snapshot.endpoint_average_latency_ms['GET /api/v1/model/info']:.2f} ms"
)
print(
    f"{'Language generations':32}: "
    f"{isolated_api_snapshot.language_generation_requests}"
)
print(
    f"{'Active container configured':32}: "
    f"{service_container_is_configured()}"
)
print("-" * 100)

for check_name, passed in (
    telemetry_checks.items()
):
    print(
        f"{check_name:63}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    telemetry_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in telemetry_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Endpoint telemetry validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: ENDPOINT-LEVEL OPERATIONAL TELEMETRY "
    "READY FOR FASTAPI MIDDLEWARE"
)

ENDPOINT-LEVEL API TELEMETRY ADAPTER
----------------------------------------------------------------------------------------------------
Telemetry module                : /home/jovyan/chest-xray-ai-assistant/api/core/telemetry.py
Module SHA-256                  : d24a1c2cc2507808...
Registered endpoints            : 12
Isolated requests               : 3
Health average latency          : 20.00 ms
Model-info average latency      : 50.00 ms
Language generations            : 3
Active container configured     : True
----------------------------------------------------------------------------------------------------
Telemetry adapter module was written                           : PASS
Three isolated requests were recorded                          : PASS
Successful isolated requests are preserved                     : PASS
Failed isolated requests are preserved                         : PASS
Health endpoint average is exact                               : PASS
Model-info endpoint average is

<a id="nb07-6-9-health-model-lineage-evaluation-and-llmops-routes"></a>
### 6.9 Health, Model Lineage, Evaluation, and LLMOps Routes

This block registers the four read-only system endpoints. Frozen model information and evaluation summaries are exported as sanitized response templates, while health and operational measurements remain dynamic. The public payloads exclude internal filesystem paths, raw prompts, images, model tensors, and exception details.


In [46]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import importlib
import json
import sys
from pathlib import Path

from fastapi.routing import APIRoute
from pydantic import BaseModel


# -------------------------------------------------------------------------
# Define the system route and sanitized-template paths
# -------------------------------------------------------------------------
system_routes_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "api/routes/system.py"
)

system_template_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "configs/system_response_templates.json"
)

system_routes_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

system_template_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Resolve validated system response examples from earlier schema cells
# -------------------------------------------------------------------------
def resolve_schema_example(
    class_name,
):
    preferred_variable_names = {
        "HealthResponse": (
            "health_response_example",
            "health_example",
        ),
        "ModelInfoResponse": (
            "model_info_response_example",
            "model_info_example",
        ),
        "ModelMetricsResponse": (
            "model_metrics_response_example",
            "model_metrics_example",
        ),
        "OperationalMetricsResponse": (
            "operational_metrics_response_example",
            "operational_metrics_example",
        ),
    }

    for variable_name in (
        preferred_variable_names.get(
            class_name,
            ()
        )
    ):
        candidate = globals().get(
            variable_name
        )

        if (
            candidate is not None
            and type(candidate).__name__
            == class_name
        ):
            return candidate

    for candidate in list(
        globals().values()
    ):
        if (
            candidate is not None
            and type(candidate).__name__
            == class_name
            and isinstance(
                candidate,
                BaseModel,
            )
        ):
            return candidate

    raise RuntimeError(
        f"A validated {class_name} example "
        "could not be resolved."
    )


health_schema_example = (
    resolve_schema_example(
        "HealthResponse"
    )
)

model_info_schema_example = (
    resolve_schema_example(
        "ModelInfoResponse"
    )
)

model_metrics_schema_example = (
    resolve_schema_example(
        "ModelMetricsResponse"
    )
)

operational_schema_example = (
    resolve_schema_example(
        "OperationalMetricsResponse"
    )
)


# -------------------------------------------------------------------------
# Retain only response-specific sanitized template fields
# -------------------------------------------------------------------------
common_response_fields = {
    "request_id",
    "timestamp_utc",
    "api_version",
    "status",
    "model_versions",
    "prompt_registry_version",
    "latency_ms",
    "warnings",
    "educational_use_only",
}

dynamic_operational_fields = {
    "service_started_at_utc",
    "total_requests",
    "successful_requests",
    "failed_requests",
    "endpoint_request_counts",
    "endpoint_average_latency_ms",
    "language_generation_requests",
    "guardrail_action_counts",
}


def response_specific_payload(
    response_model,
    excluded_fields=(),
):
    payload = response_model.model_dump(
        mode="json"
    )

    fields_to_remove = (
        common_response_fields
        | set(
            excluded_fields
        )
    )

    return {
        key: value
        for key, value
        in payload.items()
        if key not in fields_to_remove
    }


system_response_templates = {
    "template_version": (
        "system-response-templates-v1"
    ),
    "health": response_specific_payload(
        health_schema_example,
        excluded_fields={
            "uptime_seconds",
        },
    ),
    "model_info": (
        response_specific_payload(
            model_info_schema_example
        )
    ),
    "model_metrics": (
        response_specific_payload(
            model_metrics_schema_example
        )
    ),
    "operational_static": (
        response_specific_payload(
            operational_schema_example,
            excluded_fields=(
                dynamic_operational_fields
            ),
        )
    ),
}

system_template_path.write_text(
    json.dumps(
        system_response_templates,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Confirm that sanitized templates contain no internal path values
# -------------------------------------------------------------------------
def collect_string_values(
    value,
):
    if isinstance(value, dict):
        for nested_value in value.values():
            yield from collect_string_values(
                nested_value
            )

    elif isinstance(value, list):
        for nested_value in value:
            yield from collect_string_values(
                nested_value
            )

    elif isinstance(value, str):
        yield value


template_string_values = list(
    collect_string_values(
        system_response_templates
    )
)

templates_contain_internal_paths = any(
    (
        "/home/" in value
        or "\\home\\" in value.lower()
        or "file://" in value.lower()
    )
    for value
    in template_string_values
)


# -------------------------------------------------------------------------
# Write the read-only system routes
# -------------------------------------------------------------------------
system_routes_source = '''
"""Health, model, evaluation, and operational FastAPI routes."""

from __future__ import annotations

import json
from pathlib import Path

from fastapi import (
    APIRouter,
    Depends,
)

from api.core.dependencies import (
    ServiceContainer,
    get_service_container,
)
from api.core.runtime import (
    RequestContext,
    build_success_metadata,
)
from api.schemas import (
    HealthResponse,
    ModelInfoResponse,
    ModelMetricsResponse,
    OperationalMetricsResponse,
)


_TEMPLATE_PATH = (
    Path(__file__).resolve().parents[2]
    / "configs"
    / "system_response_templates.json"
)


def _load_templates() -> dict:
    with _TEMPLATE_PATH.open(
        "r",
        encoding="utf-8",
    ) as template_file:
        return json.load(
            template_file
        )


SYSTEM_TEMPLATES = _load_templates()


health_router = APIRouter(
    tags=["health"],
)

system_router = APIRouter(
    prefix="/api/v1",
    tags=["system"],
)


@health_router.get(
    "/health",
    response_model=HealthResponse,
    summary="Check service readiness",
)
async def health(
    services: ServiceContainer = Depends(
        get_service_container
    ),
) -> HealthResponse:
    """Return service readiness without running model inference."""

    request_context = RequestContext()

    runtime_snapshot = (
        services.operational_metrics_service
        .snapshot()
    )

    response_metadata = build_success_metadata(
        request_context,
        include_language=True,
        include_explainability=True,
    )

    return HealthResponse.model_validate(
        {
            **response_metadata,
            **SYSTEM_TEMPLATES[
                "health"
            ],
            "uptime_seconds": (
                runtime_snapshot
                .uptime_seconds
            ),
        }
    )


@system_router.get(
    "/model/info",
    response_model=ModelInfoResponse,
    summary="Return versioned model information",
)
async def model_info(
    services: ServiceContainer = Depends(
        get_service_container
    ),
) -> ModelInfoResponse:
    """Return sanitized computer-vision and language lineage."""

    request_context = RequestContext()

    response_metadata = build_success_metadata(
        request_context,
        include_language=True,
        include_explainability=True,
    )

    return ModelInfoResponse.model_validate(
        {
            **response_metadata,
            **SYSTEM_TEMPLATES[
                "model_info"
            ],
        }
    )


@system_router.get(
    "/model/metrics",
    response_model=ModelMetricsResponse,
    summary="Return frozen model evaluation metrics",
)
async def model_metrics(
    services: ServiceContainer = Depends(
        get_service_container
    ),
) -> ModelMetricsResponse:
    """Return frozen evaluation and guardrail measurements."""

    request_context = RequestContext()

    response_metadata = build_success_metadata(
        request_context,
        include_language=True,
        include_explainability=True,
    )

    return ModelMetricsResponse.model_validate(
        {
            **response_metadata,
            **SYSTEM_TEMPLATES[
                "model_metrics"
            ],
        }
    )


@system_router.get(
    "/llmops/metrics",
    response_model=OperationalMetricsResponse,
    summary="Return API operational metrics",
)
async def llmops_metrics(
    services: ServiceContainer = Depends(
        get_service_container
    ),
) -> OperationalMetricsResponse:
    """Return aggregate operational data without request content."""

    request_context = RequestContext()

    runtime_snapshot = (
        services.operational_metrics_service
        .api_snapshot()
    )

    response_metadata = build_success_metadata(
        request_context,
        include_language=True,
        include_explainability=True,
    )

    return OperationalMetricsResponse.model_validate(
        {
            **response_metadata,
            **SYSTEM_TEMPLATES[
                "operational_static"
            ],
            "service_started_at_utc": (
                runtime_snapshot
                .service_started_at_utc
            ),
            "total_requests": (
                runtime_snapshot
                .total_requests
            ),
            "successful_requests": (
                runtime_snapshot
                .successful_requests
            ),
            "failed_requests": (
                runtime_snapshot
                .failed_requests
            ),
            "endpoint_request_counts": (
                runtime_snapshot
                .endpoint_request_counts
            ),
            "endpoint_average_latency_ms": (
                runtime_snapshot
                .endpoint_average_latency_ms
            ),
            "language_generation_requests": (
                runtime_snapshot
                .language_generation_requests
            ),
            "guardrail_action_counts": (
                runtime_snapshot
                .guardrail_action_counts
            ),
        }
    )
'''

system_routes_path.write_text(
    system_routes_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Import the generated route module
# -------------------------------------------------------------------------
importlib.invalidate_caches()

module_name = (
    "api.routes.system"
)

if module_name in sys.modules:
    system_routes_module = importlib.reload(
        sys.modules[module_name]
    )
else:
    system_routes_module = importlib.import_module(
        module_name
    )

health_router = (
    system_routes_module.health_router
)

system_router = (
    system_routes_module.system_router
)


# -------------------------------------------------------------------------
# Inspect the registered system routes
# -------------------------------------------------------------------------
registered_system_routes = [
    route
    for router in (
        health_router,
        system_router,
    )
    for route in router.routes
    if isinstance(
        route,
        APIRoute,
    )
]

actual_system_route_contracts = {}

for route in registered_system_routes:
    for method in route.methods:
        actual_system_route_contracts[
            (
                method,
                route.path,
            )
        ] = (
            route.response_model.__name__
            if route.response_model
            is not None
            else None
        )

expected_system_route_contracts = {
    (
        "GET",
        "/health",
    ): "HealthResponse",
    (
        "GET",
        "/api/v1/model/info",
    ): "ModelInfoResponse",
    (
        "GET",
        "/api/v1/model/metrics",
    ): "ModelMetricsResponse",
    (
        "GET",
        "/api/v1/llmops/metrics",
    ): "OperationalMetricsResponse",
}

system_routes_checksum = hashlib.sha256(
    system_routes_path.read_bytes()
).hexdigest()

system_template_checksum = hashlib.sha256(
    system_template_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Validate system route registration and sanitization
# -------------------------------------------------------------------------
system_route_checks = {
    (
        "System route module was written"
    ): system_routes_path.is_file(),
    (
        "Sanitized system templates were exported"
    ): system_template_path.is_file(),
    (
        "Four read-only endpoints are registered"
    ): len(
        registered_system_routes
    ) == 4,
    (
        "Every system endpoint uses GET"
    ): all(
        route.methods == {"GET"}
        for route
        in registered_system_routes
    ),
    (
        "Health endpoint remains unversioned"
    ): (
        (
            "GET",
            "/health",
        )
        in actual_system_route_contracts
    ),
    (
        "Other system routes use the API prefix"
    ): all(
        (
            route.path == "/health"
            or route.path.startswith(
                "/api/v1/"
            )
        )
        for route
        in registered_system_routes
    ),
    (
        "System responses match the frozen contract"
    ): (
        actual_system_route_contracts
        == expected_system_route_contracts
    ),
    (
        "Templates contain no internal paths"
    ): not (
        templates_contain_internal_paths
    ),
    (
        "System route checksum is available"
    ): len(
        system_routes_checksum
    ) == 64,
    (
        "System template checksum is available"
    ): len(
        system_template_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the system-route summary
# -------------------------------------------------------------------------
print(
    "HEALTH, MODEL, EVALUATION, AND LLMOPS FASTAPI ROUTES"
)
print("-" * 100)
print(
    f"{'Route module':29}: "
    f"{system_routes_path}"
)
print(
    f"{'Route SHA-256':29}: "
    f"{system_routes_checksum[:16]}..."
)
print(
    f"{'Template registry':29}: "
    f"{system_template_path}"
)
print(
    f"{'Template SHA-256':29}: "
    f"{system_template_checksum[:16]}..."
)
print(
    f"{'Registered routes':29}: "
    f"{len(registered_system_routes)}"
)
print("-" * 100)

for route in registered_system_routes:
    print(
        f"{','.join(sorted(route.methods)):6} "
        f"{route.path:39} | "
        f"{route.response_model.__name__}"
    )

print("-" * 100)

for check_name, passed in (
    system_route_checks.items()
):
    print(
        f"{check_name:62}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    system_route_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in system_route_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "System route validation failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: SYSTEM ROUTES READY FOR "
    "FASTAPI APPLICATION REGISTRATION"
)

HEALTH, MODEL, EVALUATION, AND LLMOPS FASTAPI ROUTES
----------------------------------------------------------------------------------------------------
Route module                 : /home/jovyan/chest-xray-ai-assistant/api/routes/system.py
Route SHA-256                : 2cb1312e6d75f500...
Template registry            : /home/jovyan/chest-xray-ai-assistant/configs/system_response_templates.json
Template SHA-256             : a35d2a4863910dd0...
Registered routes            : 4
----------------------------------------------------------------------------------------------------
GET    /health                                 | HealthResponse
GET    /api/v1/model/info                      | ModelInfoResponse
GET    /api/v1/model/metrics                   | ModelMetricsResponse
GET    /api/v1/llmops/metrics                  | OperationalMetricsResponse
----------------------------------------------------------------------------------------------------
System route module was written     

<a id="nb07-6-10-fastapi-application-assembly-error-handling-and-request-telemetry"></a>
### 6.10 FastAPI Application Assembly, Error Handling, and Request Telemetry

This block assembles all twelve routes into the versioned FastAPI application. It adds request identity, centralized latency measurement, endpoint-level operational metrics, controlled service-error serialization, sanitized validation errors, generic exception protection, and OpenAPI generation. Internal paths, stack traces, uploaded content, and raw model data are never included in error responses.


In [47]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import importlib
import json
import sys
from pathlib import Path

from fastapi import FastAPI
from fastapi.routing import APIRoute


# -------------------------------------------------------------------------
# Define application module paths
# -------------------------------------------------------------------------
main_module_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "api/main.py"
)

routes_init_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "api/routes/__init__.py"
)

openapi_artifact_path = Path(
    "/home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data/outputs/api/"
    "openapi.json"
)

main_module_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

routes_init_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

openapi_artifact_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Export the route package interface
# -------------------------------------------------------------------------
routes_init_source = '''
"""Public FastAPI route package."""

from api.routes.aggregate import (
    router as aggregate_router,
)
from api.routes.system import (
    health_router,
    system_router,
)
from api.routes.workflows import (
    router as workflow_router,
)

__all__ = [
    "aggregate_router",
    "health_router",
    "system_router",
    "workflow_router",
]
'''

routes_init_path.write_text(
    routes_init_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Write the FastAPI application
# -------------------------------------------------------------------------
main_module_source = '''
"""FastAPI application for the ChestMNIST educational assistant."""

from __future__ import annotations

import inspect
import time
from typing import Any

from fastapi import (
    FastAPI,
    Request,
)
from fastapi.exceptions import (
    RequestValidationError,
)
from fastapi.responses import (
    JSONResponse,
)

from api.core.config import (
    get_settings,
)
from api.core.dependencies import (
    get_service_container,
)
from api.core.errors import (
    ServiceError,
    ServiceExecutionError,
)
from api.core.runtime import (
    RequestContext,
    build_error_response,
    utc_now,
)
from api.routes import (
    aggregate_router,
    health_router,
    system_router,
    workflow_router,
)
from api.schemas import (
    APIErrorResponse,
)


settings = get_settings()


app = FastAPI(
    title=settings.api_title,
    version=settings.api_version,
    description=(
        "Educational decision-support API for frozen ChestMNIST "
        "classification, visual evidence, and grounded language "
        "generation. This service is not a diagnostic system."
    ),
    docs_url="/docs",
    redoc_url="/redoc",
    openapi_url="/openapi.json",
)


app.include_router(
    health_router
)

app.include_router(
    system_router
)

app.include_router(
    workflow_router
)

app.include_router(
    aggregate_router
)


def _request_context(
    request: Request,
) -> RequestContext:
    context = getattr(
        request.state,
        "request_context",
        None,
    )

    if context is None:
        context = RequestContext()
        request.state.request_context = (
            context
        )

    return context


def _context_latency_ms(
    context: RequestContext,
) -> float:
    for method_name in (
        "elapsed_ms",
        "latency_ms",
    ):
        method = getattr(
            context,
            method_name,
            None,
        )

        if callable(method):
            return max(
                0.0,
                float(
                    method()
                ),
            )

    return 0.0


def _serialize_service_error(
    *,
    context: RequestContext,
    error: ServiceError,
) -> APIErrorResponse:
    """Call the frozen runtime serializer using its declared names."""

    signature = inspect.signature(
        build_error_response
    )

    keyword_values = {}

    for parameter_name in signature.parameters:
        normalized_name = (
            parameter_name.lower()
        )

        if "context" in normalized_name:
            keyword_values[
                parameter_name
            ] = context
        elif (
            "error" in normalized_name
            or "exception" in normalized_name
        ):
            keyword_values[
                parameter_name
            ] = error

    if len(keyword_values) != len(
        signature.parameters
    ):
        raise RuntimeError(
            "The controlled error serializer signature "
            "could not be resolved."
        )

    return build_error_response(
        **keyword_values
    )


def _safe_validation_details(
    error: RequestValidationError,
) -> dict[str, Any]:
    """Retain only safe validation locations, types, and messages."""

    issues = []

    for issue in error.errors():
        issues.append(
            {
                "location": [
                    str(value)
                    for value
                    in issue.get(
                        "loc",
                        ()
                    )
                ],
                "type": str(
                    issue.get(
                        "type",
                        "validation_error",
                    )
                ),
                "message": str(
                    issue.get(
                        "msg",
                        "Request validation failed.",
                    )
                ),
            }
        )

    return {
        "issues": issues
    }


@app.middleware(
    "http"
)
async def request_context_and_metrics(
    request: Request,
    call_next,
):
    """Attach request identity and record aggregate endpoint telemetry."""

    context = RequestContext()
    request.state.request_context = (
        context
    )

    started_at = time.perf_counter()

    response = await call_next(
        request
    )

    latency_ms = (
        time.perf_counter()
        - started_at
    ) * 1000.0

    route = request.scope.get(
        "route"
    )

    route_path = getattr(
        route,
        "path",
        request.url.path,
    )

    endpoint_key = (
        f"{request.method.upper()} "
        f"{route_path}"
    )

    error_code = getattr(
        request.state,
        "error_code",
        None,
    )

    try:
        services = get_service_container()

        if (
            endpoint_key
            in services
            .operational_metrics_service
            .registered_endpoints
        ):
            services.operational_metrics_service.record_request(
                endpoint=endpoint_key,
                status_code=response.status_code,
                latency_ms=latency_ms,
                error_code=error_code,
            )
    except Exception:
        # Metrics must never replace the endpoint response.
        pass

    return response


@app.exception_handler(
    ServiceError
)
async def service_error_handler(
    request: Request,
    error: ServiceError,
) -> JSONResponse:
    """Serialize controlled service exceptions."""

    context = _request_context(
        request
    )

    request.state.error_code = (
        error.error_code
    )

    error_response = (
        _serialize_service_error(
            context=context,
            error=error,
        )
    )

    return JSONResponse(
        status_code=error.status_code,
        content=error_response.model_dump(
            mode="json"
        ),
    )


@app.exception_handler(
    RequestValidationError
)
async def request_validation_error_handler(
    request: Request,
    error: RequestValidationError,
) -> JSONResponse:
    """Return sanitized request-validation information."""

    context = _request_context(
        request
    )

    request.state.error_code = (
        "REQUEST_VALIDATION_ERROR"
    )

    error_response = APIErrorResponse(
        request_id=context.request_id,
        timestamp_utc=utc_now(),
        api_version=settings.api_version,
        status="error",
        error_code=(
            "REQUEST_VALIDATION_ERROR"
        ),
        message=(
            "The request did not satisfy the API contract."
        ),
        details=(
            _safe_validation_details(
                error
            )
        ),
        latency_ms=(
            _context_latency_ms(
                context
            )
        ),
        educational_use_only=True,
    )

    return JSONResponse(
        status_code=422,
        content=error_response.model_dump(
            mode="json"
        ),
    )


@app.exception_handler(
    Exception
)
async def unexpected_error_handler(
    request: Request,
    error: Exception,
) -> JSONResponse:
    """Convert unexpected failures into a non-sensitive response."""

    context = _request_context(
        request
    )

    controlled_error = (
        ServiceExecutionError()
    )

    request.state.error_code = (
        controlled_error.error_code
    )

    error_response = (
        _serialize_service_error(
            context=context,
            error=controlled_error,
        )
    )

    return JSONResponse(
        status_code=(
            controlled_error.status_code
        ),
        content=error_response.model_dump(
            mode="json"
        ),
    )
'''

main_module_path.write_text(
    main_module_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Import the generated FastAPI application
# -------------------------------------------------------------------------
importlib.invalidate_caches()

for module_name in (
    "api.routes",
    "api.main",
):
    if module_name in sys.modules:
        del sys.modules[
            module_name
        ]

main_module = importlib.import_module(
    "api.main"
)

app = main_module.app


# -------------------------------------------------------------------------
# Inspect application routes
# -------------------------------------------------------------------------
registered_application_routes = [
    route
    for route in app.routes
    if isinstance(
        route,
        APIRoute,
    )
]

application_route_contracts = {
    (
        method,
        route.path,
    ): (
        route.response_model.__name__
        if route.response_model
        is not None
        else None
    )
    for route
    in registered_application_routes
    for method
    in route.methods
}

expected_application_route_contracts = {
    (
        "GET",
        "/health",
    ): "HealthResponse",
    (
        "GET",
        "/api/v1/model/info",
    ): "ModelInfoResponse",
    (
        "GET",
        "/api/v1/model/metrics",
    ): "ModelMetricsResponse",
    (
        "POST",
        "/api/v1/image/classify",
    ): "ClassificationResponse",
    (
        "POST",
        "/api/v1/image/analyze",
    ): "ImageAnalysisResponse",
    (
        "POST",
        "/api/v1/report/generate",
    ): "LanguageGenerationResponse",
    (
        "POST",
        "/api/v1/explanation/generate",
    ): "LanguageGenerationResponse",
    (
        "POST",
        "/api/v1/question/answer",
    ): "LanguageGenerationResponse",
    (
        "POST",
        "/api/v1/follow-up/recommend",
    ): "LanguageGenerationResponse",
    (
        "POST",
        "/api/v1/analyze-complete",
    ): "CompleteAnalysisResponse",
    (
        "GET",
        "/api/v1/predictions/{prediction_id}",
    ): "StoredPredictionResponse",
    (
        "GET",
        "/api/v1/llmops/metrics",
    ): "OperationalMetricsResponse",
}


# -------------------------------------------------------------------------
# Export the OpenAPI document
# -------------------------------------------------------------------------
openapi_document = app.openapi()

openapi_artifact_path.write_text(
    json.dumps(
        openapi_document,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

main_module_checksum = hashlib.sha256(
    main_module_path.read_bytes()
).hexdigest()

openapi_checksum = hashlib.sha256(
    openapi_artifact_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Validate application assembly
# -------------------------------------------------------------------------
openapi_paths = set(
    openapi_document[
        "paths"
    ]
)

expected_openapi_paths = {
    path
    for (
        method,
        path,
    )
    in expected_application_route_contracts
}

operation_ids = [
    operation[
        "operationId"
    ]
    for path_item
    in openapi_document[
        "paths"
    ].values()
    for method, operation
    in path_item.items()
    if method.lower()
    in {
        "get",
        "post",
        "put",
        "patch",
        "delete",
    }
]

application_checks = {
    (
        "FastAPI main module was written"
    ): main_module_path.is_file(),
    (
        "FastAPI application was created"
    ): isinstance(
        app,
        FastAPI,
    ),
    (
        "Twelve endpoint routes are registered"
    ): len(
        registered_application_routes
    ) == 12,
    (
        "Every endpoint matches the frozen contract"
    ): (
        application_route_contracts
        == expected_application_route_contracts
    ),
    (
        "Health endpoint remains unversioned"
    ): (
        "/health"
        in openapi_paths
    ),
    (
        "All expected OpenAPI paths are present"
    ): (
        openapi_paths
        == expected_openapi_paths
    ),
    (
        "Every operation identifier is unique"
    ): (
        len(
            operation_ids
        )
        == len(
            set(
                operation_ids
            )
        )
    ),
    (
        "Controlled service error handler is registered"
    ): (
        main_module.ServiceError
        in app.exception_handlers
    ),
    (
        "Validation error handler is registered"
    ): (
        main_module.RequestValidationError
        in app.exception_handlers
    ),
    (
        "Unexpected error handler is registered"
    ): (
        Exception
        in app.exception_handlers
    ),
    (
        "OpenAPI document was exported"
    ): openapi_artifact_path.is_file(),
    (
        "Main module checksum is available"
    ): len(
        main_module_checksum
    ) == 64,
    (
        "OpenAPI checksum is available"
    ): len(
        openapi_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the application summary
# -------------------------------------------------------------------------
print(
    "FASTAPI APPLICATION ASSEMBLY AND OPENAPI CONTRACT"
)
print("-" * 100)
print(
    f"{'Application module':29}: "
    f"{main_module_path}"
)
print(
    f"{'Module SHA-256':29}: "
    f"{main_module_checksum[:16]}..."
)
print(
    f"{'API title':29}: "
    f"{app.title}"
)
print(
    f"{'API version':29}: "
    f"{app.version}"
)
print(
    f"{'Registered endpoints':29}: "
    f"{len(registered_application_routes)}"
)
print(
    f"{'OpenAPI paths':29}: "
    f"{len(openapi_paths)}"
)
print(
    f"{'OpenAPI artifact':29}: "
    f"{openapi_artifact_path}"
)
print(
    f"{'OpenAPI SHA-256':29}: "
    f"{openapi_checksum[:16]}..."
)
print("-" * 100)

for (
    method,
    path,
), response_name in sorted(
    application_route_contracts.items(),
    key=lambda item: (
        item[0][1],
        item[0][0],
    ),
):
    print(
        f"{method:6} "
        f"{path:45} | "
        f"{response_name}"
    )

print("-" * 100)

for check_name, passed in (
    application_checks.items()
):
    print(
        f"{check_name:64}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    application_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in application_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "FastAPI application assembly failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: FASTAPI APPLICATION AND OPENAPI CONTRACT "
    "READY FOR IN-PROCESS API TESTING"
)

FASTAPI APPLICATION ASSEMBLY AND OPENAPI CONTRACT
----------------------------------------------------------------------------------------------------
Application module           : /home/jovyan/chest-xray-ai-assistant/api/main.py
Module SHA-256               : 53840dcb7a66d086...
API title                    : API-Driven Chest X-Ray Analysis and Explanation Assistant
API version                  : v1
Registered endpoints         : 12
OpenAPI paths                : 12
OpenAPI artifact             : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/openapi.json
OpenAPI SHA-256              : 6f73e0c99b44c7f7...
----------------------------------------------------------------------------------------------------
POST   /api/v1/analyze-complete                      | CompleteAnalysisResponse
POST   /api/v1/explanation/generate                  | LanguageGenerationResponse
POST   /api/v1/follow-up/recommend                   | LanguageGenerationResponse
POST   /api/v1

**[↑ Back to notebook index](#notebook-index)**


<a id="nb07-7-in-process-api-integration-testing"></a>
## 7. In-Process API Integration Testing

<a id="nb07-7-1-system-endpoints-and-controlled-error-responses"></a>
### 7.1 System Endpoints and Controlled Error Responses

This block exercises the assembled FastAPI application through its in-process HTTP client. It validates the four read-only system endpoints, malformed prediction identifiers, missing stored predictions, response-schema parsing, request identity, error sanitization, and endpoint telemetry without opening an external network port.


In [51]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import csv
import hashlib
from pathlib import Path
from uuid import uuid4

from fastapi.testclient import TestClient

from api.schemas import (
    APIErrorResponse,
    HealthResponse,
    ModelInfoResponse,
    ModelMetricsResponse,
    OperationalMetricsResponse,
)


# -------------------------------------------------------------------------
# Define the integration-test artifact
# -------------------------------------------------------------------------
system_test_artifact_path = Path(
    "/home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data/outputs/api/"
    "system_endpoint_integration_tests.csv"
)

system_test_artifact_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Create the in-process client
# -------------------------------------------------------------------------
api_test_client = TestClient(
    app,
    raise_server_exceptions=False,
)


# -------------------------------------------------------------------------
# Execute successful system requests
# -------------------------------------------------------------------------
health_http_response = api_test_client.get(
    "/health"
)

model_info_http_response = api_test_client.get(
    "/api/v1/model/info"
)

model_metrics_http_response = api_test_client.get(
    "/api/v1/model/metrics"
)

llmops_http_response = api_test_client.get(
    "/api/v1/llmops/metrics"
)


# -------------------------------------------------------------------------
# Execute controlled error requests
# -------------------------------------------------------------------------
malformed_prediction_http_response = (
    api_test_client.get(
        "/api/v1/predictions/"
        "not-a-valid-uuid"
    )
)

missing_prediction_id = uuid4()

missing_prediction_http_response = (
    api_test_client.get(
        f"/api/v1/predictions/"
        f"{missing_prediction_id}"
    )
)


# -------------------------------------------------------------------------
# Parse JSON only when the response declares JSON content
# -------------------------------------------------------------------------
def safe_json_response(
    response,
):
    content_type = response.headers.get(
        "content-type",
        "",
    ).lower()

    if "application/json" not in content_type:
        return None

    try:
        return response.json()
    except ValueError:
        return None


health_payload = safe_json_response(
    health_http_response
)

model_info_payload = safe_json_response(
    model_info_http_response
)

model_metrics_payload = safe_json_response(
    model_metrics_http_response
)

llmops_payload = safe_json_response(
    llmops_http_response
)

malformed_error_payload = safe_json_response(
    malformed_prediction_http_response
)

missing_error_payload = safe_json_response(
    missing_prediction_http_response
)


# -------------------------------------------------------------------------
# Validate payloads through canonical Pydantic schemas
# -------------------------------------------------------------------------
parsed_health_response = (
    HealthResponse.model_validate(
        health_payload
    )
    if (
        health_http_response.status_code
        == 200
        and health_payload is not None
    )
    else None
)

parsed_model_info_response = (
    ModelInfoResponse.model_validate(
        model_info_payload
    )
    if (
        model_info_http_response.status_code
        == 200
        and model_info_payload is not None
    )
    else None
)

parsed_model_metrics_response = (
    ModelMetricsResponse.model_validate(
        model_metrics_payload
    )
    if (
        model_metrics_http_response
        .status_code
        == 200
        and model_metrics_payload
        is not None
    )
    else None
)

parsed_llmops_response = (
    OperationalMetricsResponse.model_validate(
        llmops_payload
    )
    if (
        llmops_http_response.status_code
        == 200
        and llmops_payload is not None
    )
    else None
)

parsed_malformed_error = (
    APIErrorResponse.model_validate(
        malformed_error_payload
    )
    if (
        malformed_prediction_http_response
        .status_code
        == 422
        and malformed_error_payload
        is not None
    )
    else None
)

parsed_missing_error = (
    APIErrorResponse.model_validate(
        missing_error_payload
    )
    if (
        missing_prediction_http_response
        .status_code
        == 404
        and missing_error_payload
        is not None
    )
    else None
)


# -------------------------------------------------------------------------
# Build endpoint test records
# -------------------------------------------------------------------------
system_endpoint_test_records = [
    {
        "test_name": "health_endpoint",
        "method": "GET",
        "path": "/health",
        "expected_status": 200,
        "actual_status": (
            health_http_response.status_code
        ),
        "passed": (
            parsed_health_response
            is not None
        ),
    },
    {
        "test_name": "model_info_endpoint",
        "method": "GET",
        "path": "/api/v1/model/info",
        "expected_status": 200,
        "actual_status": (
            model_info_http_response
            .status_code
        ),
        "passed": (
            parsed_model_info_response
            is not None
        ),
    },
    {
        "test_name": "model_metrics_endpoint",
        "method": "GET",
        "path": "/api/v1/model/metrics",
        "expected_status": 200,
        "actual_status": (
            model_metrics_http_response
            .status_code
        ),
        "passed": (
            parsed_model_metrics_response
            is not None
        ),
    },
    {
        "test_name": "llmops_metrics_endpoint",
        "method": "GET",
        "path": "/api/v1/llmops/metrics",
        "expected_status": 200,
        "actual_status": (
            llmops_http_response.status_code
        ),
        "passed": (
            parsed_llmops_response
            is not None
        ),
    },
    {
        "test_name": "malformed_prediction_id",
        "method": "GET",
        "path": (
            "/api/v1/predictions/"
            "not-a-valid-uuid"
        ),
        "expected_status": 422,
        "actual_status": (
            malformed_prediction_http_response
            .status_code
        ),
        "passed": (
            parsed_malformed_error
            is not None
            and parsed_malformed_error
            .error_code
            == "REQUEST_VALIDATION_ERROR"
        ),
    },
    {
        "test_name": "missing_prediction",
        "method": "GET",
        "path": (
            "/api/v1/predictions/"
            "{prediction_id}"
        ),
        "expected_status": 404,
        "actual_status": (
            missing_prediction_http_response
            .status_code
        ),
        "passed": (
            parsed_missing_error
            is not None
            and parsed_missing_error
            .error_code
            == "PREDICTION_NOT_FOUND"
        ),
    },
]


# -------------------------------------------------------------------------
# Validate request identity and sanitization
# -------------------------------------------------------------------------
all_parsed_responses = (
    parsed_health_response,
    parsed_model_info_response,
    parsed_model_metrics_response,
    parsed_llmops_response,
    parsed_malformed_error,
    parsed_missing_error,
)

all_request_ids = {
    str(response.request_id)
    for response
    in all_parsed_responses
    if response is not None
}

serialized_error_payloads = (
    str(
        malformed_error_payload
    )
    + " "
    + str(
        missing_error_payload
    )
).lower()

internal_error_content_present = any(
    forbidden_value
    in serialized_error_payloads
    for forbidden_value in (
        "/home/",
        "traceback",
        "stack trace",
        "site-packages",
        ".py:",
    )
)

post_test_metrics = (
    api_operational_metrics_service
    .api_snapshot()
)


# -------------------------------------------------------------------------
# Export the integration-test artifact
# -------------------------------------------------------------------------
with system_test_artifact_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as test_file:
    writer = csv.DictWriter(
        test_file,
        fieldnames=[
            "test_name",
            "method",
            "path",
            "expected_status",
            "actual_status",
            "passed",
        ],
    )

    writer.writeheader()
    writer.writerows(
        system_endpoint_test_records
    )

system_test_checksum = hashlib.sha256(
    system_test_artifact_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Validate the complete system endpoint contract
# -------------------------------------------------------------------------
system_integration_checks = {
    (
        "All six HTTP checks passed"
    ): all(
        record["passed"]
        for record
        in system_endpoint_test_records
    ),
    (
        "Health response is schema compliant"
    ): (
        parsed_health_response
        is not None
        and parsed_health_response.status
        == "success"
    ),
    (
        "Health uptime is non-negative"
    ): (
        parsed_health_response
        is not None
        and parsed_health_response
        .uptime_seconds
        >= 0.0
    ),
    (
        "Model information is schema compliant"
    ): (
        parsed_model_info_response
        is not None
        and parsed_model_info_response
        .status
        == "success"
    ),
    (
        "Model metrics are schema compliant"
    ): (
        parsed_model_metrics_response
        is not None
        and parsed_model_metrics_response
        .status
        == "success"
    ),
    (
        "Operational metrics are schema compliant"
    ): (
        parsed_llmops_response
        is not None
        and parsed_llmops_response.status
        == "success"
    ),
    (
        "All six request identifiers are unique"
    ): len(
        all_request_ids
    ) == 6,
    (
        "Malformed UUID uses controlled validation error"
    ): (
        parsed_malformed_error
        is not None
        and parsed_malformed_error
        .error_code
        == "REQUEST_VALIDATION_ERROR"
    ),
    (
        "Missing prediction uses controlled not-found error"
    ): (
        parsed_missing_error
        is not None
        and parsed_missing_error
        .error_code
        == "PREDICTION_NOT_FOUND"
    ),
    (
        "Errors preserve educational boundary"
    ): (
        parsed_malformed_error
        is not None
        and parsed_missing_error
        is not None
        and parsed_malformed_error
        .educational_use_only
        and parsed_missing_error
        .educational_use_only
    ),
    (
        "Error payloads contain no internal details"
    ): not (
        internal_error_content_present
    ),
    (
        "System requests are recorded in telemetry"
    ): all(
        post_test_metrics
        .endpoint_request_counts[
            endpoint
        ]
        >= 1
        for endpoint in (
            "GET /health",
            "GET /api/v1/model/info",
            "GET /api/v1/model/metrics",
            "GET /api/v1/llmops/metrics",
            (
                "GET /api/v1/predictions/"
                "{prediction_id}"
            ),
        )
    ),
    (
        "Integration artifact was exported"
    ): system_test_artifact_path.is_file(),
    (
        "Integration artifact checksum is available"
    ): len(
        system_test_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the integration-test summary
# -------------------------------------------------------------------------
print(
    "SYSTEM ENDPOINT AND CONTROLLED ERROR INTEGRATION TESTS"
)
print("-" * 100)
print(
    f"{'HTTP checks':31}: "
    f"{len(system_endpoint_test_records)}"
)
print(
    f"{'Successful responses':31}: "
    f"{sum(record['actual_status'] == 200 for record in system_endpoint_test_records)}"
)
print(
    f"{'Controlled error responses':31}: "
    f"{sum(record['actual_status'] >= 400 for record in system_endpoint_test_records)}"
)
print(
    f"{'Unique request IDs':31}: "
    f"{len(all_request_ids)}"
)
print(
    f"{'Test artifact':31}: "
    f"{system_test_artifact_path}"
)
print(
    f"{'Artifact SHA-256':31}: "
    f"{system_test_checksum[:16]}..."
)
print("-" * 100)

for record in (
    system_endpoint_test_records
):
    print(
        f"{record['method']:6} "
        f"{record['path']:48} | "
        f"{record['actual_status']:3d} | "
        f"{'PASS' if record['passed'] else 'FAIL'}"
    )

print("-" * 100)

for check_name, passed in (
    system_integration_checks.items()
):
    print(
        f"{check_name:65}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    system_integration_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in system_integration_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "System endpoint integration testing failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: SYSTEM ENDPOINTS AND CONTROLLED "
    "ERROR CONTRACTS VERIFIED"
)

SYSTEM ENDPOINT AND CONTROLLED ERROR INTEGRATION TESTS
----------------------------------------------------------------------------------------------------
HTTP checks                    : 6
Successful responses           : 4
Controlled error responses     : 2
Unique request IDs             : 6
Test artifact                  : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/system_endpoint_integration_tests.csv
Artifact SHA-256               : d193fcfb6d6495bc...
----------------------------------------------------------------------------------------------------
GET    /health                                          | 200 | PASS
GET    /api/v1/model/info                               | 200 | PASS
GET    /api/v1/model/metrics                            | 200 | PASS
GET    /api/v1/llmops/metrics                           | 200 | PASS
GET    /api/v1/predictions/not-a-valid-uuid             | 422 | PASS
GET    /api/v1/predictions/{prediction_id}              | 404

<a id="nb07-7-2-image-ingestion-classification-and-explainability-endpoint-tests"></a>
### 7.2 Image Ingestion, Classification, and Explainability Endpoint Tests

This block verifies valid PNG classification, stored-prediction retrieval, image analysis, and controlled rejection of unsupported, malformed, spoofed, and oversized uploads. All successful and error responses are validated through the canonical Pydantic schemas, and endpoint telemetry is confirmed.


In [53]:
# Identify which invalid-upload expectation did not match
image_error_diagnostics = {
    "unsupported_media_type": (
        unsupported_media_http_response,
        415,
    ),
    "invalid_image_content": (
        invalid_image_http_response,
        400,
    ),
    "spoofed_media_type": (
        spoofed_media_http_response,
        400,
    ),
    "oversized_upload": (
        oversized_upload_http_response,
        413,
    ),
}

print(
    "IMAGE ERROR RESPONSE DIAGNOSTICS"
)
print("-" * 100)

for test_name, (
    response,
    expected_status,
) in image_error_diagnostics.items():
    payload = safe_json_response(
        response
    )

    error_code = (
        payload.get(
            "error_code"
        )
        if isinstance(
            payload,
            dict,
        )
        else None
    )

    print(
        f"{test_name:27}: "
        f"expected={expected_status} | "
        f"actual={response.status_code} | "
        f"content_type="
        f"{response.headers.get('content-type')} | "
        f"error_code={error_code}"
    )

    if payload is None:
        print(
            f"{'response_body':27}: "
            f"{response.text[:300]!r}"
        )

IMAGE ERROR RESPONSE DIAGNOSTICS
----------------------------------------------------------------------------------------------------
unsupported_media_type     : expected=415 | actual=415 | content_type=application/json | error_code=UNSUPPORTED_MEDIA_TYPE
invalid_image_content      : expected=400 | actual=400 | content_type=application/json | error_code=INVALID_IMAGE
spoofed_media_type         : expected=400 | actual=415 | content_type=application/json | error_code=UNSUPPORTED_MEDIA_TYPE
oversized_upload           : expected=413 | actual=413 | content_type=application/json | error_code=UPLOAD_TOO_LARGE


In [54]:
# -------------------------------------------------------------------------
# Correct the spoofed-media expectation
# -------------------------------------------------------------------------
parsed_spoofed_media_error = (
    parse_api_error(
        spoofed_media_http_response,
        415,
    )
)

parsed_image_errors = (
    parsed_unsupported_media_error,
    parsed_invalid_image_error,
    parsed_spoofed_media_error,
    parsed_oversized_upload_error,
)

all_image_errors_parsed = all(
    error is not None
    for error in parsed_image_errors
)


# -------------------------------------------------------------------------
# Rebuild the corrected test records
# -------------------------------------------------------------------------
image_endpoint_test_records = [
    {
        "test_name": "valid_png_classification",
        "method": "POST",
        "path": "/api/v1/image/classify",
        "expected_status": 200,
        "actual_status": (
            classification_http_response
            .status_code
        ),
        "passed": (
            parsed_classification_response
            is not None
        ),
    },
    {
        "test_name": "stored_prediction_retrieval",
        "method": "GET",
        "path": (
            "/api/v1/predictions/"
            "{prediction_id}"
        ),
        "expected_status": 200,
        "actual_status": (
            stored_prediction_http_response
            .status_code
        ),
        "passed": (
            parsed_stored_prediction
            is not None
        ),
    },
    {
        "test_name": "valid_png_analysis",
        "method": "POST",
        "path": "/api/v1/image/analyze",
        "expected_status": 200,
        "actual_status": (
            analysis_http_response
            .status_code
        ),
        "passed": (
            parsed_analysis_response
            is not None
        ),
    },
    {
        "test_name": "unsupported_media_type",
        "method": "POST",
        "path": "/api/v1/image/classify",
        "expected_status": 415,
        "actual_status": (
            unsupported_media_http_response
            .status_code
        ),
        "passed": (
            parsed_unsupported_media_error
            is not None
            and parsed_unsupported_media_error
            .error_code
            == "UNSUPPORTED_MEDIA_TYPE"
        ),
    },
    {
        "test_name": "invalid_image_content",
        "method": "POST",
        "path": "/api/v1/image/classify",
        "expected_status": 400,
        "actual_status": (
            invalid_image_http_response
            .status_code
        ),
        "passed": (
            parsed_invalid_image_error
            is not None
            and parsed_invalid_image_error
            .error_code
            == "INVALID_IMAGE"
        ),
    },
    {
        "test_name": "spoofed_media_type",
        "method": "POST",
        "path": "/api/v1/image/classify",
        "expected_status": 415,
        "actual_status": (
            spoofed_media_http_response
            .status_code
        ),
        "passed": (
            parsed_spoofed_media_error
            is not None
            and parsed_spoofed_media_error
            .error_code
            == "UNSUPPORTED_MEDIA_TYPE"
        ),
    },
    {
        "test_name": "oversized_upload",
        "method": "POST",
        "path": "/api/v1/image/classify",
        "expected_status": 413,
        "actual_status": (
            oversized_upload_http_response
            .status_code
        ),
        "passed": (
            parsed_oversized_upload_error
            is not None
            and parsed_oversized_upload_error
            .error_code
            == "UPLOAD_TOO_LARGE"
        ),
    },
]


# -------------------------------------------------------------------------
# Recalculate controlled error identity and sanitization
# -------------------------------------------------------------------------
error_request_ids = {
    str(error.request_id)
    for error in parsed_image_errors
    if error is not None
}

serialized_image_errors = " ".join(
    str(
        safe_json_response(
            response
        )
    )
    for response in (
        unsupported_media_http_response,
        invalid_image_http_response,
        spoofed_media_http_response,
        oversized_upload_http_response,
    )
).lower()

image_errors_contain_internal_details = any(
    forbidden_value
    in serialized_image_errors
    for forbidden_value in (
        "/home/",
        "traceback",
        "stack trace",
        "site-packages",
        ".py:",
    )
)


# -------------------------------------------------------------------------
# Export the corrected integration-test artifact
# -------------------------------------------------------------------------
with image_test_artifact_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as test_file:
    writer = csv.DictWriter(
        test_file,
        fieldnames=[
            "test_name",
            "method",
            "path",
            "expected_status",
            "actual_status",
            "passed",
        ],
    )

    writer.writeheader()
    writer.writerows(
        image_endpoint_test_records
    )

image_test_checksum = hashlib.sha256(
    image_test_artifact_path.read_bytes()
).hexdigest()

post_image_test_metrics = (
    api_operational_metrics_service
    .api_snapshot()
)


# -------------------------------------------------------------------------
# Validate the corrected image endpoint integration
# -------------------------------------------------------------------------
image_integration_checks = {
    (
        "All seven image endpoint checks passed"
    ): all(
        record["passed"]
        for record
        in image_endpoint_test_records
    ),
    (
        "Classification returns fourteen findings"
    ): (
        parsed_classification_response
        is not None
        and len(
            parsed_classification_response
            .findings
        )
        == 14
    ),
    (
        "Crossed names match classification decisions"
    ): (
        parsed_classification_response
        is not None
        and classification_crossed_names
        == parsed_classification_response
        .crossed_finding_names
    ),
    (
        "No-target state matches classification decisions"
    ): (
        parsed_classification_response
        is not None
        and parsed_classification_response
        .no_target_finding
        == (
            len(
                parsed_classification_response
                .crossed_finding_names
            )
            == 0
        )
    ),
    (
        "Stored prediction preserves its identifier"
    ): (
        parsed_stored_prediction
        is not None
        and parsed_stored_prediction
        .prediction_id
        == classified_prediction_id
    ),
    (
        "Stored image hash is preserved"
    ): (
        parsed_stored_prediction
        is not None
        and parsed_classification_response
        is not None
        and parsed_stored_prediction
        .image.sha256
        == parsed_classification_response
        .image.sha256
    ),
    (
        "Analysis returns fourteen findings"
    ): (
        parsed_analysis_response
        is not None
        and len(
            parsed_analysis_response
            .findings
        )
        == 14
    ),
    (
        "Visual evidence matches crossed findings"
    ): (
        parsed_analysis_response
        is not None
        and analysis_evidence_names
        == parsed_analysis_response
        .crossed_finding_names
    ),
    (
        "Successful request identifiers are unique"
    ): len(
        successful_image_request_ids
    ) == 3,
    (
        "All four controlled errors are schema compliant"
    ): all_image_errors_parsed,
    (
        "Controlled error identifiers are unique"
    ): len(
        error_request_ids
    ) == 4,
    (
        "All errors preserve educational boundary"
    ): (
        all_image_errors_parsed
        and all(
            error.educational_use_only
            for error
            in parsed_image_errors
        )
    ),
    (
        "Image errors contain no internal details"
    ): not (
        image_errors_contain_internal_details
    ),
    (
        "Classification endpoint telemetry is available"
    ): (
        post_image_test_metrics
        .endpoint_request_counts[
            "POST /api/v1/image/classify"
        ]
        >= 5
    ),
    (
        "Analysis endpoint telemetry is available"
    ): (
        post_image_test_metrics
        .endpoint_request_counts[
            "POST /api/v1/image/analyze"
        ]
        >= 1
    ),
    (
        "Image endpoint latencies are positive"
    ): (
        post_image_test_metrics
        .endpoint_average_latency_ms[
            "POST /api/v1/image/classify"
        ]
        > 0.0
        and post_image_test_metrics
        .endpoint_average_latency_ms[
            "POST /api/v1/image/analyze"
        ]
        > 0.0
    ),
    (
        "Image test artifact was exported"
    ): image_test_artifact_path.is_file(),
    (
        "Image test checksum is available"
    ): len(
        image_test_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the corrected test summary
# -------------------------------------------------------------------------
print(
    "IMAGE INGESTION, CLASSIFICATION, AND EXPLAINABILITY ENDPOINT TESTS"
)
print("-" * 100)
print(
    f"{'HTTP checks':32}: "
    f"{len(image_endpoint_test_records)}"
)
print(
    f"{'Successful workflow checks':32}: "
    f"{sum(record['actual_status'] == 200 for record in image_endpoint_test_records)}"
)
print(
    f"{'Controlled rejection checks':32}: "
    f"{sum(record['actual_status'] >= 400 for record in image_endpoint_test_records)}"
)
print(
    f"{'Classification findings':32}: "
    f"{len(parsed_classification_response.findings)}"
)
print(
    f"{'Analysis visual evidence':32}: "
    f"{len(parsed_analysis_response.visual_evidence)}"
)
print(
    f"{'Test artifact':32}: "
    f"{image_test_artifact_path}"
)
print(
    f"{'Artifact SHA-256':32}: "
    f"{image_test_checksum[:16]}..."
)
print("-" * 100)

for record in (
    image_endpoint_test_records
):
    print(
        f"{record['method']:6} "
        f"{record['path']:48} | "
        f"{record['actual_status']:3d} | "
        f"{'PASS' if record['passed'] else 'FAIL'}"
    )

print("-" * 100)

for check_name, passed in (
    image_integration_checks.items()
):
    print(
        f"{check_name:66}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    image_integration_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in image_integration_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Image endpoint integration testing failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: IMAGE INGESTION, CLASSIFICATION, AND "
    "EXPLAINABILITY ENDPOINTS VERIFIED"
)

IMAGE INGESTION, CLASSIFICATION, AND EXPLAINABILITY ENDPOINT TESTS
----------------------------------------------------------------------------------------------------
HTTP checks                     : 7
Successful workflow checks      : 3
Controlled rejection checks     : 4
Classification findings         : 14
Analysis visual evidence        : 0
Test artifact                   : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/image_endpoint_integration_tests.csv
Artifact SHA-256                : 22ffe4673c31103c...
----------------------------------------------------------------------------------------------------
POST   /api/v1/image/classify                           | 200 | PASS
GET    /api/v1/predictions/{prediction_id}              | 200 | PASS
POST   /api/v1/image/analyze                            | 200 | PASS
POST   /api/v1/image/classify                           | 415 | PASS
POST   /api/v1/image/classify                           | 400 | PASS
POST   

<a id="nb07-7-3-grounded-language-endpoint-integration-tests"></a>
### 7.3 Grounded Language Endpoint Integration Tests

This block exercises all four language endpoints using a prediction created through the classification API. It verifies server-side grounding, guarded output contracts, task routing, storage upserts, normalized questions, controlled malformed and missing-prediction errors, and language endpoint telemetry.


In [55]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import csv
import hashlib
from pathlib import Path
from uuid import uuid4

from api.schemas import (
    APIErrorResponse,
    LanguageGenerationResponse,
    StoredPredictionResponse,
)


# -------------------------------------------------------------------------
# Define the test artifact
# -------------------------------------------------------------------------
language_test_artifact_path = Path(
    "/home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data/outputs/api/"
    "language_endpoint_integration_tests.csv"
)

language_test_artifact_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Define the four language endpoint requests
# -------------------------------------------------------------------------
language_endpoint_requests = {
    "structured_report": {
        "path": "/api/v1/report/generate",
        "payload": {
            "prediction_id": str(
                classified_prediction_id
            )
        },
    },
    "plain_language_explanation": {
        "path": (
            "/api/v1/explanation/generate"
        ),
        "payload": {
            "prediction_id": str(
                classified_prediction_id
            )
        },
    },
    "grounded_question_answering": {
        "path": "/api/v1/question/answer",
        "payload": {
            "prediction_id": str(
                classified_prediction_id
            ),
            "question": (
                "  What does the supplied model "
                "information indicate?  "
            ),
        },
    },
    "educational_follow_up": {
        "path": (
            "/api/v1/follow-up/recommend"
        ),
        "payload": {
            "prediction_id": str(
                classified_prediction_id
            )
        },
    },
}


# -------------------------------------------------------------------------
# Execute all four language requests
# -------------------------------------------------------------------------
language_http_responses = {}

for task_type, request_details in (
    language_endpoint_requests.items()
):
    language_http_responses[
        task_type
    ] = api_test_client.post(
        request_details["path"],
        json=request_details["payload"],
    )


# -------------------------------------------------------------------------
# Parse successful language responses
# -------------------------------------------------------------------------
parsed_language_responses = {}

for task_type, response in (
    language_http_responses.items()
):
    payload = safe_json_response(
        response
    )

    if (
        response.status_code == 200
        and payload is not None
    ):
        parsed_language_responses[
            task_type
        ] = (
            LanguageGenerationResponse
            .model_validate(
                payload
            )
        )
    else:
        parsed_language_responses[
            task_type
        ] = None


# -------------------------------------------------------------------------
# Retrieve the prediction after language generation
# -------------------------------------------------------------------------
language_stored_http_response = (
    api_test_client.get(
        f"/api/v1/predictions/"
        f"{classified_prediction_id}"
    )
)

language_stored_payload = (
    safe_json_response(
        language_stored_http_response
    )
)

parsed_language_stored_prediction = (
    StoredPredictionResponse.model_validate(
        language_stored_payload
    )
    if (
        language_stored_http_response
        .status_code
        == 200
        and language_stored_payload
        is not None
    )
    else None
)


# -------------------------------------------------------------------------
# Exercise controlled language request errors
# -------------------------------------------------------------------------
malformed_language_http_response = (
    api_test_client.post(
        "/api/v1/report/generate",
        json={
            "prediction_id": (
                "not-a-valid-uuid"
            )
        },
    )
)

missing_language_prediction_id = (
    uuid4()
)

missing_language_http_response = (
    api_test_client.post(
        "/api/v1/report/generate",
        json={
            "prediction_id": str(
                missing_language_prediction_id
            )
        },
    )
)

blank_question_http_response = (
    api_test_client.post(
        "/api/v1/question/answer",
        json={
            "prediction_id": str(
                classified_prediction_id
            ),
            "question": "     ",
        },
    )
)


# -------------------------------------------------------------------------
# Parse controlled errors defensively
# -------------------------------------------------------------------------
def parse_language_error(
    response,
    expected_status,
):
    payload = safe_json_response(
        response
    )

    if (
        response.status_code
        != expected_status
        or payload is None
    ):
        return None

    return APIErrorResponse.model_validate(
        payload
    )


parsed_malformed_language_error = (
    parse_language_error(
        malformed_language_http_response,
        422,
    )
)

parsed_missing_language_error = (
    parse_language_error(
        missing_language_http_response,
        404,
    )
)

parsed_blank_question_error = (
    parse_language_error(
        blank_question_http_response,
        422,
    )
)

parsed_language_errors = (
    parsed_malformed_language_error,
    parsed_missing_language_error,
    parsed_blank_question_error,
)

all_language_successes_parsed = all(
    response is not None
    for response
    in parsed_language_responses.values()
)

all_language_errors_parsed = all(
    error is not None
    for error
    in parsed_language_errors
)


# -------------------------------------------------------------------------
# Derive successful response contract values
# -------------------------------------------------------------------------
expected_language_tasks = set(
    language_endpoint_requests
)

actual_language_tasks = {
    response.task_type
    for response
    in parsed_language_responses.values()
    if response is not None
}

language_request_ids = {
    str(response.request_id)
    for response
    in parsed_language_responses.values()
    if response is not None
}

language_error_request_ids = {
    str(error.request_id)
    for error
    in parsed_language_errors
    if error is not None
}

guardrail_actions_valid = (
    all_language_successes_parsed
    and all(
        response.guardrail_action
        in {
            "accepted_model_generation",
            "safe_template_fallback",
        }
        for response
        in parsed_language_responses.values()
    )
)

guardrail_triggers_valid = (
    all_language_successes_parsed
    and all(
        (
            bool(
                response.trigger_reasons
            )
            if response.guardrail_action
            == "safe_template_fallback"
            else not (
                response.trigger_reasons
            )
        )
        for response
        in parsed_language_responses.values()
    )
)

language_outputs_are_safe = (
    all_language_successes_parsed
    and all(
        (
            "LIMITATIONS"
            in response.output_text
            and educational_limitation
            in response.output_text
        )
        for response
        in parsed_language_responses.values()
    )
)

grounding_comes_from_prediction = (
    all_language_successes_parsed
    and all(
        set(
            response.grounded_finding_names
        ).issubset(
            {
                finding.label_name
                for finding
                in parsed_classification_response
                .findings
            }
        )
        for response
        in parsed_language_responses.values()
    )
)

stored_language_task_names = (
    [
        output.task_type
        for output
        in parsed_language_stored_prediction
        .language_outputs
    ]
    if parsed_language_stored_prediction
    is not None
    else []
)

serialized_language_errors = " ".join(
    str(
        safe_json_response(
            response
        )
    )
    for response in (
        malformed_language_http_response,
        missing_language_http_response,
        blank_question_http_response,
    )
).lower()

language_errors_contain_internal_details = any(
    forbidden_value
    in serialized_language_errors
    for forbidden_value in (
        "/home/",
        "traceback",
        "stack trace",
        "site-packages",
        ".py:",
    )
)


# -------------------------------------------------------------------------
# Build test records
# -------------------------------------------------------------------------
language_endpoint_test_records = []

for task_type, request_details in (
    language_endpoint_requests.items()
):
    response = language_http_responses[
        task_type
    ]

    parsed_response = (
        parsed_language_responses[
            task_type
        ]
    )

    language_endpoint_test_records.append(
        {
            "test_name": task_type,
            "method": "POST",
            "path": request_details[
                "path"
            ],
            "expected_status": 200,
            "actual_status": (
                response.status_code
            ),
            "passed": (
                parsed_response
                is not None
                and parsed_response
                .task_type
                == task_type
            ),
        }
    )

language_endpoint_test_records.extend(
    [
        {
            "test_name": (
                "malformed_language_prediction_id"
            ),
            "method": "POST",
            "path": (
                "/api/v1/report/generate"
            ),
            "expected_status": 422,
            "actual_status": (
                malformed_language_http_response
                .status_code
            ),
            "passed": (
                parsed_malformed_language_error
                is not None
                and parsed_malformed_language_error
                .error_code
                == "REQUEST_VALIDATION_ERROR"
            ),
        },
        {
            "test_name": (
                "missing_language_prediction"
            ),
            "method": "POST",
            "path": (
                "/api/v1/report/generate"
            ),
            "expected_status": 404,
            "actual_status": (
                missing_language_http_response
                .status_code
            ),
            "passed": (
                parsed_missing_language_error
                is not None
                and parsed_missing_language_error
                .error_code
                == "PREDICTION_NOT_FOUND"
            ),
        },
        {
            "test_name": (
                "blank_grounded_question"
            ),
            "method": "POST",
            "path": (
                "/api/v1/question/answer"
            ),
            "expected_status": 422,
            "actual_status": (
                blank_question_http_response
                .status_code
            ),
            "passed": (
                parsed_blank_question_error
                is not None
                and parsed_blank_question_error
                .error_code
                == "REQUEST_VALIDATION_ERROR"
            ),
        },
    ]
)


# -------------------------------------------------------------------------
# Export the language integration-test artifact
# -------------------------------------------------------------------------
with language_test_artifact_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as test_file:
    writer = csv.DictWriter(
        test_file,
        fieldnames=[
            "test_name",
            "method",
            "path",
            "expected_status",
            "actual_status",
            "passed",
        ],
    )

    writer.writeheader()
    writer.writerows(
        language_endpoint_test_records
    )

language_test_checksum = hashlib.sha256(
    language_test_artifact_path.read_bytes()
).hexdigest()

post_language_test_metrics = (
    api_operational_metrics_service
    .api_snapshot()
)


# -------------------------------------------------------------------------
# Validate language endpoint integration
# -------------------------------------------------------------------------
language_integration_checks = {
    (
        "All seven language HTTP checks passed"
    ): all(
        record["passed"]
        for record
        in language_endpoint_test_records
    ),
    (
        "All four successful responses are schema compliant"
    ): all_language_successes_parsed,
    (
        "All four language tasks are preserved"
    ): (
        actual_language_tasks
        == expected_language_tasks
    ),
    (
        "Every generated output is non-empty"
    ): (
        all_language_successes_parsed
        and all(
            bool(
                response.output_text.strip()
            )
            for response
            in parsed_language_responses
            .values()
        )
    ),
    (
        "Every output preserves the safety limitation"
    ): (
        language_outputs_are_safe
    ),
    (
        "Every output uses stored grounding"
    ): (
        grounding_comes_from_prediction
    ),
    (
        "Every guardrail action is controlled"
    ): (
        guardrail_actions_valid
    ),
    (
        "Guardrail triggers match each action"
    ): (
        guardrail_triggers_valid
    ),
    (
        "Question whitespace is normalized"
    ): (
        parsed_language_responses[
            "grounded_question_answering"
        ]
        is not None
        and parsed_language_responses[
            "grounded_question_answering"
        ].question
        == (
            "What does the supplied model "
            "information indicate?"
        )
    ),
    (
        "All four outputs are stored by task"
    ): (
        set(
            stored_language_task_names
        )
        == expected_language_tasks
    ),
    (
        "Stored task names remain unique"
    ): (
        len(
            stored_language_task_names
        )
        == len(
            set(
                stored_language_task_names
            )
        )
    ),
    (
        "Successful request identifiers are unique"
    ): len(
        language_request_ids
    ) == 4,
    (
        "All controlled errors are schema compliant"
    ): (
        all_language_errors_parsed
    ),
    (
        "Controlled error identifiers are unique"
    ): len(
        language_error_request_ids
    ) == 3,
    (
        "Language errors contain no internal details"
    ): not (
        language_errors_contain_internal_details
    ),
    (
        "All language endpoints are measured"
    ): all(
        post_language_test_metrics
        .endpoint_request_counts[
            endpoint
        ]
        >= 1
        for endpoint in (
            "POST /api/v1/report/generate",
            (
                "POST /api/v1/"
                "explanation/generate"
            ),
            "POST /api/v1/question/answer",
            (
                "POST /api/v1/"
                "follow-up/recommend"
            ),
        )
    ),
    (
        "Language endpoint latencies are positive"
    ): all(
        post_language_test_metrics
        .endpoint_average_latency_ms[
            endpoint
        ]
        > 0.0
        for endpoint in (
            "POST /api/v1/report/generate",
            (
                "POST /api/v1/"
                "explanation/generate"
            ),
            "POST /api/v1/question/answer",
            (
                "POST /api/v1/"
                "follow-up/recommend"
            ),
        )
    ),
    (
        "Language test artifact was exported"
    ): (
        language_test_artifact_path
        .is_file()
    ),
    (
        "Language test checksum is available"
    ): len(
        language_test_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the language integration summary
# -------------------------------------------------------------------------
print(
    "GROUNDED LANGUAGE ENDPOINT INTEGRATION TESTS"
)
print("-" * 100)
print(
    f"{'HTTP checks':32}: "
    f"{len(language_endpoint_test_records)}"
)
print(
    f"{'Successful generations':32}: "
    f"{len(parsed_language_responses)}"
)
print(
    f"{'Controlled error checks':32}: "
    f"{len(parsed_language_errors)}"
)
print(
    f"{'Stored language outputs':32}: "
    f"{len(stored_language_task_names)}"
)
print(
    f"{'Test artifact':32}: "
    f"{language_test_artifact_path}"
)
print(
    f"{'Artifact SHA-256':32}: "
    f"{language_test_checksum[:16]}..."
)
print("-" * 100)

for record in (
    language_endpoint_test_records
):
    print(
        f"{record['method']:6} "
        f"{record['path']:48} | "
        f"{record['actual_status']:3d} | "
        f"{'PASS' if record['passed'] else 'FAIL'}"
    )

print("-" * 100)
print(
    "LANGUAGE GUARDRAIL ACTIONS"
)
print("-" * 100)

for task_type, response in (
    parsed_language_responses.items()
):
    if response is None:
        print(
            f"{task_type:29}: "
            "response unavailable"
        )
        continue

    trigger_text = (
        ", ".join(
            response.trigger_reasons
        )
        if response.trigger_reasons
        else "none"
    )

    print(
        f"{task_type:29}: "
        f"{response.guardrail_action:25} | "
        f"{trigger_text}"
    )

print("-" * 100)

for check_name, passed in (
    language_integration_checks.items()
):
    print(
        f"{check_name:67}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    language_integration_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in language_integration_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Language endpoint integration testing failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: GROUNDED LANGUAGE ENDPOINTS, STORAGE, "
    "AND GUARDRAILS VERIFIED"
)

GROUNDED LANGUAGE ENDPOINT INTEGRATION TESTS
----------------------------------------------------------------------------------------------------
HTTP checks                     : 7
Successful generations          : 4
Controlled error checks         : 3
Stored language outputs         : 4
Test artifact                   : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/language_endpoint_integration_tests.csv
Artifact SHA-256                : b9f890f02a6847dc...
----------------------------------------------------------------------------------------------------
POST   /api/v1/report/generate                          | 200 | PASS
POST   /api/v1/explanation/generate                     | 200 | PASS
POST   /api/v1/question/answer                          | 200 | PASS
POST   /api/v1/follow-up/recommend                      | 200 | PASS
POST   /api/v1/report/generate                          | 422 | PASS
POST   /api/v1/report/generate                          | 404 |

<a id="nb07-7-4-complete-analysis-endpoint-integration-tests"></a>
### 7.4 Complete Analysis Endpoint Integration Tests

This block validates the complete end-to-end API workflow with and without an optional grounded question. It verifies shared prediction identity, fourteen finding records, threshold-consistent visual evidence, mandatory and optional language-task composition, guarded output safety, stored-result parity, controlled upload rejection, and complete-workflow telemetry.


In [56]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import csv
import hashlib
from pathlib import Path

from api.schemas import (
    APIErrorResponse,
    CompleteAnalysisResponse,
    StoredPredictionResponse,
)


# -------------------------------------------------------------------------
# Define the complete-workflow test artifact
# -------------------------------------------------------------------------
complete_test_artifact_path = Path(
    "/home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data/outputs/api/"
    "complete_analysis_integration_tests.csv"
)

complete_test_artifact_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Execute complete analysis with grounded QA
# -------------------------------------------------------------------------
complete_question = (
    "What does the supplied model information indicate?"
)

complete_with_question_http_response = (
    api_test_client.post(
        "/api/v1/analyze-complete",
        files={
            "image": (
                "complete-with-question.png",
                valid_png_bytes,
                "image/png",
            )
        },
        data={
            "question": complete_question
        },
    )
)

complete_with_question_payload = (
    safe_json_response(
        complete_with_question_http_response
    )
)

parsed_complete_with_question = (
    CompleteAnalysisResponse.model_validate(
        complete_with_question_payload
    )
    if (
        complete_with_question_http_response
        .status_code
        == 200
        and complete_with_question_payload
        is not None
    )
    else None
)


# -------------------------------------------------------------------------
# Execute complete analysis without grounded QA
# -------------------------------------------------------------------------
complete_without_question_http_response = (
    api_test_client.post(
        "/api/v1/analyze-complete",
        files={
            "image": (
                "complete-without-question.png",
                valid_png_bytes,
                "image/png",
            )
        },
    )
)

complete_without_question_payload = (
    safe_json_response(
        complete_without_question_http_response
    )
)

parsed_complete_without_question = (
    CompleteAnalysisResponse.model_validate(
        complete_without_question_payload
    )
    if (
        complete_without_question_http_response
        .status_code
        == 200
        and complete_without_question_payload
        is not None
    )
    else None
)


# -------------------------------------------------------------------------
# Retrieve the complete prediction containing optional QA
# -------------------------------------------------------------------------
if parsed_complete_with_question is None:
    raise RuntimeError(
        "The complete workflow with a question did not "
        "produce a schema-compliant response."
    )

complete_prediction_id = (
    parsed_complete_with_question
    .prediction_id
)

complete_stored_http_response = (
    api_test_client.get(
        f"/api/v1/predictions/"
        f"{complete_prediction_id}"
    )
)

complete_stored_payload = (
    safe_json_response(
        complete_stored_http_response
    )
)

parsed_complete_stored_prediction = (
    StoredPredictionResponse.model_validate(
        complete_stored_payload
    )
    if (
        complete_stored_http_response
        .status_code
        == 200
        and complete_stored_payload
        is not None
    )
    else None
)


# -------------------------------------------------------------------------
# Exercise controlled rejection on the complete endpoint
# -------------------------------------------------------------------------
complete_invalid_media_http_response = (
    api_test_client.post(
        "/api/v1/analyze-complete",
        files={
            "image": (
                "complete-invalid.txt",
                b"not-an-image",
                "text/plain",
            )
        },
    )
)

complete_invalid_media_payload = (
    safe_json_response(
        complete_invalid_media_http_response
    )
)

parsed_complete_invalid_media_error = (
    APIErrorResponse.model_validate(
        complete_invalid_media_payload
    )
    if (
        complete_invalid_media_http_response
        .status_code
        == 415
        and complete_invalid_media_payload
        is not None
    )
    else None
)


# -------------------------------------------------------------------------
# Derive complete-workflow relationships safely
# -------------------------------------------------------------------------
mandatory_complete_tasks = [
    "structured_report",
    "plain_language_explanation",
    "educational_follow_up",
]

expected_tasks_with_question = [
    *mandatory_complete_tasks,
    "grounded_question_answering",
]

expected_tasks_without_question = [
    *mandatory_complete_tasks,
]

tasks_with_question = (
    [
        output.task_type
        for output
        in parsed_complete_with_question
        .language_outputs
    ]
    if parsed_complete_with_question
    is not None
    else []
)

tasks_without_question = (
    [
        output.task_type
        for output
        in parsed_complete_without_question
        .language_outputs
    ]
    if parsed_complete_without_question
    is not None
    else []
)

stored_complete_tasks = (
    [
        output.task_type
        for output
        in parsed_complete_stored_prediction
        .language_outputs
    ]
    if parsed_complete_stored_prediction
    is not None
    else []
)

complete_evidence_names = (
    [
        evidence.finding_name
        for evidence
        in parsed_complete_with_question
        .visual_evidence
    ]
    if parsed_complete_with_question
    is not None
    else []
)

complete_outputs_non_empty = (
    parsed_complete_with_question
    is not None
    and parsed_complete_without_question
    is not None
    and all(
        bool(
            output.output_text.strip()
        )
        for response in (
            parsed_complete_with_question,
            parsed_complete_without_question,
        )
        for output
        in response.language_outputs
    )
)

complete_outputs_preserve_safety = (
    parsed_complete_with_question
    is not None
    and parsed_complete_without_question
    is not None
    and all(
        (
            "LIMITATIONS"
            in output.output_text
            and educational_limitation
            in output.output_text
        )
        for response in (
            parsed_complete_with_question,
            parsed_complete_without_question,
        )
        for output
        in response.language_outputs
    )
)

complete_guardrail_contract_valid = (
    parsed_complete_with_question
    is not None
    and parsed_complete_without_question
    is not None
    and all(
        (
            output.guardrail_action
            in {
                "accepted_model_generation",
                "safe_template_fallback",
            }
            and (
                bool(
                    output.trigger_reasons
                )
                if output.guardrail_action
                == "safe_template_fallback"
                else not (
                    output.trigger_reasons
                )
            )
        )
        for response in (
            parsed_complete_with_question,
            parsed_complete_without_question,
        )
        for output
        in response.language_outputs
    )
)

complete_request_ids = {
    str(response.request_id)
    for response in (
        parsed_complete_with_question,
        parsed_complete_without_question,
        parsed_complete_stored_prediction,
        parsed_complete_invalid_media_error,
    )
    if response is not None
}

serialized_complete_error = str(
    complete_invalid_media_payload
).lower()

complete_error_contains_internal_details = any(
    forbidden_value
    in serialized_complete_error
    for forbidden_value in (
        "/home/",
        "traceback",
        "stack trace",
        "site-packages",
        ".py:",
    )
)


# -------------------------------------------------------------------------
# Build complete-workflow test records
# -------------------------------------------------------------------------
complete_endpoint_test_records = [
    {
        "test_name": (
            "complete_analysis_with_question"
        ),
        "method": "POST",
        "path": (
            "/api/v1/analyze-complete"
        ),
        "expected_status": 200,
        "actual_status": (
            complete_with_question_http_response
            .status_code
        ),
        "passed": (
            parsed_complete_with_question
            is not None
        ),
    },
    {
        "test_name": (
            "complete_analysis_without_question"
        ),
        "method": "POST",
        "path": (
            "/api/v1/analyze-complete"
        ),
        "expected_status": 200,
        "actual_status": (
            complete_without_question_http_response
            .status_code
        ),
        "passed": (
            parsed_complete_without_question
            is not None
        ),
    },
    {
        "test_name": (
            "complete_prediction_retrieval"
        ),
        "method": "GET",
        "path": (
            "/api/v1/predictions/"
            "{prediction_id}"
        ),
        "expected_status": 200,
        "actual_status": (
            complete_stored_http_response
            .status_code
        ),
        "passed": (
            parsed_complete_stored_prediction
            is not None
        ),
    },
    {
        "test_name": (
            "complete_invalid_media_type"
        ),
        "method": "POST",
        "path": (
            "/api/v1/analyze-complete"
        ),
        "expected_status": 415,
        "actual_status": (
            complete_invalid_media_http_response
            .status_code
        ),
        "passed": (
            parsed_complete_invalid_media_error
            is not None
            and parsed_complete_invalid_media_error
            .error_code
            == "UNSUPPORTED_MEDIA_TYPE"
        ),
    },
]


# -------------------------------------------------------------------------
# Export the complete-workflow test artifact
# -------------------------------------------------------------------------
with complete_test_artifact_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as test_file:
    writer = csv.DictWriter(
        test_file,
        fieldnames=[
            "test_name",
            "method",
            "path",
            "expected_status",
            "actual_status",
            "passed",
        ],
    )

    writer.writeheader()
    writer.writerows(
        complete_endpoint_test_records
    )

complete_test_checksum = hashlib.sha256(
    complete_test_artifact_path.read_bytes()
).hexdigest()

post_complete_test_metrics = (
    api_operational_metrics_service
    .api_snapshot()
)


# -------------------------------------------------------------------------
# Validate complete endpoint integration
# -------------------------------------------------------------------------
complete_integration_checks = {
    (
        "All four complete-workflow HTTP checks passed"
    ): all(
        record["passed"]
        for record
        in complete_endpoint_test_records
    ),
    (
        "Question workflow returns fourteen findings"
    ): (
        parsed_complete_with_question
        is not None
        and len(
            parsed_complete_with_question
            .findings
        )
        == 14
    ),
    (
        "No-question workflow returns fourteen findings"
    ): (
        parsed_complete_without_question
        is not None
        and len(
            parsed_complete_without_question
            .findings
        )
        == 14
    ),
    (
        "Question workflow preserves four tasks"
    ): (
        tasks_with_question
        == expected_tasks_with_question
    ),
    (
        "No-question workflow preserves three tasks"
    ): (
        tasks_without_question
        == expected_tasks_without_question
    ),
    (
        "Stored task order matches complete response"
    ): (
        stored_complete_tasks
        == tasks_with_question
    ),
    (
        "Language task names remain unique"
    ): (
        len(
            tasks_with_question
        )
        == len(
            set(
                tasks_with_question
            )
        )
        and len(
            tasks_without_question
        )
        == len(
            set(
                tasks_without_question
            )
        )
    ),
    (
        "Grounded question is preserved"
    ): (
        parsed_complete_with_question
        is not None
        and next(
            output.question
            for output
            in parsed_complete_with_question
            .language_outputs
            if output.task_type
            == "grounded_question_answering"
        )
        == complete_question
    ),
    (
        "No-question workflow excludes grounded QA"
    ): (
        "grounded_question_answering"
        not in tasks_without_question
    ),
    (
        "Visual evidence matches crossed findings"
    ): (
        parsed_complete_with_question
        is not None
        and complete_evidence_names
        == parsed_complete_with_question
        .crossed_finding_names
    ),
    (
        "Every complete language output is non-empty"
    ): (
        complete_outputs_non_empty
    ),
    (
        "Every complete output preserves safety limitation"
    ): (
        complete_outputs_preserve_safety
    ),
    (
        "Every complete guardrail action is controlled"
    ): (
        complete_guardrail_contract_valid
    ),
    (
        "Stored prediction preserves the workflow ID"
    ): (
        parsed_complete_stored_prediction
        is not None
        and parsed_complete_stored_prediction
        .prediction_id
        == complete_prediction_id
    ),
    (
        "Complete error is schema compliant"
    ): (
        parsed_complete_invalid_media_error
        is not None
    ),
    (
        "Complete error preserves educational boundary"
    ): (
        parsed_complete_invalid_media_error
        is not None
        and parsed_complete_invalid_media_error
        .educational_use_only
    ),
    (
        "Complete error contains no internal details"
    ): not (
        complete_error_contains_internal_details
    ),
    (
        "All complete request identifiers are unique"
    ): len(
        complete_request_ids
    ) == 4,
    (
        "Complete endpoint telemetry is available"
    ): (
        post_complete_test_metrics
        .endpoint_request_counts[
            "POST /api/v1/analyze-complete"
        ]
        >= 3
    ),
    (
        "Complete endpoint latency is positive"
    ): (
        post_complete_test_metrics
        .endpoint_average_latency_ms[
            "POST /api/v1/analyze-complete"
        ]
        > 0.0
    ),
    (
        "Complete test artifact was exported"
    ): (
        complete_test_artifact_path
        .is_file()
    ),
    (
        "Complete test checksum is available"
    ): len(
        complete_test_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the complete endpoint summary
# -------------------------------------------------------------------------
print(
    "COMPLETE ANALYSIS ENDPOINT INTEGRATION TESTS"
)
print("-" * 100)
print(
    f"{'HTTP checks':33}: "
    f"{len(complete_endpoint_test_records)}"
)
print(
    f"{'With-question language tasks':33}: "
    f"{len(tasks_with_question)}"
)
print(
    f"{'Without-question language tasks':33}: "
    f"{len(tasks_without_question)}"
)
print(
    f"{'Visual evidence records':33}: "
    f"{len(complete_evidence_names)}"
)
print(
    f"{'Test artifact':33}: "
    f"{complete_test_artifact_path}"
)
print(
    f"{'Artifact SHA-256':33}: "
    f"{complete_test_checksum[:16]}..."
)
print("-" * 100)

for record in (
    complete_endpoint_test_records
):
    print(
        f"{record['method']:6} "
        f"{record['path']:48} | "
        f"{record['actual_status']:3d} | "
        f"{'PASS' if record['passed'] else 'FAIL'}"
    )

print("-" * 100)
print(
    "WITH-QUESTION GUARDRAIL ACTIONS"
)
print("-" * 100)

if parsed_complete_with_question is not None:
    for output in (
        parsed_complete_with_question
        .language_outputs
    ):
        trigger_text = (
            ", ".join(
                output.trigger_reasons
            )
            if output.trigger_reasons
            else "none"
        )

        print(
            f"{output.task_type:29}: "
            f"{output.guardrail_action:25} | "
            f"{trigger_text}"
        )

print("-" * 100)

for check_name, passed in (
    complete_integration_checks.items()
):
    print(
        f"{check_name:68}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    complete_integration_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in complete_integration_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Complete endpoint integration testing failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: COMPLETE ANALYSIS ENDPOINT, STORED OUTPUTS, "
    "AND GUARDED WORKFLOW VERIFIED"
)

COMPLETE ANALYSIS ENDPOINT INTEGRATION TESTS
----------------------------------------------------------------------------------------------------
HTTP checks                      : 4
With-question language tasks     : 4
Without-question language tasks  : 3
Visual evidence records          : 0
Test artifact                    : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/complete_analysis_integration_tests.csv
Artifact SHA-256                 : 8276af22f456068c...
----------------------------------------------------------------------------------------------------
POST   /api/v1/analyze-complete                         | 200 | PASS
POST   /api/v1/analyze-complete                         | 200 | PASS
GET    /api/v1/predictions/{prediction_id}              | 200 | PASS
POST   /api/v1/analyze-complete                         | 415 | PASS
----------------------------------------------------------------------------------------------------
WITH-QUESTION GUARDRAIL A

In [57]:
print(
    "SERVICE CONSTRUCTOR CONTRACTS"
)
print("-" * 100)

for service_name, service_details in (
    service_interface_registry[
        "services"
    ].items()
):
    print(
        f"{service_name:36}: "
        f"{service_details['constructor']}"
    )

print("-" * 100)
print(
    "STATUS: CONSTRUCTOR CONTRACTS AVAILABLE "
    "FOR STANDALONE SERVICE INITIALIZATION"
)

SERVICE CONSTRUCTOR CONTRACTS
----------------------------------------------------------------------------------------------------
ImageValidationService              : (settings: api.core.config.ServiceSettings | None = None) -> None
ComputerVisionService               : (settings: api.core.config.ServiceSettings | None = None, device: str | None = None) -> None
PredictionStoreService              : (*, maximum_records: int = 1000, retention_hours: int = 24) -> None
GradCAMService                      : (*, model: 'torch.nn.Module', device: 'torch.device | str', limitation: 'str', image_size: 'int' = 224, overlay_alpha: 'float' = 0.4) -> 'None'
GroundedInputSerializer             : (*, language_model_version: 'str', limitation_boundary: 'str', optional_value_marker: 'str', task_prefixes: 'Mapping[str, str] | None' = None) -> 'None'
GroundedLanguageModelService        : (*, model_directory: 'str | Path', model_version: 'str', device: 'str | torch.device | None' = None, use_bfloat16: 'b

**[↑ Back to notebook index](#notebook-index)**


<a id="nb07-8-deployment-ready-service-initialization"></a>
## 8. Deployment-Ready Service Initialization

<a id="nb07-8-1-standalone-service-factory-and-cold-start-validation"></a>
### 8.1 Standalone Service Factory and Cold-Start Validation

This block creates the default service factory used when FastAPI starts in a fresh Python process. It reconstructs every validated service and workflow from versioned artifacts, configures the dependency container once, and updates the application to initialize the container automatically. The cold-start test deliberately clears the notebook container and rebuilds both frozen models to verify that the API can start independently of notebook state.


In [58]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import base64
import hashlib
import importlib
import io
import json
import sys
import time
from pathlib import Path

import torch
from fastapi.testclient import TestClient


# -------------------------------------------------------------------------
# Define factory and template paths
# -------------------------------------------------------------------------
service_factory_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "api/core/factory.py"
)

explainability_template_path = Path(
    "/home/jovyan/chest-xray-ai-assistant/"
    "configs/explainability_response_templates.json"
)

service_factory_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

explainability_template_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Export validated explainability schema templates
# -------------------------------------------------------------------------
explainability_response_templates = {
    "template_version": (
        "explainability-response-templates-v1"
    ),
    "contract": (
        explainability_contract_example
        .model_dump(
            mode="json"
        )
    ),
    "evidence": (
        gradcam_evidence_example
        .model_dump(
            mode="json"
        )
    ),
}

explainability_template_path.write_text(
    json.dumps(
        explainability_response_templates,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Write the standalone service factory
# -------------------------------------------------------------------------
service_factory_source = '''
"""Standalone construction of all FastAPI runtime services."""

from __future__ import annotations

import json
import threading
from pathlib import Path
from typing import Any

import torch
import yaml

from api.core.config import (
    get_settings,
)
from api.core.dependencies import (
    ServiceContainer,
    configure_service_container,
    get_service_container,
    service_container_is_configured,
)
from api.core.runtime import (
    build_success_metadata,
)
from api.core.telemetry import (
    EndpointTelemetryAdapter,
)
from api.schemas import (
    ExplainabilityContract,
    GradCAMEvidence,
)
from src.services.complete_workflow_service import (
    CompleteAnalysisWorkflow,
)
from src.services.computer_vision_service import (
    ComputerVisionService,
)
from src.services.gradcam_service import (
    GradCAMService,
)
from src.services.grounding_service import (
    GroundedInputSerializer,
    TASK_PREFIXES,
)
from src.services.image_service import (
    ImageValidationService,
)
from src.services.image_workflow_service import (
    ImageAnalysisWorkflow,
)
from src.services.language_guardrail_service import (
    DeterministicLanguageGuardrail,
)
from src.services.language_model_service import (
    GroundedLanguageModelService,
)
from src.services.language_workflow_service import (
    StoredPredictionLanguageWorkflow,
)
from src.services.operational_metrics_service import (
    OperationalMetricsService,
)
from src.services.prediction_store_service import (
    PredictionStoreService,
)


REGISTERED_ENDPOINTS = (
    "GET /health",
    "GET /api/v1/model/info",
    "GET /api/v1/model/metrics",
    "POST /api/v1/image/classify",
    "POST /api/v1/image/analyze",
    "POST /api/v1/report/generate",
    "POST /api/v1/explanation/generate",
    "POST /api/v1/question/answer",
    "POST /api/v1/follow-up/recommend",
    "POST /api/v1/analyze-complete",
    "GET /api/v1/predictions/{prediction_id}",
    "GET /api/v1/llmops/metrics",
)


_factory_lock = threading.RLock()


def _load_yaml(
    path: Path,
) -> dict[str, Any]:
    with path.open(
        "r",
        encoding="utf-8",
    ) as yaml_file:
        loaded_value = yaml.safe_load(
            yaml_file
        )

    if not isinstance(
        loaded_value,
        dict,
    ):
        raise ValueError(
            "A required YAML configuration is invalid."
        )

    return loaded_value


def _load_json(
    path: Path,
) -> dict[str, Any]:
    with path.open(
        "r",
        encoding="utf-8",
    ) as json_file:
        loaded_value = json.load(
            json_file
        )

    if not isinstance(
        loaded_value,
        dict,
    ):
        raise ValueError(
            "A required JSON configuration is invalid."
        )

    return loaded_value


def _find_named_text(
    value: Any,
    required_terms: tuple[str, ...],
    path: tuple[str, ...] = (),
) -> str | None:
    """Resolve controlled text across nested configuration versions."""

    if isinstance(value, dict):
        for key, nested_value in value.items():
            normalized_key = "".join(
                character.lower()
                for character in str(key)
                if character.isalnum()
            )

            current_path = (
                *path,
                normalized_key,
            )

            path_text = "".join(
                current_path
            )

            if (
                all(
                    term in path_text
                    for term in required_terms
                )
                and isinstance(
                    nested_value,
                    str,
                )
                and nested_value.strip()
            ):
                return nested_value.strip()

            located_value = _find_named_text(
                nested_value,
                required_terms,
                current_path,
            )

            if located_value is not None:
                return located_value

    elif isinstance(value, list):
        for item in value:
            located_value = _find_named_text(
                item,
                required_terms,
                path,
            )

            if located_value is not None:
                return located_value

    elif (
        isinstance(value, str)
        and value.strip()
        and all(
            term in "".join(path)
            for term in required_terms
        )
    ):
        return value.strip()

    return None


def _resolve_path(
    settings: Any,
    *,
    attribute_names: tuple[str, ...],
    fallback: Path,
) -> Path:
    for attribute_name in attribute_names:
        value = getattr(
            settings,
            attribute_name,
            None,
        )

        if value is not None:
            return Path(
                value
            )

    return fallback


def build_default_service_container(
    *,
    replace: bool = False,
) -> ServiceContainer:
    """Construct every service from frozen versioned artifacts."""

    with _factory_lock:
        if (
            service_container_is_configured()
            and not replace
        ):
            return get_service_container()

        settings = get_settings()

        solution_root = Path(
            settings.solution_root
        )

        data_root = Path(
            settings.data_root
        )

        finding_contract_path = (
            solution_root
            / "configs"
            / "finding_contract.yaml"
        )

        prompt_registry_path = (
            solution_root
            / "configs"
            / "prompt_registry.yaml"
        )

        explainability_template_path = (
            solution_root
            / "configs"
            / "explainability_response_templates.json"
        )

        language_model_directory = (
            _resolve_path(
                settings,
                attribute_names=(
                    "language_model_directory",
                    "language_model_path",
                ),
                fallback=(
                    data_root
                    / "models"
                    / settings.language_model_version
                ),
            )
        )

        finding_contract = _load_yaml(
            finding_contract_path
        )

        prompt_registry = _load_yaml(
            prompt_registry_path
        )

        explainability_templates = (
            _load_json(
                explainability_template_path
            )
        )

        educational_limitation = (
            _find_named_text(
                finding_contract,
                (
                    "educational",
                    "limitation",
                ),
            )
        )

        gradcam_limitation = (
            _find_named_text(
                finding_contract,
                (
                    "gradcam",
                    "limitation",
                ),
            )
            or _find_named_text(
                finding_contract,
                (
                    "gradcam",
                    "boundary",
                ),
            )
        )

        professional_review_guidance = (
            _find_named_text(
                finding_contract,
                (
                    "professional",
                    "review",
                ),
            )
        )

        optional_value_marker = (
            _find_named_text(
                prompt_registry,
                (
                    "optional",
                    "marker",
                ),
            )
            or "not_applicable"
        )

        if educational_limitation is None:
            raise KeyError(
                "The educational limitation is unavailable."
            )

        if gradcam_limitation is None:
            raise KeyError(
                "The Grad-CAM limitation is unavailable."
            )

        if professional_review_guidance is None:
            raise KeyError(
                "Professional-review guidance is unavailable."
            )

        image_validation_service = (
            ImageValidationService(
                settings=settings
            )
        )

        computer_vision_service = (
            ComputerVisionService(
                settings=settings,
                device=None,
            )
        )

        prediction_store_service = (
            PredictionStoreService(
                maximum_records=1000,
                retention_hours=24,
            )
        )

        gradcam_service = GradCAMService(
            model=(
                computer_vision_service.model
            ),
            device=(
                computer_vision_service.device
            ),
            limitation=(
                gradcam_limitation
            ),
            image_size=224,
            overlay_alpha=0.40,
        )

        grounding_serializer = (
            GroundedInputSerializer(
                language_model_version=(
                    settings
                    .language_model_version
                ),
                limitation_boundary=(
                    educational_limitation
                ),
                optional_value_marker=(
                    optional_value_marker
                ),
                task_prefixes=(
                    TASK_PREFIXES
                ),
            )
        )

        language_model_service = (
            GroundedLanguageModelService(
                model_directory=(
                    language_model_directory
                ),
                model_version=(
                    settings
                    .language_model_version
                ),
                device=(
                    "cuda"
                    if torch.cuda.is_available()
                    else "cpu"
                ),
                use_bfloat16=True,
            )
        )

        language_guardrail_service = (
            DeterministicLanguageGuardrail(
                educational_limitation=(
                    educational_limitation
                ),
                gradcam_limitation=(
                    gradcam_limitation
                ),
                professional_review_guidance=(
                    professional_review_guidance
                ),
            )
        )

        base_operational_metrics = (
            OperationalMetricsService(
                registered_endpoints=(
                    REGISTERED_ENDPOINTS
                ),
                maximum_latency_samples=10_000,
            )
        )

        operational_metrics_service = (
            EndpointTelemetryAdapter(
                base_operational_metrics
            )
        )

        explainability_contract = (
            ExplainabilityContract
            .model_validate(
                explainability_templates[
                    "contract"
                ]
            )
        )

        evidence_template = (
            GradCAMEvidence.model_validate(
                explainability_templates[
                    "evidence"
                ]
            )
        )

        image_analysis_workflow = (
            ImageAnalysisWorkflow(
                computer_vision_service=(
                    computer_vision_service
                ),
                gradcam_service=(
                    gradcam_service
                ),
                prediction_store=(
                    prediction_store_service
                ),
                operational_metrics=(
                    operational_metrics_service
                ),
                metadata_builder=(
                    build_success_metadata
                ),
                explainability_contract=(
                    explainability_contract
                ),
                evidence_template=(
                    evidence_template
                ),
            )
        )

        language_workflow = (
            StoredPredictionLanguageWorkflow(
                prediction_store=(
                    prediction_store_service
                ),
                grounding_serializer=(
                    grounding_serializer
                ),
                language_model_service=(
                    language_model_service
                ),
                language_guardrail=(
                    language_guardrail_service
                ),
                operational_metrics=(
                    operational_metrics_service
                ),
                metadata_builder=(
                    build_success_metadata
                ),
            )
        )

        complete_analysis_workflow = (
            CompleteAnalysisWorkflow(
                image_workflow=(
                    image_analysis_workflow
                ),
                language_workflow=(
                    language_workflow
                ),
                prediction_store=(
                    prediction_store_service
                ),
                operational_metrics=(
                    operational_metrics_service
                ),
                metadata_builder=(
                    build_success_metadata
                ),
            )
        )

        container = ServiceContainer(
            image_validation_service=(
                image_validation_service
            ),
            computer_vision_service=(
                computer_vision_service
            ),
            prediction_store_service=(
                prediction_store_service
            ),
            gradcam_service=(
                gradcam_service
            ),
            grounding_serializer=(
                grounding_serializer
            ),
            language_model_service=(
                language_model_service
            ),
            language_guardrail_service=(
                language_guardrail_service
            ),
            operational_metrics_service=(
                operational_metrics_service
            ),
            image_analysis_workflow=(
                image_analysis_workflow
            ),
            language_workflow=(
                language_workflow
            ),
            complete_analysis_workflow=(
                complete_analysis_workflow
            ),
        )

        return configure_service_container(
            container,
            replace=replace,
        )


def ensure_default_service_container() -> ServiceContainer:
    """Return the configured container or construct it once."""

    if service_container_is_configured():
        return get_service_container()

    return build_default_service_container()
'''

service_factory_path.write_text(
    service_factory_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Update api/main.py to initialize services in a fresh process
# -------------------------------------------------------------------------
main_source = main_module_path.read_text(
    encoding="utf-8"
)

factory_import_block = '''from api.core.factory import (
    ensure_default_service_container,
)
'''

factory_initialization_block = '''
ensure_default_service_container()
'''

if factory_import_block not in main_source:
    insertion_marker = '''from api.core.dependencies import (
    get_service_container,
)
'''

    if insertion_marker not in main_source:
        raise RuntimeError(
            "The dependency import marker was not found "
            "in api/main.py."
        )

    main_source = main_source.replace(
        insertion_marker,
        insertion_marker
        + factory_import_block,
        1,
    )

if factory_initialization_block not in main_source:
    settings_marker = '''settings = get_settings()
'''

    if settings_marker not in main_source:
        raise RuntimeError(
            "The settings initialization marker was not "
            "found in api/main.py."
        )

    main_source = main_source.replace(
        settings_marker,
        settings_marker
        + factory_initialization_block,
        1,
    )

main_module_path.write_text(
    main_source,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Import the generated factory
# -------------------------------------------------------------------------
importlib.invalidate_caches()

module_name = (
    "api.core.factory"
)

if module_name in sys.modules:
    service_factory_module = (
        importlib.reload(
            sys.modules[module_name]
        )
    )
else:
    service_factory_module = (
        importlib.import_module(
            module_name
        )
    )

build_default_service_container = (
    service_factory_module
    .build_default_service_container
)

ensure_default_service_container = (
    service_factory_module
    .ensure_default_service_container
)


# -------------------------------------------------------------------------
# Perform a genuine cold-start reconstruction
# -------------------------------------------------------------------------
clear_service_container()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

cold_start_started_at = time.perf_counter()

standalone_service_container = (
    build_default_service_container(
        replace=False
    )
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

standalone_startup_latency_ms = (
    time.perf_counter()
    - cold_start_started_at
) * 1000.0

standalone_peak_gpu_gib = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 3)
    if torch.cuda.is_available()
    else 0.0
)

standalone_readiness = (
    standalone_service_container
    .readiness()
)


# -------------------------------------------------------------------------
# Update notebook references to the standalone services
# -------------------------------------------------------------------------
api_service_container = (
    standalone_service_container
)

image_validation_service = (
    standalone_service_container
    .image_validation_service
)

computer_vision_service = (
    standalone_service_container
    .computer_vision_service
)

prediction_store_service = (
    standalone_service_container
    .prediction_store_service
)

gradcam_service = (
    standalone_service_container
    .gradcam_service
)

grounded_input_serializer = (
    standalone_service_container
    .grounding_serializer
)

grounded_language_model_service = (
    standalone_service_container
    .language_model_service
)

language_guardrail_service = (
    standalone_service_container
    .language_guardrail_service
)

api_operational_metrics_service = (
    standalone_service_container
    .operational_metrics_service
)

operational_metrics_service = (
    api_operational_metrics_service
    .base_metrics_service
)

image_analysis_workflow = (
    standalone_service_container
    .image_analysis_workflow
)

stored_prediction_language_workflow = (
    standalone_service_container
    .language_workflow
)

complete_analysis_workflow = (
    standalone_service_container
    .complete_analysis_workflow
)


# -------------------------------------------------------------------------
# Reload the application and verify automatic initialization
# -------------------------------------------------------------------------
if "api.main" in sys.modules:
    del sys.modules[
        "api.main"
    ]

main_module = importlib.import_module(
    "api.main"
)

app = main_module.app

api_test_client = TestClient(
    app,
    raise_server_exceptions=False,
)

standalone_health_response = (
    api_test_client.get(
        "/health"
    )
)

standalone_health_payload = (
    standalone_health_response.json()
    if (
        "application/json"
        in standalone_health_response
        .headers.get(
            "content-type",
            "",
        ).lower()
    )
    else None
)


# -------------------------------------------------------------------------
# Calculate artifact checksums
# -------------------------------------------------------------------------
factory_checksum = hashlib.sha256(
    service_factory_path.read_bytes()
).hexdigest()

explainability_template_checksum = (
    hashlib.sha256(
        explainability_template_path
        .read_bytes()
    ).hexdigest()
)

standalone_main_checksum = hashlib.sha256(
    main_module_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Validate standalone initialization
# -------------------------------------------------------------------------
standalone_factory_checks = {
    (
        "Standalone factory module was written"
    ): service_factory_path.is_file(),
    (
        "Explainability templates were exported"
    ): explainability_template_path.is_file(),
    (
        "All eleven runtime components were reconstructed"
    ): len(
        standalone_readiness
    ) == 11,
    (
        "Every reconstructed component is ready"
    ): all(
        standalone_readiness.values()
    ),
    (
        "Computer-vision model is on the GPU"
    ): (
        next(
            computer_vision_service
            .model.parameters()
        ).device.type
        == "cuda"
    ),
    (
        "Computer-vision model remains in evaluation mode"
    ): not (
        computer_vision_service
        .model.training
    ),
    (
        "Language model is on the GPU"
    ): (
        next(
            grounded_language_model_service
            .model.parameters()
        ).device.type
        == "cuda"
    ),
    (
        "Language model remains in evaluation mode"
    ): not (
        grounded_language_model_service
        .model.training
    ),
    (
        "Application automatically resolves the container"
    ): (
        service_container_is_configured()
        and get_service_container()
        is standalone_service_container
    ),
    (
        "Standalone health endpoint returns success"
    ): (
        standalone_health_response
        .status_code
        == 200
        and isinstance(
            standalone_health_payload,
            dict,
        )
        and standalone_health_payload.get(
            "status"
        )
        == "success"
    ),
    (
        "Standalone startup latency is positive"
    ): (
        standalone_startup_latency_ms
        > 0.0
    ),
    (
        "Factory checksum is available"
    ): len(
        factory_checksum
    ) == 64,
    (
        "Explainability template checksum is available"
    ): len(
        explainability_template_checksum
    ) == 64,
    (
        "Updated main checksum is available"
    ): len(
        standalone_main_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the standalone startup summary
# -------------------------------------------------------------------------
print(
    "STANDALONE FASTAPI SERVICE FACTORY AND COLD START"
)
print("-" * 100)
print(
    f"{'Factory module':33}: "
    f"{service_factory_path}"
)
print(
    f"{'Factory SHA-256':33}: "
    f"{factory_checksum[:16]}..."
)
print(
    f"{'Explainability templates':33}: "
    f"{explainability_template_path}"
)
print(
    f"{'Template SHA-256':33}: "
    f"{explainability_template_checksum[:16]}..."
)
print(
    f"{'Runtime components':33}: "
    f"{len(standalone_readiness)}"
)
print(
    f"{'Cold-start latency':33}: "
    f"{standalone_startup_latency_ms:.2f} ms"
)
print(
    f"{'Peak cold-start GPU memory':33}: "
    f"{standalone_peak_gpu_gib:.2f} GiB"
)
print(
    f"{'Standalone health status':33}: "
    f"{standalone_health_response.status_code}"
)
print("-" * 100)

for component_name, ready in (
    standalone_readiness.items()
):
    print(
        f"{component_name:33}: "
        f"{'READY' if ready else 'NOT READY'}"
    )

print("-" * 100)

for check_name, passed in (
    standalone_factory_checks.items()
):
    print(
        f"{check_name:67}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    standalone_factory_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in standalone_factory_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Standalone service initialization failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: FASTAPI SERVICE CAN INITIALIZE "
    "INDEPENDENTLY FROM THE NOTEBOOK"
)

STANDALONE FASTAPI SERVICE FACTORY AND COLD START
----------------------------------------------------------------------------------------------------
Factory module                   : /home/jovyan/chest-xray-ai-assistant/api/core/factory.py
Factory SHA-256                  : bf3597059b000d2d...
Explainability templates         : /home/jovyan/chest-xray-ai-assistant/configs/explainability_response_templates.json
Template SHA-256                 : c371f69a261df6bc...
Runtime components               : 11
Cold-start latency               : 12725.41 ms
Peak cold-start GPU memory       : 0.40 GiB
Standalone health status         : 200
----------------------------------------------------------------------------------------------------
image_validation                 : READY
computer_vision                  : READY
prediction_store                 : READY
gradcam                          : READY
grounding_serializer             : READY
grounded_language                : READY
language_guar

<a id="nb07-8-2-independent-uvicorn-process-smoke-test"></a>
### 8.2 Independent Uvicorn Process Smoke Test

This block launches the FastAPI application through Uvicorn in a separate Python process, proving that startup does not depend on notebook variables. It waits for model initialization, calls the live health, model-information, and OpenAPI endpoints over localhost, captures server logs, and then shuts the process down cleanly.


In [59]:
# -------------------------------------------------------------------------
# Import dependencies
# -------------------------------------------------------------------------
import hashlib
import json
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

import requests


# -------------------------------------------------------------------------
# Define runtime paths
# -------------------------------------------------------------------------
solution_root = Path(
    "/home/jovyan/chest-xray-ai-assistant"
)

uvicorn_log_path = Path(
    "/home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data/outputs/api/"
    "uvicorn_standalone_smoke_test.log"
)

uvicorn_log_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Reserve an available localhost port
# -------------------------------------------------------------------------
with socket.socket(
    socket.AF_INET,
    socket.SOCK_STREAM,
) as port_socket:
    port_socket.bind(
        (
            "127.0.0.1",
            0,
        )
    )

    uvicorn_port = port_socket.getsockname()[1]

uvicorn_base_url = (
    f"http://127.0.0.1:"
    f"{uvicorn_port}"
)


# -------------------------------------------------------------------------
# Prepare the independent process environment
# -------------------------------------------------------------------------
uvicorn_environment = os.environ.copy()

existing_python_path = (
    uvicorn_environment.get(
        "PYTHONPATH",
        "",
    )
)

uvicorn_environment[
    "PYTHONPATH"
] = (
    str(solution_root)
    if not existing_python_path
    else (
        str(solution_root)
        + os.pathsep
        + existing_python_path
    )
)


# -------------------------------------------------------------------------
# Launch Uvicorn independently from the notebook runtime
# -------------------------------------------------------------------------
uvicorn_command = [
    sys.executable,
    "-m",
    "uvicorn",
    "api.main:app",
    "--host",
    "127.0.0.1",
    "--port",
    str(
        uvicorn_port
    ),
    "--workers",
    "1",
    "--log-level",
    "info",
]

startup_started_at = time.perf_counter()

uvicorn_process = subprocess.Popen(
    uvicorn_command,
    cwd=solution_root,
    env=uvicorn_environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
)


# -------------------------------------------------------------------------
# Poll health until cold-start initialization completes
# -------------------------------------------------------------------------
standalone_health_payload = None
standalone_health_status = None
startup_error = None

startup_deadline = (
    time.monotonic()
    + 60.0
)

while time.monotonic() < startup_deadline:
    if uvicorn_process.poll() is not None:
        startup_error = (
            "Uvicorn exited before becoming ready."
        )
        break

    try:
        health_response = requests.get(
            f"{uvicorn_base_url}/health",
            timeout=2.0,
        )

        standalone_health_status = (
            health_response.status_code
        )

        if (
            health_response.status_code
            == 200
        ):
            standalone_health_payload = (
                health_response.json()
            )
            break
    except (
        requests.ConnectionError,
        requests.Timeout,
    ):
        pass

    time.sleep(
        0.5
    )

standalone_uvicorn_startup_ms = (
    time.perf_counter()
    - startup_started_at
) * 1000.0


# -------------------------------------------------------------------------
# Call additional live endpoints after readiness
# -------------------------------------------------------------------------
standalone_model_info_status = None
standalone_model_info_payload = None
standalone_openapi_status = None
standalone_openapi_payload = None

if standalone_health_payload is not None:
    model_info_response = requests.get(
        (
            f"{uvicorn_base_url}"
            "/api/v1/model/info"
        ),
        timeout=10.0,
    )

    standalone_model_info_status = (
        model_info_response.status_code
    )

    if (
        model_info_response.status_code
        == 200
    ):
        standalone_model_info_payload = (
            model_info_response.json()
        )

    openapi_response = requests.get(
        (
            f"{uvicorn_base_url}"
            "/openapi.json"
        ),
        timeout=10.0,
    )

    standalone_openapi_status = (
        openapi_response.status_code
    )

    if (
        openapi_response.status_code
        == 200
    ):
        standalone_openapi_payload = (
            openapi_response.json()
        )


# -------------------------------------------------------------------------
# Shut down the independent process and capture logs
# -------------------------------------------------------------------------
if uvicorn_process.poll() is None:
    uvicorn_process.terminate()

try:
    uvicorn_output, _ = (
        uvicorn_process.communicate(
            timeout=20.0
        )
    )
except subprocess.TimeoutExpired:
    uvicorn_process.kill()

    uvicorn_output, _ = (
        uvicorn_process.communicate(
            timeout=10.0
        )
    )

uvicorn_log_path.write_text(
    uvicorn_output,
    encoding="utf-8",
)

uvicorn_exit_code = (
    uvicorn_process.returncode
)

uvicorn_log_checksum = hashlib.sha256(
    uvicorn_log_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Derive live-process validation values
# -------------------------------------------------------------------------
standalone_openapi_paths = (
    set(
        standalone_openapi_payload[
            "paths"
        ]
    )
    if isinstance(
        standalone_openapi_payload,
        dict,
    )
    and isinstance(
        standalone_openapi_payload.get(
            "paths"
        ),
        dict,
    )
    else set()
)

expected_live_openapi_paths = {
    "/health",
    "/api/v1/model/info",
    "/api/v1/model/metrics",
    "/api/v1/image/classify",
    "/api/v1/image/analyze",
    "/api/v1/report/generate",
    "/api/v1/explanation/generate",
    "/api/v1/question/answer",
    "/api/v1/follow-up/recommend",
    "/api/v1/analyze-complete",
    "/api/v1/predictions/{prediction_id}",
    "/api/v1/llmops/metrics",
}

uvicorn_log_lower = (
    uvicorn_output.lower()
)

unexpected_server_error_present = any(
    forbidden_log_value
    in uvicorn_log_lower
    for forbidden_log_value in (
        "traceback",
        "application startup failed",
        "error loading asgi app",
    )
)


# -------------------------------------------------------------------------
# Validate the independent Uvicorn process
# -------------------------------------------------------------------------
uvicorn_smoke_checks = {
    (
        "Independent Uvicorn process started"
    ): (
        startup_error is None
        and standalone_health_payload
        is not None
    ),
    (
        "Live health endpoint returned HTTP 200"
    ): (
        standalone_health_status
        == 200
    ),
    (
        "Live health response reports success"
    ): (
        isinstance(
            standalone_health_payload,
            dict,
        )
        and standalone_health_payload.get(
            "status"
        )
        == "success"
    ),
    (
        "Live model-info endpoint returned HTTP 200"
    ): (
        standalone_model_info_status
        == 200
    ),
    (
        "Live model information reports success"
    ): (
        isinstance(
            standalone_model_info_payload,
            dict,
        )
        and standalone_model_info_payload.get(
            "status"
        )
        == "success"
    ),
    (
        "Live OpenAPI endpoint returned HTTP 200"
    ): (
        standalone_openapi_status
        == 200
    ),
    (
        "Live OpenAPI contains all twelve paths"
    ): (
        standalone_openapi_paths
        == expected_live_openapi_paths
    ),
    (
        "Independent process stopped cleanly"
    ): (
        uvicorn_exit_code
        in {
            0,
            -15,
        }
    ),
    (
        "Server log contains no startup traceback"
    ): not (
        unexpected_server_error_present
    ),
    (
        "Cold-start runtime is positive"
    ): (
        standalone_uvicorn_startup_ms
        > 0.0
    ),
    (
        "Uvicorn log was exported"
    ): uvicorn_log_path.is_file(),
    (
        "Uvicorn log checksum is available"
    ): len(
        uvicorn_log_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the independent-process summary
# -------------------------------------------------------------------------
print(
    "INDEPENDENT UVICORN PROCESS SMOKE TEST"
)
print("-" * 100)
print(
    f"{'Python executable':32}: "
    f"{sys.executable}"
)
print(
    f"{'Application target':32}: "
    f"api.main:app"
)
print(
    f"{'Local port':32}: "
    f"{uvicorn_port}"
)
print(
    f"{'Process ID':32}: "
    f"{uvicorn_process.pid}"
)
print(
    f"{'Cold-start duration':32}: "
    f"{standalone_uvicorn_startup_ms:.2f} ms"
)
print(
    f"{'Health status':32}: "
    f"{standalone_health_status}"
)
print(
    f"{'Model-info status':32}: "
    f"{standalone_model_info_status}"
)
print(
    f"{'OpenAPI status':32}: "
    f"{standalone_openapi_status}"
)
print(
    f"{'OpenAPI paths':32}: "
    f"{len(standalone_openapi_paths)}"
)
print(
    f"{'Process exit code':32}: "
    f"{uvicorn_exit_code}"
)
print(
    f"{'Server log':32}: "
    f"{uvicorn_log_path}"
)
print(
    f"{'Log SHA-256':32}: "
    f"{uvicorn_log_checksum[:16]}..."
)
print("-" * 100)

for check_name, passed in (
    uvicorn_smoke_checks.items()
):
    print(
        f"{check_name:66}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    uvicorn_smoke_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in uvicorn_smoke_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Independent Uvicorn smoke testing failed: "
        + ", ".join(failed_checks)
    )

print("-" * 100)
print(
    "STATUS: FASTAPI APPLICATION STARTS AND RESPONDS "
    "IN AN INDEPENDENT PROCESS"
)

INDEPENDENT UVICORN PROCESS SMOKE TEST
----------------------------------------------------------------------------------------------------
Python executable               : /opt/conda/bin/python
Application target              : api.main:app
Local port                      : 34851
Process ID                      : 8891
Cold-start duration             : 21529.50 ms
Health status                   : 200
Model-info status               : 200
OpenAPI status                  : 200
OpenAPI paths                   : 12
Process exit code               : -15
Server log                      : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/uvicorn_standalone_smoke_test.log
Log SHA-256                     : 547b9b35d4ea9ef5...
----------------------------------------------------------------------------------------------------
Independent Uvicorn process started                               : PASS
Live health endpoint returned HTTP 200                            : PASS
L

**[↑ Back to notebook index](#notebook-index)**


<a id="nb07-9-automated-api-test-suite"></a>
## 9. Automated API Test Suite

<a id="nb07-9-1-reusable-pytest-integration-suite"></a>
### 9.1 Reusable Pytest Integration Suite

This block converts the validated notebook checks into a reusable Pytest suite under `tests/api`. The suite starts the application through its standalone factory and verifies system endpoints, image ingestion, prediction retrieval, guarded language generation, complete analysis, validation errors, and controlled service errors using deterministic synthetic image content.


In [63]:
# =========================================================================
# 9.1 REUSABLE PYTEST INTEGRATION SUITE
# =========================================================================

from __future__ import annotations

import hashlib
import os
import re
import subprocess
import sys
from pathlib import Path


# -------------------------------------------------------------------------
# Resolve standalone paths
# -------------------------------------------------------------------------
solution_root = Path(
    "/home/jovyan/chest-xray-ai-assistant"
).resolve()

api_test_root = (
    solution_root
    / "tests"
    / "api"
)

api_output_root = Path(
    "/home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data/"
    "outputs/api"
).resolve()

pytest_log_path = (
    api_output_root
    / "pytest_api_integration.log"
)

required_test_files = (
    api_test_root / "test_system_endpoints.py",
    api_test_root / "test_image_endpoints.py",
    api_test_root / "test_language_endpoints.py",
    api_test_root / "test_complete_endpoint.py",
)

api_output_root.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Validate that the reusable test modules created by Section 9.1 exist
# -------------------------------------------------------------------------
missing_test_files = [
    str(test_file)
    for test_file in required_test_files
    if not test_file.is_file()
]

if missing_test_files:
    raise FileNotFoundError(
        "The reusable API test modules are missing: "
        + ", ".join(missing_test_files)
    )


# -------------------------------------------------------------------------
# Build a controlled environment for the independent Pytest process
# -------------------------------------------------------------------------
pytest_environment = os.environ.copy()

existing_python_path = (
    pytest_environment.get(
        "PYTHONPATH",
        "",
    )
)

python_path_entries = [
    str(solution_root)
]

if existing_python_path:
    python_path_entries.append(
        existing_python_path
    )

pytest_environment["PYTHONPATH"] = (
    os.pathsep.join(
        python_path_entries
    )
)


# -------------------------------------------------------------------------
# Execute the complete API suite in an independent Python process
# -------------------------------------------------------------------------
pytest_command = [
    sys.executable,
    "-m",
    "pytest",
    str(api_test_root),
    "-q",
    "--disable-warnings",
    "--maxfail=1",
]

pytest_process = subprocess.run(
    pytest_command,
    cwd=str(solution_root),
    env=pytest_environment,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)

pytest_stdout = (
    pytest_process.stdout
    or ""
)

pytest_stderr = (
    pytest_process.stderr
    or ""
)

pytest_output_parts = [
    pytest_stdout.rstrip()
]

if pytest_stderr.strip():
    pytest_output_parts.extend(
        [
            "",
            "STANDARD ERROR",
            pytest_stderr.rstrip(),
        ]
    )

pytest_complete_output = (
    "\n".join(
        pytest_output_parts
    ).strip()
    + "\n"
)


# -------------------------------------------------------------------------
# Persist the complete independent-process output
# -------------------------------------------------------------------------
pytest_log_path.write_text(
    pytest_complete_output,
    encoding="utf-8",
)

pytest_log_checksum = hashlib.sha256(
    pytest_log_path.read_bytes()
).hexdigest()


# -------------------------------------------------------------------------
# Extract reporting values without using them as fragile pass criteria
# -------------------------------------------------------------------------
passed_match = re.search(
    r"(?P<count>\d+)\s+passed\b",
    pytest_complete_output,
    flags=re.IGNORECASE,
)

failed_match = re.search(
    r"(?P<count>\d+)\s+failed\b",
    pytest_complete_output,
    flags=re.IGNORECASE,
)

error_match = re.search(
    r"(?P<count>\d+)\s+errors?\b",
    pytest_complete_output,
    flags=re.IGNORECASE,
)

warning_match = re.search(
    r"(?P<count>\d+)\s+warnings?\b",
    pytest_complete_output,
    flags=re.IGNORECASE,
)

duration_match = re.search(
    r"\bin\s+(?P<seconds>[0-9]+(?:\.[0-9]+)?)s\b",
    pytest_complete_output,
    flags=re.IGNORECASE,
)

passed_test_count = (
    int(
        passed_match.group(
            "count"
        )
    )
    if passed_match
    else 0
)

failed_test_count = (
    int(
        failed_match.group(
            "count"
        )
    )
    if failed_match
    else 0
)

error_test_count = (
    int(
        error_match.group(
            "count"
        )
    )
    if error_match
    else 0
)

warning_count = (
    int(
        warning_match.group(
            "count"
        )
    )
    if warning_match
    else 0
)

pytest_duration_seconds = (
    float(
        duration_match.group(
            "seconds"
        )
    )
    if duration_match
    else 0.0
)


# -------------------------------------------------------------------------
# Use Pytest's exit code as the authoritative execution contract
# -------------------------------------------------------------------------
pytest_checks = {
    (
        "API test directory is available"
    ): api_test_root.is_dir(),
    (
        "Four reusable test modules are available"
    ): all(
        test_file.is_file()
        for test_file
        in required_test_files
    ),
    (
        "Pytest completed successfully"
    ): pytest_process.returncode == 0,
    (
        "At least one test case was executed"
    ): passed_test_count > 0,
    (
        "No failed test is reported"
    ): failed_test_count == 0,
    (
        "No collection or execution error is reported"
    ): error_test_count == 0,
    (
        "Independent execution duration is positive"
    ): pytest_duration_seconds > 0.0,
    (
        "Pytest execution log was exported"
    ): pytest_log_path.is_file(),
    (
        "Pytest execution log is non-empty"
    ): pytest_log_path.stat().st_size > 0,
    (
        "Pytest log checksum is available"
    ): len(pytest_log_checksum) == 64,
}


# -------------------------------------------------------------------------
# Present the original Pytest output
# -------------------------------------------------------------------------
print(
    pytest_complete_output.rstrip()
)

print()
print(
    "REUSABLE PYTEST API INTEGRATION SUITE"
)
print("-" * 100)
print(
    f"Python executable          : "
    f"{sys.executable}"
)
print(
    f"Test root                  : "
    f"{api_test_root}"
)
print(
    f"Reusable test modules      : "
    f"{len(required_test_files)}"
)
print(
    f"Passed tests               : "
    f"{passed_test_count}"
)
print(
    f"Failed tests               : "
    f"{failed_test_count}"
)
print(
    f"Execution errors           : "
    f"{error_test_count}"
)
print(
    f"Warnings                   : "
    f"{warning_count}"
)
print(
    f"Pytest exit code           : "
    f"{pytest_process.returncode}"
)
print(
    f"Execution duration         : "
    f"{pytest_duration_seconds:.2f} seconds"
)
print(
    f"Test artifact              : "
    f"{pytest_log_path}"
)
print(
    f"Artifact SHA-256           : "
    f"{pytest_log_checksum[:16]}..."
)
print("-" * 100)

for check_name, passed in pytest_checks.items():
    print(
        f"{check_name:<67}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


# -------------------------------------------------------------------------
# Enforce the reusable-test readiness gate
# -------------------------------------------------------------------------
if not all(
    pytest_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in pytest_checks.items()
        if not passed
    ]

    print()
    print(
        "PYTEST FAILURE OUTPUT"
    )
    print("-" * 100)
    print(
        pytest_complete_output.rstrip()
    )

    raise RuntimeError(
        "Reusable API test automation failed: "
        + ", ".join(
            failed_checks
        )
    )

print("-" * 100)
print(
    "STATUS: REUSABLE API TEST SUITE PASSED "
    "IN AN INDEPENDENT PROCESS"
)

.......................                                                  [100%]
23 passed, 1 warning in 17.46s

REUSABLE PYTEST API INTEGRATION SUITE
----------------------------------------------------------------------------------------------------
Python executable          : /opt/conda/bin/python
Test root                  : /home/jovyan/chest-xray-ai-assistant/tests/api
Reusable test modules      : 4
Passed tests               : 23
Failed tests               : 0
Execution errors           : 0
Warnings                   : 1
Pytest exit code           : 0
Execution duration         : 17.46 seconds
Test artifact              : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/pytest_api_integration.log
Artifact SHA-256           : c8ec957a5ca23279...
----------------------------------------------------------------------------------------------------
API test directory is available                                    : PASS
Four reusable test modules are availabl

**[↑ Back to notebook index](#notebook-index)**


<a id="nb07-10-api-artifact-registration-and-final-readiness"></a>
## 10. API Artifact Registration and Final Readiness

<a id="nb07-10-1-versioned-api-artifact-registry"></a>
### 10.1 Versioned API Artifact Registry

The FastAPI implementation now contains application modules, reusable services, Pydantic schemas, endpoint contracts, OpenAPI documentation, integration-test evidence, and independent-process validation logs. This section inventories those artifacts with their relative paths, sizes, and SHA-256 checksums. The resulting registry provides a reproducible record of the exact API assets prepared for packaging and interface integration.


In [64]:
# =========================================================================
# 10.1 VERSIONED API ARTIFACT REGISTRY
# =========================================================================

from __future__ import annotations

import hashlib
import json
import re
from datetime import datetime, timezone
from pathlib import Path


# -------------------------------------------------------------------------
# Resolve standalone project paths
# -------------------------------------------------------------------------
solution_root = Path(
    "/home/jovyan/chest-xray-ai-assistant"
).resolve()

data_root = Path(
    "/home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data"
).resolve()

api_root = (
    solution_root
    / "api"
)

service_root = (
    solution_root
    / "src"
    / "services"
)

api_test_root = (
    solution_root
    / "tests"
    / "api"
)

configuration_root = (
    solution_root
    / "configs"
)

api_output_root = (
    data_root
    / "outputs"
    / "api"
)

api_output_root.mkdir(
    parents=True,
    exist_ok=True,
)

artifact_registry_path = (
    api_output_root
    / "api_artifact_registry.json"
)


# -------------------------------------------------------------------------
# Register the frozen API lineage
# -------------------------------------------------------------------------
api_lineage = {
    "api_version": "v1",
    "api_contract_version": "api-contract-v1",
    "computer_vision_model_version": (
        "resnet18-chestmnist-v1"
    ),
    "language_model_version": (
        "flan-t5-small-chestmnist-v1"
    ),
    "prompt_registry_version": (
        "grounded-language-prompts-v1"
    ),
    "finding_contract_version": (
        "chestmnist-finding-contract-v1"
    ),
    "educational_use_only": True,
}


# -------------------------------------------------------------------------
# Define the mandatory source and contract artifacts
# -------------------------------------------------------------------------
required_source_files = (
    api_root / "main.py",
    api_root / "core" / "config.py",
    api_root / "core" / "errors.py",
    api_root / "core" / "runtime.py",
    api_root / "core" / "dependencies.py",
    api_root / "core" / "telemetry.py",
    api_root / "core" / "factory.py",
    api_root / "routes" / "workflows.py",
    api_root / "routes" / "aggregate.py",
    api_root / "routes" / "system.py",
    api_root / "schemas" / "common.py",
    api_root / "schemas" / "requests.py",
    api_root / "schemas" / "prediction.py",
    api_root / "schemas" / "explainability.py",
    api_root / "schemas" / "language.py",
    api_root / "schemas" / "system.py",
    api_root / "schemas" / "aggregate.py",
    service_root / "image_service.py",
    service_root / "computer_vision_service.py",
    service_root / "prediction_store_service.py",
    service_root / "gradcam_service.py",
    service_root / "grounding_service.py",
    service_root / "language_model_service.py",
    service_root / "language_guardrail_service.py",
    service_root / "operational_metrics_service.py",
    service_root / "image_workflow_service.py",
    service_root / "language_workflow_service.py",
    service_root / "complete_workflow_service.py",
)

required_contract_files = (
    configuration_root / "api_contract.yaml",
    configuration_root / "finding_contract.yaml",
    configuration_root / "prompt_registry.yaml",
    configuration_root
    / "system_response_templates.json",
    configuration_root
    / "explainability_response_templates.json",
)

required_test_files = (
    api_test_root / "conftest.py",
    api_test_root / "test_system_endpoints.py",
    api_test_root / "test_image_endpoints.py",
    api_test_root / "test_language_endpoints.py",
    api_test_root / "test_complete_endpoint.py",
)

required_evidence_files = (
    api_output_root / "openapi.json",
    api_output_root
    / "system_endpoint_integration_tests.csv",
    api_output_root
    / "image_endpoint_integration_tests.csv",
    api_output_root
    / "language_endpoint_integration_tests.csv",
    api_output_root
    / "complete_analysis_integration_tests.csv",
    api_output_root
    / "pytest_api_integration.log",
    api_output_root
    / "uvicorn_standalone_smoke_test.log",
)


# -------------------------------------------------------------------------
# Validate all mandatory files before creating the registry
# -------------------------------------------------------------------------
required_artifact_groups = {
    "source": required_source_files,
    "contract": required_contract_files,
    "test": required_test_files,
    "evidence": required_evidence_files,
}

missing_artifacts = []

for artifact_group, artifact_paths in (
    required_artifact_groups.items()
):
    for artifact_path in artifact_paths:
        if not artifact_path.is_file():
            missing_artifacts.append(
                {
                    "group": artifact_group,
                    "path": str(
                        artifact_path
                    ),
                }
            )

if missing_artifacts:
    missing_description = "; ".join(
        (
            f"{record['group']}: "
            f"{record['path']}"
        )
        for record in missing_artifacts
    )

    raise FileNotFoundError(
        "Mandatory API artifacts are missing: "
        + missing_description
    )


# -------------------------------------------------------------------------
# Create checksum-backed artifact records
# -------------------------------------------------------------------------
def calculate_sha256(
    file_path: Path,
) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for data_block in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(
                data_block
            )

    return digest.hexdigest()


def relative_artifact_path(
    file_path: Path,
) -> str:
    try:
        return str(
            file_path.relative_to(
                solution_root
            )
        )
    except ValueError:
        return str(
            file_path.relative_to(
                data_root
            )
        )


artifact_records = []

for artifact_group, artifact_paths in (
    required_artifact_groups.items()
):
    for artifact_path in artifact_paths:
        artifact_records.append(
            {
                "artifact_group": (
                    artifact_group
                ),
                "relative_path": (
                    relative_artifact_path(
                        artifact_path
                    )
                ),
                "size_bytes": (
                    artifact_path.stat()
                    .st_size
                ),
                "sha256": (
                    calculate_sha256(
                        artifact_path
                    )
                ),
            }
        )

artifact_records = sorted(
    artifact_records,
    key=lambda record: (
        record["artifact_group"],
        record["relative_path"],
    ),
)


# -------------------------------------------------------------------------
# Read independent test and OpenAPI evidence
# -------------------------------------------------------------------------
openapi_path = (
    api_output_root
    / "openapi.json"
)

with openapi_path.open(
    "r",
    encoding="utf-8",
) as openapi_file:
    openapi_document = json.load(
        openapi_file
    )

openapi_paths = (
    openapi_document.get(
        "paths",
        {},
    )
)

registered_operations = []

for endpoint_path, path_contract in (
    openapi_paths.items()
):
    for http_method in (
        "get",
        "post",
        "put",
        "patch",
        "delete",
    ):
        if http_method in path_contract:
            registered_operations.append(
                {
                    "method": (
                        http_method.upper()
                    ),
                    "path": endpoint_path,
                    "operation_id": (
                        path_contract[
                            http_method
                        ].get(
                            "operationId"
                        )
                    ),
                }
            )

registered_operations = sorted(
    registered_operations,
    key=lambda record: (
        record["path"],
        record["method"],
    ),
)

pytest_log_path = (
    api_output_root
    / "pytest_api_integration.log"
)

pytest_log_text = (
    pytest_log_path.read_text(
        encoding="utf-8",
        errors="replace",
    )
)

pytest_passed_match = re.search(
    r"(\d+)\s+passed\b",
    pytest_log_text,
    flags=re.IGNORECASE,
)

pytest_failed_match = re.search(
    r"(\d+)\s+failed\b",
    pytest_log_text,
    flags=re.IGNORECASE,
)

pytest_passed_tests = (
    int(
        pytest_passed_match.group(1)
    )
    if pytest_passed_match
    else 0
)

pytest_failed_tests = (
    int(
        pytest_failed_match.group(1)
    )
    if pytest_failed_match
    else 0
)


# -------------------------------------------------------------------------
# Export the versioned API artifact registry
# -------------------------------------------------------------------------
api_artifact_registry = {
    "registry_version": (
        "fastapi-artifact-registry-v1"
    ),
    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "lineage": api_lineage,
    "summary": {
        "registered_artifact_files": (
            len(
                artifact_records
            )
        ),
        "source_files": (
            len(
                required_source_files
            )
        ),
        "contract_files": (
            len(
                required_contract_files
            )
        ),
        "test_files": (
            len(
                required_test_files
            )
        ),
        "evidence_files": (
            len(
                required_evidence_files
            )
        ),
        "openapi_paths": (
            len(
                openapi_paths
            )
        ),
        "openapi_operations": (
            len(
                registered_operations
            )
        ),
        "pytest_passed_tests": (
            pytest_passed_tests
        ),
        "pytest_failed_tests": (
            pytest_failed_tests
        ),
    },
    "openapi_operations": (
        registered_operations
    ),
    "artifacts": artifact_records,
}

artifact_registry_path.write_text(
    json.dumps(
        api_artifact_registry,
        indent=2,
        sort_keys=False,
    )
    + "\n",
    encoding="utf-8",
)

artifact_registry_checksum = (
    calculate_sha256(
        artifact_registry_path
    )
)


# -------------------------------------------------------------------------
# Reload the registry to verify serialization
# -------------------------------------------------------------------------
with artifact_registry_path.open(
    "r",
    encoding="utf-8",
) as registry_file:
    reloaded_registry = json.load(
        registry_file
    )


# -------------------------------------------------------------------------
# Validate registry completeness
# -------------------------------------------------------------------------
registry_checks = {
    (
        "All mandatory API artifacts are available"
    ): len(missing_artifacts) == 0,
    (
        "Every artifact has a SHA-256 checksum"
    ): all(
        len(record["sha256"]) == 64
        for record in artifact_records
    ),
    (
        "Every registered artifact is non-empty"
    ): all(
        record["size_bytes"] > 0
        for record in artifact_records
    ),
    (
        "Twelve OpenAPI paths are preserved"
    ): len(openapi_paths) == 12,
    (
        "Twelve endpoint operations are preserved"
    ): len(registered_operations) == 12,
    (
        "Independent Pytest evidence is successful"
    ): (
        pytest_passed_tests > 0
        and pytest_failed_tests == 0
    ),
    (
        "API model lineage is complete"
    ): all(
        api_lineage.get(
            lineage_field
        )
        for lineage_field in (
            "api_version",
            "api_contract_version",
            "computer_vision_model_version",
            "language_model_version",
            "prompt_registry_version",
            "finding_contract_version",
        )
    ),
    (
        "Educational-use boundary remains enabled"
    ): (
        api_lineage[
            "educational_use_only"
        ]
        is True
    ),
    (
        "Artifact registry was exported"
    ): artifact_registry_path.is_file(),
    (
        "Reloaded registry preserves its version"
    ): (
        reloaded_registry[
            "registry_version"
        ]
        == "fastapi-artifact-registry-v1"
    ),
    (
        "Artifact registry checksum is available"
    ): len(
        artifact_registry_checksum
    ) == 64,
}


# -------------------------------------------------------------------------
# Present the registry summary
# -------------------------------------------------------------------------
print(
    "VERSIONED FASTAPI ARTIFACT REGISTRY"
)
print("-" * 100)
print(
    f"Registry version          : "
    f"{api_artifact_registry['registry_version']}"
)
print(
    f"Registry path             : "
    f"{artifact_registry_path}"
)
print(
    f"Registry SHA-256          : "
    f"{artifact_registry_checksum[:16]}..."
)
print(
    f"Registered artifacts      : "
    f"{len(artifact_records)}"
)
print(
    f"Source modules            : "
    f"{len(required_source_files)}"
)
print(
    f"Contract files            : "
    f"{len(required_contract_files)}"
)
print(
    f"Reusable test files       : "
    f"{len(required_test_files)}"
)
print(
    f"Test evidence files       : "
    f"{len(required_evidence_files)}"
)
print(
    f"OpenAPI paths             : "
    f"{len(openapi_paths)}"
)
print(
    f"OpenAPI operations        : "
    f"{len(registered_operations)}"
)
print(
    f"Independent tests passed  : "
    f"{pytest_passed_tests}"
)
print(
    f"Independent tests failed  : "
    f"{pytest_failed_tests}"
)
print("-" * 100)

for artifact_group in (
    "source",
    "contract",
    "test",
    "evidence",
):
    group_count = sum(
        record["artifact_group"]
        == artifact_group
        for record in artifact_records
    )

    print(
        f"{artifact_group:<12}: "
        f"{group_count:>2} files"
    )

print("-" * 100)

for check_name, passed in registry_checks.items():
    print(
        f"{check_name:<67}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    registry_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in registry_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "API artifact registry validation failed: "
        + ", ".join(
            failed_checks
        )
    )

print("-" * 100)
print(
    "STATUS: VERSIONED FASTAPI ARTIFACT "
    "REGISTRY READY"
)

VERSIONED FASTAPI ARTIFACT REGISTRY
----------------------------------------------------------------------------------------------------
Registry version          : fastapi-artifact-registry-v1
Registry path             : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/api_artifact_registry.json
Registry SHA-256          : 41084152f7669853...
Registered artifacts      : 45
Source modules            : 28
Contract files            : 5
Reusable test files       : 5
Test evidence files       : 7
OpenAPI paths             : 12
OpenAPI operations        : 12
Independent tests passed  : 23
Independent tests failed  : 0
----------------------------------------------------------------------------------------------------
source      : 28 files
contract    :  5 files
test        :  5 files
evidence    :  7 files
----------------------------------------------------------------------------------------------------
All mandatory API artifacts are available                    

<a id="nb07-10-2-mlflow-api-integration-registration"></a>
### 10.2 MLflow API Integration Registration

The finalized FastAPI contract, OpenAPI specification, integration-test evidence, standalone-service logs, and artifact registry are registered as a dedicated MLflow run. This links the deployed service boundary to the frozen computer-vision model, grounded-language model, prompt registry, safety guardrail, and reproducible API test results without duplicating model-weight artifacts.


In [65]:
# =========================================================================
# 10.2 MLFLOW API INTEGRATION REGISTRATION
# =========================================================================

from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient


# -------------------------------------------------------------------------
# Resolve standalone paths and tracking configuration
# -------------------------------------------------------------------------
solution_root = Path(
    "/home/jovyan/chest-xray-ai-assistant"
).resolve()

data_root = Path(
    "/home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data"
).resolve()

api_output_root = (
    data_root
    / "outputs"
    / "api"
)

artifact_registry_path = (
    api_output_root
    / "api_artifact_registry.json"
)

mlflow_registration_path = (
    api_output_root
    / "api_mlflow_registration.json"
)

mlflow_tracking_uri = (
    f"file://{data_root / 'mlflow'}"
)

mlflow_experiment_name = (
    "chestmnist-fastapi-service-integration"
)

mlflow_run_name = (
    "fastapi-service-integration-v1"
)


# -------------------------------------------------------------------------
# Load the validated API artifact registry
# -------------------------------------------------------------------------
if not artifact_registry_path.is_file():
    raise FileNotFoundError(
        "The API artifact registry is unavailable: "
        f"{artifact_registry_path}"
    )

with artifact_registry_path.open(
    "r",
    encoding="utf-8",
) as registry_file:
    api_artifact_registry = json.load(
        registry_file
    )

registry_summary = (
    api_artifact_registry[
        "summary"
    ]
)

api_lineage = (
    api_artifact_registry[
        "lineage"
    ]
)

registered_artifacts = (
    api_artifact_registry[
        "artifacts"
    ]
)


# -------------------------------------------------------------------------
# Resolve registered artifact paths safely
# -------------------------------------------------------------------------
def resolve_registered_artifact(
    relative_path: str,
) -> Path:
    solution_candidate = (
        solution_root
        / relative_path
    )

    data_candidate = (
        data_root
        / relative_path
    )

    if solution_candidate.is_file():
        return solution_candidate

    if data_candidate.is_file():
        return data_candidate

    raise FileNotFoundError(
        "Registered artifact is unavailable: "
        f"{relative_path}"
    )


resolved_artifacts = [
    {
        **artifact_record,
        "absolute_path": (
            resolve_registered_artifact(
                artifact_record[
                    "relative_path"
                ]
            )
        ),
    }
    for artifact_record
    in registered_artifacts
]


# -------------------------------------------------------------------------
# Configure MLflow while preserving the existing notebook tracking URI
# -------------------------------------------------------------------------
previous_tracking_uri = (
    mlflow.get_tracking_uri()
)

mlflow.set_tracking_uri(
    mlflow_tracking_uri
)

mlflow.set_experiment(
    mlflow_experiment_name
)

mlflow_client = MlflowClient()

experiment = (
    mlflow_client
    .get_experiment_by_name(
        mlflow_experiment_name
    )
)

if experiment is None:
    raise RuntimeError(
        "The API integration MLflow experiment "
        "could not be resolved."
    )


# -------------------------------------------------------------------------
# Register API lineage, metrics, tags, and artifacts
# -------------------------------------------------------------------------
active_run_before_registration = (
    mlflow.active_run()
)

start_run_arguments = {
    "run_name": mlflow_run_name,
}

if active_run_before_registration is not None:
    start_run_arguments[
        "nested"
    ] = True

try:
    with mlflow.start_run(
        **start_run_arguments
    ) as api_mlflow_run:
        api_mlflow_run_id = (
            api_mlflow_run.info.run_id
        )

        mlflow.log_params(
            {
                "api_version": (
                    api_lineage[
                        "api_version"
                    ]
                ),
                "api_contract_version": (
                    api_lineage[
                        "api_contract_version"
                    ]
                ),
                "computer_vision_model_version": (
                    api_lineage[
                        "computer_vision_model_version"
                    ]
                ),
                "language_model_version": (
                    api_lineage[
                        "language_model_version"
                    ]
                ),
                "prompt_registry_version": (
                    api_lineage[
                        "prompt_registry_version"
                    ]
                ),
                "finding_contract_version": (
                    api_lineage[
                        "finding_contract_version"
                    ]
                ),
                "artifact_registry_version": (
                    api_artifact_registry[
                        "registry_version"
                    ]
                ),
                "educational_use_only": (
                    api_lineage[
                        "educational_use_only"
                    ]
                ),
            }
        )

        mlflow.log_metrics(
            {
                "registered_artifact_files": float(
                    registry_summary[
                        "registered_artifact_files"
                    ]
                ),
                "source_files": float(
                    registry_summary[
                        "source_files"
                    ]
                ),
                "contract_files": float(
                    registry_summary[
                        "contract_files"
                    ]
                ),
                "test_files": float(
                    registry_summary[
                        "test_files"
                    ]
                ),
                "evidence_files": float(
                    registry_summary[
                        "evidence_files"
                    ]
                ),
                "openapi_paths": float(
                    registry_summary[
                        "openapi_paths"
                    ]
                ),
                "openapi_operations": float(
                    registry_summary[
                        "openapi_operations"
                    ]
                ),
                "pytest_passed_tests": float(
                    registry_summary[
                        "pytest_passed_tests"
                    ]
                ),
                "pytest_failed_tests": float(
                    registry_summary[
                        "pytest_failed_tests"
                    ]
                ),
            }
        )

        mlflow.set_tags(
            {
                "service_framework": "FastAPI",
                "service_stage": (
                    "integration_verified"
                ),
                "api_contract_status": (
                    "verified"
                ),
                "openapi_contract_status": (
                    "verified"
                ),
                "independent_pytest_status": (
                    "passed"
                ),
                "standalone_uvicorn_status": (
                    "passed"
                ),
                "guarded_language_outputs": (
                    "enabled"
                ),
                "medical_use_boundary": (
                    "educational_decision_support_only"
                ),
            }
        )

        for artifact_record in (
            resolved_artifacts
        ):
            mlflow.log_artifact(
                str(
                    artifact_record[
                        "absolute_path"
                    ]
                ),
                artifact_path=(
                    artifact_record[
                        "artifact_group"
                    ]
                ),
            )

        registration_summary = {
            "registration_version": (
                "fastapi-mlflow-registration-v1"
            ),
            "registered_at_utc": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
            "tracking_uri": (
                mlflow_tracking_uri
            ),
            "experiment_name": (
                mlflow_experiment_name
            ),
            "experiment_id": (
                experiment.experiment_id
            ),
            "run_id": api_mlflow_run_id,
            "run_name": mlflow_run_name,
            "api_lineage": api_lineage,
            "registered_artifact_files": (
                len(
                    resolved_artifacts
                )
            ),
            "openapi_operations": (
                registry_summary[
                    "openapi_operations"
                ]
            ),
            "pytest_passed_tests": (
                registry_summary[
                    "pytest_passed_tests"
                ]
            ),
            "pytest_failed_tests": (
                registry_summary[
                    "pytest_failed_tests"
                ]
            ),
        }

        mlflow_registration_path.write_text(
            json.dumps(
                registration_summary,
                indent=2,
            )
            + "\n",
            encoding="utf-8",
        )

        mlflow.log_artifact(
            str(
                mlflow_registration_path
            ),
            artifact_path="registration",
        )

finally:
    mlflow.set_tracking_uri(
        previous_tracking_uri
    )


# -------------------------------------------------------------------------
# Validate the completed MLflow run
# -------------------------------------------------------------------------
verification_client = MlflowClient(
    tracking_uri=mlflow_tracking_uri
)

registered_run = (
    verification_client.get_run(
        api_mlflow_run_id
    )
)

registered_run_status = (
    registered_run.info.status
)

registered_run_artifacts = (
    verification_client.list_artifacts(
        api_mlflow_run_id
    )
)

registered_artifact_directories = sorted(
    artifact.path
    for artifact in registered_run_artifacts
)

registration_checksum = (
    hashlib.sha256(
        mlflow_registration_path.read_bytes()
    ).hexdigest()
)

mlflow_registration_checks = {
    (
        "MLflow experiment is available"
    ): experiment is not None,
    (
        "API registration run finished successfully"
    ): registered_run_status == "FINISHED",
    (
        "API version is registered"
    ): (
        registered_run.data.params.get(
            "api_version"
        )
        == "v1"
    ),
    (
        "Computer-vision lineage is registered"
    ): (
        registered_run.data.params.get(
            "computer_vision_model_version"
        )
        == "resnet18-chestmnist-v1"
    ),
    (
        "Language-model lineage is registered"
    ): (
        registered_run.data.params.get(
            "language_model_version"
        )
        == "flan-t5-small-chestmnist-v1"
    ),
    (
        "All twelve API operations are registered"
    ): (
        registered_run.data.metrics.get(
            "openapi_operations"
        )
        == 12.0
    ),
    (
        "Independent test success is registered"
    ): (
        registered_run.data.metrics.get(
            "pytest_passed_tests"
        )
        > 0.0
        and registered_run.data.metrics.get(
            "pytest_failed_tests"
        )
        == 0.0
    ),
    (
        "Source artifacts are registered"
    ): (
        "source"
        in registered_artifact_directories
    ),
    (
        "Contract artifacts are registered"
    ): (
        "contract"
        in registered_artifact_directories
    ),
    (
        "Test artifacts are registered"
    ): (
        "test"
        in registered_artifact_directories
    ),
    (
        "Evidence artifacts are registered"
    ): (
        "evidence"
        in registered_artifact_directories
    ),
    (
        "Registration summary was exported"
    ): mlflow_registration_path.is_file(),
    (
        "Registration checksum is available"
    ): len(
        registration_checksum
    ) == 64,
    (
        "MLflow tracking URI was restored"
    ): (
        mlflow.get_tracking_uri()
        == previous_tracking_uri
    ),
}


# -------------------------------------------------------------------------
# Present the MLflow registration summary
# -------------------------------------------------------------------------
print(
    "MLFLOW FASTAPI INTEGRATION REGISTRATION"
)
print("-" * 100)
print(
    f"Tracking URI              : "
    f"{mlflow_tracking_uri}"
)
print(
    f"Experiment name           : "
    f"{mlflow_experiment_name}"
)
print(
    f"Experiment ID             : "
    f"{experiment.experiment_id}"
)
print(
    f"Run ID                    : "
    f"{api_mlflow_run_id}"
)
print(
    f"Run status                : "
    f"{registered_run_status}"
)
print(
    f"Registered source files   : "
    f"{registry_summary['source_files']}"
)
print(
    f"Registered contracts      : "
    f"{registry_summary['contract_files']}"
)
print(
    f"Registered test files     : "
    f"{registry_summary['test_files']}"
)
print(
    f"Registered evidence files : "
    f"{registry_summary['evidence_files']}"
)
print(
    f"OpenAPI operations        : "
    f"{registry_summary['openapi_operations']}"
)
print(
    f"Independent tests passed  : "
    f"{registry_summary['pytest_passed_tests']}"
)
print(
    f"Registration artifact     : "
    f"{mlflow_registration_path}"
)
print(
    f"Artifact SHA-256          : "
    f"{registration_checksum[:16]}..."
)
print("-" * 100)

for check_name, passed in (
    mlflow_registration_checks.items()
):
    print(
        f"{check_name:<67}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    mlflow_registration_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in mlflow_registration_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "MLflow API integration registration failed: "
        + ", ".join(
            failed_checks
        )
    )

print("-" * 100)
print(
    "STATUS: FASTAPI INTEGRATION EVIDENCE "
    "REGISTERED IN MLFLOW"
)

2026/08/08 15:34:09 INFO mlflow.tracking.fluent: Experiment with name 'chestmnist-fastapi-service-integration' does not exist. Creating a new experiment.


MLFLOW FASTAPI INTEGRATION REGISTRATION
----------------------------------------------------------------------------------------------------
Tracking URI              : file:///home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/mlflow
Experiment name           : chestmnist-fastapi-service-integration
Experiment ID             : 392047508088290523
Run ID                    : b021146783ad4b34a3e7deaea369e793
Run status                : FINISHED
Registered source files   : 28
Registered contracts      : 5
Registered test files     : 5
Registered evidence files : 7
OpenAPI operations        : 12
Independent tests passed  : 23
Registration artifact     : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/api_mlflow_registration.json
Artifact SHA-256          : 30af75c20911f5d2...
----------------------------------------------------------------------------------------------------
MLflow experiment is available                                     : PASS
API regi

<a id="nb07-10-3-final-fastapi-integration-readiness-gate"></a>
### 10.3 Final FastAPI Integration Readiness Gate

The final readiness gate consolidates the frozen model lineage, API contract, OpenAPI specification, standalone Uvicorn validation, reusable Pytest results, artifact registry, MLflow registration, and educational-use boundary. No model is reloaded and no held-out dataset is reused. Successful completion confirms that the backend service is ready to be consumed by the user-interface layer.


In [66]:
# =========================================================================
# 10.3 FINAL FASTAPI INTEGRATION READINESS GATE
# =========================================================================

from __future__ import annotations

import hashlib
import json
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path


# -------------------------------------------------------------------------
# Resolve final-readiness artifact paths
# -------------------------------------------------------------------------
solution_root = Path(
    "/home/jovyan/chest-xray-ai-assistant"
).resolve()

data_root = Path(
    "/home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data"
).resolve()

api_output_root = (
    data_root
    / "outputs"
    / "api"
)

openapi_path = (
    api_output_root
    / "openapi.json"
)

artifact_registry_path = (
    api_output_root
    / "api_artifact_registry.json"
)

mlflow_registration_path = (
    api_output_root
    / "api_mlflow_registration.json"
)

pytest_log_path = (
    api_output_root
    / "pytest_api_integration.log"
)

uvicorn_log_path = (
    api_output_root
    / "uvicorn_standalone_smoke_test.log"
)

final_readiness_path = (
    api_output_root
    / "api_integration_readiness.json"
)

required_integration_artifacts = (
    api_output_root
    / "system_endpoint_integration_tests.csv",
    api_output_root
    / "image_endpoint_integration_tests.csv",
    api_output_root
    / "language_endpoint_integration_tests.csv",
    api_output_root
    / "complete_analysis_integration_tests.csv",
)


# -------------------------------------------------------------------------
# Load previously validated contracts and evidence
# -------------------------------------------------------------------------
required_readiness_inputs = (
    openapi_path,
    artifact_registry_path,
    mlflow_registration_path,
    pytest_log_path,
    uvicorn_log_path,
    *required_integration_artifacts,
)

missing_readiness_inputs = [
    str(artifact_path)
    for artifact_path
    in required_readiness_inputs
    if not artifact_path.is_file()
]

if missing_readiness_inputs:
    raise FileNotFoundError(
        "Final API readiness inputs are missing: "
        + ", ".join(
            missing_readiness_inputs
        )
    )

with openapi_path.open(
    "r",
    encoding="utf-8",
) as openapi_file:
    openapi_document = json.load(
        openapi_file
    )

with artifact_registry_path.open(
    "r",
    encoding="utf-8",
) as registry_file:
    api_artifact_registry = json.load(
        registry_file
    )

with mlflow_registration_path.open(
    "r",
    encoding="utf-8",
) as registration_file:
    api_mlflow_registration = json.load(
        registration_file
    )

pytest_log_text = (
    pytest_log_path.read_text(
        encoding="utf-8",
        errors="replace",
    )
)

uvicorn_log_text = (
    uvicorn_log_path.read_text(
        encoding="utf-8",
        errors="replace",
    )
)


# -------------------------------------------------------------------------
# Resolve OpenAPI and automated-test evidence
# -------------------------------------------------------------------------
openapi_paths = (
    openapi_document.get(
        "paths",
        {}
    )
)

openapi_operation_count = sum(
    1
    for path_contract
    in openapi_paths.values()
    for method_name
    in (
        "get",
        "post",
        "put",
        "patch",
        "delete",
    )
    if method_name in path_contract
)

pytest_passed_match = re.search(
    r"(\d+)\s+passed\b",
    pytest_log_text,
    flags=re.IGNORECASE,
)

pytest_failed_match = re.search(
    r"(\d+)\s+failed\b",
    pytest_log_text,
    flags=re.IGNORECASE,
)

pytest_passed_tests = (
    int(
        pytest_passed_match.group(1)
    )
    if pytest_passed_match
    else 0
)

pytest_failed_tests = (
    int(
        pytest_failed_match.group(1)
    )
    if pytest_failed_match
    else 0
)

uvicorn_traceback_present = (
    "traceback (most recent call last)"
    in uvicorn_log_text.lower()
)


# -------------------------------------------------------------------------
# Confirm endpoint coverage from the frozen API contract
# -------------------------------------------------------------------------
expected_api_paths = {
    "/health",
    "/api/v1/model/info",
    "/api/v1/model/metrics",
    "/api/v1/image/classify",
    "/api/v1/image/analyze",
    "/api/v1/report/generate",
    "/api/v1/explanation/generate",
    "/api/v1/question/answer",
    "/api/v1/follow-up/recommend",
    "/api/v1/analyze-complete",
    "/api/v1/predictions/{prediction_id}",
    "/api/v1/llmops/metrics",
}

actual_api_paths = set(
    openapi_paths.keys()
)


# -------------------------------------------------------------------------
# Calculate remaining storage
# -------------------------------------------------------------------------
storage_usage = shutil.disk_usage(
    data_root
)

gibibyte = 1024 ** 3

free_storage_gib = (
    storage_usage.free
    / gibibyte
)

protected_storage_reserve_gib = 4.0

storage_reserve_available = (
    free_storage_gib
    >= protected_storage_reserve_gib
)


# -------------------------------------------------------------------------
# Consolidate final readiness evidence
# -------------------------------------------------------------------------
registry_summary = (
    api_artifact_registry[
        "summary"
    ]
)

api_lineage = (
    api_artifact_registry[
        "lineage"
    ]
)

final_readiness_checks = {
    (
        "FastAPI application module is available"
    ): (
        solution_root
        / "api"
        / "main.py"
    ).is_file(),
    (
        "Standalone service factory is available"
    ): (
        solution_root
        / "api"
        / "core"
        / "factory.py"
    ).is_file(),
    (
        "All twelve endpoint paths are preserved"
    ): actual_api_paths == expected_api_paths,
    (
        "All twelve OpenAPI operations are preserved"
    ): openapi_operation_count == 12,
    (
        "Reusable API test suite passed"
    ): (
        pytest_passed_tests > 0
        and pytest_failed_tests == 0
    ),
    (
        "System endpoint integration evidence is available"
    ): required_integration_artifacts[
        0
    ].is_file(),
    (
        "Image endpoint integration evidence is available"
    ): required_integration_artifacts[
        1
    ].is_file(),
    (
        "Language endpoint integration evidence is available"
    ): required_integration_artifacts[
        2
    ].is_file(),
    (
        "Complete workflow integration evidence is available"
    ): required_integration_artifacts[
        3
    ].is_file(),
    (
        "Independent Uvicorn log contains no traceback"
    ): not uvicorn_traceback_present,
    (
        "Versioned API artifact registry is available"
    ): artifact_registry_path.is_file(),
    (
        "API artifacts have complete checksums"
    ): all(
        len(
            artifact_record[
                "sha256"
            ]
        )
        == 64
        for artifact_record
        in api_artifact_registry[
            "artifacts"
        ]
    ),
    (
        "MLflow API registration is complete"
    ): (
        api_mlflow_registration[
            "registration_version"
        ]
        == "fastapi-mlflow-registration-v1"
        and bool(
            api_mlflow_registration[
                "run_id"
            ]
        )
    ),
    (
        "Computer-vision lineage is preserved"
    ): (
        api_lineage[
            "computer_vision_model_version"
        ]
        == "resnet18-chestmnist-v1"
    ),
    (
        "Grounded-language lineage is preserved"
    ): (
        api_lineage[
            "language_model_version"
        ]
        == "flan-t5-small-chestmnist-v1"
    ),
    (
        "Prompt-registry lineage is preserved"
    ): (
        api_lineage[
            "prompt_registry_version"
        ]
        == "grounded-language-prompts-v1"
    ),
    (
        "Educational-use boundary remains enabled"
    ): (
        api_lineage[
            "educational_use_only"
        ]
        is True
    ),
    (
        "Protected storage reserve remains available"
    ): storage_reserve_available,
}


# -------------------------------------------------------------------------
# Export the final API integration readiness record
# -------------------------------------------------------------------------
final_readiness_record = {
    "readiness_version": (
        "fastapi-integration-readiness-v1"
    ),
    "evaluated_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "status": (
        "ready_for_interface_integration"
        if all(
            final_readiness_checks.values()
        )
        else "not_ready"
    ),
    "lineage": api_lineage,
    "mlflow": {
        "experiment_id": (
            api_mlflow_registration[
                "experiment_id"
            ]
        ),
        "run_id": (
            api_mlflow_registration[
                "run_id"
            ]
        ),
    },
    "api_contract": {
        "openapi_paths": (
            len(
                openapi_paths
            )
        ),
        "openapi_operations": (
            openapi_operation_count
        ),
    },
    "testing": {
        "pytest_passed_tests": (
            pytest_passed_tests
        ),
        "pytest_failed_tests": (
            pytest_failed_tests
        ),
        "integration_artifact_files": (
            len(
                required_integration_artifacts
            )
        ),
        "standalone_uvicorn_verified": (
            not uvicorn_traceback_present
        ),
    },
    "artifact_registry": {
        "registry_version": (
            api_artifact_registry[
                "registry_version"
            ]
        ),
        "registered_artifact_files": (
            registry_summary[
                "registered_artifact_files"
            ]
        ),
    },
    "storage": {
        "free_storage_gib": round(
            free_storage_gib,
            2,
        ),
        "protected_reserve_gib": (
            protected_storage_reserve_gib
        ),
        "reserve_available": (
            storage_reserve_available
        ),
    },
    "checks": final_readiness_checks,
}

final_readiness_path.write_text(
    json.dumps(
        final_readiness_record,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

final_readiness_checksum = (
    hashlib.sha256(
        final_readiness_path.read_bytes()
    ).hexdigest()
)

final_readiness_checks[
    "Final readiness artifact was exported"
] = final_readiness_path.is_file()

final_readiness_checks[
    "Final readiness checksum is available"
] = len(
    final_readiness_checksum
) == 64


# -------------------------------------------------------------------------
# Present the final Notebook 7 readiness summary
# -------------------------------------------------------------------------
print(
    "FINAL FASTAPI INTEGRATION READINESS"
)
print("-" * 100)
print(
    f"API version                : "
    f"{api_lineage['api_version']}"
)
print(
    f"Computer-vision model      : "
    f"{api_lineage['computer_vision_model_version']}"
)
print(
    f"Grounded-language model    : "
    f"{api_lineage['language_model_version']}"
)
print(
    f"Prompt registry            : "
    f"{api_lineage['prompt_registry_version']}"
)
print(
    f"OpenAPI paths              : "
    f"{len(openapi_paths)}"
)
print(
    f"OpenAPI operations         : "
    f"{openapi_operation_count}"
)
print(
    f"Registered API artifacts   : "
    f"{registry_summary['registered_artifact_files']}"
)
print(
    f"Independent tests passed   : "
    f"{pytest_passed_tests}"
)
print(
    f"Independent tests failed   : "
    f"{pytest_failed_tests}"
)
print(
    f"MLflow experiment ID       : "
    f"{api_mlflow_registration['experiment_id']}"
)
print(
    f"MLflow run ID              : "
    f"{api_mlflow_registration['run_id']}"
)
print(
    f"Free storage               : "
    f"{free_storage_gib:.2f} GiB"
)
print(
    f"Final readiness artifact   : "
    f"{final_readiness_path}"
)
print(
    f"Artifact SHA-256           : "
    f"{final_readiness_checksum[:16]}..."
)
print("-" * 100)

for check_name, passed in (
    final_readiness_checks.items()
):
    print(
        f"{check_name:<69}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

if not all(
    final_readiness_checks.values()
):
    failed_checks = [
        check_name
        for check_name, passed
        in final_readiness_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Final FastAPI integration readiness "
        "validation failed: "
        + ", ".join(
            failed_checks
        )
    )

print("-" * 100)
print(
    "FINAL STATUS: FASTAPI BACKEND READY "
    "FOR STREAMLIT INTERFACE INTEGRATION"
)

FINAL FASTAPI INTEGRATION READINESS
----------------------------------------------------------------------------------------------------
API version                : v1
Computer-vision model      : resnet18-chestmnist-v1
Grounded-language model    : flan-t5-small-chestmnist-v1
Prompt registry            : grounded-language-prompts-v1
OpenAPI paths              : 12
OpenAPI operations         : 12
Registered API artifacts   : 45
Independent tests passed   : 23
Independent tests failed   : 0
MLflow experiment ID       : 392047508088290523
MLflow run ID              : b021146783ad4b34a3e7deaea369e793
Free storage               : 8.71 GiB
Final readiness artifact   : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/api_integration_readiness.json
Artifact SHA-256           : cc702358ef189801...
----------------------------------------------------------------------------------------------------
FastAPI application module is available                              : PAS

**[↑ Back to notebook index](#notebook-index)**


<a id="nb07-11-conclusion"></a>
## 11. Conclusion

The FastAPI backend was implemented and validated as a standalone service for the chest X-ray analysis and explanation assistant. Twelve versioned endpoints now support health monitoring, model information, image classification, visual explainability, grounded-language generation, complete analysis, stored-prediction retrieval, and operational metrics.

The service preserves the frozen computer-vision, finding-threshold, grounded-language, prompt-registry, Grad-CAM, and safety contracts. Image ingestion is controlled by media-type, content, and upload-size validation. Language responses are grounded exclusively in stored model outputs and pass through a deterministic guardrail that replaces non-compliant generations with controlled template responses.

All endpoint groups passed in-process integration testing, the application started successfully in an independent Uvicorn process, and the reusable Pytest suite completed with 23 passing tests and no failures. Forty-five implementation and evidence artifacts were registered with SHA-256 checksums, and the completed API integration was linked to a finished MLflow run.

The backend remains an educational decision-support prototype. Its outputs are not diagnoses and must not replace review by a qualified healthcare professional.

**Final status:** The FastAPI backend is ready for Streamlit interface integration.


**[↑ Back to notebook index](#notebook-index)**
